# DATA-DRIVEN 1D MECHANICAL EARTH MODEL — POSEIDON 2

## INCREMENT 6 / 6.1 / ... / 6.1.7 — GAMMA-RAY QC, SHALE-PROXY SENSITIVITY, WELL-FRAME ASSEMBLY, AND METHOD-ELIGIBILITY FRAMEWORK

**Author:** Mikael Elgo

**Project classification:** **Tier C — Screening-Level / Uncalibrated Educational**

> **This notebook is screening-level, uncalibrated, and educational. It is NOT validated against independent field measurements — no RFT, MDT, DST, FIT, LOT, XLOT or DFIT data exist for this project, and the supplied Vp/Vs text file is derived from the existing sonic curves rather than being independent calibration data. Nothing here may be used for operational drilling, well design, or any real-world decision.**

**Locked foundation:** Increment 5.1.2 (`p2mem` 0.5.2) — LAS ingestion, deviation-survey ingestion, minimum-curvature trajectory validation, MD→TVD/TVDSS depth mapping, checkshot ingestion, the time-depth framework, and survey-corrected formation tops — has passed independent technical review (482 locked tests) and is treated as **LOCKED**. This notebook does not modify any locked module, config, test, fixture, notebook, output, or figure. It changes exactly three existing files, all additively: `pyproject.toml` (version → 0.6.8), `p2mem/__init__.py` (version and scope description), and `README.md` (Increment 6 / 6.1 / ... / 6.1.7 documentation). Everything else it adds is new.

---

### Phase Objective

Answer four questions about the four approved wells, and refuse to answer anything beyond them:

1. **Which curves and depth intervals are technically usable** for later density/overburden, dynamic-elastic, and sonic-NCT workflows?
2. **How sensitive is a dimensionless GR-derived proxy** to endpoint and threshold selection?
3. **Which wells and intervals must be excluded, and why?**
4. **Does the available dataset support named lithology interpretation?**

The expected answer to question 4 is **no**, and this notebook must *demonstrate* that from data availability and sensitivity results — never assert it as a hardcoded conclusion.

### Expected Outputs

Eight deterministic tables/manifests and four QC figures under `outputs/06_petrophysics_eligibility/`:

| Output | Content |
|---|---|
| `gr_family_qc_summary.csv` | factual per-well GR-family statistics, for every well including excluded ones |
| `gr_endpoint_scenarios.csv` | the MEASURED endpoint values each configured scenario produced |
| `gr_proxy_sensitivity_summary.csv` | IGR/proxy summary and clipping counts per well per scenario |
| `method_eligibility_summary.csv` | per-well, per-mask, per-scenario eligible counts and limiting criterion |
| `eligibility_interval_register.csv` | contiguous eligible blocks on MD, TVD and TVDSS |
| `petrophysics_eligibility_issues.csv` | issues and isolated per-well assembly failures |
| `thickness_sensitivity_summary.csv` | gross/net and strict-no-gap interval sensitivity summaries |
| `petrophysics_eligibility_manifest.json` | complete run metadata, exclusions, and limitations |
| `figures/fig01…fig04` | raw GR QC, endpoint/proxy sensitivity, eligibility coverage, candidate-interval sensitivity |

**Eight deterministic tables/manifests plus four figures — twelve outputs in total.**

### Explicitly NOT Implemented In This Increment

Named lithology interpretation · environmental GR correction · Boreas ECGR rescaling · neutron–density crossplot interpretation · nonlinear Vsh transforms (Larionov, Clavier, Stieber — all deferred) · shallow-density reconstruction · density extrapolation · vertical-stress integration · hydrostatic-pressure modelling · velocity–effective-stress crossplots · NCT fitting · Eaton sonic · Eaton resistivity · Bowers · pore-pressure prediction · dynamic elastic-property calculation · static elastic conversion · rock-strength correlations · friction-angle modelling · Shmin/SHmax modelling · stress-polygon construction · wellbore-stability analysis · mud-weight recommendations. **Increment 7 has not been started.**

---

## Theory and Physical Basis

### 1. Physical basis of the gamma-ray response

A gamma-ray log records **natural** gamma radiation emitted by the formation, in API units (gAPI) defined by the API calibration pit at the University of Houston. Essentially all of that radiation comes from three naturally occurring radioisotopes:

- **Potassium-40** (⁴⁰K) — abundant in illite, mica, feldspar, glauconite and evaporite salts;
- **the Thorium-232 decay series** — concentrated in heavy minerals, clays and bauxite;
- **the Uranium-238 decay series** — associated with organic matter, phosphates, and secondary precipitation from circulating fluids.

The total count rate a standard (non-spectral) GR tool reports is the **sum** of contributions from all three, weighted by the tool's own response function and by borehole environment (hole size, mud weight, barite loading, casing, tool standoff, logging speed).

### 2. Why GR is not uniquely lithology

The mapping from mineralogy to GR is many-to-one, so the inverse — GR to rock type — is **not unique**. Concretely:

- A **clean quartz sandstone** and a **clean limestone** both read low GR, and are indistinguishable on GR alone.
- An **arkosic (feldspar-rich) sandstone** can read 60–100+ API on ⁴⁰K alone while being a perfectly good reservoir rock — the classic "hot sand" false positive.
- A **glauconitic** or **micaceous** sand reads high for the same reason.
- An **organic-rich shale** reads very high largely on uranium, while a **kaolinite-dominated shale** can read modestly, because kaolinite is potassium-poor.
- **Uranium salts precipitated in fractures** produce high GR in rock that is not clay-rich at all.

This is why every quantity derived from GR in this increment is named a **proxy** or a **screening** quantity, and why no classification produced here is permitted to contain a rock name. The code enforces this at persisted-output boundaries through closed typed-label, statement and template registries plus exact artifact schemas. The prohibited-vocabulary list is a supplementary diagnostic linter only; it authorizes nothing, and free-form notebook narrative remains subject to manual scientific review.

### 3. Raw GR, normalized GR index, and a Vsh proxy — three different things

These are routinely conflated, and this project keeps them separate by construction:

| Quantity | Definition | Units | What it actually is |
|---|---|---|---|
| **Raw GR** | the logged count rate | API | a **measurement** (subject to tool and borehole effects) |
| **GR index (IGR)** | $\mathrm{IGR}=\dfrac{\mathrm{GR}-\mathrm{GR}_{\text{low}}}{\mathrm{GR}_{\text{high}}-\mathrm{GR}_{\text{low}}}$ | dimensionless | the measurement's **position between two assumed endpoints** |
| **Vsh proxy** | a transform of IGR (here: the linear identity) | dimensionless fraction | an **interpretation** whose accuracy depends entirely on assumptions no data here constrains |

Increment 6 computes the first two honestly and the third only under the name `VSH_GR_linear_proxy_frac` — "proxy" is part of the identifier itself, so the quantity cannot be referenced downstream without carrying its own disclaimer. It is **never** called Vshale, shale volume, or clay volume, because a *calibrated* shale volume requires core, XRD, or spectral-GR control that this project does not have.

### 4. Endpoint uncertainty

$\mathrm{GR}_{\text{low}}$ and $\mathrm{GR}_{\text{high}}$ are supposed to represent the "clean" and "most argillaceous" formation responses. In practice they are **chosen**, not measured, and the choice propagates directly into every downstream number:

$$\frac{\partial\,\mathrm{IGR}}{\partial\,\mathrm{GR}_{\text{low}}}=\frac{\mathrm{IGR}-1}{\mathrm{GR}_{\text{high}}-\mathrm{GR}_{\text{low}}},\qquad\frac{\partial\,\mathrm{IGR}}{\partial\,\mathrm{GR}_{\text{high}}}=\frac{-\,\mathrm{IGR}}{\mathrm{GR}_{\text{high}}-\mathrm{GR}_{\text{low}}}$$

Both sensitivities scale as the **inverse of the endpoint separation**, so a narrow bracket amplifies both the proxy's magnitude and its uncertainty. That is exactly why this increment carries **three** endpoint scenarios (`low`, `base`, `high`) rather than one preferred pair, estimates them **per well from that well's own samples** (never one universal cross-well pair — these are four differently named tools with no calibration tie), and records the **measured** endpoint values each rule produced. Every scenario carries `evidence_class = "assumed_configured"`; no code path in this project can promote one to "calibrated".

### 5. Consequences of clipping

$\mathrm{IGR}_{\text{clipped}}=\mathrm{clip}(\mathrm{IGR},0,1)$ is what any downstream mask may consume, because a proxy fraction outside $[0,1]$ is meaningless. But clipping is **lossy in a specific and dangerous way**: it converts "this sample fell outside the assumed endpoint bracket" into a clean-looking 0 or 1, hiding the very evidence that the endpoints were wrong. This project therefore **retains both** the clipped and unclipped indices side by side in memory, and exports the clipped-low/clipped-high counts, so the fraction of data that ran past the bracket is always visible and auditable.

### 6. Threshold sensitivity

A sonic-NCT screening threshold (e.g. "proxy ≥ 0.60") is a second **assumed** choice layered on top of the first. Since the proxy is monotone in raw GR, raising the threshold can only shrink the candidate set — but *by how much* is a property of the data, not of the method, and it compounds with endpoint choice. This increment measures the full endpoint × threshold grid rather than reporting one case.

### 7. Missing-data masks

Every curve carries gaps. This project's rule is absolute: **an invalid sample is masked, never deleted, filled, interpolated, or reordered.** Sample count and file order are preserved exactly, so array position remains directly comparable across curves, scenarios and wells. A NaN never becomes 0, an endpoint, or a neighbour's value; an infinite value is explicitly *invalidated* rather than normalized into a ±∞ index that clipping would quietly launder into a clean 0 or 1.

### 8. Contiguous-interval logic

Downstream methods need *intervals*, not scattered samples. Two different notions of continuity must not be confused:

- **Sample-count continuity** — how many consecutive array positions pass a mask;
- **Physical-depth continuity** — whether those positions actually span an unbroken depth range.

A two-sample gap across a 400 m depth jump is **not** a continuous interval. This increment therefore bridges a gap only when **both** the sample gap and the physical depth span are within explicitly configured, tested tolerances, and always reports the bridged sample count separately from the genuinely eligible count.

### 9. Method eligibility versus method validity

**Eligibility is a necessary, not a sufficient, condition.** A mask says a sample is technically admissible as *input*; it says nothing about whether the method would be appropriate, or its result defensible. The clearest example in this dataset: density becomes eligible only from roughly 3,900–4,800 m TVDSS downward in all four wells, so no amount of per-sample eligibility can support an overburden integral from surface — the shallow section simply is not logged. Reporting eligibility separately from validity is what keeps that gap visible instead of buried inside a later calculation.

### 10. Why candidate NCT intervals are not proof of normal compaction

A normal compaction trend is a claim about **effective-stress history**: that a shale's porosity (and hence transit time) has decreased monotonically with burial under hydrostatic pore pressure, with no unloading, no cementation overprint, and no lithological change along the trend. A `eligible_sonic_nct_candidate` mask establishes none of that. It establishes only that, at these samples, a sonic reading exists, a depth is mapped, and a GR-derived *proxy* — itself uncalibrated and endpoint-dependent — sits above an assumed threshold. Calling such an interval "normally compacted shale" would smuggle in three unproven claims at once: that the interval *is* shale (GR cannot establish it), that it *is* normally compacted (no pressure data exist to test it), and that the threshold that selected it was correct (no calibration constrains it). This increment therefore fits nothing, selects no donor interval, and labels every candidate output **"candidate data only — no NCT fitted"**.

### 11. The Vp/Vs domain, stated precisely (Increment 6.1 correction)

Increment 6 rejected every Vp/Vs ratio at or below $\sqrt{2}$ and labelled the whole excluded set "non-physical". That was scientifically inaccurate, and it is corrected here. For an isotropic elastic solid with $r = V_p/V_s$:

$$\nu = \frac{r^{2}-2}{2\left(r^{2}-1\right)}, \qquad K = \rho\left(V_p^{2}-\tfrac{4}{3}V_s^{2}\right)$$

Four **distinct** regimes follow, and collapsing them into one label destroys the distinction that matters:

| Regime | Bulk modulus $K$ | Poisson's ratio $\nu$ | Correct description |
|---|---|---|---|
| $r \le \sqrt{4/3} \approx 1.1547$ | $K \le 0$ | — | **Genuinely outside the isotropic elastic model.** The only regime that warrants "non-physical". |
| $\sqrt{4/3} < r < \sqrt{2}$ | $K > 0$ | $\nu < 0$ | Positive bulk modulus, negative Poisson's ratio. **Unusual, and outside this project's conservative policy — but not physically impossible.** Auxetic elastic behavior is real, if rare. |
| $r = \sqrt{2}$ | $K > 0$ | $\nu = 0$ **exactly** | **Must be ACCEPTED** by a policy requiring a *non-negative* Poisson's ratio. Increment 6's exclusive bound wrongly rejected it. |
| $r > 4$ (configured) | $K > 0$ | $\nu \approx 0.467$ | An entirely ordinary Poisson's ratio. This is a **configured plausibility limit**, not a Poisson-domain boundary. |

The condition is therefore renamed a **CONFIGURED NON-NEGATIVE-POISSON-RATIO APPLICABILITY SCREEN** — a conservative *policy* about what this project will admit to a later dynamic-elastic calculation, not a claim about what is physically possible. The $\sqrt{2}$ bound is **inclusive**.

**A numerical detail that matters at exactly the boundary.** The screen is applied to the *ratio*, compared inclusively against the configured constant — not to a floating-point evaluation of $\nu \ge 0$. Evaluating the $\nu$ expression at $r = \sqrt{2}$ in IEEE-754 returns $\approx 2.2\times10^{-16}$ rather than exactly zero, so a naive test on $\nu$ would be decided by rounding noise precisely where the policy is defined.

Exclusions are reported **by regime** — `n_ratio_nonpositive_bulk_modulus`, `n_ratio_positive_bulk_but_negative_poisson`, `n_ratio_above_configured_plausibility_max`, `n_vp_not_greater_than_vs` — and are **never aggregated** into a single "non-physical" count. On the real data this matters concretely: Poseidon North 1's previously reported aggregate of 7 decomposes into **4** genuinely non-positive-bulk-modulus samples and **3** positive-bulk/negative-Poisson samples, which are different findings.

### 12. Gross, net, and strict interval thickness (Increment 6.1 correction)

A block's **gross** thickness is its endpoint span, first eligible sample to last. Under the configured contiguity policy that span may *contain* explicitly bridged ineligible samples, so calling it simply "eligible thickness" overstates it. Three quantities are therefore reported separately and named explicitly:

- **gross** — endpoint span, bridging-inclusive;
- **net** — the sum of the maximal *strictly contiguous* eligible sub-run spans inside the block, i.e. gross minus the bridged gaps;
- **strict-no-gap** — the whole decomposition redone with no bridging at all, giving the conservative comparison.

With sub-runs $[s,\,p-1]$ and $[p+g,\,e]$ around a gap of length $g$ starting at $p$, the identity is exact:

$$\text{gross} - \text{net} \;=\; y_{p+g} - y_{p-1}$$

the depth span from the last eligible sample before the gap to the first after it. Every sensitivity case additionally reports how many bridged samples and interrupted blocks its gross figure absorbed, so the difference is never invisible.

### 13. Counting what a bridged block actually contains (Increment 6.1.1 correction)

Increment 6.1 reported two quantities per bridged block, `n_interruptions` and `n_interrupted_subruns`, and both were defective. The first was set to 1 whenever *any* bridging occurred — all 327 bridged blocks in the real data reported exactly 1 — so it was a boolean flag presented in the grammatical form of a count, which invites the reader to add it up. The second never said what it counted. Three separately named, independently meaningful quantities replace them:

- **`n_bridged_samples`** — the total number of ineligible samples absorbed inside the gross block;
- **`n_bridged_gaps`** — the number of *distinct* bridged runs of ineligible samples;
- **`n_eligible_subruns`** — the number of maximal strictly contiguous eligible sub-runs.

These are not redundant: one gap may absorb several samples, so `n_bridged_samples` and `n_bridged_gaps` differ in general. Cutting a block into $k$ pieces requires exactly $k-1$ cuts, so wherever a block exists the identity

$$n_{\text{bridged gaps}} \;=\; n_{\text{eligible subruns}} - 1$$

holds exactly, and it is **enforced at construction** — an interval record that violates it is rejected rather than exported. A block with no gap reports 0 samples, 0 gaps, 1 sub-run; a block with two separate bridged gaps reports 2 gaps and 3 sub-runs. The sensitivity summary likewise separates `n_bridged_samples_in_qualifying_blocks`, `n_bridged_gaps_in_qualifying_blocks`, and `n_interrupted_qualifying_blocks` (blocks with at least one gap), which are three different questions about the same population. The configured bridging tolerances and the qualifying-block policy are **unchanged** — only the reporting is corrected.

### 14. What an allowlist can and cannot be asked to do (Increment 6.1.2 correction)

This project prohibits naming a lithology, but it must still be able to *name the method* — it computes a **shale proxy**, and it explicitly denies computing a calibrated **shale volume**. Those two phrases must therefore be sayable in prose while the underlying rock name stays prohibited. Increment 6.1.1 implemented that allowance with a sentence-wide assertion-cue list and a fixed look-behind window, and an independent audit showed both were structurally wrong — in **both** directions:

| Sentence | 6.1.1 verdict | Correct verdict |
|---|---|---|
| "Poseidon 2's shale volume is high." | **passed** (0 violations) | violation |
| "Poseidon 2's shale volume is 70 percent." | **passed** | violation |
| "The interval's shale volume exceeds 60 percent." | **passed** | violation |
| "This method contains a shale proxy calculation." | **violation** | pass |
| "The analysis shows no shale volume was computed." | **violation** | pass |

The two failure modes have a single root cause: the allowance was treated as a property of the *sentence* rather than of the *phrase occurrence*. A look-behind window cannot see `is high` or `exceeds 60 percent`, because those sit to the **right**; and the normalizer replaced every non-alphanumeric character with a space, which erased the possessive `'s` — the very construction that attaches a rock property to a body of rock. In the other direction, a sentence-wide cue fires on `contains` without ever asking *what contains what*: the subject of "This method contains …" is a calculation, not a formation.

**The corrected rule is grammatical, not lexical.** Each occurrence of an allowed phrase is classified from its own local context:

$$\text{exempt}(o) \;=\; \neg\big[\underbrace{\text{poss}(o)}_{\text{(a)}} \vee \underbrace{\text{mod}(o)}_{\text{(b)}} \vee \underbrace{\text{pred}(o)}_{\text{(c)}} \vee \underbrace{\text{comp}(s)}_{\text{(d)}}\big]$$

- **(a) possessor** — is the phrase possessed by a geological entity? *"the interval's shale volume"*, *"Poseidon 2's shale volume"*. Apostrophes are now parsed, including typographic and plural forms.
- **(b) modifier** — is it immediately modified by a magnitude? *"a **high** shale volume"*, *"**significant** shale volume"*. A negation in the same noun phrase suppresses this, because *"**no** shale volume"* denies rather than asserts.
- **(c) predicate** — what is asserted *of* it to the right? A magnitude or dominance verb (*exceeds*, *dominates*, *reaches*), or a copula **followed by a magnitude** (*is high*, *is 70 percent*). A bare copula is deliberately not enough: *"the shale proxy **is** dimensionless"* states a property of the **method**.
- **(d) composition** — does a containment verb in this sentence take a *geological subject*? The subject head is resolved by walking left past determiners and numerals, so *"The unit comprises …"* asserts while *"This method contains …"* does not.

Sentence boundaries — now including line breaks — are respected in both directions: a clean method sentence cannot license an assertion elsewhere, and an assertion elsewhere cannot condemn a clean method sentence. Where a project-specific sentence is genuinely ambiguous, the rule **fails closed**. `SCOPE_LABEL` is untouched by all of this: a label is a verdict and gets no latitude at all, so `SHALE_PROXY_HIGH` remains a violation.

### 15. What a count record must satisfy to exist (Increment 6.1.2 correction)

Section 13 introduced three counts per block. Increment 6.1.1 stated they were enforced at construction; in fact only the gap/sub-run identity was checked, and only when both values were present and `n_eligible_subruns > 0`. Records asserting five bridged samples inside zero gaps, or two gaps inside a block with no eligible sub-runs, or negative counts, were all constructible. A constructor that accepts an impossible record is not an invariant.

The three counts are now validated as **one coherent record**, each condition independently motivated:

| Rule | Why it must hold |
|---|---|
| all three present | a record missing a count is not a decomposition of anything |
| strictly integral (no `bool`, `str`, complex, fractional float) | a count says *how many*; `True` is the flag confusion that made `n_interruptions` useless |
| $n_{\text{samples}} \ge 0,\; n_{\text{gaps}} \ge 0$ | counts cannot be negative |
| $n_{\text{subruns}} \ge 1$ | a constructed record describes a block that exists, so it has at least one eligible sub-run |
| $n_{\text{gaps}} = n_{\text{subruns}} - 1$ | cutting a block into $k$ pieces takes exactly $k-1$ cuts |
| $n_{\text{samples}} = 0 \iff n_{\text{gaps}} = 0$ | bridged samples exist precisely when a bridged gap exists |
| $n_{\text{samples}} \ge n_{\text{gaps}}$ | every distinct gap holds at least one ineligible sample |

Each violation raises `PetrophysicsInputError` naming the offending field or relationship, and the removed `n_interruptions` / `n_interrupted_subruns` names are rejected outright rather than silently ignored — a stale caller fails loudly instead of writing a record with two of its three counts unset. **No configured tolerance, threshold or policy changes here**: every interval record this notebook produces already satisfied every rule above, which is exactly why all twelve outputs and figures are byte-identical to Increment 6.1.1.

### 16. Why this notebook stopped trying to read English (Increment 6.1.3 correction)

Sections 14 and 15 described a validator that decided, from grammar, whether a sentence containing a rock name was *naming a method* or *asserting geology*. Two independent audits broke two successive implementations of that idea, in both directions each time:

| Sentence | 6.1.1 | 6.1.2 | Correct |
|---|---|---|---|
| "Poseidon 2's shale volume is high." | **passed** | violation | violation |
| "The interval has a shale volume." | passed | **passed** | violation |
| "The shale volume in Poseidon 2 exceeds 60 percent." | passed | **passed** | violation |
| "Poseidon 2 shale volume was determined to be high." | passed | **passed** | violation |
| "This method contains a shale proxy calculation." | **violation** | pass | pass |
| "The well contains no shale volume estimate." | violation | **violation** | pass |

Each 6.1.2 escape has a one-line cause: `has` was not in the predicate vocabulary; `in Poseidon 2` is a *prepositional* possessor where only the genitive was parsed; `was determined to be high` puts the magnitude four tokens right of a three-token look-ahead. Each is a list gap or a window edge. The tempting response — add `has`, add prepositions, widen the window — is what produced this table in the first place, twice.

**The space is unbounded, so no finite grammar closes it.** That is the finding. A validator that must decide, from arbitrary English, whether a rock name is being *used* or *mentioned* is attempting an open-ended natural-language task inside a geomechanics deliverable, and it will keep passing whatever examples it was shown and failing the next one.

Increment 6.1.3 therefore removes the question rather than answering it better. Every scope is decided by **set membership or exact equality**:

$$\text{violation}(x, s) = \begin{cases}
\text{terms}(x) \neq \emptyset & s \in \{\text{LABEL},\ \text{INTERPRETIVE}\}\\[2pt]
x \notin \mathcal{R} & s = \text{METHOD}\\[2pt]
\exists\, \sigma \in x:\ \text{terms}(\sigma) \neq \emptyset \wedge \text{refs}(\sigma) \neq \emptyset & s = \text{EXPLANATORY}
\end{cases}$$

where $\mathcal{R}$ is a closed registry of five provenance-tagged statements. Project-specific prose has **zero allowance** — there is no exemption path, so there is nothing to bypass — and the project still states its own boundaries, through $\mathcal{R}$, by registered identity rather than by phrasing.

The registry was **measured, not designed**: scanning every string literal in active packaged source and every persisted field returned the strings that carry a rock term for a legitimate reason. **Increment 6.1.6 supersedes the count this paragraph previously quoted.** `n_fields_checked` described the constructed validation *scope*, not the records written to disk, and is replaced by `scope_object_fields_checked` plus an `emitted_field_coverage` block that counts every emitted string-field **occurrence** across all eight artifacts.

What replaces the example lists is a **closure proof**: every prohibited term, in every position, inside carrier prose built from the exact constructions that defeated both previous implementations. That is finite and enumerable, which a sentence list never was — and it is why the deletion of the grammar machinery, not its improvement, is the correction.

### 17. Finishing the argument (Increment 6.1.4 correction)

Section 16 replaced grammar with membership — but only in three of four scopes. Explanatory text still decided its verdict from a finite list of *project-reference* tokens: a rock name was admitted whenever the sentence appeared to mention none of this project's wells or rock bodies. That is the same shape of rule as the two that had just been deleted, and it failed in the same way:

| Explanatory text | 6.1.3 | Correct |
|---|---|---|
| "The member is shale." | **passed** | violation |
| "The group is limestone." | **passed** | violation |
| "The package is a clean sandstone." | **passed** | violation |
| "The play is shale-dominated." | **passed** | violation |
| "The prospect is carbonate." | **passed** | violation |
| "The target is sandstone." | **passed** | violation |
| "The upper member is a clean sandstone reservoir." | violation | violation |

The last row is the instructive one. It failed *only* because `reservoir` happened to be in the list while `member` was not — an accident of vocabulary, not a judgement about geology. `member` and `group` are formal lithostratigraphic ranks; `play`, `prospect` and `target` are the words an exploration report actually uses. Any of them would have carried an assertion straight through.

**The rule is inverted, and becomes identical in kind to the method rule:**

$$\text{violation}(x,\ \text{EXPLANATORY}) \;=\; \text{terms}(x) \neq \emptyset \;\wedge\; x \notin \mathcal{G}$$

Text carrying no rock name is not a question the validator has to answer. Text that carries one is admitted **only** as a member of the closed registry $\mathcal{G}$. This **fails closed**: an unrecognised construction is refused rather than admitted, so no gap in any vocabulary can ever let an assertion through — because nothing is admitted by how it reads. `PROJECT_WELL_NAME_TOKENS`, `PROJECT_ROCK_BODY_TOKENS` and `PROJECT_REFERENCE_TOKENS` are deleted; nothing in the validator now enumerates what a project reference looks like.

$\mathcal{G}$ is **empty**, and that is a measured fact rather than an omission: this project persists exactly one field in explanatory scope — the manifest's own `named_lithology_statement` — and it carries no prohibited term at all. The mechanism is nevertheless live and tested, because a later increment explaining gamma-ray non-uniqueness may genuinely need to write *"a clean sandstone and a clean limestone read alike on GR"*. When that day comes it is a registered statement with recorded provenance, reviewed once — not a sentence admitted because a rule failed to recognise it as an assertion.

**All four scopes are now closed by membership or exact equality**, and the closure proof runs over all of them. The cost of this discipline is real and worth stating: legitimate generic science now requires a registry entry. That friction is the mechanism, not a side effect — it converts an unbounded judgement, made silently on every string, into a finite decision made once, in the open.

### 18. Why Boreas 1 is excluded rather than corrected

Boreas 1's `ECGR` curve shows a scale/acquisition anomaly that this notebook measures independently below. Three response options exist, and only one is defensible here:

1. **Rescale it** to match the other wells — requires a calibration reference. None exists: no tool header, no calibration record, no environmental-correction metadata, no core or spectral control, and no overlapping interval logged by two tools. A rescaling factor would be *invented*, and would then propagate silently into every shale proxy, every candidate interval, and every later NCT or pore-pressure result derived from them.
2. **Use it as-is** alongside the others — asserts an equivalence between curves whose central tendencies differ by a factor of four to five, which the data actively contradict.
3. **Exclude it from GR-derived work, retain it for factual QC** — states exactly what is known (the curve exists, here are its statistics, its behaviour is anomalous and unexplained) and exactly what is not (why).

Option 3 is the only one that adds no fabricated information, so it is what this project does. The exclusion is machine-readable (`BOREAS_ECGR_SCALE_UNRESOLVED`), enforced in code as a raised exception rather than a silent no-op, and tested. Boreas 1 still appears in raw-GR QC displays and availability reporting — exclusion from *interpretation* is not exclusion from *disclosure*.

---

## Environment and Locked-Foundation Setup

#### Step 1 — Mount Google Drive

**Technical objective:** re-attach the persistent project folder from prior increments.

**Inputs/outputs:** no project inputs are read here; this only establishes the `/content/drive` mount point.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')


#### Step 2 — Verify the Increment 5.1.2 foundation is present and locked

**Technical objective:** confirm the locked LAS-ingestion, deviation-survey, checkshot/time-depth, and formation-top layers already exist in this Drive project folder BEFORE this notebook adds anything on top. Increment 6 does not modify any of them.

**Failure behavior:** raises `RuntimeError` naming exactly which expected file is missing. This is a hard gate, not a warning.

In [ ]:
import os
from pathlib import Path

PROJECT_ROOT = "/content/drive/MyDrive/Poseidon_1D_MEM"

_required_locked_files = [
    "pyproject.toml",
    os.path.join("p2mem", "__init__.py"),
    os.path.join("p2mem", "units.py"),
    os.path.join("p2mem", "models.py"),
    os.path.join("p2mem", "deviation_models.py"),
    os.path.join("p2mem", "trajectory.py"),
    os.path.join("p2mem", "depth_mapping.py"),
    os.path.join("p2mem", "checkshot_models.py"),
    os.path.join("p2mem", "time_depth.py"),
    os.path.join("p2mem", "top_models.py"),
    os.path.join("p2mem", "io", "las.py"),
    os.path.join("p2mem", "io", "inventory.py"),
    os.path.join("p2mem", "io", "deviation.py"),
    os.path.join("p2mem", "io", "deviation_inventory.py"),
    os.path.join("p2mem", "io", "checkshot.py"),
    os.path.join("p2mem", "io", "checkshot_inventory.py"),
    os.path.join("p2mem", "io", "tops.py"),
    os.path.join("p2mem", "io", "tops_inventory.py"),
    os.path.join("config", "las_curve_contracts.yml"),
    os.path.join("config", "deviation_survey_contracts.yml"),
    os.path.join("config", "checkshot_contracts.yml"),
    os.path.join("config", "formation_top_contracts.yml"),
]
_missing = [f for f in _required_locked_files if not os.path.exists(os.path.join(PROJECT_ROOT, f))]
if _missing:
    raise RuntimeError(
        "Locked Increment 5.1.2 foundation is missing file(s): "
        + ", ".join(_missing)
        + ". Run the Increment 5 notebook first; Increment 6 does not "
        "reconstruct the locked foundation from memory."
    )
print("Locked Increment 5.1.2 foundation confirmed present:")
for f in _required_locked_files:
    print("  -", f)


#### Step 2b — Enter the project root before any relative file write

**Established execution-order rule:** Colab starts in `/content`, not `PROJECT_ROOT`, and every `%%writefile` cell below writes a RELATIVE path. This cell performs the `chdir` immediately after `PROJECT_ROOT` is defined and the locked foundation is confirmed (Step 2), and strictly before the first relative `%%writefile`.

**Failure behavior:** raises `RuntimeError` if the working directory does not resolve to `PROJECT_ROOT`, or if `p2mem` is not a directory there. Never silently creates a substitute directory.

In [ ]:
os.chdir(PROJECT_ROOT)

if Path.cwd().resolve() != Path(PROJECT_ROOT).resolve():
    raise RuntimeError(
        f"Failed to enter the project root. "
        f"Expected {PROJECT_ROOT}, actual working directory: {Path.cwd()}"
    )

if not Path("p2mem").is_dir():
    raise RuntimeError(
        f"Required p2mem directory is missing under {PROJECT_ROOT}. "
        "Extract the approved Increment package before running this notebook."
    )

print("Working directory confirmed:", Path.cwd())


#### Step 3 — Create the Increment 6 directory additions

**Technical objective:** create only the new directories this increment needs, without touching any existing directory.

In [ ]:
for d in ["outputs/06_petrophysics_eligibility", "outputs/06_petrophysics_eligibility/figures"]:
    os.makedirs(os.path.join(PROJECT_ROOT, d), exist_ok=True)
print("Increment 6 directories ready.")


#### Step 4 — Verify the approved LAS and deviation-survey filenames, and their SHA-256

**Technical objective:** require the exact approved filenames — four LAS files and four deviation surveys — and stop cleanly, naming every missing file, if any are absent. No substitute file is ever accepted, and no well's data is ever used in place of another's.

**Note on the deviation filenames:** these contain literal spaces (`Poseidon 2_dev.txt`), matching the project's supplied files exactly. They are never renamed.

**Failure behavior:** raises `RuntimeError` listing every missing filename by its exact expected name.

In [ ]:
import hashlib

LAS_DIR = os.path.join(PROJECT_ROOT, "data", "raw", "las")
DEV_DIR = os.path.join(PROJECT_ROOT, "data", "raw", "deviation")

LAS_FILES = {
    "Boreas_1": "Boreas_1_logs.las",
    "Poseidon_2": "Poseidon_2_logs.las",
    "Poseidon_North_1": "Poseidon_North_1_logs.las",
    "Proteus_1ST2": "Proteus_1ST2_logs.las",
}
DEV_FILES = {
    "Boreas_1": "Boreas 1_dev.txt",
    "Poseidon_2": "Poseidon 2_dev.txt",
    "Poseidon_North_1": "Poseidon North 1_dev.txt",
    "Proteus_1ST2": "Proteus 1ST2_dev.txt",
}

LAS_PATHS = {k: os.path.join(LAS_DIR, v) for k, v in LAS_FILES.items()}
SURVEY_PATHS = {k: os.path.join(DEV_DIR, v) for k, v in DEV_FILES.items()}

_missing_inputs = [p for p in list(LAS_PATHS.values()) + list(SURVEY_PATHS.values())
                   if not os.path.exists(p)]
if _missing_inputs:
    raise RuntimeError(
        "Missing required approved input file(s): "
        + ", ".join(os.path.basename(p) for p in _missing_inputs)
        + ". Increment 6 requires exactly these four LAS files and four deviation surveys, "
        "under these exact names - no other file is accepted as a substitute."
    )

print("All eight approved input files found. SHA-256:")
for key in sorted(LAS_PATHS):
    for p in (LAS_PATHS[key], SURVEY_PATHS[key]):
        print(f"  {key} ({os.path.basename(p)}): "
              f"{hashlib.sha256(open(p, 'rb').read()).hexdigest()}")


#### Step 5 — Install dependencies

**Technical objective:** install exactly the packages this increment's code and notebook display/plotting cells need. No new runtime dependency is added — `p2mem` itself still depends only on NumPy and PyYAML (see `pyproject.toml`); `pandas`/`matplotlib` remain notebook-only, unchanged from prior increments.

In [ ]:
!pip install -q numpy pyyaml pytest pandas matplotlib


---

## Increment 6 Source Files

#### Step 6 — Write the Increment 6 package files

**Technical objective:** write every new/updated source file for this increment, verbatim from the tested files on disk (this build script never hand-retypes code into notebook cells).

**Updated (version/documentation only — package version bumped to 0.6.8 for the Increment 6.1.7 corrective patch):** `pyproject.toml`, `p2mem/__init__.py`, `README.md`. **These are the only three existing files Increment 6 touches.** Every locked module, config, test, fixture, notebook, output, and figure from Increments 1–5.1.2 is intentionally NOT rewritten here.

In [ ]:
%%writefile pyproject.toml
[build-system]
requires = ["setuptools>=68.0"]
build-backend = "setuptools.build_meta"

[project]
name = "p2mem"
version = "0.6.8"
description = "Screening-level 1D Mechanical Earth Model workflow for Poseidon 2 (Tier C, uncalibrated / educational)."
readme = "README.md"
requires-python = ">=3.9"
license = { text = "All Rights Reserved. Copyright (c) 2026 Mikael Elgo. This is a personal portfolio project; no license is granted for reuse, redistribution, or commercial use without the author's explicit written permission." }
authors = [
    { name = "Mikael Elgo" }
]
keywords = ["geomechanics", "mechanical-earth-model", "pore-pressure", "wellbore-stability", "portfolio-project"]
classifiers = [
    "Development Status :: 3 - Alpha",
    "Programming Language :: Python :: 3",
    "Intended Audience :: Science/Research",
    "Topic :: Scientific/Engineering",
    "License :: Other/Proprietary License",
]

# Runtime dependencies are deliberately minimal. No unit-handling libraries
# (e.g. Pint) are used: unit conversions are implemented explicitly in
# p2mem.units so that every conversion factor is visible, documented, and
# testable rather than delegated to a third-party unit registry. PyYAML is
# added in Increment 2 for exactly one purpose: parsing the human-authored,
# human-reviewable per-file LAS curve contracts in
# config/las_curve_contracts.yml - a plain-text, diffable format was judged
# preferable to a hand-rolled config parser or a hard-coded Python dict.
dependencies = [
    "numpy>=1.24",
    "pyyaml>=6.0",
]

[project.optional-dependencies]
dev = [
    "pytest>=7.4",
]

[tool.setuptools.packages.find]
include = ["p2mem*"]

[tool.pytest.ini_options]
testpaths = ["tests"]
python_files = ["test_*.py"]


In [ ]:
%%writefile p2mem/__init__.py
"""
p2mem - Poseidon 2 1D Mechanical Earth Model workflow package.

Project classification: Tier C - Screening-Level / Uncalibrated Educational
1D Mechanical Earth Model (see project design review, Rev 1). Nothing in
this package should be presented as a calibrated, operational, or
field-validated result unless an explicit independent calibration record
is attached to that specific output.

This package is under incremental, gated construction.

* Increment 1 / 1.1 delivered the project skeleton and the unit-control
  system (``p2mem.units``).
* Increment 2 added an auditable LAS-ingestion layer with explicit
  per-file curve contracts (``p2mem.io.las``, ``p2mem.io.inventory``,
  ``p2mem.models``) for the four approved wells (Poseidon 2, Boreas 1,
  Poseidon North 1, Proteus 1ST2). It performs LAS parsing, curve-identity
  resolution, NULL-sentinel handling, and factual inventory generation
  ONLY - no deviation-survey processing, MD-to-TVD/TVDSS transformation,
  checkshot processing, formation-top correction, petrophysical
  interpretation, or any later-phase geomechanical calculation.
* Increment 2.1 / 2.1.1 are corrective patches to Increment 2, applied
  after independent technical audits, WITHOUT changing scope or the
  underlying LAS-parsing/curve-resolution architecture (which both audits
  found sound). 2.1 corrected: canonical array naming (every array is now
  explicitly unit-suffixed, e.g. ``VP_m_s`` rather than ``DTCO``, so a
  name can never be mistaken for the wrong physical quantity or unit);
  the measured-depth curve is now located via an explicit contract role
  rather than by matching a canonical name spelled "DEPT"; several
  file-identity checks (filename, SHA-256, WELL, VERS, WRAP, NULL) that
  were not previously blocking now are; curve-coverage statistics now
  report raw AND canonical values with explicit units; and per-well
  batch failures are now typed (``p2mem.models.IngestionFailure``)
  instead of bare caught exceptions. 2.1.1 corrected a packaging-only gap
  (three notebook ``%%writefile`` cells that had drifted from their
  packaged source files). See ``INCREMENT_02_v2.1_MANIFEST.md`` and
  ``INCREMENT_02_v2.1.1_MANIFEST.md`` for the full audits and
  corrected-file checksums.
* Increment 3 adds Petrel deviation-survey ingestion with explicit
  per-file contracts (``p2mem.io.deviation``), a standard minimum-
  curvature trajectory engine with a numerically stable ratio-factor
  limit (``p2mem.trajectory``), an explicit MD-referenced/TVD-referenced/
  TVDSS depth-reference framework and MD-to-TVD/TVDSS interpolation with
  no silent extrapolation (``p2mem.depth_mapping``), and typed dataclasses
  for all of the above (``p2mem.deviation_models``) - for the same four
  approved wells. It independently reproduces the Petrel-supplied
  trajectory to millimetre scale for three of the four wells and
  discloses (rather than resolves) a real, larger trajectory-
  reconstruction discrepancy found in Proteus 1ST2's deeper section - see
  ``INCREMENT_03_MANIFEST.md``. It performs deviation-survey ingestion,
  trajectory validation, and depth mapping ONLY - no checkshot
  processing, formation-top correction, petrophysical interpretation, or
  any later-phase geomechanical calculation.
* Increment 3.1 is a corrective patch to Increment 3, applied after an
  independent technical audit, WITHOUT changing scope, equations, real
  well data, or the locked LAS/Increment-2.1.1 foundation. It corrected
  four defects: (1) the four deviation-survey source filenames are the
  exact, literal names as they exist in Google Drive, which contain
  spaces (e.g. ``"Poseidon 2_dev.txt"``) - Increment 3 had incorrectly
  substituted underscores in the contract keys, notebook mapping, and
  tests, which would have failed to resolve against the real files;
  internal well keys (e.g. ``Poseidon_2``) remain underscored and are
  unaffected; (2) every exported CSV/JSON/manifest field is now
  guaranteed to carry a basename only, never a full environment-dependent
  build path (runtime-only diagnostic objects may still retain one);
  (3) the previously undisclosed inference that the supplied ``DLS``
  column is normalized as degrees per 30 metres is now explicitly flagged
  with a new, independently per-file-verified
  ``DLS_NORMALIZATION_INFERRED_AS_DEG_PER_30M`` WARNING (mirroring the
  pre-existing MD-unit-inference warning); (4) the dogleg angle between
  successive stations is now computed with a numerically stable
  ``arctan2(||cross||, dot)`` vector formulation (``p2mem.trajectory``)
  instead of ``arccos``, which was ill-conditioned near a zero dogleg and
  previously reported a spurious ~1e-6-degree value for two stations with
  identical inclination/azimuth. The real four-well data, station counts,
  tolerances, and the unresolved Proteus 1ST2 trajectory discrepancy are
  all unchanged by this patch. See ``INCREMENT_03_1_MANIFEST.md`` for the
  full audit and re-verification record.

* Increment 4 adds checkshot (velocity survey) ingestion with explicit
  per-file contracts (``p2mem.io.checkshot``, ``config/checkshot_
  contracts.yml``), typed checkshot dataclasses (``p2mem.checkshot_
  models``), deterministic checkshot inventory/QC-table builders
  (``p2mem.io.checkshot_inventory``), and a numerical time-depth layer
  (``p2mem.time_depth``): duplicate-tie detection/conditioning, average/
  interval velocity diagnostics, checkshot-vs-locked-survey depth-
  reference comparison, forward/inverse piecewise-linear time-depth
  interpolation with explicit coverage masking (no extrapolation), LAS
  MD-to-checkshot-time mapping within validated checkshot coverage only,
  and a Poseidon-2-only sonic-checkshot drift diagnostic (trapezoidal
  integration of sonic slowness vs. the checkshot-interpolated OWT
  increment over the same MD/Depth interval). Three checkshot files are
  admitted: ``Poseidon2-Checkshot.txt`` (Poseidon 2 - the ONLY checkshot
  approved to define a primary time-depth relationship),
  ``Boreas1-Checkshot.txt`` and ``Proteus1-Checkshot.txt`` (Boreas 1 and
  Proteus 1ST2 - supporting QC data only, newly admitted in this
  increment, never transferred into Poseidon 2 as a substitute time-depth
  model). Proteus 1ST2's association with ``Proteus1-Checkshot.txt`` is
  explicitly disclosed as inferred/unverified (no embedded well
  identifier). Poseidon North 1 has no approved checkshot file
  (``checkshot_availability: NOT_AVAILABLE`` - a factual data gap, not an
  ingestion failure). This increment reuses the LOCKED Increment 1
  ``owt_to_twt``/``twt_to_owt`` unit functions and the LOCKED Increment
  3/3.1.1 ``petrel_source_trace`` survey trajectory unchanged; it performs
  checkshot QC and time-depth framework work ONLY - no formation-top
  correction, lithology interpretation, density modelling, pore-pressure
  prediction, elastic properties, rock strength, stress modelling, or
  wellbore-stability analysis. See ``INCREMENT_04_MANIFEST.md`` for the
  full technical detail and independently recomputed statistics. NOTE:
  ``INCREMENT_04_MANIFEST.md`` contained one identified defect, corrected
  by Increment 4.1 below - do not rely on its original, uncorrected
  statement that TVDSS is strictly increasing after Depth-tie conditioning
  for all three wells.

* Increment 4.1 is a narrowly scoped corrective patch to Increment 4,
  applied after an independent technical/numerical-method audit, WITHOUT
  beginning Increment 5 or any formation-top/petrophysics/pore-pressure/
  mechanical-properties/stress/wellbore-stability work. It corrected two
  defects and hardened two numerical contracts: (1) ``tvdss_to_owt``/
  ``owt_to_tvdss`` (and their TWT equivalents) previously resolved a
  repeated (tied) value on the axis being inverted by silently keeping
  whichever tied row appeared first in the Depth-conditioned table and
  discarding the other (``p2mem.time_depth._build_strictly_increasing_
  table``, REMOVED) - an ORDER-DEPENDENT tie-break with no audit trail
  beyond a bare count. This is replaced by ``build_axis_conditioned_
  lookup_table``/``build_axis_conditioned_tables_for_well``: an explicit,
  ORDER-INVARIANT policy that groups every tied value by exact equality
  regardless of parse order, registers every tied row in a new typed
  audit register (``AxisTimeDepthTieRegisterEntry`` /
  ``checkshot_time_axis_tie_register.csv`` - separate from, and never
  confused with, the pre-existing Depth-axis ``DuplicateTieRegisterEntry``
  register), and uses the tied group's dependent-value MEDIAN as the
  conditioned representative (order-invariant; disclosed as reducing to
  the arithmetic mean for the size-2 groups observed in this project's
  real data). A genuine reversal (not a tie) in the axis being inverted
  raises ``TimeDepthError`` rather than being sorted, discarded, or forced
  monotonic. (2) ``INCREMENT_04_MANIFEST.md``'s statement that TVDSS is
  strictly increasing after Depth-tie conditioning for all three wells was
  INCORRECT - independently reproduced counts (Poseidon 2: two TVDSS-axis
  and two OWT-axis ties; Boreas 1: one TVDSS-axis tie, zero OWT-axis ties;
  Proteus 1ST2: none of either) are now documented in
  ``INCREMENT_04_1_MANIFEST.md`` and reflected in this module's own
  docstrings. (3) ``trapezoidal_integrate`` and (4) ``compute_sonic_
  checkshot_drift`` are hardened to validate their numerical
  preconditions (finite, one-dimensional, equal-length, strictly
  increasing MD/x where required) rather than silently integrating
  invalid input - most notably, a decreasing or duplicate MD run can no
  longer silently produce a physically invalid NEGATIVE transit time; it
  now raises ``TimeDepthError``. None of this hardening changes the
  already-verified real Poseidon 2 sonic-drift result, which is
  bit-for-bit unchanged. See ``INCREMENT_04_1_MANIFEST.md`` for the full
  audit, corrected statistics, and re-verification record.

* Increment 4.1.1 is a narrowly scoped numerical-validation corrective
  patch to Increment 4.1, applied after an independent numerical-method/
  software-QA audit, WITHOUT beginning Increment 5 or any formation-top/
  petrophysics/pore-pressure/mechanical-properties/stress/wellbore-
  stability work. It corrected four blocking defects and one input-safety
  gap, none of which altered any previously verified REAL Poseidon
  2/Boreas 1/Proteus 1ST2 result: (1) ``build_axis_conditioned_lookup_
  table`` grouped ALL occurrences of an identical axis value together
  GLOBALLY before checking for a reversal, so a reversal that returned to
  an already-seen value (e.g. ``[100.0, 200.0, 100.0]``) was silently
  hidden rather than raising ``TimeDepthError`` - it now evaluates the
  ORIGINAL, ungrouped sequence's successive differences for negativity
  BEFORE any grouping is attempted, which is provably equivalent to the
  4.1 behavior for every legitimate adjacent tie and strictly stronger
  against a non-adjacent reversal. (2) ``compute_sonic_checkshot_drift``/
  ``find_longest_finite_positive_run`` validated MD monotonicity only
  within the selected finite-positive-VP run, so a decreasing or
  duplicate MD value outside that run (e.g. at a NaN-VP station) could
  pass silently - the COMPLETE canonical ``md_m`` array is now required
  finite and strictly increasing before run-selection (the real Poseidon
  2 MD array, 31,897 samples, was independently re-verified to already
  satisfy this). (3) ``compare_checkshot_to_survey`` reached an untyped
  NumPy ``ValueError`` ("zero-size array to reduction operation") if
  every checkshot Depth row fell outside the locked survey's own MD
  coverage - it now raises a typed ``TimeDepthError`` naming the well,
  the checkshot Depth range, and the survey MD coverage. (4)
  ``p2mem.io.checkshot.load_checkshot_surveys`` did not catch
  ``TimeDepthError`` raised during numerical conditioning, so a defect in
  one well's data could stop the entire batch - it is now caught per well
  (never via a blanket ``except Exception``) and recorded as a typed
  ``CheckshotIngestionFailure(error_type="numerical_conditioning_
  failure")``, isolated exactly like every other expected per-well
  failure. (5) ``seconds_to_milliseconds``/``milliseconds_to_seconds``
  coerced their input directly, unlike ``p2mem.units``'s Increment-1
  input-safety policy, so a boolean, numeric-looking string, or complex
  value would be silently reinterpreted rather than rejected - both now
  reject such input with ``TypeError`` via a local, documented copy of
  ``p2mem.units``'s identical private dtype check (``p2mem/units.py``
  itself remains LOCKED and unmodified). See
  ``INCREMENT_04_1_1_MANIFEST.md`` for the full audit, the regression-test
  list, and the re-verification record.

* Increment 5 adds contract-driven formation-top ingestion, HRS-versus-
  selected-readable source RECONCILIATION, and survey-corrected
  stratigraphic depth mapping (``p2mem.top_models``, ``p2mem.io.tops``,
  ``p2mem.io.tops_inventory``) for the two approved wells with formation-
  top data (Poseidon 2, Boreas 1). Each well has TWO independently
  supplied top files - an "HRS" file (``Top_Name``/``MDRT_m`` only, no
  well name in the file body) and a "selected readable" file
  (``TOP_NAME``/``MDRT_M``/``TVDSS_M``/``NOTE``, with ``#``-comment
  headers and a dashed separator line, deliberately parsed rather than
  treated as data) - and neither is silently preferred: markers are
  matched by exact/normalized name or an explicit human-authored alias
  contract (never fuzzy matching), and MDRT is cross-checked between the
  two sources within a documented 0.005 m tolerance; a marker on which
  the two sources disagree beyond that tolerance is excluded from depth
  mapping (``mapping_status == "not_mapped_mdrt_unresolved"``) rather
  than resolved by picking one file. Reconciled MDRT is mapped through
  the LOCKED Increment 3.1.1 ``petrel_source_trace`` survey trajectory
  using the existing, unmodified ``p2mem.depth_mapping.
  map_las_md_to_tvd_tvdss`` (per marker, so one out-of-coverage marker
  never blocks the rest of the well; extrapolation is never performed),
  producing ``TVD_survey_m``/``TVDSS_survey_corrected_m`` alongside an
  explicit, unambiguous ``TVDSS_residual_source_minus_survey_m =
  TVDSS_source_m - TVDSS_survey_corrected_m`` residual field. Both file
  representations for both wells are classified
  ``well_identity_evidence_status = "inferred_unverified"`` (filename-only
  or in-file-comment association, never independently content-verified -
  a more conservative classification than Increment 4's checkshot files).
  Poseidon North 1 and Proteus 1ST2 have no approved formation-top file
  and are recorded as ``FormationTopAvailabilityRecord(..., "NOT_
  AVAILABLE")`` - never substituted or depth-correlated from another
  well. Independently recomputing (never hardcoding) this increment's two
  required regression findings against the real approved files confirmed:
  Poseidon 2's "selected readable" file's supplied TVDSS equals
  ``MDRT - 21.8 m`` EXACTLY for every one of its 9 markers (21.8 m is the
  well's own rotary-table elevation) - a literal vertical-well-assumption
  depth-reference defect, corrected in the derived
  ``TVDSS_survey_corrected_m`` representation while the raw
  ``TVDSS_source_m`` column is preserved unmodified; survey-corrected
  residuals reproduce Sea Bed at ~0 m, Plover Fm (Top Reservoir) at
  +1.68 m, and TD at +2.49 m, exactly as the approved Rev 1 design
  anticipated. Boreas 1's supplied TVDSS does NOT follow that pattern and
  its maximum absolute source-versus-survey residual is 0.0416 m, below
  the approved 0.05 m tolerance - confirming it was generated from the
  well's real surveyed trajectory, not a vertical-well shortcut, and it is
  therefore NOT "corrected" the way Poseidon 2 is. See
  ``INCREMENT_05_MANIFEST.md`` for the full audit, the per-file contracts,
  and the complete reconciliation/mapping results.

* Increment 5.1 is a narrowly scoped corrective patch to Increment 5,
  applied after an independent technical/software-QA audit, WITHOUT
  beginning Increment 6 or any petrophysics/pore-pressure/mechanical-
  properties/stress/wellbore-stability work, and WITHOUT changing any
  previously verified real Poseidon 2/Boreas 1 formation-top result. It
  corrected three defects and one documentation-accuracy gap: (1)
  ``reconcile_formation_top_sources`` documented "zero canonical markers
  in common between the two sources" as a fatal ``NO_COMMON_MARKERS``
  ERROR, but actually tested the emptiness of the UNION of both sources'
  canonical names - so two entirely DISJOINT, non-empty marker sets (e.g.
  HRS names sharing nothing with the readable file's names) were silently
  accepted as one-sided ``NOT_COMPARABLE`` entries instead of being
  rejected; the check now explicitly evaluates the INTERSECTION of the two
  sources' canonical names and raises the documented ERROR (and
  ``load_formation_top_well`` consequently raises
  ``TopSourceReconciliationError``) whenever both sources are non-empty but
  share nothing, while a legitimate one-sided marker (with at least one
  marker genuinely shared) remains the pre-existing, non-fatal
  ``NOT_COMPARABLE`` case. (2) ``load_formation_top_surveys`` recorded only
  the HRS file's path in every ``TopIngestionFailure.source_path``,
  regardless of which file/stage actually failed - so a failure originating
  from the "selected readable" file (missing file, malformed content, or a
  contract mismatch) could leak that file's own absolute path, unsanitized,
  into exported issues/availability/manifest rows (the sanitizer only ever
  stripped the recorded, and in that case WRONG, HRS path).
  ``TopIngestionFailure`` now carries an explicit ``failure_origin``
  ("hrs"/"readable"/"reconciliation"/"mapping"/"unknown") and both
  ``hrs_path``/``readable_path`` fields, ``load_formation_top_well`` tags
  each raised exception with the stage that actually failed, and every
  exporting function in ``p2mem.io.tops_inventory`` now sanitizes BOTH
  candidate paths (literal substring replacement only, never a regex) and
  reports the correctly identified failing file's basename as context. (3)
  ``reconcile_formation_top_sources`` - the public in-memory API, as
  distinct from the file parsers, which already enforced this - did not
  validate its own documented contract before any numerical comparison:
  non-finite (NaN/Inf) or negative MDRT/TVDSS values, a shorter
  ``NOTE_source`` tuple (previously reaching an untyped ``IndexError``), a
  non-1-dimensional array, and an unvalidated ``mdrt_agreement_tolerance_m``
  keyword (previously accepting ``NaN``, a negative value, or a boolean,
  reaching an incidental ``TypeError`` only for a string) were all silently
  accepted or reached an undocumented incidental error. All of these are
  now rejected before any comparison, with deliberate, documented
  exceptions (``TopParsingError`` for a value/structural defect,
  ``TypeError`` for a type-class defect - matching this module's existing
  split), while valid Python ``int``/``float`` and NumPy integer/floating
  scalars (including 0-d/size-1 arrays) continue to work. (4)
  ``INCREMENT_05_MANIFEST.md`` stated that no ``/home/``, ``/root/``,
  ``/content/``, or absolute path exists ANYWHERE in the package - this was
  inaccurate, since existing synthetic tests and historical documentation
  intentionally contain fake absolute-path strings as test inputs; the
  precise, narrowly scoped claim ("no environment-dependent build path
  appears in exported CSV/JSON outputs") is now stated explicitly in
  ``INCREMENT_05_1_MANIFEST.md``, which also acknowledges the prior
  wording was overbroad. ``INCREMENT_05_MANIFEST.md`` itself is a locked
  historical record and is NOT rewritten. See ``INCREMENT_05_1_MANIFEST.md``
  for the full audit, the regression-test list, and the real-data
  non-regression verification record.

* Increment 5.1.1 is a further narrowly scoped corrective patch to
  Increment 5.1, applied after an independent technical/software-QA audit,
  WITHOUT beginning Increment 6 and WITHOUT changing any scientific result,
  tolerance, depth-mapping method, or formation-top contract. It corrected
  one remaining reconciliation edge case and three documentation-accuracy
  gaps. (1) Increment 5.1 correctly rejected two non-empty, disjoint
  marker sets as a fatal ``NO_COMMON_MARKERS`` ERROR, but its condition
  (``both sources non-empty AND intersection empty``, plus a separate
  both-empty case) still silently accepted the remaining zero-common-
  markers configuration: exactly ONE source entirely empty and the other
  non-empty - the intersection of an empty set with anything is itself
  empty, so this was still a zero-common-markers condition, but the
  ``hrs_by_canon and readable_by_canon`` non-empty guard skipped it.
  ``reconcile_formation_top_sources`` now uses the single, strictly
  correct check ``if not common_markers`` (the intersection of the two
  sources' canonical names), which is empty in every zero-common-markers
  configuration - both empty, either side alone empty, or both non-empty
  and disjoint - and is never empty whenever at least one canonical marker
  is genuinely shared, so a legitimate one-sided marker alongside at least
  one shared marker remains the pre-existing, non-fatal ``NOT_COMPARABLE``
  case. (2) ``INCREMENT_05_1_MANIFEST.md`` stated that the real Increment
  5 baseline ZIP's SHA-256 "matches" a governing-prompt-supplied hash that
  was, in fact, a two-character truncation of the real 64-character value
  - a mathematical impossibility that is now stated transparently in
  ``INCREMENT_05_1_1_MANIFEST.md`` as a documentation/input typo, never as
  baseline corruption or uncertainty. (3) ``INCREMENT_05_1_MANIFEST.md``
  Section 7 stated that ``outputs/`` is excluded from the delivered ZIP;
  this was false - the delivered Increment 5.1 ZIP packages the
  ``outputs/`` tree (including the byte-identical Increment 5 formation-
  top outputs/figures) exactly as every prior increment's ZIP has; only
  the dev-only regenerated-comparison directories and private raw source
  files are excluded, and this is now stated accurately. (4) the same
  manifest's clean-room section stated that no ``*.las`` file exists in
  the package; this was false because small, intentionally packaged
  synthetic LAS/checkshot/deviation/top fixtures exist under
  ``tests/fixtures/`` for portable testing - the corrected wording
  distinguishes these fictional fixtures (never leaked private data) from
  the genuinely excluded real/private project LAS, deviation, checkshot,
  and formation-top source files. See ``INCREMENT_05_1_1_MANIFEST.md`` for
  the full audit, the regression-test list, and the real-data
  non-regression verification record.

Increment 6 (this release, v0.6.0) adds the gamma-ray QC, shale-proxy
sensitivity, well-frame assembly, and method-eligibility framework, on top
of the LOCKED Increment 1-5.1.2 foundation (no locked module, config,
test, fixture, notebook, output, or figure is modified by it). It
contributes:

* ``p2mem.wellframe_models`` / ``p2mem.wellframe`` - a typed, auditable
  per-well assembly of the locked canonical LAS curve arrays alongside the
  locked MD->TVD/TVDSS mapping, with per-sample validity masks, per-curve
  provenance, QC flags, and evidence classification. Sample count and
  order are preserved exactly; invalidity is expressed only through masks;
  no depth is ever extrapolated (a LAS sample outside the survey's own MD
  coverage is recorded as depth-unmapped, never clamped or held).
* ``config/petrophysics_eligibility.yml`` - the human-authored, reviewable
  per-well gamma-ray-family disposition, endpoint-sensitivity policy, and
  eligibility rules. Boreas 1 is formally excluded from every
  lithology-dependent, GR-normalized, shale-proxy and NCT-candidate
  calculation under the machine-readable reason
  ``BOREAS_ECGR_SCALE_UNRESOLVED``; it is EXCLUDED, never corrected or
  rescaled, because no calibration evidence exists to support any
  correction.
* ``p2mem.petrophysics_models`` / ``p2mem.petrophysics`` - factual
  GR-family QC statistics for every well (including excluded ones),
  per-well low/base/high endpoint scenarios whose measured endpoint values
  are recorded, and the dimensionless GR index with clipped and unclipped
  results retained side by side. The only permitted shale-proxy transform
  is the linear identity of the clipped index, exported under the
  self-labelling name ``VSH_GR_linear_proxy_frac``; every nonlinear Vsh
  transform (Larionov, Clavier, Stieber, ...) is deliberately DEFERRED
  pending a retrieved, verified primary-source method record.
* ``p2mem.method_eligibility`` - three input-admissibility masks
  (``eligible_density_for_sv``, ``eligible_dynamic_elastic``,
  ``eligible_sonic_nct_candidate``) plus contiguous-interval registers on
  MD/TVD/TVDSS with explicit, configured, tested gap tolerances that
  distinguish sample-count continuity from physical-depth continuity.
  ELIGIBILITY IS NOT VALIDITY: none of the gated methods is implemented,
  fitted, or validated here.
* ``p2mem.io.petrophysics_inventory`` - deterministic summary/interval/
  manifest builders that never export a per-sample real-data array and
  never emit an absolute path.

Increment 6 assigns NO named lithology anywhere. Gamma-ray response is not
uniquely diagnostic of rock type and no independent lithological evidence
exists in this project, so every classification it produces describes DATA
AND PROXY CONFIDENCE ONLY, guarded by an enforced prohibited-vocabulary
check.

Increment 6.1 (v0.6.1) is a narrowly scoped corrective patch
to Increment 6, applied after an independent geomechanics/rock-physics and
software audit. It changes no architecture and starts no new phase. Four
findings were corrected:

* **Vp/Vs terminology and boundaries.** Increment 6 labelled every excluded
  Vp/Vs ratio "non-physical" and used a strictly exclusive sqrt(2) bound.
  Both were wrong. With r = Vp/Vs, nu = (r^2 - 2) / (2 (r^2 - 1)) and
  K = rho (Vp^2 - (4/3) Vs^2), so: r = sqrt(2) gives nu = 0 EXACTLY and must
  be ACCEPTED by a non-negative-nu policy (the bound is now INCLUSIVE);
  sqrt(4/3) < r < sqrt(2) gives POSITIVE bulk modulus with negative Poisson
  ratio - unusual and outside this project's conservative policy, but not
  physically impossible; only r <= sqrt(4/3) implies a non-positive bulk
  modulus and is genuinely outside the isotropic elastic model; and r > 4 is
  a CONFIGURED PLAUSIBILITY LIMIT, not a Poisson-domain boundary (nu ~= 0.467
  there). The condition is renamed a CONFIGURED NON-NEGATIVE-POISSON-RATIO
  APPLICABILITY SCREEN, the three config bounds are separately named, and
  exclusions are diagnosed by regime (`n_ratio_nonpositive_bulk_modulus`,
  `n_ratio_positive_bulk_but_negative_poisson`,
  `n_ratio_above_configured_plausibility_max`, `n_vp_not_greater_than_vs`) -
  never aggregated under a single "non-physical" count.
* **Poseidon North 1 depth-tied status.** Its machine-readable `use_status`
  was `screening_proxy_allowed` while it has no approved formation tops,
  contradicting both the status vocabulary and the prose limitation. It is
  now `screening_proxy_allowed_depth_tied`, and a config invariant makes any
  contradictory has-tops/use-status pairing fail config loading loudly.
* **Unsupported lithology claims and a circular gate.** Project-specific
  "clastic-dominated" assertions and the unsupported "regionally persistent"
  cross-well claim are removed (two of the three GR-eligible wells have no
  approved tops, so cross-well stratigraphic persistence cannot be
  established). `clastic` and related rock-class terms join the prohibited
  vocabulary, and `named_lithology_assigned` is now DERIVED from an actual
  validation pass over every persisted label, per-well note, mask name and
  manifest statement - injecting a prohibited term makes the validation, the
  manifest flag and the completion gate fail together. Validation is
  three-tier: LABELS admit no prohibited term at all, per-well PROSE may name
  the method category ("shale proxy") but not assert a rock, and EXPLANATORY
  text may use rock names as generic examples of gamma-ray non-uniqueness
  but never in a sentence naming one of this project's wells.
* **Gross versus net/strict interval thickness.** The reported "qualifying
  thickness" summed block endpoint spans that may contain explicitly bridged
  ineligible samples. Thickness is now exported only under qualified names -
  `gross_thickness_*_m` (endpoint span, bridging-inclusive) and
  `net_thickness_*_m` (bridged gaps removed) - every mask is additionally
  decomposed under a `strict_no_gap` contiguity policy for comparison, every
  sensitivity case reports its bridged-sample and interrupted-block counts,
  and Figure 4 now draws qualifying and rejected sub-threshold blocks
  distinctly with the counted population stated.

Increment 6.1.1 (v0.6.2) is a second, narrower corrective
patch, applied after an independent audit of Increment 6.1. It starts no
new phase, changes no architecture, and alters no configured tolerance,
threshold, or qualifying-block policy. Five findings were corrected:

* **Named-lithology validator bypass.** The method-term allowlist that lets
  per-well prose name a METHOD ("shale proxy") contained "shale gas", which
  is not a method-category phrase but can be a direct geological/hydrocarbon
  assertion; sentences such as "This interval contains shale gas." therefore
  passed with zero violations. The allowlist is reduced to the two phrases
  that are genuinely method/quantity names required to describe this
  project's boundaries - ``shale proxy`` and ``shale volume`` - and the
  allowance is now evaluated PER SENTENCE and suppressed entirely in any
  sentence carrying a geological-assertion cue (``contains``, ``comprises``,
  ``bearing``, ``facies``, ...) or where the phrase is immediately preceded
  by a quantity cue (``has a high shale volume``). Regression tests prove
  the allowlist can no longer hide a geological assertion and that injecting
  either bypass case into manifest content makes ``named_lithology_assigned``
  true, ``lithology_validation.n_violations`` non-zero, and the completion
  gate fail.
* **Stale Vp/Vs scientific description.** The active top-level docstring of
  ``p2mem.method_eligibility`` still described the screen as keeping samples
  within the Poisson domain under a strictly EXCLUSIVE sqrt(2) ratio bound,
  contradicting the corrected implementation. It now states the configured
  non-negative-Poisson-ratio applicability screen with an INCLUSIVE
  ``Vp/Vs >= sqrt(2)`` bound and says explicitly that this is neither a
  physical-possibility test nor a boundary of the mathematical Poisson
  domain. A regression test asserts the corrected wording against the live
  module documentation.
* **Ambiguous interruption counts.** ``n_interruptions`` was a boolean-like
  flag reported as a count (always 1 for every bridged block), and
  ``n_interrupted_subruns`` did not describe what it counted. Interval
  records now carry three separately named, independently meaningful
  quantities: ``n_bridged_samples`` (ineligible samples absorbed inside the
  gross block), ``n_bridged_gaps`` (distinct bridged runs) and
  ``n_eligible_subruns`` (strictly contiguous eligible sub-runs), with the
  identity ``n_bridged_gaps == n_eligible_subruns - 1`` enforced at
  construction. The ambiguous aliases are gone from all active exports.
* **Notebook output-count error.** The notebook creates and checks eight
  deterministic CSV/JSON outputs but its gate text and printed label said
  seven. Both now say eight, and the gate asserts both the declared count
  and the existence of all eight files. Four figures remain separate,
  giving twelve outputs/figures in total.
* **Incorrect Increment 6.1 delta arithmetic.** The Increment 6.1 completion
  record stated "Changed (12)"; a clean recursive comparison gives 18
  changed, 3 added and 0 removed (21 path differences). That statement is
  explicitly superseded - not silently rewritten - in the Increment 6.1.1
  manifest and completion record, which report both measured deltas file by
  file.

Increment 6.1.2 (v0.6.3) is a third corrective patch, applied
after an independent audit of Increment 6.1.1. It starts no new phase, changes
no architecture, alters no scientific threshold, tolerance, endpoint scenario,
contiguity policy, GR disposition, depth-mapping rule, or real-data
interpretation, and adds no runtime dependency (NumPy and PyYAML remain the
only two). Two residual findings were corrected:

* **The named-lithology validator was bidirectionally incorrect.** Increment
  6.1.1 decided the method-phrase allowance from a SENTENCE-WIDE assertion-cue
  list plus a fixed LOOK-BEHIND window. Both were structurally wrong. Because
  nothing to the RIGHT of an allowed phrase was ever inspected, and because
  the normalizer destroyed possessives, direct assertions such as "Poseidon
  2's shale volume is high.", "Poseidon 2's shale volume is 70 percent." and
  "The interval's shale volume exceeds 60 percent." passed with ZERO
  violations. Conversely, because a sentence-wide cue fires without asking
  what the cue word is predicated OF, legitimate method and boundary
  statements such as "This method contains a shale proxy calculation." and
  "The analysis shows no shale volume was computed." were reported as
  lithological assertions. The allowance is no longer a property of the
  sentence: it is decided for EACH OCCURRENCE of an allowed phrase from that
  occurrence's own local grammar - whether it is possessed by a geological
  entity (``the interval's shale volume``), modified by a magnitude (``a high
  shale volume``), predicated with an amount or a dominance (``is 70
  percent``, ``exceeds 60 percent``, ``dominates the interval``), or sits in a
  sentence whose composition verb takes a GEOLOGICAL subject (``the unit
  comprises ...``, but not ``this method contains ...``). Apostrophes and
  possessives are parsed rather than erased, sentence boundaries include line
  breaks, and genuinely ambiguous project-specific sentences fail closed.
  ``SCOPE_LABEL`` remains absolute: a label gets no latitude at all.

* **Interval-record invariants were only partially enforced.** Increment 6.1.1
  claimed the interruption-count identities were enforced at construction, but
  the only check was ``n_bridged_gaps == n_eligible_subruns - 1``, skipped
  whenever either value was ``None`` or ``n_eligible_subruns`` was 0. Records
  with missing counts, negative counts, zero sub-runs, boolean counts, or
  bridged samples without a bridged gap were all constructible. All three
  counts are now validated together as one coherent record: each is required,
  must be a true integer count (booleans, strings, complex values and
  fractional floats are rejected, never coerced), ``n_bridged_samples >= 0``,
  ``n_bridged_gaps >= 0``, ``n_eligible_subruns >= 1``, ``n_bridged_gaps ==
  n_eligible_subruns - 1``, ``n_bridged_samples == 0`` if and only if
  ``n_bridged_gaps == 0``, and ``n_bridged_samples >= n_bridged_gaps`` because
  every distinct gap holds at least one sample. Each violation raises
  ``PetrophysicsInputError`` naming the offending field or relationship, and
  the removed ``n_interruptions`` / ``n_interrupted_subruns`` field names are
  rejected outright rather than silently ignored.

Every scientific result is unchanged by this patch: the same eight
deterministic outputs and four figures are produced, byte for byte.

Increment 6.1.3 (v0.6.4) is the fourth and architectural
corrective patch, applied after an independent audit of Increment 6.1.2. It
starts no new phase, changes no scientific threshold, tolerance, endpoint
scenario, contiguity policy, GR disposition, depth-mapping rule, or real-data
interpretation, and adds no runtime dependency (NumPy and PyYAML remain the
only two).

**The validator no longer tries to understand English.** Increments 6.1.1 and
6.1.2 each attempted to decide, from grammar, whether a sentence containing a
rock name was NAMING A METHOD or ASSERTING GEOLOGY - 6.1.1 with a
sentence-wide cue list and a fixed look-behind window, 6.1.2 with
per-occurrence possessive/modifier/predicate/subject analysis. Both were
audited and both failed in BOTH directions. 6.1.2 still missed "The interval
has a shale volume.", "The shale volume in Poseidon 2 exceeds 60 percent." and
"Poseidon 2 shale volume was determined to be high.", while wrongly rejecting
"The well contains no shale volume estimate." Every one of those is a list gap
or a window edge. The lesson is not that the lists were too small: free-text
geological-assertion detection is unbounded, and no finite grammar closes it.

Increment 6.1.3 replaces judgement with membership. Every scope is now decided
by set membership or exact equality:

* ``SCOPE_LABEL`` - verdicts. Zero allowance.
* ``SCOPE_INTERPRETIVE`` - project-specific prose. **ZERO ALLOWANCE.** There is
  no exemption path at all, so there is nothing to bypass. The
  ``allow_method_phrases`` parameter is gone.
* ``SCOPE_METHOD`` (new) - method and limitation statements. The text must
  EQUAL a member of the closed, provenance-tagged ``METHOD_STATEMENTS``
  registry. Not matched, not scored - equal. The registry has five members,
  measured rather than guessed by scanning every string literal in active
  packaged source and every persisted field under a zero-allowance rule.
* ``SCOPE_EXPLANATORY`` - generic scientific text. A rock name is permitted
  only when the sentence refers to no project well AND no project rock body.
  6.1.2 checked well names alone, which is why an assertion about "the
  interval" behaved inconsistently between scopes.

The deletion is the correction: ``ALLOWED_METHOD_TERM_PHRASES``,
``GEOLOGICAL_ENTITY_TOKENS``, ``COMPOSITION_PREDICATES``,
``MAGNITUDE_PREDICATES``, ``MAGNITUDE_WORDS``, ``COPULAR_VERBS``,
``ATTRIBUTIVE_ROCK_TOKENS``, ``NEGATION_TOKENS``, the look-behind and
look-ahead windows, subject resolution and occurrence classification are all
GONE, and a regression test asserts none of them is importable. Correctness is
proved by CLOSURE - every prohibited term, in every position, inside carrier
prose built from the exact constructions that defeated both previous
implementations - not by a list of example sentences.

This patch also closes a gap neither audit reported: three strings this project
WRITES INTO PACKAGED EXPORTS had never been inside the validated scope in any
increment, including a ``calibration_status`` value literally containing
``not_a_shale_volume`` persisted to every row of
``gr_proxy_sensitivity_summary.csv``. All three are now scanned in
``SCOPE_METHOD``, which raises ``n_fields_checked`` from 138 to 143.

**Interval-record type gate.** Increment 6.1.2 enforced the relational rules
but let three type defects through: ``0.0 / 0.0 / 1.0`` was accepted although
the contract requires integers; ``NaN`` and ``Inf`` escaped as bare
``ValueError`` / ``OverflowError`` from ``int()``; and an unknown keyword such
as a misspelled count name was silently ignored, leaving the real count unset.
Keyword acceptance is now a WHITELIST over the declared slots, floats are
rejected outright including whole-valued ones, and non-finite values raise a
typed ``PetrophysicsInputError`` before any conversion is attempted.

All eleven scientific outputs and figures are byte-identical to Increment
6.1.2; only the manifest changes, by the single scalar named above.

Increment 6.1.4 (v0.6.5) completes the architecture Increment
6.1.3 began. It is deliberately narrow: one scope rule, no other change.

Increment 6.1.3 closed LABEL and INTERPRETIVE by removing every exemption and
closed METHOD by exact registry membership - but left EXPLANATORY decided by a
FINITE TOKEN LIST of project references. That is the same shape of rule that
failed in 6.1.1 and 6.1.2, and it had the same defect. Ordinary stratigraphic
and exploration nouns were absent from the list, so in explanatory scope

    "The member is shale."               passed
    "The group is limestone."            passed
    "The package is a clean sandstone."  passed
    "The play is shale-dominated."       passed
    "The prospect is carbonate."         passed
    "The target is sandstone."           passed

while "The upper member is a clean sandstone reservoir." failed only because
``reservoir`` happened to be listed - an accident, which is what a list-shaped
rule produces.

The rule is INVERTED and made identical in kind to SCOPE_METHOD: explanatory
text carrying a prohibited term is a violation UNLESS the text is a member of
the closed ``GENERIC_EXPLANATORY_STATEMENTS`` registry. It FAILS CLOSED, so no
vocabulary gap can admit anything. ``PROJECT_WELL_NAME_TOKENS``,
``PROJECT_ROCK_BODY_TOKENS`` and ``PROJECT_REFERENCE_TOKENS`` are deleted, and
a regression test asserts none of them is importable. Nothing in the validator
now enumerates what a project reference looks like.

The generic registry is **empty**, as a measured fact rather than an omission:
this project persists exactly one explanatory field,
``manifest.named_lithology_statement``, and it carries no prohibited term at
all. The mechanism is nevertheless live and tested, because later increments
explaining gamma-ray non-uniqueness may genuinely need to write "a clean
sandstone and a clean limestone read alike on GR" - and when they do, that is
a registered, reviewable statement rather than a sentence admitted by shape.

All four validation scopes are now closed by membership or exact equality, and
the closure proof covers all of them. All twelve outputs and figures are
byte-identical to Increment 6.1.3.

Increment 6.1.5 (v0.6.6) replaces blacklist-based acceptance
with POSITIVE AUTHORIZATION, and is the final Increment 6 corrective patch.

Increments 6.1 through 6.1.4 all asked the same question in different ways:
"does this text contain a geological assertion?" All four shared one
structural assumption - that content is ACCEPTABLE BY DEFAULT and becomes
unacceptable only when a recognizer fires - and all four were defeated,
finally by a word no recognizer had been given. In the 6.1.4 package these
all passed label, interpretive and explanatory scope:

    "The interval is chalk."         "The interval is chert."
    "The interval is halite."        "The interval is tuff."
    "The interval is gypsum."        "The interval is basalt."
    "The interval is conglomerate."  "The interval is dolostone."
    "The interval is lignite."       "The interval is calcareous."

and so did "The interval is qxzite.", a word that does not exist. They already
failed in METHOD scope, which was the only scope then requiring registration -
and that is the clue this patch acts on. A longer blacklist would have caught
the first ten and still missed the eleventh, so lengthening it is neither a
completion criterion nor the mechanism this package's assurance rests on.

The model is inverted. Nothing is acceptable by default:

* ``SCOPE_LABEL`` - the value must be a member of ``APPROVED_LABELS``, a
  registry of typed, enumerated label values each declaring its field kind,
  purpose and provenance. Arbitrary caller-supplied label text is rejected
  even when it contains no recognizable rock name at all.
* ``SCOPE_INTERPRETIVE`` - the text must resolve to a registered
  ``statement_id``, or to a reviewed ``template_id`` whose substitutions are
  strictly typed. Unregistered free text is rejected unconditionally.
* ``SCOPE_METHOD`` - the text must resolve to a registered ``statement_id``,
  and the id and the exact rendered text are validated TOGETHER, so neither a
  renamed id nor an edited sentence passes on the strength of the other.
* ``SCOPE_EXPLANATORY`` - identical, for every non-empty statement, whether or
  not any prohibited term is detected. The early-pass behaviour equivalent to
  "if no prohibited term is found: accept" is gone from every scope.

The registries were MEASURED from the actual persisted Increment 6 export, not
designed: 17 approved label values, 17 registered statements (11 interpretive,
5 method, 1 explanatory) and 1 controlled template covering the three per-well
confidence rationales, whose only variable parts are three decimal literals.
Every registered entry carries a stable id, its scope, its exact text or
controlled template, a scientific purpose, a provenance justification, and its
permitted typed substitutions. Duplicate ids fail at import.

``PROHIBITED_LITHOLOGY_TERMS`` survives ONLY as a supplementary diagnostic
linter. It authorizes nothing, and it is never cited as evidence that all
named lithologies have been detected - it cannot be: its 28 terms include
neither ``chalk`` nor ``chert`` nor ``qxzite``, and every rejection listed
above happens with that linter returning empty.

THE DEFENSIBLE ASSURANCE STATEMENT, which supersedes the wording of every
earlier Increment 6 manifest: every persisted project-specific classification
and interpretive statement is generated from an approved typed value, a
controlled template, or a registered statement, and arbitrary free text cannot
enter these controlled fields. This is NOT a claim that the software
understands or exhaustively recognizes natural-language lithology; it does
not, and no earlier version did. Free-form notebook narrative and
documentation lie outside these controlled fields and remain subject to
ordinary manual scientific review.

All twelve outputs and figures are byte-identical to Increment 6.1.4.

Increment 6.1.6 (previous corrective patch, v0.6.7) enforces authorization at the EMISSION
BOUNDARY. Increment 6.1.5's positive-authorization model was sound but was
applied to a scope the manifest builder RECONSTRUCTED from dispositions,
confidences and masks - a parallel object, not the records written to disk. A
`GrEndpointScenario` carrying `description="The interval is chalk."` was
persisted verbatim by `build_gr_endpoint_scenario_rows()` while never entering
that 143-field scope, so it could not move `named_lithology_assigned`. The unit
validator was closed; the export path was not.

Five findings, all reproduced first:

* **Exported endpoint description bypass** - closed. The row builder now
  authorizes the value it is about to place in the record and raises
  ``PetrophysicsInputError`` if it cannot.
* **Incomplete output coverage** - closed. ``p2mem.io.output_policy`` declares a
  category for EVERY string column and JSON path of all eight artifacts (105
  entries). Nothing is unclassified; unknown artifacts, columns, JSON paths and
  categories fail closed. Categories: structural enum, identifier, filename,
  typed label, registered statement, controlled template, structured diagnostic,
  sanitized diagnostic. There is no general category admitting arbitrary prose.
* **Field-kind label mismatch** - closed. ``APPROVED_LABELS`` is keyed by
  ``(field_kind, value)``; ``use_status="GR"`` and ``mask_name="measured"`` now
  fail, the field kind is supplied by the caller and never parsed from a context
  string, and duplicate pairs fail at import.
* **Stale exported derivation** - closed. The superseded prohibited-term wording
  is replaced by a registered statement describing the model actually
  implemented, so the assurance prose is itself authorized.
* **Duplicated manifest statement** - closed. The manifest takes
  ``REGISTERED_STATEMENTS["named_lithology_statement"].text``; no second literal
  exists to diverge from it.

Export is two-stage: build the exact pre-serialization records, authorize every
string occurrence, write, then RE-READ the written bytes and authorize again.
A value mutated after authorization is caught before delivery, and nothing is
written at all if any field fails.

The assurance metrics now say what they count. ``n_fields_checked=143`` is gone;
the manifest reports ``scope_object_fields_checked`` alongside an
``emitted_field_coverage`` block giving total emitted string-field occurrences,
controlled occurrences, structural occurrences, occurrences outside the
guarantee, unclassified fields, unauthorized fields and field-kind mismatches.
Machine diagnostics and the operator-facing issue text are counted separately
and are explicitly OUTSIDE the controlled-interpretation guarantee.

All seven CSV artifacts and all four figures are byte-identical to
Increment 6.1.5. ``petrophysics_eligibility_manifest.json`` changes, and only
inside ``lithology_validation``: the derivation is corrected and the coverage
metrics become truthful. No numerical, disposition, mask, threshold or depth
value moves.

Increment 6.1.7 (this release, v0.6.8) corrects the remaining export-boundary
assurance defect. Increment 6.1.6 discovered fields from non-empty string
values, so missing controlled columns, empty or numeric-looking controlled
strings, non-string substitutions and unknown fields with non-prose values
could escape collection. It also wrote into the official directory before the
post-write check and compared aggregate coverage counts, allowing an
authorized value to be changed into a different authorized value without
detection and leaving partial artifacts after a rejected write.

Increment 6.1.7 defines exact schemas for all eight artifacts and validates
artifact inventory, ordered CSV columns, JSON keys, requiredness, strict types
and finite numerics independently of prose authorization. Every schema-declared
string occurrence is then authorized, including empty and numeric-looking
strings. Candidate files are written only to an isolated sibling directory,
re-read and re-validated, and every typed field and row is compared against the
authorized pre-serialization record before failure-atomic publication. Unknown
well identifiers, stale output artifacts, serializer failures, field additions
or deletions, row reordering, type changes and authorized-to-authorized value
changes all fail closed while leaving the official destination unchanged.
This is failure-atomic for handled process errors; it is not a claim of
multi-file atomicity across power loss or operating-system failure.

Subsequent increments (pore pressure, elastic property calculation,
strength, stress, and wellbore-stability screening) are added one
validated phase at a time and are intentionally absent from this version -
importing them will fail until they exist.
"""

__version__ = "0.6.8"

# Fixed project-wide assurance tier. Referenced by later modules (reporting,
# plotting) so that every generated output can stamp its own classification
# without each module re-declaring the string. This value must not be
# changed without a documented calibration event (e.g. a verified RFT/MDT,
# LOT/XLOT, or core-calibrated log tie) recorded in the method-and-citation
# register.
ASSURANCE_TIER = "Tier C - Screening-Level / Uncalibrated Educational"

__all__ = ["__version__", "ASSURANCE_TIER"]


In [ ]:
%%writefile README.md
# Poseidon 2 — 1D Mechanical Earth Model

**Author:** Mikael Elgo

**Project classification:** Tier C — Screening-Level / Uncalibrated Educational 1D Mechanical Earth Model (MEM)

> **This project is screening-level, uncalibrated, and educational in nature. It is NOT validated against independent field measurements (no confirmed RFT/MDT pressure points, LOT/XLOT tests, or core-calibrated log ties are currently incorporated), and it must NOT be used for operational drilling, well-design, or any real-world decision-making. It exists to demonstrate a technically defensible, transparent, modular geomechanics workflow — not to produce field-ready predictions.**

---

## Purpose and technical scope

This repository implements a modular, reproducible 1D Mechanical Earth Model workflow for the Poseidon 2 well, built from well logs, deviation surveys, checkshot data, formation tops, and Vp/Vs data supplied for the project. The intended end-to-end scope (delivered incrementally, one validated phase at a time) covers:

- data quality control and depth alignment across LAS logs, deviation surveys, and checkshot data
- pore-pressure prediction (Eaton-family methods, contingent on a defensible normal compaction trend)
- elastic properties (dynamic Vp/Vs-derived Poisson's ratio, and density-dependent moduli where density coverage permits)
- rock-strength estimation
- vertical-stress (overburden) modelling
- horizontal-stress and wellbore-stability screening (Kirsch elastic wall-stress equations with Mohr–Coulomb/Mogi–Coulomb failure criteria)
- uncertainty treatment via deterministic low/base/high scenarios and one-at-a-time sensitivity (tornado) analysis, rather than unsupported probabilistic distributions

Every empirical or correlation-based relationship used anywhere in this project (Eaton, Bowers, Gardner, Castagna, etc.) is required to have a recorded source, stated units, applicability range, and calibration status in the project's method-and-citation register *before* it is implemented in code. Nothing is fabricated or assumed silently: missing measurements, missing calibration points, and unavailable data are always reported as unavailable rather than filled in.

This is a personal portfolio project intended to demonstrate scientific rigor, reproducibility, and honest handling of data limitations — not a commercial or operational deliverable.

## Current implementation status

**Increment 6 / 6.1 / … / 6.1.7 (this release, v0.6.8): Gamma-Ray QC, Shale-Proxy Sensitivity, Well-Frame Assembly, and Method-Eligibility Framework.** Increment 6.1.7 replaces value-driven output discovery with exact schemas for all eight artifacts. Artifact inventory, ordered CSV columns, JSON keys, requiredness, strict types and finite numeric values are validated independently of prose authorization; every declared string occurrence is then authorized, including empty and numeric-looking strings. Export writes a complete candidate set to an isolated staging directory, re-reads and re-validates it, and compares every typed field and row before failure-atomic publication. Missing/unknown fields, type substitution, authorized-to-authorized mutation, row reordering, stale artifacts, unknown well keys and serializer failures now fail closed without changing the official destination. This corrects assurance/export behavior only; no scientific calculation, threshold, disposition or result changes. See `INCREMENT_06_1_7_MANIFEST.md`.

**Increment 6.1.6 (previous corrective patch, v0.6.7): Gamma-Ray QC, Shale-Proxy Sensitivity, Well-Frame Assembly, and Method-Eligibility Framework.** Increment 6.1.6 enforces authorization at the **emission boundary** — see `INCREMENT_06_1_6_MANIFEST.md`. Increment 6.1.5's model was sound but validated a *reconstructed* scope rather than the records written to disk, so an endpoint `description` carrying an unauthorized sentence was persisted verbatim without ever entering that scope. Every string column and JSON path of all eight artifacts now carries a declared policy classification, labels are authorized by `(field_kind, value)` together, and export is two-stage: authorize the exact records, write, then re-authorize the written bytes. Eleven of twelve outputs are byte-identical to 6.1.5; the manifest changes only inside `lithology_validation`, where the derivation is corrected and the coverage metrics become truthful. Increment 6.1.5 is the preceding corrective patch — see `INCREMENT_06_1_5_MANIFEST.md`. It replaces blacklist-based acceptance with **positive authorization**: every persisted label must be an approved typed value, and every persisted interpretive, method and explanatory statement must resolve to a registered `statement_id` or a controlled `template_id` with strictly typed substitutions. Unregistered free text is rejected unconditionally, so an unknown lithology — or an invented word — cannot enter a controlled field. `PROHIBITED_LITHOLOGY_TERMS` is demoted to a supplementary diagnostic linter that authorizes nothing. All twelve outputs/figures are byte-identical to 6.1.4 and no scientific value changed. Increment 6.1.4 completes the architecture Increment 6.1.3 began — see `INCREMENT_06_1_4_MANIFEST.md`. 6.1.3 closed three of four validation scopes by membership or exact equality but left explanatory scope decided by a finite token list of project references, which omitted ordinary stratigraphic nouns (`member`, `group`, `package`, `play`, `prospect`, `target`) and therefore admitted assertions built on them. That rule is inverted and closed by registry, failing closed; the three token lists are deleted. All four scopes are now closed, all twelve outputs/figures are byte-identical to 6.1.3, and no scientific value changed. Increment 6.1.3 is the preceding architectural corrective patch, applied after an independent audit of Increment 6.1.2 — see `INCREMENT_06_1_3_MANIFEST.md` and the "Increment 6.1.3 update" bullet under Scientific limitations. It stops trying to detect geological assertions in free text at all: every validation scope is now decided by set membership or exact equality, project-specific prose has **zero allowance**, and method wording is admitted only by exact membership of a closed five-entry provenance registry. It also hardens the interval-record type gate and brings three previously unvalidated exported strings into scope. No scientific threshold, tolerance, policy, disposition or result changed; eleven of twelve outputs/figures are byte-identical to Increment 6.1.2 and the twelfth differs by one scalar. Increment 6.1.2 is the preceding corrective patch applied after an independent audit of Increment 6.1.1 — see `INCREMENT_06_1_2_MANIFEST.md` and the "Increment 6.1.2 update" bullet under Scientific limitations. It redesigns the named-lithology validator so the method-phrase allowance is decided per occurrence from local grammar rather than from a sentence-wide cue list and a fixed look-behind window (closing demonstrated false negatives AND demonstrated false positives), and enforces the interval-record interruption-count invariants completely at construction. No scientific threshold, tolerance, policy, disposition or result changed, and all twelve outputs/figures are byte-identical to Increment 6.1.1. Increment 6.1.1 is the preceding corrective patch applied after an independent audit of Increment 6.1 — see `INCREMENT_06_1_1_MANIFEST.md` and the "Increment 6.1.1 update" bullet under Scientific limitations. It closes a named-lithology validator bypass, corrects a stale Vp/Vs description in the active module documentation, replaces ambiguous interruption counts with three separately named quantities, corrects the notebook's output-file count, and supersedes an incorrect Increment 6.1 delta statement. Increment 6.1 is the preceding corrective patch applied after an independent geomechanics/rock-physics and software audit — see `INCREMENT_06_1_MANIFEST.md` and the "Increment 6.1 update" bullet under Scientific limitations. It corrects Vp/Vs domain terminology and boundaries, the Poseidon North 1 depth-tied status, unsupported lithology/correlation claims and a circular named-lithology gate, and gross-versus-net interval-thickness reporting. No architecture changed and no new phase was started. Builds on the LOCKED Increment 1-5.1.2 foundation (no locked module, config, test, fixture, notebook, output, or figure is modified) by adding an auditable per-well frame assembly, factual gamma-ray-family QC, an endpoint-sensitivity framework for a dimensionless screening proxy, and three method-eligibility masks with contiguous-interval registers. See `INCREMENT_06_MANIFEST.md` for the full technical design, the independently recomputed real-data findings, and the complete verification record; see "Increment 6" below for the scientific boundaries it deliberately does not cross. **Increment 7 has not been started.**

**Increment 5 / 5.1 / 5.1.1 / 5.1.2 (previous release, v0.5.2): Formation-Top Ingestion, Source Reconciliation, and Survey-Corrected Stratigraphic Depth Framework.** Builds on the LOCKED Increment 4.1.2 checkshot/time-depth layer, the LOCKED Increment 3.1.1 deviation-survey/depth-mapping layer, and the LOCKED Increment 2.1.1 LAS-ingestion layer (all unmodified) by adding contract-driven formation-top file ingestion for the two approved wells with formation-top data (Poseidon 2, Boreas 1), explicit reconciliation between each well's two independently supplied top-file representations, and mapping of reconciled marker depths through the locked survey trajectory to produce a corrected, auditable stratigraphic marker table. See `INCREMENT_05_MANIFEST.md` for the full technical design, the independently recomputed real-data findings, and the complete verification record; `INCREMENT_05_1_MANIFEST.md` for the Increment 5.1 corrective patch (a zero-common-marker reconciliation defect, readable-file absolute-path leakage, incomplete in-memory numerical validation, and a manifest-language correction — see "Increment 5.1 update" below); and `INCREMENT_05_1_1_MANIFEST.md` for the Increment 5.1.1 corrective patch (a remaining one-empty-source zero-common-marker edge case, plus three documentation-accuracy corrections — see "Increment 5.1.1 update" below). Formation-top ingestion/reconciliation/depth-correction only — no gamma-ray normalization, shale-volume calculation, named lithology classification, petrophysical interpretation, method-eligibility masks, shallow-density modelling, overburden-stress integration, NCT fitting, pore-pressure prediction, elastic properties, rock strength, horizontal stresses, or wellbore-stability analysis is performed in this increment.

New in Increment 6:
- `p2mem/wellframe_models.py` / `p2mem/wellframe.py` — a typed, auditable per-well assembly of the LOCKED canonical LAS curve arrays alongside the LOCKED MD→TVD/TVDSS mapping, carrying per-sample validity masks, per-curve provenance (source curve name, raw mnemonic, raw and canonical unit, conversion function, source filename), QC flags, and an evidence classification. Sample count and original file order are preserved exactly; an invalid sample is expressed only through a mask, never deleted, filled, interpolated, or reordered; every curve array is exposed read-only so a downstream consumer cannot mutate a locked loader's data through a frame. **No depth is ever extrapolated:** the locked `map_las_md_to_tvd_tvdss` is deliberately all-or-nothing, so this layer computes the in-coverage mask from the locked trajectory's own MD range and calls the locked mapper on the in-coverage subset only, leaving out-of-coverage samples as NaN with `depth_valid_mask == False` (`n_extrapolated` is 0 by construction, and the honest coverage gap is reported as `n_depth_unmapped` instead).
- `config/petrophysics_eligibility.yml` — the human-authored, human-reviewable per-well gamma-ray-family disposition, endpoint-sensitivity policy, physical-plausibility bounds, contiguity tolerances, and eligibility rules. The four wells carry four DISTINCT GR-family curves (`GR_api`, `GRD_api`, `ECGR_api`, and a second, independent `GR_api`) that are never merged, renamed, rescaled, or treated as geologically equivalent — Poseidon 2's and Proteus 1ST2's curves canonicalize to the same name but remain different tools in different wells with no cross-well calibration tie, so endpoints are always estimated per well from that well's own samples (a single universal cross-well endpoint pair is rejected at config-load time).
- `p2mem/petrophysics_models.py` / `p2mem/petrophysics.py` — factual GR-family QC statistics for EVERY well including excluded ones (the numbers justifying an exclusion must themselves be published), per-well low/base/high endpoint scenarios whose MEASURED endpoint values are recorded, and the dimensionless GR index `IGR = (GR − GR_low)/(GR_high − GR_low)` with clipped and unclipped results retained side by side so the amount of clipping — i.e. how far real data fell outside the assumed bracket — stays visible. Every endpoint carries `evidence_class = "assumed_configured"` and an explicit uncalibrated status; no code path can promote one to calibrated. The only permitted shale-proxy transform is the linear identity of the clipped index, exported under the self-labelling name `VSH_GR_linear_proxy_frac`.
- `p2mem/method_eligibility.py` — three input-admissibility masks (`eligible_density_for_sv`, `eligible_dynamic_elastic`, `eligible_sonic_nct_candidate`) with per-criterion pass counts and a named limiting criterion, plus contiguous-interval registers reported on MD, TVD and TVDSS. Gap bridging requires BOTH a small sample gap AND a small physical depth span, so sample-count continuity is never confused with physical-depth continuity; bridged samples are always disclosed separately from genuinely eligible ones.
- `p2mem/io/petrophysics_inventory.py` — deterministic summary/scenario/interval/manifest builders that never export a per-sample real-data array (which would effectively reproduce the private source logs) and never emit an absolute path, sanitizing every failure message against BOTH candidate source paths.
- **Key real-data findings (independently measured, not asserted):** Boreas 1's ECGR carries an unresolved scale/acquisition anomaly — median **8.40 API** against 36.06 / 36.37 / 41.41 API for the other three wells (4.3×–4.9× lower), a range of **[−0.0001, 519.18] API** including 4 negative and 42 exactly-zero samples, and **2,059 samples above the well's own declared seabed marker**. With no independent tool header, calibration record, or environmental-correction metadata available to adjudicate the cause, Boreas 1 is formally EXCLUDED (`BOREAS_ECGR_SCALE_UNRESOLVED`) from every lithology-dependent, GR-normalized, shale-proxy and NCT-candidate calculation — **excluded, never corrected**, since any shift, gain, or normalization would fabricate a calibration that does not exist and would silently propagate into every downstream result. Zero endpoint scenarios, zero proxies and zero lithology-dependent masks are computed for it; it remains available for factual raw-GR QC display and availability reporting only. Endpoint choice alone moves the screening-proxy median by up to 0.12 (dimensionless), and endpoint × threshold choice moves the qualifying sonic-NCT-CANDIDATE thickness by a factor of **8.0** in Poseidon 2 (144 m to 1,159 m), 3.5× in Poseidon North 1 and 2.3× in Proteus 1ST2 — quantifying exactly how much a later NCT result would depend on choices no data in this project constrains.
- **No named lithology is assigned anywhere in Increment 6, and the available data do not support assigning one.** Low-GR intervals occur independently in each of the three GR-eligible wells; cross-well stratigraphic persistence is NOT established, and cannot be, because Poseidon North 1 and Proteus 1ST2 have no approved formation tops to correlate within. Gamma-ray response is not uniquely diagnostic of rock type, and no core, cuttings description, image log, spectral GR, or calibrated multi-mineral solution exists in this project to adjudicate it. Every classification this increment produces (`GR_PROXY_HIGH` / `GR_PROXY_INTERMEDIATE` / `GR_PROXY_LOW` / `GR_NOT_AVAILABLE` / `GR_EXCLUDED_UNRESOLVED_SCALE`) describes DATA AND PROXY CONFIDENCE ONLY, and is machine-checked against an enforced prohibited-rock-name vocabulary.
- Still not implemented: named lithology interpretation, environmental GR correction, Boreas ECGR rescaling, neutron-density crossplot interpretation, any nonlinear Vsh transform (Larionov, Clavier, Stieber — all DEFERRED pending a retrieved, verified primary-source method record), shallow-density reconstruction, density extrapolation, vertical-stress integration, hydrostatic-pressure modelling, NCT fitting, Eaton/Bowers pore-pressure prediction, dynamic or static elastic-property calculation, rock-strength or friction-angle correlation, Shmin/SHmax modelling, stress-polygon construction, wellbore-stability analysis, and mud-weight recommendation.


New in Increment 5:
- `p2mem/top_models.py` — typed, frozen dataclasses for every formation-top header/contract/raw-row/reconciliation/corrected-marker result object, mirroring the checkshot/deviation layers' design philosophy. Two source representations exist per well — an "HRS" file (`Top_Name`/`MDRT_m` only, no well name in the file body) and a "selected readable" file (`TOP_NAME`/`MDRT_M`/`TVDSS_M`/`NOTE`, `#`-comment headers, a dashed separator line) — and every raw (`_source_`) value from both is preserved separately; nothing is ever silently overwritten, sorted, deduplicated, or repaired.
- `p2mem/io/tops.py` — an auditable formation-top parser, per-file contract resolver, and HRS-versus-readable source reconciler for the four approved files, keyed by their exact literal source filenames. Markers are matched between the two files by exact name, whitespace-normalized name, or an explicit human-authored alias contract (`config/formation_top_contracts.yml`'s `marker_name_aliases` — currently empty; no fuzzy/similarity matching of any kind is ever performed). MDRT is cross-checked between the two sources within a documented 0.005 m tolerance; a marker on which the two sources disagree beyond that tolerance is registered (`mdrt_status = "MISMATCH"`) and excluded from depth mapping (`mdrt_authority_basis = "disagreement_unresolved"`, `mapping_status = "not_mapped_mdrt_unresolved"`) rather than resolved by silently picking one file's value. Reconciled MDRT is mapped through the LOCKED Increment 3.1.1 `petrel_source_trace` survey trajectory using the existing, unmodified `p2mem.depth_mapping.map_las_md_to_tvd_tvdss` — reused unmodified, never reimplemented — called once per marker so that one out-of-coverage marker never blocks mapping of the rest of the well, and extrapolation is never performed (a marker outside survey MD coverage is reported as `mapping_status = "rejected_outside_coverage"`, never extrapolated).
- `p2mem/io/tops_inventory.py` — deterministic, metadata-only inventory/QC-table builders for the Increment 5 outputs (file inventory, marker/source register, HRS-versus-readable reconciliation table, survey-corrected marker table, ingestion issues, formation-top availability, JSON manifest) — never raw per-marker arrays beyond a single scalar per field, and never a full environment-dependent build path, only a basename.
- `config/formation_top_contracts.yml` — the human-authored, human-reviewable per-file formation-top contract for each of the four approved files, including each file's `well_identity_evidence_status` (both representations for both wells are `inferred_unverified` — the HRS files carry no well name in their body at all, association resting on the filename alone, and the readable files' well name appears only in a project-supplied `#` comment, not independently verified content; this is a deliberately MORE conservative classification than Increment 4's checkshot files, explicitly justified in `INCREMENT_05_MANIFEST.md`).
- **Key real-data findings (disclosed):** every one of Poseidon 2's 9 "selected readable" TVDSS values equals `MDRT_M - 21.8 m` EXACTLY (21.8 m is the well's own rotary-table elevation) — a literal vertical-well-assumption depth-reference defect, since a real TVDSS should differ from MD by more than a constant datum shift once a well deviates. Survey-corrected residuals (`TVDSS_source_m - TVDSS_survey_corrected_m`) reproduce Sea Bed at ≈0 m, Plover Fm (Top Reservoir) at ≈+1.68 m, and TD at ≈+2.49 m — confirming the anticipated defect and correcting it in the derived `TVDSS_survey_corrected_m` representation while `TVDSS_source_m` itself is preserved unmodified. Boreas 1's supplied TVDSS does NOT follow the `MDRT - 21.8` pattern and its maximum absolute source-versus-survey residual across all 10 markers is 0.0416 m, below the approved 0.05 m tolerance — confirming it was generated from the well's real surveyed trajectory, and it is therefore NOT "corrected" the way Poseidon 2 is; evidence is applied per file, per well, never by analogy. Poseidon North 1 and Proteus 1ST2 have no approved formation-top file and are recorded as `formation_top_availability: NOT_AVAILABLE`, a factual data gap, never substituted with another well's tops or correlated by depth alone.
- Still not implemented: gamma-ray normalization, shale-volume calculation, named lithology classification, petrophysical interpretation, method-eligibility masks, shallow-density modelling, overburden-stress integration, NCT fitting, pore-pressure prediction, elastic-property calculation, rock-strength estimation, horizontal stresses, or wellbore-stability calculations. Those remain explicitly out of scope for this increment.

New in the Increment 5.1 corrective patch (see "Increment 5.1 update" under Scientific limitations for the full defect list): `reconcile_formation_top_sources` now correctly rejects two disjoint, non-empty marker sets as a fatal `NO_COMMON_MARKERS` ERROR (previously silently accepted); `TopIngestionFailure` now carries an explicit `failure_origin` and both `hrs_path`/`readable_path`, so a failure originating from the "selected readable" file can no longer leak that file's own absolute path into any exported CSV/JSON field; and `reconcile_formation_top_sources` now fully validates its numeric-array and `mdrt_agreement_tolerance_m` inputs before any comparison, with deliberate typed exceptions rather than an incidental `IndexError`/`TypeError`. None of the real Poseidon 2 / Boreas 1 formation-top results above changed.

New in the Increment 5.1.1 corrective patch (see "Increment 5.1.1 update" under Scientific limitations for the full defect list): `reconcile_formation_top_sources` now rejects the one remaining zero-common-markers configuration Increment 5.1 missed — exactly one source entirely empty, the other non-empty — via a single, strictly correct intersection check (`if not common_markers`) that covers every zero-common-markers case at once, while a legitimate one-sided marker alongside at least one genuinely shared marker remains the pre-existing, non-fatal `NOT_COMPARABLE` case. `INCREMENT_05_1_MANIFEST.md`'s baseline-hash, packaged-outputs, and packaged-LAS-fixture wording is also corrected (documentation-only; see "Increment 5.1.1 update"). None of the real Poseidon 2 / Boreas 1 formation-top results above changed.

**Increment 4 / 4.1 / 4.1.1 / 4.1.2 (v0.4.1.1, LOCKED as of Increment 5): Checkshot (Velocity Survey) Ingestion, Duplicate-Tie Conditioning, and Time–Depth Framework.** Builds on the LOCKED Increment 3.1.1 deviation-survey/depth-mapping layer and the LOCKED Increment 2.1.1 LAS-ingestion layer (both unmodified - see below) by adding auditable checkshot parsing with explicit per-file contracts, raw-vs-conditioned duplicate-tie handling, average/interval velocity diagnostics, checkshot-vs-locked-survey depth-reference comparison, a coverage-masked forward/inverse piecewise-linear time-depth interpolation layer, LAS MD-to-checkshot-time mapping within validated coverage only, and a Poseidon-2-only sonic-checkshot drift diagnostic. See `INCREMENT_04_MANIFEST.md` for the full technical design and the independently recomputed real-data statistics, `INCREMENT_04_1_MANIFEST.md` for the Increment 4.1 corrective patch (order-invariant axis-tie conditioning for TVDSS↔OWT/TWT inversion, replacing an order-dependent defect; numerical-validation hardening — see "Increment 4.1 update" below), `INCREMENT_04_1_1_MANIFEST.md` for the Increment 4.1.1 numerical-validation corrective patch (a hidden-reversal grouping defect, incomplete full-MD validation, an untyped zero-coverage crash, missing batch isolation for numerical failures, and a unit-helper input-safety gap — see "Increment 4.1.1 update" below), and `INCREMENT_04_1_2_MANIFEST.md` for the Increment 4.1.2 packaging-only corrective patch (Colab line-ending reproducibility and a truthful notebook completion gate; no scientific, numerical, or version change). Checkshot data QC, duplicate-tie conditioning, and time-depth interpolation only — no formation-top correction, lithology interpretation, density modelling, pore-pressure prediction, elastic properties, rock strength, stress modelling, or wellbore-stability analysis is performed in this increment.

New in Increment 4:
- `p2mem/checkshot_models.py` — typed, frozen dataclasses for every checkshot header/contract/raw-row/duplicate-tie/velocity-diagnostic/depth-comparison/time-mapping/sonic-drift result object, mirroring the deviation-survey layer's design philosophy. Raw (`_source_`) values are always kept explicitly separate from conditioned (`_conditioned_`) values — never overwritten, never mixed.
- `p2mem/io/checkshot.py` — an auditable checkshot (velocity-survey) parser and per-file contract resolver for the three approved checkshot files (`Poseidon2-Checkshot.txt`, `Boreas1-Checkshot.txt`, `Proteus1-Checkshot.txt`), keyed by their exact literal source filenames. File-identity checks (filename, SHA-256, header/survey-statement text, column order, row width, numeric structure) are enforced as blocking `ERROR`s before any time-depth computation is attempted. Raises disclosure `WARNING`s including `DEPTH_BASIS_NOT_EXPLICITLY_DECLARED` (the source file's first column is labelled only `Depth`, never assumed to be MD without evidence) and, for Proteus 1ST2, `WELL_IDENTITY_INFERRED_UNVERIFIED` (the file carries no embedded well identifier tying it to Proteus 1ST2).
- `p2mem/time_depth.py` — the numerical time-depth layer: duplicate/repeated-tie detection and deterministic, disclosed median-based conditioning (raw rows always preserved and separately registered; ties never silently averaged, never force-monotonized with artificial epsilon increments); average velocity (`Vavg = TVDSS / OWT`) and interval velocity (`Vint = ΔTVDSS / ΔOWT`, NaN — never infinite or negative — for any invalid, zero, or non-increasing interval); checkshot-vs-locked-survey depth-reference comparison (residual = survey-interpolated TVDSS − checkshot-supplied TVDSS); forward/inverse piecewise-linear time-depth interpolation with explicit coverage masking (points outside validated checkshot coverage are reported as not-mapped, never extrapolated) via an order-invariant, median-based axis-tie-conditioned lookup table for TVDSS↔OWT/TWT inversion (Increment 4.1 — every tied value on the axis being inverted is grouped and registered, never resolved by an order-dependent "first wins" tie-break); LAS MD-to-checkshot-time mapping for Poseidon 2 within validated coverage only; and a Poseidon-2-only sonic-checkshot drift diagnostic (a local, version-independent trapezoidal integration of sonic slowness over the longest valid continuous MD interval, compared against the checkshot-interpolated OWT increment over the same interval — this diagnostic never modifies `VP_m_s`, `DTCO`, checkshot OWT, or the time-depth curve itself).
- `p2mem/io/checkshot_inventory.py` — deterministic, metadata-only inventory/QC-table builders for the Increment 4 / 4.1 outputs (file inventory, ingestion issues, Depth-axis duplicate-tie register, the Increment 4.1 TVDSS/OWT-axis tie register, depth-tie QC, velocity summary, sonic-checkshot drift summary, time-depth mapping summary, JSON manifest) — never raw per-sample checkshot or LAS arrays, and never a full environment-dependent build path, only a basename.
- `config/checkshot_contracts.yml` — the human-authored, human-reviewable per-file checkshot contract for each of the three approved files, including each well's `model_use_status` (`primary_model` for Poseidon 2 only; `qc_only` for Boreas 1 and Proteus 1ST2) and `identity_evidence_status` (`verified` for Poseidon 2 and Boreas 1; `inferred_unverified` for Proteus 1ST2).
- **Key real-data findings (disclosed):** Poseidon 2's raw checkshot file contains 5 repeated Depth ties, 6 non-increasing TVDSS steps, and 4 non-increasing OWT steps, all independently detected and conditioned (never silently smoothed); its checkshot-vs-survey TVDSS comparison shows a maximum absolute residual of ≈0.089 m with no systematic offset pattern. Boreas 1 shows 3 repeated Depth ties and 4 non-increasing TVDSS steps (OWT strictly increasing throughout) and a checkshot-vs-survey comparison with a near-constant offset of ≈−0.69 m, reported as an observed datum-like offset pattern — not a proven datum error. Proteus 1ST2's file is strictly increasing in all three columns with no repeated ties, and shows a near-constant offset of ≈+0.30 m against the locked survey trajectory; its association with Proteus 1ST2 remains `inferred_unverified` throughout every output. Poseidon 2's sonic-checkshot drift over its longest valid continuous sonic interval (≈MD 2449–4064 m) is ≈+16.7 ms (≈+4.4% of the checkshot-interpolated one-way transit time (OWT) increment over that interval — NOT two-way time; the diagnostic compares the integrated sonic transit time directly against the checkshot's OWT increment over the identical interval), reported as a diagnostic only. Poseidon North 1 has no approved checkshot file; this is recorded as `checkshot_availability: NOT_AVAILABLE`, a factual data gap, not an ingestion failure. Depth-tie conditioning alone does NOT guarantee TVDSS or OWT is itself strictly increasing (required for TVDSS↔OWT/TWT inversion) — see the Increment 4.1 update below for the corrected, order-invariant handling of this. See `INCREMENT_04_MANIFEST.md` for the full statistics, tables, and figures.
- Still not implemented: formation-top correction, lithology interpretation, density modelling, sonic/checkshot drift *correction*, synthetic extension of the time-depth relationship beyond measured checkshot coverage, pore-pressure prediction, elastic-property calculation, rock-strength estimation, overburden/horizontal stresses, or wellbore-stability calculations. Those remain explicitly out of scope for this increment.

**Increment 3 / 3.1 / 3.1.1: Deviation-Survey Ingestion, Minimum-Curvature Validation, and MD–TVD–TVDSS Depth Framework.** Builds on the LOCKED Increment 2.1.1 LAS-ingestion layer (unmodified - see below) by adding Petrel deviation-survey parsing, an explicit per-file survey contract, a standard minimum-curvature trajectory engine, an explicit depth-reference (MD/TVD/TVDSS) framework, and MD-to-TVD/TVDSS mapping of the existing LAS `MD_m` arrays. See `INCREMENT_03_MANIFEST.md` for the Increment 3 technical design and real four-well integration results, `INCREMENT_03_1_MANIFEST.md` for the Increment 3.1 corrective patch (four audit findings: source-filenames-with-spaces, absolute-path leakage, DLS-normalization disclosure, dogleg numerical stability — see "Increment 3.1 update" below), and `INCREMENT_03_1_1_MANIFEST.md` for the Increment 3.1.1 packaging-only corrective patch (notebook/source `%%writefile` synchronization; no scientific or numerical change). The Proteus 1ST2 trajectory-discrepancy finding is disclosed, not resolved, in any of these releases. This layer, and the Increment 2.1.1 LAS-ingestion layer beneath it, are LOCKED as of Increment 4 and reused unmodified.

Locked from Increment 2.1.1 (unmodified in Increment 3 unless a blocking defect is documented - none was found):
- `p2mem/units.py` — an explicit, NumPy-based unit-conversion layer (no external unit-registry dependency such as Pint) implementing 21 public conversion functions between oilfield and SI-internal units. Unchanged since Increment 1.1. See the module docstring and `tests/test_units.py`.
- `p2mem/models.py`, `p2mem/io/las.py`, `p2mem/io/inventory.py`, `config/las_curve_contracts.yml` — the auditable LAS 2.0 parser, per-file curve-contract resolver, and inventory builders for the four approved wells, corrected and independently re-verified through Increment 2.1.1 (158 tests passing, 4/4 real wells loading with zero ingestion errors). See `INCREMENT_02_v2.1.1_MANIFEST.md`.

New in Increment 3:
- `p2mem/deviation_models.py` — typed, frozen dataclasses for every deviation-survey/trajectory/depth-mapping result object (header info, per-file contract, raw station data, minimum-curvature result, trajectory-validation residuals, depth-basis selection, typed batch failures, LAS depth-mapping result), mirroring the LAS layer's design philosophy. Every source (`_source_`) array is kept explicitly separate from every independently computed (`_mc_`) array — never overwritten, never mixed.
- `p2mem/io/deviation.py` — an auditable Petrel deviation-survey (well-trace) parser and per-file contract resolver for the same four wells, keyed by their exact, literal source filenames (which contain spaces, e.g. `"Poseidon 2_dev.txt"` — corrected in Increment 3.1; see below). Extracts and preserves the full header block (well/survey identity, wellhead X/Y, datum and its MSL reference, coordinate-reference-system text, declared angle/depth/coordinate conventions) and the exact 11-column station table, with file-identity checks (filename, SHA-256, well/survey identifier, wellhead/datum values, coordinate system, column order, station count, MD coverage) all enforced as blocking `ERROR`s before any trajectory computation is attempted. Also raises two disclosure `WARNING`s for every successfully loaded file: the pre-existing `MD_UNIT_NOT_EXPLICITLY_DECLARED`, and the Increment 3.1 `DLS_NORMALIZATION_INFERRED_AS_DEG_PER_30M` (the supplied `DLS` column's degrees-per-30-m normalization is inferred, not header-declared, and is independently verified per file against a recomputation from that file's own inclination/azimuth).
- `p2mem/trajectory.py` — the standard minimum-curvature method (a numerically stable `arctan2(||cross||, dot)` dogleg-angle formulation — Increment 3.1 correction, see below — a ratio factor with an explicit Taylor-series limit as the dogleg approaches zero, TVD/northing/easting displacement, dogleg severity in degrees per 30 m), implemented explicitly and transparently with no third-party survey-computation library.
- `p2mem/depth_mapping.py` — MD-to-TVD/TVDSS interpolation of the locked LAS `MD_m` array against the explicitly selected depth-trajectory basis, using a documented, deterministic piecewise-linear station interpolation (never a per-sample minimum-curvature recomputation) that never extrapolates silently.
- `p2mem/io/deviation_inventory.py` — deterministic, metadata-only inventory-table builders for the Increment 3 outputs (file inventory, trajectory-validation summary, depth-reference register, LAS depth-mapping summary, ingestion issues, JSON manifest) — never raw per-sample station or LAS arrays, and (Increment 3.1 correction) never a full environment-dependent build path, only a basename.
- `config/deviation_survey_contracts.yml` — the human-authored, human-reviewable per-file deviation-survey contract for each of the four wells, keyed by the exact literal source filename (`"Poseidon 2_dev.txt"`, `"Boreas 1_dev.txt"`, `"Poseidon North 1_dev.txt"`, `"Proteus 1ST2_dev.txt"` — corrected in Increment 3.1), including an explicit, uniformly applied `depth_basis_policy` (`petrel_source_trace`, the conservative default given the Proteus 1ST2 finding below) and residual-comparison tolerances declared once and applied identically to every well (never tuned per well to force a pass/fail outcome).
- **Key real-data finding (disclosed, not resolved):** independent minimum-curvature reconstruction of Poseidon 2, Boreas 1, and Poseidon North 1 agrees with their Petrel-supplied TVD to approximately millimetre scale. Proteus 1ST2 shows a materially larger discrepancy (~0.19 m TVD, ~1.6 m easting at maximum) concentrated in its deeper section (below ~MD 4200 m), even though its own supplied dogleg-severity column is internally consistent with an independent recomputation from its own inclination/azimuth at every station. This is reported as a visible trajectory-validation `WARNING`, not corrected, hidden, or used to justify loosening every well's tolerance — see `INCREMENT_03_MANIFEST.md` Section 6 for the full investigation and the evidence pattern observed. Unaffected by the Increment 3.1 patch.
- Still not implemented (as of Increment 3/3.1/3.1.1): checkshot ingestion, time-depth conversion, formation-top correction, petrophysical interpretation, gamma-ray normalization, shale-volume calculation, lithology classification, normal-compaction-trend fitting, pore-pressure prediction, elastic-property calculation, rock-strength estimation, overburden/horizontal stresses, or wellbore-stability calculations. Those were explicitly out of scope for this increment. Checkshot ingestion and the time-depth framework were subsequently added in Increment 4 (see above); the remainder are added one gated increment at a time in later releases.

## Installation

Requires Python 3.9 or later.

```bash
# from the project root (the directory containing pyproject.toml)
pip install -e .
```

This installs the `p2mem` package in editable mode along with its runtime dependencies: NumPy (`numpy>=1.24`) and, as of Increment 2, PyYAML (`pyyaml>=6.0`) — used for parsing the human-authored curve contracts in `config/las_curve_contracts.yml`, `config/deviation_survey_contracts.yml` (Increment 3), `config/checkshot_contracts.yml` (Increment 4), and (new in Increment 5) `config/formation_top_contracts.yml`. No new runtime dependency was added in Increment 3, 4, or 5: the minimum-curvature engine, depth-mapping interpolation, duplicate-tie conditioning, velocity diagnostics, time-depth interpolation, and formation-top reconciliation/mapping all use only NumPy (including a local, version-independent trapezoidal-integration helper in `p2mem/time_depth.py`, added because `numpy.trapz`/`numpy.trapezoid` are not consistently available across supported NumPy versions). Matplotlib and pandas are used only for notebook display and QC-figure generation (`run_integration_03.py`/`run_integration_04.py`/`run_integration_05.py`, the Increment 3/4/5 notebooks) — never imported by the installable `p2mem` package itself. To also install the test dependency:

```bash
pip install -e ".[dev]"
```

## Running the tests

```bash
pytest -v
```

The suite in `tests/test_units.py` validates `p2mem/units.py` (unchanged since Increment 1.1) against analytical reference values, round-trip consistency, scalar/array inputs, NaN preservation, and rejection of invalid/nonphysical/ambiguous inputs. The suite in `tests/test_las.py` validates `p2mem/io/las.py` (locked since Increment 2.1.1) against small synthetic LAS fixtures. The suites in `tests/test_trajectory.py`, `tests/test_deviation.py`, and `tests/test_depth_mapping.py` (new in Increment 3; extended in Increment 3.1 with dogleg numerical-stability, DLS-normalization-disclosure, and real-filename-with-spaces regression/negative tests) validate the minimum-curvature engine, the Petrel deviation-survey parser/contract resolver, and the MD-to-TVD/TVDSS mapping respectively, against small synthetic fixtures under `tests/fixtures/` and in-memory synthetic data (this layer is locked, unmodified, as of Increment 4). `tests/test_deviation_inventory.py` (new in Increment 3.1) validates that no exported inventory/issues/manifest row, for a successful or a failed well, ever embeds a full environment-dependent build path. The suites in `tests/test_checkshot.py`, `tests/test_time_depth.py`, and `tests/test_checkshot_inventory.py` (new in Increment 4) validate the checkshot parser/contract resolver, the duplicate-tie conditioning/velocity-diagnostic/time-depth-interpolation/sonic-drift numerical layer, and the deterministic inventory/QC-table builders respectively, against small synthetic fixtures under `tests/fixtures/` (including a CRLF fixture used to verify line-ending detection) and in-memory synthetic/analytic data — including an analytic constant-velocity case used to independently verify the trapezoidal-integration helper. The suites in `tests/test_tops.py` and `tests/test_tops_inventory.py` (new in Increment 5) validate the formation-top parser/contract resolver/HRS-versus-readable reconciliation logic and the deterministic inventory/QC-table builders respectively, against small synthetic fixtures under `tests/fixtures/` and in-memory synthetic data, including synthetic analogs of both real regression findings (a vertical-assumption depth-reference defect with a growing residual, and a survey-consistent well with a near-zero residual). None of these suites require the private/raw project LAS, deviation, checkshot, or formation-top files, so the full suite runs the same way for anyone who clones this repository. Run the command above and read the reported pass/fail count directly — this document does not assert a fixed expected count, since that must always be read from the actual `pytest` output for the code currently on disk. Real integration validation (which DOES require the raw LAS/deviation/checkshot/formation-top files, not included in this repository) is a separate notebook run — see `02_LAS_Ingestion_and_Curve_Contracts.ipynb`, `03_Deviation_Survey_and_Depth_Framework.ipynb`, `04_Checkshot_QC_and_Time_Depth_Framework.ipynb`, and `05_Formation_Tops_and_Stratigraphic_Depth_Framework.ipynb`.

## Directory structure

```
Poseidon_1D_MEM/
├── README.md
├── pyproject.toml
├── p2mem/
│   ├── __init__.py
│   ├── models.py
│   ├── units.py
│   ├── deviation_models.py
│   ├── trajectory.py
│   ├── depth_mapping.py
│   ├── checkshot_models.py
│   ├── time_depth.py
│   ├── top_models.py
│   └── io/
│       ├── __init__.py
│       ├── las.py
│       ├── inventory.py
│       ├── deviation.py
│       ├── deviation_inventory.py
│       ├── checkshot.py
│       ├── checkshot_inventory.py
│       ├── tops.py
│       └── tops_inventory.py
├── tests/
│   ├── test_units.py
│   ├── test_las.py
│   ├── test_trajectory.py
│   ├── test_deviation.py
│   ├── test_depth_mapping.py
│   ├── test_deviation_inventory.py
│   ├── test_checkshot.py
│   ├── test_time_depth.py
│   ├── test_checkshot_inventory.py
│   ├── test_tops.py
│   ├── test_tops_inventory.py
│   └── fixtures/         (small synthetic LAS + deviation-survey + checkshot + formation-top files; no project raw data)
├── config/
│   ├── las_curve_contracts.yml
│   ├── deviation_survey_contracts.yml
│   ├── checkshot_contracts.yml
│   └── formation_top_contracts.yml
├── data/
│   └── raw/
│       ├── logs/         (the four raw LAS files - NOT included in this repository; immutable inputs)
│       ├── deviation/    (the four raw deviation-survey files, exact filenames contain spaces, e.g. "Poseidon 2_dev.txt" - NOT included in this repository; immutable inputs)
│       ├── checkshot/    (the three raw checkshot files, exact filenames e.g. "Poseidon2-Checkshot.txt" - NOT included in this repository; immutable inputs, never rewritten/renamed/"cleaned")
│       └── tops/         (the four raw formation-top files, e.g. "Poseidon_2_HRS_tops_no_wellname_MDRT.txt" - NOT included in this repository; immutable inputs)
├── notebooks/  (reserved for later increments)
└── outputs/
    ├── 02_las_inventory/            (Increment 2.1.1 real four-well run: CSV/JSON metadata only, no raw log samples)
    ├── 03_deviation_depth/          (Increment 3 real four-well run: CSV/JSON metadata + QC figures, no raw station/log samples)
    ├── 04_checkshot_time_depth/     (Increment 4 real three-file checkshot run: CSV/JSON metadata + QC figures, no raw per-sample checkshot/LAS arrays)
    └── 05_formation_tops/           (Increment 5 real four-file formation-top run: CSV/JSON metadata + QC figures, no raw per-marker arrays beyond scalar fields)
```

`notebooks/` is created empty by the project-setup notebook cell and is not yet populated in-repo (the increment notebooks themselves are delivered as top-level files, e.g. `02_LAS_Ingestion_and_Curve_Contracts.ipynb`, `03_Deviation_Survey_and_Depth_Framework.ipynb`, `04_Checkshot_QC_and_Time_Depth_Framework.ipynb`, `05_Formation_Tops_and_Stratigraphic_Depth_Framework.ipynb`, `06_GR_QC_Shale_Proxy_and_Method_Eligibility.ipynb`, and are meant to be run from Google Drive per their own directory-setup cells).

## Scientific limitations

These limitations are specific to the Poseidon 2 dataset and this project's current increment, and are carried forward here so they are visible outside the conversation in which they were identified:

- **RHOB (bulk density) coverage in Poseidon 2 ends at approximately 5,296.85 m MD.** Sonic and other curves continue deeper, so Vp/Vs and dynamic Poisson's ratio remain computable below that depth, but density-dependent properties (Young's modulus, shear modulus, bulk modulus, acoustic impedance, shear impedance) are unavailable below it unless density is explicitly estimated and flagged as such — never silently substituted.
- **No reliable shale-based normal compaction trend (NCT) exists from Poseidon 2 alone.** A provisional, transferred candidate NCT identified in offset well Poseidon North 1 is a *candidate*, not a validated trend, and must not be presented as calibrated.
- **Independently measured Vp/Vs quality flags:** approximately 3.38% of Poseidon 2 Vp/Vs values fall below 1.5, and approximately 0.53% fall below the physical validity cutoff of √2 (≈1.4142) required for a non-negative dynamic Poisson's ratio.
- **No independent calibration data (RFT/MDT pressure points, LOT/XLOT tests, or core data) has been supplied or incorporated.** Any pore-pressure or stress output in later increments must be presented as a bounded or theoretical estimate, not a validated field prediction.
- **Empirical/correlation equations are not implemented until their governing equation, units, applicability range, and calibration status are recorded in the project's method-and-citation register.** Several candidate methods remain in "pending" status and are intentionally absent from the codebase for that reason, not because they were overlooked.
- Additional open items (offset-well GR/ECGR scale adjudication, missing formation tops for one offset well) are tracked in the project's design-review documentation and gate specific later phases (lithology and pore-pressure), not this increment.
- **Increment 2 update:** LAS ingestion independently reconfirms (does not newly discover, and does not act on) two previously-flagged anomalies from the Rev 1 design review: Boreas 1's ECGR curve (canonical name `ECGR_api`) ranges from approximately −0.0001 to 519.18 API (vs. roughly 5–205 API for the other three wells' GR-family curves) with 96.98% valid coverage; and Proteus 1ST2's LAS log file places its neutron-porosity curve (canonical name `NPHI_pct`) at column position 5 rather than the last position (8) used by the other three wells. Both are reported as ingestion facts (see `outputs/02_las_inventory/`); neither is rescaled, reinterpreted, or otherwise acted on by this increment.
- **Increment 2.1 / 2.1.1 update:** corrective patches addressing independent audits' naming, reporting, contract-validation, and notebook/source-synchronization findings — see `INCREMENT_02_v2.1_MANIFEST.md` and `INCREMENT_02_v2.1.1_MANIFEST.md`. No new scientific finding was made in either patch; the two anomalies above are unaffected and remain open items for a later, explicitly-scoped increment.
- **Increment 3 update:** deviation-survey ingestion independently reconfirms the Petrel-supplied trajectory for Poseidon 2, Boreas 1, and Poseidon North 1 to approximately millimetre scale via minimum curvature, and additionally DISCOVERS (not merely reconfirms) a real trajectory-reconstruction discrepancy in Proteus 1ST2's deeper section (~0.19 m TVD, ~1.6 m easting at maximum, concentrated below ~MD 4200 m) — see "Current implementation status" above and `INCREMENT_03_MANIFEST.md` Section 6 for the full investigation. This is disclosed as an open item, not corrected or hidden; downstream MD-to-TVD/TVDSS mapping for Proteus 1ST2 conservatively uses the Petrel-supplied source trajectory (not the disagreeing minimum-curvature trajectory) as a result.
- **Increment 3.1 update:** a corrective patch addressing an independent audit's findings on source-filename handling, output environment-independence, an undisclosed normalization inference, and dogleg-angle numerical conditioning — see `INCREMENT_03_1_MANIFEST.md` for the full audit and re-verification record. No new scientific finding was made in this patch; the Proteus 1ST2 discrepancy above is unaffected, remains disclosed exactly as before, and was neither corrected nor concealed. The only numerical changes are at the floating-point noise floor of the diagnostic `dogleg_deg`/`dls_deg_per_30m`/TVD-and-offset-residual fields (at most ~9×10⁻¹³ m for TVD, ~7×10⁻¹⁵ m for easting/northing, across all four real wells), with zero change to any well's PASS/WARNING status.
- **Increment 3.1.1 update:** a packaging-only corrective patch (three notebook `%%writefile` cells that had drifted from their packaged source files, mirroring the earlier Increment 2.1.1 finding). No scientific, numerical, or real-data change of any kind — see `INCREMENT_03_1_1_MANIFEST.md`.
- **Increment 4 update:** checkshot ingestion and the time-depth framework independently reproduce every raw-data anomaly the project design anticipated (Poseidon 2: 5 repeated Depth ties, 6 non-increasing TVDSS steps, 4 non-increasing OWT steps, and a ≈257 m gap in Depth coverage between 1313.1 m and 1570.1 m; Boreas 1: 3 repeated Depth ties and 4 non-increasing TVDSS steps with OWT strictly increasing; Proteus 1ST2: strictly increasing in all three raw columns) and DISCLOSES (not resolves) two further items: (1) Boreas 1's and Proteus 1ST2's checkshot-vs-locked-survey TVDSS comparisons each show a near-constant offset (≈−0.69 m and ≈+0.30 m respectively) — reported as an observed datum-like offset pattern, not a proven datum error, with both source references preserved unmodified; (2) `Proteus1-Checkshot.txt` carries no embedded well identifier, so its association with Proteus 1ST2 is recorded as `identity_status: inferred_unverified` and used for QC only, never as a substitute time-depth model for any other well. Poseidon 2's sonic-checkshot drift over its longest valid continuous sonic interval is a diagnostic finding only (≈+16.7 ms, ≈+4.4%) and does not trigger any correction of `VP_m_s`, `DTCO`, checkshot OWT, or the time-depth curve. Poseidon North 1 has no approved checkshot file and is recorded as a factual data gap (`checkshot_availability: NOT_AVAILABLE`), not substituted with another well's data. See `INCREMENT_04_MANIFEST.md` for the full statistics, tables, and figures. **NOTE:** `INCREMENT_04_MANIFEST.md` incorrectly stated TVDSS is strictly increasing after Depth-tie conditioning for all three wells; this was corrected by Increment 4.1 (see below) — do not rely on that original statement.
- **Increment 4.1 update:** a narrowly scoped corrective patch to Increment 4, applied after an independent numerical-method audit, that does NOT begin Increment 5 or any later-phase work. It corrected an order-dependent tie-break: `tvdss_to_owt`/`owt_to_tvdss` (and their TWT equivalents) previously resolved a repeated value on the axis being inverted by silently keeping whichever tied row was encountered first in the Depth-conditioned table and discarding the other, with no audit trail beyond a bare count. This is replaced by an explicit, order-invariant policy (`p2mem.time_depth.build_axis_conditioned_lookup_table`/`build_axis_conditioned_tables_for_well`): every tied value is grouped by exact equality regardless of parse order, every tied row is registered in a new audit register (`checkshot_time_axis_tie_register.csv` — separate from, and never confused with, the pre-existing Depth-axis `checkshot_duplicate_tie_register.csv`), and the group's dependent-value MEDIAN becomes the conditioned representative (order-invariant; a screening-level choice, not proof the original TVDSS↔OWT relationship was single-valued at that tied value). A genuine reversal (not a tie) raises a typed error rather than being sorted or forced monotonic. Independently reproduced real-data counts: Poseidon 2 has two TVDSS-axis and two OWT-axis tie groups after Depth-tie conditioning; Boreas 1 has one TVDSS-axis tie group and zero OWT-axis tie groups; Proteus 1ST2 has none of either — correcting `INCREMENT_04_MANIFEST.md`'s original, incorrect "strictly increasing for all three wells" statement. This patch also hardens `trapezoidal_integrate` and `compute_sonic_checkshot_drift` to validate their numerical preconditions (finite, one-dimensional, equal-length, strictly increasing MD/x where required) — most notably, a decreasing or duplicate MD run can no longer silently produce a physically invalid negative transit time; it now raises a typed error instead. None of this hardening changes the already-verified real Poseidon 2 sonic-drift result, which is bit-for-bit unchanged. See `INCREMENT_04_1_MANIFEST.md` for the full audit, corrected statistics, and re-verification record.

- **Increment 4.1.1 update:** a narrowly scoped numerical-validation corrective patch to Increment 4.1, applied after an independent numerical-method/software-QA audit, that does NOT begin Increment 5 or any later-phase work, and does NOT alter any previously verified real Poseidon 2/Boreas 1/Proteus 1ST2 result. It corrected four blocking defects and one input-safety gap. (1) `build_axis_conditioned_lookup_table` grouped ALL occurrences of an identical axis value together GLOBALLY before checking for a reversal, so a reversal that returned to an already-seen value (e.g. `[100.0, 200.0, 100.0]`) was silently hidden rather than raising a typed error; it now evaluates the ORIGINAL, ungrouped sequence's successive differences for negativity BEFORE any grouping is attempted — provably equivalent to the 4.1 behavior for every legitimate adjacent tie, and strictly stronger against a non-adjacent reversal. (2) `compute_sonic_checkshot_drift`/`find_longest_finite_positive_run` validated MD monotonicity only within the selected finite-positive-VP run, so a decreasing or duplicate MD value outside that run (e.g. at a NaN-VP station) could pass silently; the COMPLETE canonical `md_m` array is now required finite and strictly increasing before run-selection (the real Poseidon 2 MD array, 31,897 samples, was independently re-verified to already satisfy this — the real sonic-drift result is bit-for-bit unchanged). (3) `compare_checkshot_to_survey` reached an untyped NumPy `ValueError` ("zero-size array to reduction operation") if every checkshot Depth row fell outside the locked survey's own MD coverage; it now raises a typed error naming the well, the checkshot Depth range, and the survey MD coverage. (4) `p2mem.io.checkshot.load_checkshot_surveys` did not catch the typed numerical-conditioning error, so a defect in one well's data could stop the entire batch; it is now caught per well (never via a blanket exception handler) and recorded as a typed, isolated per-well failure, exactly like every other expected failure mode. (5) `seconds_to_milliseconds`/`milliseconds_to_seconds` coerced their input directly, unlike `p2mem.units`'s Increment-1 input-safety policy, so a boolean, numeric-looking string, or complex value would be silently reinterpreted rather than rejected; both now reject such input with a typed error via a local, documented copy of `p2mem.units`'s identical private dtype check (`p2mem/units.py` itself remains LOCKED and unmodified). See `INCREMENT_04_1_1_MANIFEST.md` for the full audit, the regression-test list, and the re-verification record.

- **Increment 5 update:** formation-top ingestion independently reproduces both required regression findings from the real approved files. Poseidon 2's "selected readable" file's supplied TVDSS equals `MDRT - 21.8 m` EXACTLY for every one of its 9 markers (21.8 m is the well's own rotary-table elevation) — a vertical-well-assumption depth-reference defect that is corrected in the derived `TVDSS_survey_corrected_m` representation, reproducing residuals of ≈0 m at Sea Bed, ≈+1.68 m at Plover Fm (Top Reservoir), and ≈+2.49 m at TD. Boreas 1's supplied TVDSS does not follow that pattern (maximum absolute residual 0.0416 m, below the 0.05 m tolerance) and is therefore NOT corrected — this is applied per file, per well, never by analogy from Poseidon 2's defect. Poseidon North 1 and Proteus 1ST2 have no approved formation-top file and are recorded as a factual data gap (`formation_top_availability: NOT_AVAILABLE`), never substituted with another well's tops or correlated by depth alone. Both formation-top file representations for both wells are classified `well_identity_evidence_status: inferred_unverified` — a deliberately more conservative classification than Increment 4's checkshot files, since neither the HRS files (no well name in the body) nor the readable files (well name only in a project-supplied comment) constitute independently verified file content; see `INCREMENT_05_MANIFEST.md` Section 2 for the full rationale. No lithology interpretation, petrophysical calculation, or geological correlation is performed on these markers — the Increment 5 marker-depth comparison figures are explicitly labelled as depth comparisons only.

- **Increment 5.1 update:** a narrowly scoped corrective patch to Increment 5, applied after an independent technical/software-QA audit, that does NOT begin Increment 6 or any later-phase work, and does NOT alter any previously verified real Poseidon 2/Boreas 1 formation-top result above. It corrected three defects and one documentation-accuracy gap. (1) `reconcile_formation_top_sources` documented "zero canonical markers in common between the two sources" as a fatal `NO_COMMON_MARKERS` ERROR, but actually tested the emptiness of the UNION of both sources' canonical names, so two entirely disjoint, non-empty marker sets were silently accepted as one-sided `NOT_COMPARABLE` entries instead of being rejected; the check now explicitly evaluates the INTERSECTION of the two sources' canonical names, and `load_formation_top_well` consequently raises `TopSourceReconciliationError` for this condition, while a legitimate one-sided marker (with at least one marker genuinely shared) remains the pre-existing, non-fatal `NOT_COMPARABLE` case. (2) `load_formation_top_surveys` recorded only the HRS file's path in every `TopIngestionFailure.source_path`, regardless of which file/stage actually failed, so a failure originating from the "selected readable" file (a missing file, malformed content, or a contract mismatch) could leak that file's own absolute path, unsanitized, into exported issues/availability/manifest rows; `TopIngestionFailure` now carries an explicit `failure_origin` and both `hrs_path`/`readable_path` fields, and every exporting function in `p2mem.io.tops_inventory` now sanitizes both candidate paths (literal substring replacement only, never a regex). (3) `reconcile_formation_top_sources` — the public in-memory API, as distinct from the file parsers, which already enforced this — did not validate its own documented contract before any numerical comparison: non-finite or negative MDRT/TVDSS values, a shorter `NOTE_source` tuple (previously reaching an untyped `IndexError`), a non-1-dimensional array, and an unvalidated `mdrt_agreement_tolerance_m` keyword (previously accepting NaN, a negative value, or a boolean) were all silently accepted or reached an undocumented incidental error; all are now rejected before any comparison, with deliberate, documented exceptions. (4) `INCREMENT_05_MANIFEST.md`'s statement that no absolute path exists ANYWHERE in the package was overbroad and inaccurate, since existing synthetic tests and historical documentation intentionally contain fake absolute-path strings as test inputs; the precise, narrowly scoped claim ("no environment-dependent build path appears in exported CSV/JSON outputs") is now stated explicitly in `INCREMENT_05_1_MANIFEST.md`, which also acknowledges the prior wording was overbroad — `INCREMENT_05_MANIFEST.md` itself is a locked historical record and is NOT rewritten. See `INCREMENT_05_1_MANIFEST.md` for the full audit, the regression-test list, and the real-data non-regression verification record.

- **Increment 5.1.1 update:** a further narrowly scoped corrective patch to Increment 5.1, applied after an independent technical/software-QA audit, that does NOT begin Increment 6 or any later-phase work, and does NOT alter any scientific result, tolerance, depth-mapping method, formation-top contract, or real-data output. It corrected one remaining reconciliation edge case and three documentation-accuracy gaps. (1) Increment 5.1 correctly rejected two non-empty, disjoint marker sets, but its condition ("both sources non-empty AND intersection empty", plus a separate both-empty case) still silently accepted the one remaining zero-common-markers configuration: exactly one source entirely empty and the other non-empty (the intersection of an empty set with anything is itself empty, so this was still a zero-common-markers condition). `reconcile_formation_top_sources` now uses the single, strictly correct check `if not common_markers` — empty in every zero-common-markers case (both empty, either side alone empty, or both non-empty and disjoint) and never empty whenever at least one canonical marker is genuinely shared, so a legitimate one-sided marker alongside at least one shared marker remains the pre-existing, non-fatal `NOT_COMPARABLE` case. (2) `INCREMENT_05_1_MANIFEST.md` stated that the real Increment 5 baseline ZIP's SHA-256 "matches" a governing-instruction-supplied hash that was, in fact, a two-character truncation of the real 64-character value — a mathematical impossibility, now stated transparently as a documentation/input typo, never as baseline corruption or uncertainty. (3) `INCREMENT_05_1_MANIFEST.md` Section 7 stated `outputs/` is excluded from the delivered ZIP; this was false — the delivered Increment 5.1 ZIP packages the `outputs/` tree (including the byte-identical Increment 5 formation-top outputs/figures), exactly as every prior increment's ZIP has; only dev-only regenerated-comparison directories and private raw source files are excluded, now stated accurately. (4) the same manifest's clean-room section stated no `*.las` file exists in the package; this was false because small, intentionally packaged synthetic LAS/checkshot/deviation/top fixtures exist under `tests/fixtures/` for portable testing — the corrected wording distinguishes these fictional fixtures from the genuinely excluded real/private project source files. See `INCREMENT_05_1_1_MANIFEST.md` for the full audit, the regression-test list, and the real-data non-regression verification record.

- **Increment 6 update:** the gamma-ray QC, screening-proxy sensitivity, well-frame and method-eligibility framework adds NO calibration and NO interpretation. Four specific limitations are newly quantified and disclosed. (1) **Boreas 1's ECGR is excluded, not corrected.** Its independently measured median (8.40 API) sits 4.3×–4.9× below the other three wells' GR medians, its range spans [−0.0001, 519.18] API including values at and below zero, and 2,059 of its samples lie above the well's own declared seabed marker. No tool header, calibration record, or environmental-correction metadata exists to adjudicate whether this is a scale, unit, tool-type, or acquisition problem, so the curve is formally excluded from every GR-derived calculation under the machine-readable reason `BOREAS_ECGR_SCALE_UNRESOLVED`. Rescaling it would fabricate a calibration this project does not have. (2) **The screening proxy is not a shale volume and is strongly endpoint-dependent.** `VSH_GR_linear_proxy_frac` is the linear identity of a clipped GR index computed under ASSUMED per-well percentile endpoints; across the three configured scenarios the proxy median moves by up to 0.12 dimensionless, and the qualifying sonic-NCT-candidate thickness moves by a factor of 8.0 in Poseidon 2 (144 m to 1,159 m), 3.5 in Poseidon North 1 and 2.3 in Proteus 1ST2. Any later NCT or pore-pressure result built on a single endpoint/threshold choice would inherit that full range as unquantified uncertainty. (3) **Eligibility is not validity.** The three masks record only whether a sample is technically admissible as INPUT to a later method; none of those methods is implemented, fitted, or validated here, and a sonic-NCT-candidate interval is emphatically not evidence that any interval is normally compacted, is a selected donor interval, or is any named lithology. Density and dynamic-elastic eligibility in all four wells begins only around 3,900–4,800 m TVDSS, so a later overburden integration cannot be supported from surface by these logs alone regardless of per-sample eligibility. (4) **Poseidon North 1 and Proteus 1ST2 have no approved formation tops**, so their above-seabed sample counts are reported as *not determinable* (never as 0) and all their results remain depth-tied and stratigraphically unvalidated. See `INCREMENT_06_MANIFEST.md` for the full measured statistics, sensitivity tables, and verification record.

- **Increment 6.1 update:** a narrowly scoped corrective patch to Increment 6, applied after an independent geomechanics/rock-physics and software audit. It does NOT start Increment 7 and implements no pore pressure, NCT fitting, elastic-property calculation, rock strength, stress, or wellbore-stability work. Four findings were corrected. (1) **Vp/Vs terminology and boundaries were scientifically inaccurate.** Increment 6 called every excluded Vp/Vs ratio "non-physical" and used a strictly exclusive √2 bound. With *r* = Vp/Vs, ν = (r²−2)/(2(r²−1)) and K = ρ(Vp²−(4/3)Vs²): *r* = √2 gives ν = 0 **exactly** and must be accepted by a non-negative-ν policy, so the bound is now **inclusive**; √(4/3) < *r* < √2 gives a **positive** bulk modulus with a negative Poisson's ratio — unusual and outside this project's conservative policy but *not* physically impossible; only *r* ≤ √(4/3) implies a non-positive bulk modulus and is genuinely outside the isotropic elastic model; and *r* > 4 is a **configured plausibility limit**, not a Poisson-domain boundary (ν ≈ 0.467 there). The condition is renamed a *configured non-negative-Poisson-ratio applicability screen*, the three bounds are separately named in config, and exclusions are diagnosed **by regime** rather than aggregated. Independently re-measured on the real data, the previously aggregated counts decompose as: Boreas 1 — 1 positive-K/negative-ν, 0 non-positive-K; Poseidon 2 — 22 and 0; Poseidon North 1 — **3 and 4** (the former single count of 7); Proteus 1ST2 — 3 and 0. No real sample sits exactly at √2, so the inclusive-bound correction changes no eligible count in this dataset — it corrects the policy, not the numbers. (2) **Poseidon North 1's machine-readable `use_status` contradicted its own data.** It read `screening_proxy_allowed` while the well has no approved formation tops; it is now `screening_proxy_allowed_depth_tied`, and a config invariant makes any contradictory has-tops/use-status pairing fail config loading loudly. (3) **Unsupported lithology and correlation claims were removed, and the named-lithology gate is no longer circular.** Project-specific "clastic-dominated" assertions are gone (a disclaimer does not undo an assertion), and the "regionally persistent" cross-well claim is replaced by the factual statement that low-GR intervals occur *independently* in each of the three GR-eligible wells with cross-well stratigraphic persistence **not established** — it cannot be, since two of those wells have no approved tops. `named_lithology_assigned` is now **derived** from an actual validation pass over every persisted label, per-well note, mask name and manifest statement rather than hardcoded; injecting a prohibited term makes the validation, the manifest flag and the completion gate fail together. (4) **Gross versus net interval thickness was ambiguous.** The reported "qualifying thickness" summed block endpoint spans that may contain explicitly bridged ineligible samples. Thickness is now exported only as `gross_thickness_*_m` (endpoint span, bridging-inclusive) or `net_thickness_*_m` (bridged gaps removed), every mask is additionally decomposed under a strict-no-gap policy, every sensitivity case reports its bridged-sample and interrupted-block counts, and Figure 4 distinguishes qualifying from rejected sub-threshold blocks with the counted population stated. Measured for Poseidon 2: configured **gross** 144.4–1,159.2 m (factor 8.03) versus **strict-no-gap** 140.0–1,110.2 m (factor 7.93). See `INCREMENT_06_1_MANIFEST.md` for the full audit and re-verification record.
- **Increment 6.1.1 update:** a second, narrower corrective patch, applied after an independent audit of Increment 6.1. It does NOT start Increment 7 and implements no pore pressure, NCT fitting, elastic-property calculation, rock strength, stress, or wellbore-stability work; it changes no configured tolerance, threshold, or qualifying-block policy. Five findings were corrected. (1) **The named-lithology validator could be bypassed.** The allowlist that lets per-well prose name a *method* contained `"shale gas"`, which is not a method-category phrase but can be a direct geological/hydrocarbon assertion — `"This interval contains shale gas."` passed with zero violations. The allowlist is reduced to `shale proxy` and `shale volume`, the only two phrases that are genuine method/quantity names required to state this project's boundaries, and the allowance is now evaluated **per sentence** and suppressed wherever the sentence carries a geological-assertion cue (`contains`, `comprises`, `bearing`, `facies`, …) or the phrase is immediately preceded by a quantity cue (`has a high shale volume`). Explanatory use of `shale proxy` / `shale volume` as a method or quantity name is preserved; use in any label remains prohibited outright. Regression tests prove the allowlist can no longer hide a geological assertion, and that injecting either bypass case into manifest content makes `named_lithology_assigned` true, `lithology_validation.n_violations` non-zero, and the completion gate fail. (2) **The active Vp/Vs description was stale.** The live top-level docstring of `p2mem.method_eligibility` still described the screen as keeping samples "inside the Poisson domain" under a strictly exclusive `Vp/Vs > √2`, contradicting the implementation corrected in 6.1. It now states the *configured non-negative-Poisson-ratio applicability screen* with an **inclusive** `Vp/Vs ≥ √2` bound and says explicitly that this is neither a physical-possibility test nor a boundary of the mathematical Poisson domain; a regression test asserts the corrected wording against the live module documentation. Superseded wording is retained only where it is clearly preserved as historical record. (3) **Interruption counts were ambiguous or wrong.** `n_interruptions` was a boolean-like flag reported as a count — all 327 bridged blocks reported exactly 1 — and `n_interrupted_subruns` did not describe what it counted. Interval records now carry `n_bridged_samples` (ineligible samples absorbed inside the gross block), `n_bridged_gaps` (distinct bridged runs) and `n_eligible_subruns` (strictly contiguous eligible sub-runs), with `n_bridged_gaps = n_eligible_subruns − 1` enforced at construction; a block with no gap reports 0/0/1 and a block with two separate bridged gaps reports 2 gaps and 3 sub-runs. The sensitivity summary distinguishes `n_bridged_samples_in_qualifying_blocks`, `n_bridged_gaps_in_qualifying_blocks` and `n_interrupted_qualifying_blocks`. No ambiguous alias survives in any active export. (4) **The notebook's output count was wrong.** It creates and checks eight deterministic CSV/JSON outputs while its gate text and printed label said seven; both now say eight and the gate asserts the declared count and the existence of all eight. Four figures remain separate — twelve outputs/figures in total. (5) **The Increment 6.1 delta arithmetic was incorrect.** That record stated "Changed (12)"; a clean recursive comparison gives **18 changed, 3 added, 0 removed (21 path differences)**. The locked 6.1 record is not rewritten; the statement is explicitly superseded, and both deltas are reported file by file, in `INCREMENT_06_1_1_MANIFEST.md`.
- **Increment 6.1.2 update:** a third, narrower corrective patch, applied after an independent audit of Increment 6.1.1. It does NOT start Increment 7, implements no pore pressure, NCT fitting, elastic-property calculation, rock strength, stress, or wellbore-stability work, changes no scientific threshold, tolerance, endpoint scenario, contiguity policy, GR disposition, depth-mapping rule, or real-data interpretation, and adds no runtime dependency. Two residual findings were corrected. (1) **The named-lithology validator was wrong in both directions.** Increment 6.1.1 decided the method-phrase allowance from a sentence-wide assertion-cue list plus a fixed look-behind window. Because nothing to the *right* of an allowed phrase was inspected, and because the normalizer erased possessives, `"Poseidon 2's shale volume is high."`, `"Poseidon 2's shale volume is 70 percent."` and `"The interval's shale volume exceeds 60 percent."` all passed with **zero violations** — the assertion lives in the predicate and the possessive, neither of which was ever read. Conversely, because a sentence-wide cue fires without asking what the cue word is predicated *of*, `"This method contains a shale proxy calculation."` and `"The analysis shows no shale volume was computed."` were reported as **false** lithological assertions. The allowance is now a property of each *occurrence*, decided from its own local grammar: a possessive geological entity, an adjacent magnitude modifier, a right-hand magnitude or dominance predicate, or a composition verb whose resolved subject head is a geological entity each withdraw it. A bare copula does not — `"the shale proxy is dimensionless"` describes a method, not an amount of rock. Apostrophes are parsed rather than erased, line breaks end sentences, ambiguous project-specific sentences fail closed, and labels keep zero latitude. A table-driven matrix of 59 discrimination rows across all three scopes, plus gate-path injections, pins the behaviour in both directions. (2) **Interval-record invariants were only partially enforced.** The 6.1.1 constructor checked only `n_bridged_gaps == n_eligible_subruns − 1`, and skipped even that whenever a value was `None` or `n_eligible_subruns` was 0 — so records with missing, negative, boolean or contradictory counts were constructible. All three counts are now validated together: required, strictly integral (booleans, strings, complex values and fractional floats rejected, never coerced), non-negative, `n_eligible_subruns ≥ 1`, `n_bridged_gaps = n_eligible_subruns − 1`, `n_bridged_samples = 0` **iff** `n_bridged_gaps = 0`, and `n_bridged_samples ≥ n_bridged_gaps`. Removed legacy field names are rejected outright rather than ignored. See `INCREMENT_06_1_2_MANIFEST.md` for the full audit and re-verification record.
- **Increment 6.1.3 update:** the fourth corrective patch, and an architectural one. It does NOT start Increment 7, changes no scientific threshold, tolerance, endpoint scenario, contiguity policy, GR disposition, depth-mapping rule, or real-data interpretation, and adds no runtime dependency. **(1) The validator no longer parses English.** Increments 6.1.1 and 6.1.2 each tried to decide from grammar whether a sentence containing a rock name named a *method* or asserted *geology*. Both were audited; both failed in both directions. 6.1.2 still missed `"The interval has a shale volume."`, `"The shale volume in Poseidon 2 exceeds 60 percent."` and `"Poseidon 2 shale volume was determined to be high."` — a missing predicate verb, a prepositional possessor, and a magnitude beyond the look-ahead window — while wrongly rejecting `"The well contains no shale volume estimate."` Each is a list gap or a window edge, and no finite grammar closes an unbounded space. Every scope is now decided by membership or equality: labels and project-specific prose have **zero allowance** (the `allow_method_phrases` parameter is deleted, so there is no exemption path left to bypass); method and limitation wording is admitted only when the text is **exactly equal** to a member of the closed, provenance-tagged `METHOD_STATEMENTS` registry, whose five members were *measured* by scanning every string literal in active packaged source under a zero-allowance rule; and explanatory text may use a rock name generically but never in a sentence referring to a project well **or a project rock body** (6.1.2 checked well names alone). All the grammar machinery — cue lists, magnitude vocabularies, look-behind and look-ahead windows, subject resolution, occurrence classification — is **deleted**, and a test asserts none of it is importable. Correctness is proved by **closure** over every prohibited term in every position inside carrier prose built from the exact constructions that defeated both previous implementations, not by a list of example sentences. **(2) Three exported strings had never been validated at all** — including a `calibration_status` value literally containing `not_a_shale_volume`, written into every row of `gr_proxy_sensitivity_summary.csv`. All three are now scanned, raising `n_fields_checked` from 138 to 143; that single scalar is the only difference in any output. **(3) The interval-record type gate is hardened**: keyword acceptance is a whitelist over declared slots (a misspelled count name no longer leaves the real count silently unset), floats are rejected outright including whole-valued ones, and `NaN`/`Inf` raise a typed `PetrophysicsInputError` before any conversion is attempted. See `INCREMENT_06_1_3_MANIFEST.md` for the full audit and re-verification record.
- **Increment 6.1.4 update:** a deliberately narrow completion of the Increment 6.1.3 architecture — one scope rule, no other change. It does NOT start Increment 7 and changes no scientific threshold, tolerance, policy, disposition or result. 6.1.3 closed labels and project-specific prose by removing every exemption, and closed method wording by exact registry membership — but left **explanatory** scope decided by a finite token list of project references. That is the same shape of rule that failed in 6.1.1 and 6.1.2, and it failed the same way: ordinary stratigraphic and exploration nouns were missing from the list, so `"The member is shale."`, `"The group is limestone."`, `"The package is a clean sandstone."`, `"The play is shale-dominated."`, `"The prospect is carbonate."` and `"The target is sandstone."` all passed — while `"The upper member is a clean sandstone reservoir."` failed only because `reservoir` happened to be listed, an accident that shows what a list-shaped rule produces. The rule is **inverted**: explanatory text carrying a rock term is a violation unless the text is a member of the closed `GENERIC_EXPLANATORY_STATEMENTS` registry. It **fails closed**, so no vocabulary gap can admit anything, and `PROJECT_WELL_NAME_TOKENS`, `PROJECT_ROCK_BODY_TOKENS` and `PROJECT_REFERENCE_TOKENS` are deleted. The generic registry is **empty** as a measured fact — this project persists exactly one explanatory field and it carries no rock name — but the mechanism is live and tested, because later increments explaining gamma-ray non-uniqueness will need it. **All four scopes are now closed by membership or exact equality**, and the closure proof covers every one of them. See `INCREMENT_06_1_4_MANIFEST.md` for the full record.
- **Increment 6.1.5 update:** the final Increment 6 corrective patch, and the one that changes the security model rather than the recognizer. Increments 6.1–6.1.4 all asked *"does this text contain a geological assertion?"* and all four shared one assumption — content is acceptable by default and becomes unacceptable only when a recognizer fires. All four were defeated, finally by a word no recognizer had been given: in the 6.1.4 package `"The interval is chalk."`, `"…halite."`, `"…gypsum."`, `"…conglomerate."`, `"…chert."`, `"…tuff."`, `"…basalt."`, `"…dolostone."`, `"…lignite."`, `"…calcareous."` all passed label, interpretive and explanatory scope, and so did `"The interval is qxzite."` — a word that does not exist. (They already failed in *method* scope, the one scope that then required registration; that contrast is what this patch generalises.) **A longer blacklist is not the fix and is not the mechanism the assurance claim rests on.** The model is inverted: labels must be members of a typed `APPROVED_LABELS` registry; interpretive, method and explanatory statements must resolve to a registered `statement_id`, or — interpretive only — a reviewed `template_id` whose substitutions are restricted to declared types. Id and exact rendered text are validated together, so unknown ids, id/text mismatches, case changes, near-misses, duplicate ids and undeclared template fields all fail. The registries were **measured** from the actual persisted export: 17 approved labels, 17 registered statements, 1 controlled template. `PROHIBITED_LITHOLOGY_TERMS` remains only as a diagnostic linter that authorizes nothing — every rejection listed above occurs with that linter returning **empty**. **The defensible assurance statement**, superseding the wording of every earlier Increment 6 manifest: *every persisted project-specific classification and interpretive statement is generated from an approved typed value, controlled template, or registered statement; arbitrary free text cannot enter these controlled fields.* This is **not** a claim that the software understands or exhaustively recognizes natural-language lithology — it does not, and no earlier version did. Free-form notebook narrative lies outside these controlled fields and remains subject to manual scientific review. See `INCREMENT_06_1_5_MANIFEST.md`.
- **Increment 6.1.6 update:** authorization is now enforced where records are actually EMITTED. 6.1.5 closed the unit validator but validated a scope the manifest builder *reconstructed*; a `GrEndpointScenario` with `description="The interval is chalk."` was persisted verbatim while absent from that 143-field scope, so it could not move `named_lithology_assigned`. Five findings, all reproduced first: the exported endpoint-description bypass; incomplete output coverage (endpoint `description`, `limitations`, `purpose`, `population_statement`, `limiting_criterion`, calibration/evidence labels, diagnostic text and manifest prose were not consistently validated); a field-kind mismatch letting `use_status="GR"` pass because `APPROVED_LABELS` was keyed by value alone; a stale exported `derivation` still describing prohibited-term validation; and a duplicated `named_lithology_statement` literal that could diverge from its registered copy while the gate stayed green. `p2mem.io.output_policy` now declares a category for **every** string column and JSON path of all eight artifacts (105 entries, zero unclassified) across eight closed categories, with **no** general category admitting arbitrary prose; labels are authorized by `(field_kind, value)`; the manifest statement comes from its registered id; and export authorizes the exact pre-serialization records, writes, then **re-authorizes the written bytes** so a post-authorization mutation is caught. Assurance metrics say what they count — `n_fields_checked=143` is replaced by `scope_object_fields_checked` plus an `emitted_field_coverage` block reporting total emitted occurrences, controlled occurrences, structural occurrences, occurrences outside the guarantee, unclassified fields, unauthorized fields and field-kind mismatches. Machine diagnostics and operator-facing issue text are counted separately and are explicitly outside the controlled-interpretation guarantee. See `INCREMENT_06_1_6_MANIFEST.md`.
- **Increment 6.1.7 update:** the 6.1.6 gate still inferred field presence from non-empty string values and wrote to the official directory before its post-write check. Exact schemas now validate all eight artifacts independently of content, including inventory, ordered CSV columns, JSON keys, requiredness, strict types, finite numerics and approved dynamic well identifiers. Every declared string occurrence is authorized even when empty or numeric-looking. The complete candidate set is written to an isolated sibling directory, re-read, re-validated and compared field-by-field and row-by-row using a typed canonical representation before publication. Missing/unknown fields, post-write type changes, row reordering, authorized-to-authorized substitutions, stale artifacts and serializer errors fail closed without altering the official destination. Publication is failure-atomic for handled process errors; no claim is made about multi-file atomicity across power loss or operating-system failure. No scientific calculation, threshold, disposition or result changed. See `INCREMENT_06_1_7_MANIFEST.md`.

## Screening-level statement

**This 1D Mechanical Earth Model is a screening-level, uncalibrated, educational work product.** It has not been validated against independent field measurements and does not carry the assurance level required for drilling engineering, well design, casing/mud-weight selection, or any other operational decision. Any numerical result produced by this codebase should be read as illustrative of a defensible methodology applied to the available data, not as a certified or field-ready prediction.


#### `config/petrophysics_eligibility.yml` — the human-authored dispositions and policy

**Why this file is human-authored:** every decision in it is a reviewable *judgement*, not a data-derived fact. Whether a gamma-ray curve is fit for a shale-proxy calculation depends on acquisition metadata, tool type, hole conditions and calibration history that these files do not carry. Putting those judgements in a plain-text, diffable, version-controlled file lets an auditor see and challenge each one without reading code.

**Two policy invariants are enforced as hard load-time errors**, because violating either would silently cross an Increment 6 scientific boundary: `cross_well_shared_endpoints_allowed` must be false (a shared endpoint pair would assert a tool equivalence no evidence supports), and `nonlinear_vsh_transforms_enabled` must be false (no nonlinear Vsh transform has a closed primary-source method record in this project).

In [ ]:
%%writefile config/petrophysics_eligibility.yml
# =============================================================================
# config/petrophysics_eligibility.yml
#
# Poseidon 2 1D MEM - Increment 6
# Human-authored gamma-ray-family disposition, endpoint-sensitivity policy,
# and method-eligibility rules.
#
# Tier C - Screening-Level / Uncalibrated Educational.
#
# WHY THIS FILE IS HUMAN-AUTHORED
# -------------------------------
# Every decision below is a REVIEWABLE JUDGEMENT, not a data-derived fact.
# This project never adjudicates tool identity, curve equivalence, or well
# usability from the numbers alone: a gamma-ray curve's fitness for a
# screening-proxy calculation depends on acquisition metadata, tool type, hole
# conditions, and calibration history that these files do not carry. Putting
# those judgements in a plain-text, diffable, version-controlled file means an
# auditor can see and challenge each one without reading code.
#
# WHAT THIS FILE MAY NOT DO
# -------------------------
# * It may not name a lithology. No key or value here asserts that any
#   interval is shale, sand, carbonate, or any other rock type.
# * It may not declare any endpoint, threshold, or scenario "calibrated".
#   Every endpoint below is an ASSUMED, CONFIGURED value chosen to bracket
#   uncertainty, and every result derived from it is a SCREENING PROXY.
# * It may not rescale, correct, or repair a curve. There is no correction
#   factor, no shift, and no normalization-to-a-reference-well anywhere in
#   this file, by design (see Boreas 1 below).
#
# UNITS
# -----
# All gamma-ray endpoints and thresholds are in API units (gAPI) as recorded
# in each file's own header and preserved by the LOCKED Increment 2.1.1
# per-file curve contracts. Depth values are metres. IGR and the linear
# screening proxy are dimensionless fractions in [0, 1] after clipping.
# =============================================================================

schema_version: "6.0"
increment: 6
assurance_tier: "Tier C - Screening-Level / Uncalibrated Educational"

# -----------------------------------------------------------------------------
# GLOBAL POLICY
# -----------------------------------------------------------------------------
policy:

  # Endpoints are estimated per well, from that well's OWN valid GR-family
  # samples, using percentile rules. A single universal cross-well endpoint
  # pair is explicitly PROHIBITED: these are four different wells logged with
  # four differently named tools, with no cross-well calibration tie, so a
  # shared endpoint pair would silently assert an equivalence no evidence
  # supports.
  endpoint_estimation_method: "per_well_percentile_of_own_valid_samples"
  cross_well_shared_endpoints_allowed: false

  # Endpoints are estimated from samples that are BOTH finite AND
  # depth-mapped. An unmapped sample has no defensible depth and must not
  # influence a depth-resolved endpoint.
  endpoint_sample_basis: "finite_and_depth_mapped_samples_only"

  # The clipped IGR is what any downstream mask may consume; the unclipped
  # IGR is always retained alongside it so that the amount of clipping - i.e.
  # how far the real data fell outside the assumed endpoint bracket - stays
  # visible and auditable rather than being silently absorbed.
  clipping_policy: "retain_unclipped_and_clipped_side_by_side"
  clip_lower: 0.0
  clip_upper: 1.0

  # Minimum separation between the low and high endpoint. Below this the
  # normalization is numerically meaningless (a near-zero denominator turns
  # ordinary log noise into full-scale IGR swings), so it is rejected with a
  # typed error rather than producing an explosive result.
  min_endpoint_separation_api: 1.0

  # A linear IGR->proxy identity is the ONLY screening-proxy transform permitted
  # in Increment 6. Larionov, Clavier, Stieber and every other nonlinear Vsh
  # transform are DEFERRED: each would require retrieving and verifying its
  # primary method source and closing a method-register entry before use, and
  # none of that evidence exists in this project yet.
  shale_proxy_transform: "linear_identity_of_clipped_igr"
  nonlinear_vsh_transforms_enabled: false
  nonlinear_vsh_deferral_reason: >-
    No primary-source method record has been retrieved, verified, or closed for
    any nonlinear Vsh transform in this project. A nonlinear transform would
    change every downstream eligible thickness while adding no new measurement,
    so it is deferred rather than assumed.

  # Contiguity rules for interval analysis. `max_gap_samples` permits a short
  # run of ineligible samples to be bridged INSIDE one reported block only
  # when the physical depth span of that gap is also within
  # `max_gap_depth_m`; both conditions must hold. This distinguishes
  # sample-count continuity from physical-depth continuity - a 2-sample gap
  # across a 400 m depth jump is not a continuous interval.
  contiguity:
    max_gap_samples: 2
    max_gap_depth_m: 1.0
    min_block_samples: 20
    min_block_thickness_m: 5.0

  # Sonic-NCT CANDIDATE screening thresholds. A sample is a candidate only if
  # its clipped screening proxy is at or above the threshold. These are
  # SENSITIVITY CASES, not a calibrated cut-off: the whole point of carrying
  # three of them is to measure how much the candidate thickness moves.
  nct_candidate_proxy_thresholds: [0.50, 0.60, 0.70]

  # Physical plausibility bounds used by the eligibility masks. These reject
  # implausible or out-of-policy values; they never repair them.
  physical_bounds:
    rhob_min_kg_m3: 1000.0
    rhob_max_kg_m3: 3500.0
    vp_min_m_s: 1000.0
    vp_max_m_s: 8000.0
    vs_min_m_s: 300.0
    vs_max_m_s: 5000.0

    # -------------------------------------------------------------------
    # Vp/Vs bounds (Increment 6.1 correction)
    # -------------------------------------------------------------------
    # For an isotropic elastic solid with r = Vp/Vs:
    #     nu = (r^2 - 2) / (2 * (r^2 - 1))
    #     K  = rho * (Vp^2 - (4/3) * Vs^2)
    #
    # Increment 6 collapsed three DIFFERENT conditions into one bound
    # labelled "non-physical". They are separated here, with names that
    # state what each one actually is.
    #
    # CONFIGURED NON-NEGATIVE-POISSON-RATIO APPLICABILITY SCREEN.
    # r = sqrt(2) gives nu = 0 EXACTLY, so a policy requiring a
    # non-negative Poisson's ratio must ACCEPT r = sqrt(2). This bound is
    # therefore INCLUSIVE (Increment 6 wrongly used an exclusive bound and
    # rejected nu = 0). This is a conservative PROJECT POLICY choice about
    # what to admit to a later dynamic-elastic calculation - it is not a
    # claim that r < sqrt(2) is physically impossible.
    vp_vs_ratio_nonnegative_poisson_min_inclusive: 1.4142135623730951

    # POSITIVE-BULK-MODULUS BOUNDARY. K > 0 requires r > sqrt(4/3).
    # r <= sqrt(4/3) implies a non-positive bulk modulus, which IS outside
    # the isotropic elastic model - the one regime that genuinely warrants
    # the word "non-physical" under that model. Samples between this bound
    # and the screen bound above have a POSITIVE bulk modulus and a
    # NEGATIVE Poisson's ratio: unusual, outside this project's policy,
    # but not physically impossible, and diagnosed separately.
    vp_vs_ratio_positive_bulk_modulus_min_exclusive: 1.1547005383792515

    # CONFIGURED PLAUSIBILITY LIMIT - not a Poisson-domain boundary.
    # At r = 4, nu ~= 0.467, an entirely ordinary Poisson's ratio. This
    # bound expresses what this project is willing to treat as a credible
    # logged ratio, nothing more.
    vp_vs_ratio_plausibility_max: 4.0

# -----------------------------------------------------------------------------
# ENDPOINT SENSITIVITY SCENARIOS
# -----------------------------------------------------------------------------
# Three scenarios bracket the endpoint choice. Each is defined by a percentile
# pair applied to a well's own valid, depth-mapped GR-family samples. NONE of
# them is calibrated; "base" is a conventional mid-range choice, not a
# preferred or validated one, and results must always be reported across all
# three rather than from "base" alone.
endpoint_scenarios:

  - scenario_name: "low"
    description: >-
      Narrow bracket: a high low-endpoint and a low high-endpoint. This
      compresses the normalization range, so more samples clip at both ends
      and the proxy saturates sooner. Reported to show the upper bound of
      apparent proxy magnitude.
    low_percentile: 10.0
    high_percentile: 85.0

  - scenario_name: "base"
    description: >-
      Conventional mid-range bracket. Chosen for comparability with common
      screening practice, NOT because any evidence in this project supports
      it over the other two.
    low_percentile: 5.0
    high_percentile: 95.0

  - scenario_name: "high"
    description: >-
      Wide bracket: a low low-endpoint and a high high-endpoint. This expands
      the normalization range, so fewer samples clip and the proxy is damped.
      Reported to show the lower bound of apparent proxy magnitude.
    low_percentile: 1.0
    high_percentile: 99.0

# -----------------------------------------------------------------------------
# PER-WELL GR-FAMILY DISPOSITION
# -----------------------------------------------------------------------------
# `use_status` vocabulary:
#   screening_proxy_allowed            - a dimensionless screening proxy may be
#                                        computed from this well's own GR-family
#                                        curve, under its own per-well endpoints.
#   screening_proxy_allowed_depth_tied - as above, but every result remains
#                                        depth-tied and unvalidated because this
#                                        well has no approved formation tops to
#                                        anchor it stratigraphically.
#   qc_only_excluded                   - the curve may appear in factual raw QC
#                                        displays and availability reports ONLY.
#                                        No GR index, no proxy, no flag, no
#                                        lithology-dependent mask, no NCT-donor
#                                        status may be computed for it.
wells:

  Poseidon_2:
    source_las_filename: "Poseidon_2_logs.las"
    gr_family_canonical_name: "GR_api"
    gr_family_source_curve_name: "GR"
    use_status: "screening_proxy_allowed"
    exclusion_reason: null
    evidence_class: "measured"
    has_approved_formation_tops: true
    notes: >-
      The recorded values span a range and central tendency typical of
      conventionally API-scaled gamma-ray logs. This is a statement about the
      NUMERIC distribution and the scaling convention only; no depositional
      setting, rock type, or lithology is asserted or implied. This well has
      approved, survey-corrected Increment 5 formation tops, so its results can
      be annotated against locked stratigraphic markers.

  Poseidon_North_1:
    source_las_filename: "Poseidon_North_1_logs.las"
    gr_family_canonical_name: "GRD_api"
    gr_family_source_curve_name: "GRD"
    # Increment 6.1 correction: this well has no approved formation tops, so
    # its machine-readable status must be the depth-tied variant. The previous
    # value ("screening_proxy_allowed") contradicted both the config's own
    # status vocabulary and the prose limitation, and is now rejected at
    # config-load time by an explicit invariant.
    use_status: "screening_proxy_allowed_depth_tied"
    exclusion_reason: null
    evidence_class: "measured"
    has_approved_formation_tops: false
    notes: >-
      GRD is a differently named GR-family curve and is NEVER treated as
      interchangeable with GR or ECGR. This well has NO approved formation
      tops, so every result for it is depth-tied and stratigraphically
      unvalidated, and any future use as an NCT donor would remain so. That
      limitation is recorded here, in the machine-readable use_status, not
      only in prose.

  Proteus_1ST2:
    source_las_filename: "Proteus_1ST2_logs.las"
    gr_family_canonical_name: "GR_api"
    gr_family_source_curve_name: "GR"
    use_status: "screening_proxy_allowed_depth_tied"
    exclusion_reason: null
    evidence_class: "measured"
    has_approved_formation_tops: false
    notes: >-
      This well's GR canonicalizes to the same name as Poseidon 2's (GR_api),
      but it is a different tool run in a different well with no cross-well
      calibration tie. Endpoints are estimated from this well's own samples
      only. No approved formation tops exist, so results are explicitly
      depth-tied and unvalidated.

  Boreas_1:
    source_las_filename: "Boreas_1_logs.las"
    gr_family_canonical_name: "ECGR_api"
    gr_family_source_curve_name: "ECGR"
    use_status: "qc_only_excluded"
    exclusion_reason: "BOREAS_ECGR_SCALE_UNRESOLVED"
    evidence_class: "measured"
    has_approved_formation_tops: true
    notes: >-
      FORMAL EXCLUSION from every lithology-dependent, GR-normalized,
      screening-proxy (VSH_GR_linear_proxy_frac) and NCT-candidate
      calculation. The ECGR curve carries an
      unresolved scale/acquisition anomaly: its central tendency is roughly an
      order of magnitude below what an API-scaled gamma-ray log over a
      comparable section would show, its range extends across several hundred
      API, it includes values at or below zero, and it contains samples above
      the well's own declared seabed marker. No independent tool header,
      calibration record, or environmental-correction metadata is available to
      adjudicate the cause. The curve is therefore EXCLUDED, not corrected:
      applying a shift, gain, or normalization would fabricate a calibration
      that does not exist, and would silently propagate into every downstream
      screening proxy and NCT candidate interval. Boreas 1 remains available for
      factual raw-GR QC display and availability reporting only. The specific
      numeric values supporting this disposition are MEASURED INDEPENDENTLY by
      the integration run and reported there - they are deliberately not
      hardcoded in this file.

# -----------------------------------------------------------------------------
# GR-PROXY DATA-CONFIDENCE CLASSIFICATION
# -----------------------------------------------------------------------------
# These classes describe CONFIDENCE IN THE DATA AND THE PROXY, never a rock
# type. A high-confidence class means "this well's GR-family curve is well
# covered, has usable dynamic range, and its proxy is stable across the three
# endpoint scenarios" - it does NOT mean the interval is any named lithology.
gr_proxy_confidence:
  classes:
    - "GR_PROXY_HIGH"
    - "GR_PROXY_INTERMEDIATE"
    - "GR_PROXY_LOW"
    - "GR_NOT_AVAILABLE"
    - "GR_EXCLUDED_UNRESOLVED_SCALE"
  rules:
    # A well is HIGH only if it clears every one of these; otherwise
    # INTERMEDIATE if it clears the LOW bar; otherwise LOW.
    high:
      min_valid_fraction: 0.90
      min_dynamic_range_api: 60.0
      max_proxy_median_spread_across_scenarios: 0.10
    intermediate:
      min_valid_fraction: 0.70
      min_dynamic_range_api: 40.0
      max_proxy_median_spread_across_scenarios: 0.20

# -----------------------------------------------------------------------------
# METHOD-ELIGIBILITY MASK DEFINITIONS
# -----------------------------------------------------------------------------
# Each mask answers "is this sample technically ADMISSIBLE as input to a later
# method?" - never "is this method valid here?" and never "is this
# interpretation correct?". Eligibility is a necessary, not a sufficient,
# condition, and no mask in this increment computes the method it gates.
method_eligibility:

  eligible_density_for_sv:
    purpose: >-
      Marks samples technically admissible as input to a LATER vertical-stress
      (Sv) integration. Increment 6 neither fills missing density nor computes
      Sv; a sample being eligible says nothing about whether an integration
      over it would be defensible, since a density log that starts at ~470 m
      TVDSS cannot by itself support an overburden integral from surface.
    requires_finite_rhob: true
    requires_rhob_within_physical_bounds: true
    requires_mapped_depth: true
    lithology_dependent: false
    applies_to_excluded_gr_wells: true

  eligible_dynamic_elastic:
    purpose: >-
      Marks samples technically admissible as input to a LATER dynamic-elastic
      calculation. Increment 6 computes NO elastic property - no Young's
      modulus, no Poisson ratio, no bulk or shear modulus. It only records
      where the three required inputs coexist and pass the configured
      non-negative-Poisson-ratio applicability screen. Excluded Vp/Vs values
      are diagnosed BY REGIME (non-positive bulk modulus; positive bulk
      modulus with negative Poisson ratio; above the configured plausibility
      maximum) and are never aggregated under a single "non-physical" label.
    requires_finite_positive_vp: true
    requires_finite_positive_vs: true
    requires_finite_rhob: true
    requires_vp_greater_than_vs: true
    # Increment 6.1: renamed from "requires_vp_vs_ratio_in_poisson_domain",
    # which misdescribed a conservative PROJECT POLICY as a statement about
    # the mathematical Poisson domain. See physical_bounds above.
    requires_vp_vs_ratio_passes_nonnegative_poisson_screen: true
    requires_mapped_depth: true
    lithology_dependent: false
    applies_to_excluded_gr_wells: true

  eligible_sonic_nct_candidate:
    purpose: >-
      Marks samples that are CANDIDATE DATA for a LATER sonic normal-compaction
      -trend analysis. This is a data-admissibility mask and nothing more. It
      does NOT fit a trend, does NOT select a donor interval, does NOT claim
      normal compaction, does NOT claim overpressure, and must never be read as
      evidence that any interval is normally compacted or is any named
      lithology.
    requires_approved_gr_disposition: true
    requires_finite_vp: true
    requires_finite_gr_proxy: true
    requires_mapped_depth: true
    requires_proxy_at_or_above_threshold: true
    lithology_dependent: true
    applies_to_excluded_gr_wells: false


#### `p2mem/wellframe_models.py` — typed well-frame structures and the prohibited-lithology guard

Defines `CurveSlot` (one canonical curve plus its full provenance and per-sample validity mask), `WellFrame` (one well's complete assembly), and `WellFrameAssemblyFailure` (a typed, isolated per-well failure carrying BOTH candidate source paths).

It also defines `PROHIBITED_LITHOLOGY_TERMS` and `assert_no_lithology_vocabulary` — the machine-checkable expression of this increment's hardest scientific boundary. The guard lives here, at the earliest layer, precisely because the well frame is the first place a rock name could leak into a persisted field.

In [ ]:
%%writefile p2mem/wellframe_models.py
"""
p2mem.wellframe_models - Typed, immutable data structures for the
Increment 6 well-frame layer.

Scope and intent
----------------
A "well frame" is a single, auditable, in-memory assembly of ONE well's
already-validated Increment 2.1.1 canonical LAS curve arrays alongside
the Increment 3.1.1 depth mapping (MD -> TVD / TVDSS) computed from that
well's LOCKED, explicitly selected survey trajectory basis. It is a
*view-and-annotate* layer, not a new ingestion layer:

* it NEVER re-parses a LAS or deviation file;
* it NEVER recomputes minimum curvature or replaces the locked
  `petrel_source_trace` basis;
* it NEVER overwrites, resamples, reorders, interpolates, gap-fills, or
  deletes a canonical curve array - every array in a `CurveSlot` is the
  locked loader's own array (or a read-only view of it), in the original
  file order, with the original sample count;
* it NEVER extrapolates MD -> TVD/TVDSS - a LAS sample outside the
  survey's own MD coverage is recorded as depth-unmapped (masked), never
  clamped, held, or linearly extended (see `p2mem.wellframe`).

Everything this layer adds is *additive metadata*: per-sample validity
masks, per-curve provenance (which physical source curve, from which
file, under which unit conversion), depth-mapping basis and coverage,
QC flags, and an evidence classification. This separation exists so that
a later increment can never confuse "the measured log" with "what this
project decided about the measured log".

Naming discipline (Increment 6 scientific boundary)
---------------------------------------------------
Nothing in this module - and nothing any Increment 6 module may write
into a field defined here - assigns a named lithology. `EVIDENCE_CLASSES`
and the QC-flag vocabulary describe *data and provenance confidence
only*. The prohibited-vocabulary guard `PROHIBITED_LITHOLOGY_TERMS` is
defined here (rather than in the petrophysics layer) precisely because
the well frame is the earliest place a rock-name could leak into a
persisted field, and it is enforced by tests over every generated
classification string.
"""

from __future__ import annotations

from dataclasses import dataclass, field
from typing import Dict, Optional, Tuple

import numpy as np

__all__ = [
    "EVIDENCE_CLASS_MEASURED",
    "EVIDENCE_CLASS_DERIVED_LOCKED",
    "EVIDENCE_CLASS_CORRELATION_DERIVED",
    "EVIDENCE_CLASS_ASSUMED",
    "EVIDENCE_CLASS_UNAVAILABLE",
    "VALID_EVIDENCE_CLASSES",
    "DEPTH_MAP_STATUS_FULL",
    "DEPTH_MAP_STATUS_PARTIAL",
    "DEPTH_MAP_STATUS_NONE",
    "VALID_DEPTH_MAP_STATUSES",
    "VALID_WELLFRAME_FAILURE_ORIGINS",
    "PROHIBITED_LITHOLOGY_TERMS",
    "CurveSlot",
    "WellFrame",
    "WellFrameAssemblyFailure",
    "assert_no_lithology_vocabulary",
    "SCOPE_LABEL",
    "SCOPE_INTERPRETIVE",
    "SCOPE_EXPLANATORY",
    "VALID_VALIDATION_SCOPES",
    "SCOPE_METHOD",
    "CONTROLLED_SCOPES",
    "ApprovedLabel",
    "APPROVED_LABELS",
    "APPROVED_LABEL_FIELD_KINDS",
    "approved_label",
    "RegisteredStatement",
    "REGISTERED_STATEMENTS",
    "RegisteredTemplate",
    "REGISTERED_TEMPLATES",
    "FIELD_TYPES",
    "Authorization",
    "find_prohibited_lithology_terms",
    "validate_no_prohibited_interpretation",
]

# ---------------------------------------------------------------------------
# Evidence classification vocabulary
# ---------------------------------------------------------------------------
# Deliberately describes the PROVENANCE of a quantity, never a geological
# interpretation of it. "measured" means a physically logged quantity that
# survived the locked per-file LAS contract; "derived_locked" means a
# quantity computed by an already-reviewed, LOCKED prior increment (the
# MD->TVD/TVDSS mapping); "correlation_derived" is reserved for a future
# increment's empirical transforms and is NOT produced anywhere in
# Increment 6; "assumed" marks an explicitly configured, human-authored
# value that no measurement supports; "unavailable" marks a factual data
# gap that must never be silently substituted.
EVIDENCE_CLASS_MEASURED = "measured"
EVIDENCE_CLASS_DERIVED_LOCKED = "derived_locked_prior_increment"
EVIDENCE_CLASS_CORRELATION_DERIVED = "correlation_derived"
EVIDENCE_CLASS_ASSUMED = "assumed_configured"
EVIDENCE_CLASS_UNAVAILABLE = "unavailable"

VALID_EVIDENCE_CLASSES: Tuple[str, ...] = (
    EVIDENCE_CLASS_MEASURED,
    EVIDENCE_CLASS_DERIVED_LOCKED,
    EVIDENCE_CLASS_CORRELATION_DERIVED,
    EVIDENCE_CLASS_ASSUMED,
    EVIDENCE_CLASS_UNAVAILABLE,
)

# ---------------------------------------------------------------------------
# Depth-mapping status vocabulary
# ---------------------------------------------------------------------------
# "full"    - every LAS sample lies inside the survey's own MD coverage and
#             received a TVD/TVDSS value from the locked mapper.
# "partial" - some samples lie outside survey MD coverage; those samples are
#             recorded as depth-unmapped (NaN + False in `depth_valid_mask`).
#             They are NEVER extrapolated, clamped, or held constant.
# "none"    - no sample could be mapped (e.g. no overlap at all).
DEPTH_MAP_STATUS_FULL = "fully_mapped_within_survey_coverage"
DEPTH_MAP_STATUS_PARTIAL = "partially_mapped_coverage_limited"
DEPTH_MAP_STATUS_NONE = "not_mapped_no_survey_coverage_overlap"

VALID_DEPTH_MAP_STATUSES: Tuple[str, ...] = (
    DEPTH_MAP_STATUS_FULL,
    DEPTH_MAP_STATUS_PARTIAL,
    DEPTH_MAP_STATUS_NONE,
)

# Which stage of assembly produced a failure. Mirrors the Increment 5.1
# `TopIngestionFailure.failure_origin` precedent so a caller can branch on
# the kind of failure without parsing message text.
VALID_WELLFRAME_FAILURE_ORIGINS: Tuple[str, ...] = (
    "las",
    "survey",
    "depth_mapping",
    "curve_assembly",
    "unknown",
)

# ---------------------------------------------------------------------------
# Prohibited named-lithology vocabulary (Increment 6 hard scientific bound)
# ---------------------------------------------------------------------------
# Increment 6 must not convert a gamma-ray response into a rock name. This
# tuple is the machine-checkable expression of that boundary: every
# classification string this increment generates is asserted against it by
# `assert_no_lithology_vocabulary` and by the test suite. It deliberately
# contains only ROCK/LITHOLOGY names - not words like "proxy", "screening",
# or "shale_proxy_*" field names, which are permitted precisely because they
# are self-labelling as non-lithological. The bare term "shale" IS listed:
# a class named "shale" would assert a rock type, whereas a FIELD named
# `VSH_GR_linear_proxy_frac` is a dimensionless proxy quantity and is
# checked separately (see `p2mem.petrophysics`).
PROHIBITED_LITHOLOGY_TERMS: Tuple[str, ...] = (
    "shale",
    "sand",
    "sandstone",
    "carbonate",
    "limestone",
    "dolomite",
    "marl",
    "claystone",
    "siltstone",
    "mudstone",
    "coal",
    "salt",
    "anhydrite",
    "evaporite",
    "reservoir_rock",
    "non_reservoir_rock",
    "reservoir rock",
    "non-reservoir rock",
    "pay",
    "net_pay",
    # Increment 6.1 (Finding 3): depositional-system and rock-class terms are
    # prohibited for the same reason rock names are. Calling a section
    # "clastic-dominated" is a lithological assertion about these wells, and a
    # trailing disclaimer does not undo it - the claim is in the noun, not in
    # the caveat. Gamma-ray response cannot establish a depositional system any
    # more than it can establish a rock name.
    "clastic",
    "clastics",
    "siliciclastic",
    "carbonates",
    "volcanic",
    "igneous",
    "metamorphic",
    "basement",
)


def assert_no_lithology_vocabulary(value: str, context: str) -> None:
    """
    Raise `ValueError` if `value` contains any prohibited named-lithology
    term as a standalone word.

    Matching is performed on a lowercased, non-alphanumeric-normalized
    token view of `value`, so `"GR_PROXY_HIGH"` passes while
    `"SHALE_HIGH"`, `"probable-sandstone"` and `"net pay"` are rejected.
    Substring matching alone is deliberately NOT used: it would reject the
    legitimate, explicitly self-labelling field name
    `VSH_GR_linear_proxy_frac` (which contains "sh"), and would reject
    "sand" inside "thousand". Multi-word prohibited terms are checked
    against the normalized whole string.

    This guard exists so a rock name can never reach a persisted
    classification field by accident. It is intentionally strict and is
    exercised directly by the Increment 6 test suite.
    """
    if not isinstance(value, str):
        raise TypeError(f"{context}: classification value must be a string, got {type(value).__name__!r}.")
    normalized = "".join(ch.lower() if ch.isalnum() else " " for ch in value)
    tokens = set(normalized.split())
    for term in PROHIBITED_LITHOLOGY_TERMS:
        term_norm = "".join(ch.lower() if ch.isalnum() else " " for ch in term).strip()
        parts = term_norm.split()
        if len(parts) == 1:
            if parts[0] in tokens:
                raise ValueError(
                    f"{context}: value {value!r} contains prohibited named-lithology term "
                    f"{term!r}. Increment 6 classifies DATA/PROXY CONFIDENCE only and must "
                    f"never assign a named lithology."
                )
        else:
            if f" {term_norm} " in f" {' '.join(normalized.split())} ":
                raise ValueError(
                    f"{context}: value {value!r} contains prohibited named-lithology term "
                    f"{term!r}. Increment 6 classifies DATA/PROXY CONFIDENCE only and must "
                    f"never assign a named lithology."
                )


@dataclass(frozen=True)
class CurveSlot:
    """
    One canonical curve inside a well frame, with its full provenance and
    a per-sample validity mask.

    `values` is the LOCKED Increment 2.1.1 loader's own canonical array
    for this curve (NULL-sentinel already substituted to NaN, and the
    contract's exact unit conversion already applied by that locked
    loader). This layer does not copy-and-modify it: an invalid sample is
    expressed in `valid_mask`, never by deleting, replacing, or
    interpolating the value itself. `values.size` therefore always equals
    the well frame's `n_samples`, in the original file order.

    `source_curve_name` / `raw_mnemonic` / `raw_unit` preserve the
    physical identity of the column as it appeared in THIS file. Two wells
    whose contracts happen to produce the same `canonical_name` (for
    example Poseidon 2's `GR` and Proteus 1ST2's `GR`, both canonicalized
    to `GR_api`) are still distinct measurements from distinct tools in
    distinct wells; nothing in this project may treat them as
    interchangeable on the strength of a shared canonical name alone.

    `valid_mask` is True exactly where `values` is finite. It is a
    read-only boolean array (`writeable=False`) so that a downstream
    consumer cannot mutate one well frame's mask and silently affect
    another consumer holding the same object.
    """

    canonical_name: str
    source_curve_name: str
    raw_mnemonic: str
    raw_unit: str
    canonical_unit: str
    conversion_function: str
    source_filename: str
    evidence_class: str
    values: np.ndarray
    valid_mask: np.ndarray
    n_samples: int
    valid_count: int
    valid_fraction: float

    def __post_init__(self) -> None:
        if self.evidence_class not in VALID_EVIDENCE_CLASSES:
            raise ValueError(
                f"CurveSlot {self.canonical_name!r}: evidence_class {self.evidence_class!r} is not "
                f"one of {VALID_EVIDENCE_CLASSES}."
            )


@dataclass(frozen=True)
class WellFrame:
    """
    One well's complete, auditable Increment 6 assembly.

    Depth arrays
    ------------
    `MD_m` is the LOCKED loader's own canonical LAS MD array, unmodified.
    `TVD_m` / `TVDSS_m` are full-length arrays aligned sample-for-sample
    with `MD_m`; where a sample lies outside the survey's own MD coverage
    both are NaN and `depth_valid_mask` is False. `n_depth_unmapped`
    counts exactly those samples. `n_extrapolated` is always 0 by
    construction and is reported as a confirming, self-describing field
    (this layer has no code path that extrapolates).

    Formation tops
    --------------
    A well frame carries NO formation-top data. Increment 6 consumes tops
    only through the LOCKED Increment 5 survey-corrected output tables,
    and only for display/annotation - never by re-parsing, transferring
    between wells, or re-deriving them.
    """

    well_key: str
    source_las_filename: str
    source_survey_filename: str
    n_samples: int

    MD_m: np.ndarray
    TVD_m: np.ndarray
    TVDSS_m: np.ndarray
    depth_valid_mask: np.ndarray

    depth_basis_used: str
    interpolation_method: str
    datum_elevation_m: float
    depth_map_status: str
    survey_md_min_m: float
    survey_md_max_m: float
    las_md_min_m: float
    las_md_max_m: float
    n_depth_unmapped: int
    n_extrapolated: int

    curves: Dict[str, CurveSlot] = field(default_factory=dict)
    gr_family_canonical_name: Optional[str] = None
    gr_family_source_curve_name: Optional[str] = None
    well_identity_evidence_status: str = "inferred_unverified"
    qc_flags: Tuple[str, ...] = field(default_factory=tuple)

    def __post_init__(self) -> None:
        if self.depth_map_status not in VALID_DEPTH_MAP_STATUSES:
            raise ValueError(
                f"WellFrame {self.well_key!r}: depth_map_status {self.depth_map_status!r} is not "
                f"one of {VALID_DEPTH_MAP_STATUSES}."
            )

    def curve(self, canonical_name: str) -> Optional[CurveSlot]:
        """Return the named `CurveSlot`, or None if this well has no such
        contract-resolved curve. Never raises for a missing curve: a curve
        a well genuinely does not have is a factual data gap, not an
        error, and callers must be able to branch on it."""
        return self.curves.get(canonical_name)

    def values_or_none(self, canonical_name: str) -> Optional[np.ndarray]:
        """Return the canonical array for `canonical_name`, or None."""
        slot = self.curves.get(canonical_name)
        return None if slot is None else slot.values


@dataclass(frozen=True)
class WellFrameAssemblyFailure:
    """
    A typed record of one well's failed frame assembly, isolated so that
    one well's failure never stops the others (mirrors the LOCKED
    `IngestionFailure` / `DeviationIngestionFailure` / `TopIngestionFailure`
    precedents).

    `failure_origin` names the stage that actually failed, and BOTH
    candidate source paths are always retained, so an exporter can
    sanitize whichever path a message happens to contain rather than
    assuming a single one (the Increment 5.1 Finding-2 lesson, applied
    here from the start rather than retrofitted).
    """

    well_key: str
    failure_origin: str
    error_type: str
    message: str
    las_path: Optional[str] = None
    survey_path: Optional[str] = None
    exception: Optional[BaseException] = None

    def __post_init__(self) -> None:
        if self.failure_origin not in VALID_WELLFRAME_FAILURE_ORIGINS:
            raise ValueError(
                f"WellFrameAssemblyFailure {self.well_key!r}: failure_origin "
                f"{self.failure_origin!r} is not one of {VALID_WELLFRAME_FAILURE_ORIGINS}."
            )


# ---------------------------------------------------------------------------
# Increment 6.1.5: POSITIVE AUTHORIZATION for every persisted controlled field
# ---------------------------------------------------------------------------
# History, stated plainly because it is the justification for this design.
#
# Increments 6.1 through 6.1.4 all asked the same question in different ways:
# "does this text contain a geological assertion?" 6.1.1 answered it with
# sentence-wide cue lists, 6.1.2 with per-occurrence grammar, 6.1.3 with a
# prohibited-term scan, 6.1.4 with a prohibited-term scan plus a registry for
# explanatory text. Every one of them was defeated, because all of them shared
# a single structural assumption: that content is ACCEPTABLE BY DEFAULT and
# becomes unacceptable only when a recognizer fires.
#
# That assumption fails against a word the recognizer has never seen. In the
# Increment 6.1.4 package, all of the following passed label, interpretive and
# explanatory scope, because none of these words was in the blacklist:
#
#     "The interval is chalk."          "The interval is chert."
#     "The interval is halite."         "The interval is tuff."
#     "The interval is gypsum."         "The interval is basalt."
#     "The interval is conglomerate."   "The interval is dolostone."
#     "The interval is lignite."        "The interval is calcareous."
#
# and so did "The interval is qxzite." - a word that does not exist. A longer
# blacklist would have caught the first ten and still missed the eleventh, so
# lengthening it is not a completion criterion and is not the mechanism this
# module's assurance claim rests on.
#
# THE MODEL IS INVERTED. Nothing is acceptable by default. Every persisted
# controlled field must be POSITIVELY AUTHORIZED:
#
#   SCOPE_LABEL        the value must be a member of APPROVED_LABELS - a
#                      registry of typed, enumerated label values. Arbitrary
#                      caller text is rejected even when it contains no
#                      recognizable rock name at all.
#   SCOPE_INTERPRETIVE the text must resolve to a registered statement_id, or
#                      to a reviewed template_id whose substitutions are
#                      strictly typed. Unregistered free text is rejected
#                      unconditionally.
#   SCOPE_METHOD       the text must resolve to a registered statement_id, and
#                      BOTH the id and the exact rendered text are validated.
#   SCOPE_EXPLANATORY  the same, for every non-empty statement, whether or not
#                      any prohibited term is detected.
#
# There is NO path through this module that accepts arbitrary text because a
# recognizer failed to fire. `PROHIBITED_LITHOLOGY_TERMS` survives ONLY as a
# supplementary diagnostic linter: it annotates violations with any rock words
# it happens to recognize, it never authorizes anything, and it must never be
# cited as evidence that all named lithologies have been detected.
#
# WHAT THIS DOES AND DOES NOT CLAIM
#
#   It DOES claim: every persisted project-specific classification and
#   interpretive statement is generated from an approved typed value, a
#   controlled template, or a registered statement. Arbitrary free text cannot
#   enter these controlled fields.
#
#   It does NOT claim: that this software understands, recognizes, or
#   exhaustively enumerates natural-language lithology. It cannot, and no
#   version of it ever did. Free-form notebook narrative and documentation are
#   OUTSIDE these controlled fields and remain subject to ordinary manual
#   scientific review.

SCOPE_LABEL = "label"
SCOPE_INTERPRETIVE = "interpretive"
SCOPE_METHOD = "method"
SCOPE_EXPLANATORY = "explanatory"
VALID_VALIDATION_SCOPES: Tuple[str, ...] = (
    SCOPE_LABEL, SCOPE_INTERPRETIVE, SCOPE_METHOD, SCOPE_EXPLANATORY,
)

#: Scopes in which nothing is accepted without positive authorization.
CONTROLLED_SCOPES: Tuple[str, ...] = VALID_VALIDATION_SCOPES


class ApprovedLabel:
    """One authorized label value, with the field it belongs to and why it exists.

    A label is a verdict a well is stamped with. It is not prose, so it is
    enumerated rather than described: a value either is in this registry or it
    is not persisted.
    """

    __slots__ = ("value", "field_kind", "purpose", "provenance")

    def __init__(self, value, field_kind, purpose, provenance):
        self.value = value
        self.field_kind = field_kind
        self.purpose = purpose
        self.provenance = provenance

    def __repr__(self):  # pragma: no cover - diagnostic only
        return f"ApprovedLabel({self.value!r}, {self.field_kind!r})"


class RegisteredStatement:
    """One authorized exact statement, bound to a single scope.

    `text` is the canonical persisted decision. Authorization is by
    `statement_id`, and the id and the rendered text are validated together, so
    neither a renamed id nor an edited sentence can pass on the strength of the
    other.
    """

    __slots__ = ("statement_id", "scope", "text", "purpose", "provenance")

    def __init__(self, statement_id, scope, text, purpose, provenance):
        self.statement_id = statement_id
        self.scope = scope
        self.text = text
        self.purpose = purpose
        self.provenance = provenance

    def __repr__(self):  # pragma: no cover - diagnostic only
        return f"RegisteredStatement({self.statement_id!r}, {self.scope!r})"


class RegisteredTemplate:
    """One authorized statement template with strictly typed substitutions.

    Templates exist for statements that must carry MEASURED NUMBERS - a
    per-well confidence rationale, for example. The controlled part is the
    prose; the free part is restricted to values that satisfy a declared type,
    so a template can never become a channel for arbitrary text.

    `fields` maps each substitution name to a type name in `FIELD_TYPES`.
    """

    __slots__ = ("template_id", "scope", "template", "fields", "purpose", "provenance")

    def __init__(self, template_id, scope, template, fields, purpose, provenance):
        self.template_id = template_id
        self.scope = scope
        self.template = template
        self.fields = dict(fields)
        self.purpose = purpose
        self.provenance = provenance

    def render(self, values):
        return self.template.format(**values)

    def __repr__(self):  # pragma: no cover - diagnostic only
        return f"RegisteredTemplate({self.template_id!r}, {self.scope!r})"


def _is_decimal_literal(value) -> bool:
    """A finite decimal number written out, e.g. "0.8131" or "135.761".

    Deliberately a TYPE test, not a vocabulary test: it admits any number and
    no words at all, so a template substitution cannot smuggle prose.
    """
    if isinstance(value, bool) or not isinstance(value, str) or not value:
        return False
    body = value[1:] if value[0] in "+-" else value
    if not body or body.count(".") > 1:
        return False
    return all(ch.isdigit() or ch == "." for ch in body) and any(ch.isdigit() for ch in body)


#: Declared substitution types. A field type is a predicate over the SUBSTITUTED
#: STRING - never a judgement about its meaning.
FIELD_TYPES = {
    "decimal": _is_decimal_literal,
}


class Authorization:
    """The authorization a caller presents for one persisted field.

    `statement_id` cites a registered statement; `template_id` plus `fields`
    cites a registered template and its substitutions. Supplying neither means
    the caller has no authorization, and the field is rejected.
    """

    __slots__ = ("statement_id", "template_id", "fields", "field_kind")

    def __init__(self, statement_id=None, template_id=None, fields=None,
                 field_kind=None):
        self.statement_id = statement_id
        self.template_id = template_id
        self.fields = dict(fields or {})
        self.field_kind = field_kind

    def __repr__(self):  # pragma: no cover - diagnostic only
        return f"Authorization({self.statement_id or self.template_id!r})"


_APPROVED_LABEL_LIST: Tuple["ApprovedLabel", ...] = (
    ApprovedLabel(
        value="BOREAS_ECGR_SCALE_UNRESOLVED",
        field_kind="exclusion_reason",
        purpose="Machine-readable reason Boreas 1 is excluded from every GR-derived calculation.",
        provenance="Enumerated from the actual persisted Increment 6 export; every label this project writes is a member of this registry."),
    ApprovedLabel(
        value="ECGR",
        field_kind="gr_family_source_curve_name",
        purpose="Source curve mnemonic as recorded in the approved LAS file.",
        provenance="Enumerated from the actual persisted Increment 6 export; every label this project writes is a member of this registry."),
    ApprovedLabel(
        value="ECGR_api",
        field_kind="gr_family_canonical_name",
        purpose="Canonical GR-family curve identity; never treated as interchangeable.",
        provenance="Enumerated from the actual persisted Increment 6 export; every label this project writes is a member of this registry."),
    ApprovedLabel(
        value="GR",
        field_kind="gr_family_source_curve_name",
        purpose="Source curve mnemonic as recorded in the approved LAS file.",
        provenance="Enumerated from the actual persisted Increment 6 export; every label this project writes is a member of this registry."),
    ApprovedLabel(
        value="GRD",
        field_kind="gr_family_source_curve_name",
        purpose="Source curve mnemonic as recorded in the approved LAS file.",
        provenance="Enumerated from the actual persisted Increment 6 export; every label this project writes is a member of this registry."),
    ApprovedLabel(
        value="GRD_api",
        field_kind="gr_family_canonical_name",
        purpose="Canonical GR-family curve identity; never treated as interchangeable.",
        provenance="Enumerated from the actual persisted Increment 6 export; every label this project writes is a member of this registry."),
    ApprovedLabel(
        value="GR_EXCLUDED_UNRESOLVED_SCALE",
        field_kind="gr_proxy_confidence_class",
        purpose="Confidence class assigned to a well excluded for an unresolved GR scale anomaly.",
        provenance="Enumerated from the actual persisted Increment 6 export; every label this project writes is a member of this registry."),
    ApprovedLabel(
        value="GR_PROXY_HIGH",
        field_kind="gr_proxy_confidence_class",
        purpose="Confidence class describing DATA and PROXY confidence only.",
        provenance="Enumerated from the actual persisted Increment 6 export; every label this project writes is a member of this registry."),
    ApprovedLabel(
        value="GR_PROXY_INTERMEDIATE",
        field_kind="gr_proxy_confidence_class",
        purpose="Confidence class describing DATA and PROXY confidence only.",
        provenance="Enumerated from the actual persisted Increment 6 export; every label this project writes is a member of this registry."),
    ApprovedLabel(
        value="GR_api",
        field_kind="gr_family_canonical_name",
        purpose="Canonical GR-family curve identity; never treated as interchangeable.",
        provenance="Enumerated from the actual persisted Increment 6 export; every label this project writes is a member of this registry."),
    ApprovedLabel(
        value="eligible_density_for_sv",
        field_kind="mask_name",
        purpose="Name of the density input-admissibility mask.",
        provenance="Enumerated from the actual persisted Increment 6 export; every label this project writes is a member of this registry."),
    ApprovedLabel(
        value="eligible_dynamic_elastic",
        field_kind="mask_name",
        purpose="Name of the dynamic-elastic input-admissibility mask.",
        provenance="Enumerated from the actual persisted Increment 6 export; every label this project writes is a member of this registry."),
    ApprovedLabel(
        value="eligible_sonic_nct_candidate",
        field_kind="mask_name",
        purpose="Name of the sonic NCT-candidate input-admissibility mask.",
        provenance="Enumerated from the actual persisted Increment 6 export; every label this project writes is a member of this registry."),
    ApprovedLabel(
        value="assumed_configured",
        field_kind="evidence_class",
        purpose="Provenance class of a configured endpoint scenario; no code path can promote it.",
        provenance="Enumerated from the ACTUAL emitted gr_endpoint_scenarios.csv record."),
    ApprovedLabel(
        value="correlation_derived_screening_proxy_uncalibrated",
        field_kind="evidence_class",
        purpose="Provenance class of the uncalibrated screening proxy.",
        provenance="Enumerated from the ACTUAL emitted gr_proxy_sensitivity_summary.csv record."),
    ApprovedLabel(
        value="measured",
        field_kind="evidence_class",
        purpose="Provenance class: a physically logged quantity that passed the locked LAS contract.",
        provenance="Enumerated from the actual persisted Increment 6 export; every label this project writes is a member of this registry."),
    ApprovedLabel(
        value="qc_only_excluded",
        field_kind="use_status",
        purpose="Configured disposition: factual QC display and availability reporting only.",
        provenance="Enumerated from the actual persisted Increment 6 export; every label this project writes is a member of this registry."),
    ApprovedLabel(
        value="screening_proxy_allowed",
        field_kind="use_status",
        purpose="Configured disposition: screening proxy permitted, well has approved tops.",
        provenance="Enumerated from the actual persisted Increment 6 export; every label this project writes is a member of this registry."),
    ApprovedLabel(
        value="screening_proxy_allowed_depth_tied",
        field_kind="use_status",
        purpose="Configured disposition: screening proxy permitted but results are depth-tied.",
        provenance="Enumerated from the actual persisted Increment 6 export; every label this project writes is a member of this registry."),
)

#: Increment 6.1.6 (Finding 3): authorization requires field_kind AND value.
#: Increment 6.1.5 keyed this registry by VALUE alone, so `ApprovedLabel.field_kind`
#: existed but was never enforced - `use_status="GR"` passed because `GR` was
#: approved somewhere, as a curve mnemonic. The mapping is now two-level and the
#: field kind is supplied by the caller, never parsed out of a context string.
APPROVED_LABELS: Dict[str, Dict[str, "ApprovedLabel"]] = {}
for _lab in _APPROVED_LABEL_LIST:
    _bucket = APPROVED_LABELS.setdefault(_lab.field_kind, {})
    if _lab.value in _bucket:  # pragma: no cover - fails at import if violated
        raise ValueError(
            f"duplicate approved label ({_lab.field_kind!r}, {_lab.value!r})")
    _bucket[_lab.value] = _lab
del _lab, _bucket

#: Every declared label field kind. An unknown field kind fails closed.
APPROVED_LABEL_FIELD_KINDS: Tuple[str, ...] = tuple(sorted(APPROVED_LABELS))


def approved_label(field_kind, value):
    """Return the ApprovedLabel for (field_kind, value), or None.

    Both parts are required. A value approved under one field kind does not
    authorize it under another.
    """
    return APPROVED_LABELS.get(field_kind, {}).get(value)


# Every exact statement this project persists, enumerated from the ACTUAL
# Increment 6 export rather than described. Authorization is by id.
_REGISTERED_STATEMENT_LIST: Tuple["RegisteredStatement", ...] = (
    RegisteredStatement(
        statement_id="interpretive_01_candidate_data_only_no_nct_fitted_",
        scope=SCOPE_INTERPRETIVE,
        text="CANDIDATE DATA ONLY - no NCT fitted. This mask does not fit a trend, does not select a donor interval, does not claim normal compaction, and does not claim overpressure. It is not proof that any interval is normally compacted, and it assigns no lithology.",
        purpose="Persisted project-specific interpretive statement.",
        provenance="Human-authored and reviewed; enumerated from the actual persisted Increment 6 content."),
    RegisteredStatement(
        statement_id="interpretive_02_eligibility_is_a_necessary_not_suf",
        scope=SCOPE_INTERPRETIVE,
        text="Eligibility is a necessary, not sufficient, condition. Increment 6 does not fill missing density and does not compute vertical stress. A log that begins well below the seabed cannot support an overburden integral from surface regardless of how many of its own samples are eligible.",
        purpose="Persisted project-specific interpretive statement.",
        provenance="Human-authored and reviewed; enumerated from the actual persisted Increment 6 content."),
    RegisteredStatement(
        statement_id="interpretive_03_formal_exclusion_from_every_lithol",
        scope=SCOPE_INTERPRETIVE,
        text="FORMAL EXCLUSION from every lithology-dependent, GR-normalized, screening-proxy (VSH_GR_linear_proxy_frac) and NCT-candidate calculation. The ECGR curve carries an unresolved scale/acquisition anomaly: its central tendency is roughly an order of magnitude below what an API-scaled gamma-ray log over a comparable section would show, its range extends across several hundred API, it includes values at or below zero, and it contains samples above the well's own declared seabed marker. No independent tool header, calibration record, or environmental-correction metadata is available to adjudicate the cause. The curve is therefore EXCLUDED, not corrected: applying a shift, gain, or normalization would fabricate a calibration that does not exist, and would silently propagate into every downstream screening proxy and NCT candidate interval. Boreas 1 remains available for factual raw-GR QC display and availability reporting only. The specific numeric values supporting this disposition are MEASURED INDEPENDENTLY by the integration run and reported there - they are deliberately not hardcoded in this file.",
        purpose="Persisted project-specific interpretive statement.",
        provenance="Human-authored and reviewed; enumerated from the actual persisted Increment 6 content."),
    RegisteredStatement(
        statement_id="interpretive_04_formally_excluded_from_every_gr_de",
        scope=SCOPE_INTERPRETIVE,
        text="Formally excluded from every GR-derived calculation (exclusion_reason='BOREAS_ECGR_SCALE_UNRESOLVED'). The curve remains available for factual raw QC display and availability reporting only. Good coverage does not resolve an unresolved scale/acquisition anomaly, so coverage statistics do not override this classification.",
        purpose="Persisted project-specific interpretive statement.",
        provenance="Human-authored and reviewed; enumerated from the actual persisted Increment 6 content."),
    RegisteredStatement(
        statement_id="interpretive_05_grd_is_a_differently_named_gr_fami",
        scope=SCOPE_INTERPRETIVE,
        text="GRD is a differently named GR-family curve and is NEVER treated as interchangeable with GR or ECGR. This well has NO approved formation tops, so every result for it is depth-tied and stratigraphically unvalidated, and any future use as an NCT donor would remain so. That limitation is recorded here, in the machine-readable use_status, not only in prose.",
        purpose="Persisted project-specific interpretive statement.",
        provenance="Human-authored and reviewed; enumerated from the actual persisted Increment 6 content."),
    RegisteredStatement(
        statement_id="interpretive_06_input_admissibility_only_increment",
        scope=SCOPE_INTERPRETIVE,
        text="Input-admissibility only. Increment 6 computes no Young's modulus, Poisson ratio, bulk modulus, or shear modulus. The Vp/Vs condition is a CONFIGURED NON-NEGATIVE-POISSON-RATIO APPLICABILITY SCREEN (inclusive at Vp/Vs = sqrt(2), where nu = 0 exactly), not a test of physical possibility. Excluded ratios are diagnosed by regime: non-positive bulk modulus (Vp/Vs <= sqrt(4/3), genuinely outside the isotropic elastic model); positive bulk modulus with negative Poisson ratio (sqrt(4/3) < Vp/Vs < sqrt(2), unusual and outside this project's conservative policy, but NOT non-physical); and above the configured plausibility maximum (Vp/Vs > 4, a project credibility limit, not a Poisson-domain boundary). These are never aggregated into a single 'non-physical' count. No excluded sample is deleted from the well frame or corrected.",
        purpose="Persisted project-specific interpretive statement.",
        provenance="Human-authored and reviewed; enumerated from the actual persisted Increment 6 content."),
    RegisteredStatement(
        statement_id="interpretive_07_marks_samples_technically_admissib",
        scope=SCOPE_INTERPRETIVE,
        text="Marks samples technically admissible as input to a LATER dynamic-elastic calculation. Increment 6 computes NO elastic property - no Young's modulus, no Poisson ratio, no bulk or shear modulus. It only records where the three required inputs coexist and pass the configured non-negative-Poisson-ratio applicability screen. Excluded Vp/Vs values are diagnosed BY REGIME (non-positive bulk modulus; positive bulk modulus with negative Poisson ratio; above the configured plausibility maximum) and are never aggregated under a single \"non-physical\" label.",
        purpose="Persisted project-specific interpretive statement.",
        provenance="Human-authored and reviewed; enumerated from the actual persisted Increment 6 content."),
    RegisteredStatement(
        statement_id="interpretive_08_marks_samples_technically_admissib",
        scope=SCOPE_INTERPRETIVE,
        text="Marks samples technically admissible as input to a LATER vertical-stress (Sv) integration. Increment 6 neither fills missing density nor computes Sv; a sample being eligible says nothing about whether an integration over it would be defensible, since a density log that starts at ~470 m TVDSS cannot by itself support an overburden integral from surface.",
        purpose="Persisted project-specific interpretive statement.",
        provenance="Human-authored and reviewed; enumerated from the actual persisted Increment 6 content."),
    RegisteredStatement(
        statement_id="interpretive_09_marks_samples_that_are_candidate_d",
        scope=SCOPE_INTERPRETIVE,
        text="Marks samples that are CANDIDATE DATA for a LATER sonic normal-compaction -trend analysis. This is a data-admissibility mask and nothing more. It does NOT fit a trend, does NOT select a donor interval, does NOT claim normal compaction, does NOT claim overpressure, and must never be read as evidence that any interval is normally compacted or is any named lithology.",
        purpose="Persisted project-specific interpretive statement.",
        provenance="Human-authored and reviewed; enumerated from the actual persisted Increment 6 content."),
    RegisteredStatement(
        statement_id="interpretive_10_the_recorded_values_span_a_range_a",
        scope=SCOPE_INTERPRETIVE,
        text="The recorded values span a range and central tendency typical of conventionally API-scaled gamma-ray logs. This is a statement about the NUMERIC distribution and the scaling convention only; no depositional setting, rock type, or lithology is asserted or implied. This well has approved, survey-corrected Increment 5 formation tops, so its results can be annotated against locked stratigraphic markers.",
        purpose="Persisted project-specific interpretive statement.",
        provenance="Human-authored and reviewed; enumerated from the actual persisted Increment 6 content."),
    RegisteredStatement(
        statement_id="interpretive_11_this_well_s_gr_canonicalizes_to_th",
        scope=SCOPE_INTERPRETIVE,
        text="This well's GR canonicalizes to the same name as Poseidon 2's (GR_api), but it is a different tool run in a different well with no cross-well calibration tie. Endpoints are estimated from this well's own samples only. No approved formation tops exist, so results are explicitly depth-tied and unvalidated.",
        purpose="Persisted project-specific interpretive statement.",
        provenance="Human-authored and reviewed; enumerated from the actual persisted Increment 6 content."),
    RegisteredStatement(
        statement_id="proxy_limitations",
        scope=SCOPE_METHOD,
        text="IGR and the linear screening proxy are dimensionless quantities derived under ASSUMED endpoints. The proxy is NOT a calibrated shale volume and NOT a lithology. Clipped and unclipped indices are computed and retained together in memory; the clipped counts here quantify how far the real data fell outside the assumed endpoint bracket. Data/proxy confidence only - asserts no named lithology and no calibration.",
        purpose="States what VSH_GR_linear_proxy_frac is NOT; persisted as the `limitations` field of every proxy sensitivity row.",
        provenance="Enumerated from the actual persisted Increment 6 export; the project cannot state its own scientific boundary without it."),
    RegisteredStatement(
        statement_id="proxy_calibration_status",
        scope=SCOPE_METHOD,
        text="screening_proxy_uncalibrated_not_a_shale_volume",
        purpose="Self-labelling `calibration_status` value carrying the denial inside the field itself.",
        provenance="Enumerated from the actual persisted Increment 6 export; the project cannot state its own scientific boundary without it."),
    RegisteredStatement(
        statement_id="shale_proxy_transform_policy_key",
        scope=SCOPE_METHOD,
        text="shale_proxy_transform",
        purpose="Required configuration key naming the single permitted transform.",
        provenance="Enumerated from the actual persisted Increment 6 export; the project cannot state its own scientific boundary without it."),
    RegisteredStatement(
        statement_id="increment_title",
        scope=SCOPE_METHOD,
        text="Gamma-Ray QC, Shale-Proxy Sensitivity, Well-Frame Assembly, and Method-Eligibility Framework",
        purpose="The increment title persisted in the manifest.",
        provenance="Enumerated from the actual persisted Increment 6 export; the project cannot state its own scientific boundary without it."),
    RegisteredStatement(
        statement_id="proxy_transform_name",
        scope=SCOPE_METHOD,
        text="linear_identity_of_clipped_igr",
        purpose="The `transform_name` of GrProxyResult.",
        provenance="Enumerated from the actual persisted Increment 6 export; the project cannot state its own scientific boundary without it."),
    RegisteredStatement(
        statement_id="named_lithology_statement",
        scope=SCOPE_EXPLANATORY,
        text="NO named lithology is assigned anywhere in Increment 6. Gamma-ray response is not uniquely diagnostic of rock type, and no independent lithological evidence (core, cuttings description, image log, spectral GR, or calibrated multi-mineral solution) is available in this project. Low-GR intervals occur independently in each of the three GR-eligible wells; cross-well stratigraphic persistence is NOT established, and cannot be, because two of those wells have no approved formation tops. Those intervals therefore remain UNRESOLVED in lithology. All classifications in this increment describe DATA AND PROXY CONFIDENCE ONLY.",
        purpose="The manifest's own statement that no named lithology is assigned anywhere.",
        provenance="Human-authored and reviewed; the single explanatory field this project persists."),
)

REGISTERED_STATEMENTS = {r.statement_id: r for r in _REGISTERED_STATEMENT_LIST}


_REGISTERED_TEMPLATE_LIST: Tuple["RegisteredTemplate", ...] = (
    RegisteredTemplate(
        template_id="gr_proxy_confidence_rationale",
        scope=SCOPE_INTERPRETIVE,
        template="valid_fraction={valid_fraction}, dynamic_range_p05_p95={dynamic_range_p05_p95} API, proxy_median_spread_across_3_scenarios={proxy_median_spread}. Describes confidence in the DATA and the SCREENING PROXY only; asserts no lithology and no calibration.",
        fields={
            "valid_fraction": "decimal",
            "dynamic_range_p05_p95": "decimal",
            "proxy_median_spread": "decimal",
        },
        purpose="Per-well confidence rationale. The prose is fixed; only the three MEASURED numbers vary.",
        provenance="Enumerated from the actual persisted Increment 6 export. Substitutions are restricted to decimal literals, so the template cannot carry prose."),
)

REGISTERED_TEMPLATES = {t.template_id: t for t in _REGISTERED_TEMPLATE_LIST}


def _duplicate_registry_ids():
    """Ids must be unique across statements and templates; a duplicate would
    make authorization ambiguous."""
    ids = [r.statement_id for r in _REGISTERED_STATEMENT_LIST] + \
          [t.template_id for t in _REGISTERED_TEMPLATE_LIST]
    return sorted({i for i in ids if ids.count(i) > 1})


if _duplicate_registry_ids():  # pragma: no cover - fails at import if violated
    raise ValueError(f"duplicate registry ids: {_duplicate_registry_ids()}")


def _normalize(text: str) -> str:
    return " ".join(
        "".join(ch.lower() if ch.isalnum() else " " for ch in text).split()
    )


def _normalize_tokens(text: str) -> set:
    return set(_normalize(text).split())


def find_prohibited_lithology_terms(text: str) -> Tuple[str, ...]:
    """SUPPLEMENTARY DIAGNOSTIC LINTER ONLY.

    Returns any prohibited term this module happens to recognize in `text`.
    Increment 6.1.5: this function AUTHORIZES NOTHING. It is used only to
    annotate a violation with whatever rock words it can name, so a reader
    sees why a rejected string looked suspicious. Its vocabulary is a curated
    list of 28 terms and is NOT exhaustive - `chalk`, `chert` and `qxzite` are
    all absent from it - which is precisely why nothing is permitted on the
    strength of it returning empty.
"""
    if not isinstance(text, str) or not text:
        return ()
    normalized = _normalize(text)
    tokens = set(normalized.split())
    found = []
    for term in PROHIBITED_LITHOLOGY_TERMS:
        term_norm = _normalize(term)
        parts = term_norm.split()
        if len(parts) == 1:
            if parts[0] in tokens:
                found.append(term)
        elif f" {term_norm} " in f" {normalized} ":
            found.append(term)
    return tuple(found)


def _violation(context, scope, text, reason):
    """Build a violation record. `terms` is DIAGNOSTIC ONLY - it reports the
    rock words this module happens to recognize, and is frequently empty for a
    genuine violation (an unregistered statement, an unknown lithology). It is
    never the reason for the rejection."""
    return {
        "context": context,
        "scope": scope,
        "terms": sorted(set(find_prohibited_lithology_terms(text))),
        "reason": reason,
    }


def validate_no_prohibited_interpretation(scope_entries) -> Tuple[dict, ...]:
    """
    Validate that every persisted controlled field is POSITIVELY AUTHORIZED.

    `scope_entries` is an iterable of `(context, text, scope)` or
    `(context, text, scope, authorization)`. An entry with no `Authorization`
    presents no credential, and is rejected in every controlled scope - which
    is all four of them.

    Returns a tuple of violation dicts (empty when clean). Nothing is raised:
    the caller decides what a violation means, and the manifest and completion
    gate DERIVE their result from the returned list rather than assuming it.

      LABEL        the value must be a member of `APPROVED_LABELS`.
      INTERPRETIVE the text must equal a registered statement, or a registered
                   template rendered with strictly typed substitutions.
      METHOD       the text must equal the registered statement its
                   `statement_id` names - id and text validated together.
      EXPLANATORY  identical to METHOD, applied to every non-empty statement
                   whether or not any prohibited term is detected.

    There is no branch here that accepts text because a prohibited-term scan
    came back empty. An unknown lithology, and an invented word, are rejected
    for exactly the same reason as an innocent unregistered sentence: no
    authorization was presented.
    """
    violations = []
    for entry in scope_entries:
        entry = tuple(entry)
        if len(entry) == 4:
            context, text, scope, auth = entry
        elif len(entry) == 3:
            (context, text, scope), auth = entry, None
        else:
            raise ValueError(
                f"scope entry must be (context, text, scope[, authorization]); got "
                f"{len(entry)} elements: {entry!r}"
            )
        if scope not in VALID_VALIDATION_SCOPES:
            raise ValueError(
                f"{context}: unknown validation scope {scope!r}; expected one of "
                f"{VALID_VALIDATION_SCOPES}."
            )
        if not isinstance(text, str) or not text:
            # An absent optional field carries no assertion and persists no
            # claim. Every NON-EMPTY field below requires authorization.
            continue

        # ------------------------------------------------------------------
        # LABEL - typed, enumerated values only.
        # ------------------------------------------------------------------
        if scope == SCOPE_LABEL:
            field_kind = getattr(auth, "field_kind", None) if auth is not None else None
            if field_kind is None:
                violations.append(_violation(
                    context, scope, text,
                    "No field_kind presented for a persisted label. Increment 6.1.6: a "
                    "label is authorized by (field_kind, value) together, and the field "
                    "kind must be supplied by the caller - it is never parsed out of a "
                    "human-readable context string."))
                continue
            if field_kind not in APPROVED_LABELS:
                violations.append(_violation(
                    context, scope, text,
                    f"Unknown label field_kind {field_kind!r}; expected one of "
                    f"{APPROVED_LABEL_FIELD_KINDS}."))
                continue
            if approved_label(field_kind, text) is None:
                other = sorted(k for k in APPROVED_LABELS if text in APPROVED_LABELS[k])
                extra = (f" It IS approved as {other} - a value approved under one field "
                         f"kind does not authorize it under another." if other else "")
                violations.append(_violation(
                    context, scope, text,
                    f"Label value is not approved for field_kind {field_kind!r}.{extra} "
                    f"Arbitrary caller-supplied label text is rejected whether or not it "
                    f"contains a recognizable lithology term."))
            continue

        # ------------------------------------------------------------------
        # INTERPRETIVE / METHOD / EXPLANATORY - registered statement or,
        # for interpretive, a registered template with typed substitutions.
        # ------------------------------------------------------------------
        if auth is None or not isinstance(auth, Authorization):
            violations.append(_violation(
                context, scope, text,
                f"No authorization presented for a persisted {scope} field. Every "
                f"such field must cite a registered statement_id, or (interpretive "
                f"only) a registered template_id with typed substitutions. "
                f"Unregistered free text is rejected unconditionally - not because a "
                f"prohibited term was detected, but because nothing authorized it."))
            continue

        if auth.statement_id is not None and auth.template_id is not None:
            violations.append(_violation(
                context, scope, text,
                "Authorization cites BOTH a statement_id and a template_id; exactly "
                "one canonical authorization must be presented."))
            continue

        if auth.statement_id is not None:
            registered = REGISTERED_STATEMENTS.get(auth.statement_id)
            if registered is None:
                violations.append(_violation(
                    context, scope, text,
                    f"Unknown statement_id {auth.statement_id!r}: it is not a member of "
                    f"REGISTERED_STATEMENTS."))
                continue
            if registered.scope != scope:
                violations.append(_violation(
                    context, scope, text,
                    f"Statement {auth.statement_id!r} is registered for scope "
                    f"{registered.scope!r} and cannot authorize a {scope!r} field."))
                continue
            if text != registered.text:
                violations.append(_violation(
                    context, scope, text,
                    f"Text does not match registered statement {auth.statement_id!r} "
                    f"EXACTLY. Case changes, added or removed clauses, altered "
                    f"punctuation and near-misses are all rejected: the registered "
                    f"text is the canonical persisted decision."))
            continue

        if auth.template_id is not None:
            template = REGISTERED_TEMPLATES.get(auth.template_id)
            if template is None:
                violations.append(_violation(
                    context, scope, text,
                    f"Unknown template_id {auth.template_id!r}: it is not a member of "
                    f"REGISTERED_TEMPLATES."))
                continue
            if template.scope != scope:
                violations.append(_violation(
                    context, scope, text,
                    f"Template {auth.template_id!r} is registered for scope "
                    f"{template.scope!r} and cannot authorize a {scope!r} field."))
                continue
            undeclared = sorted(set(auth.fields) - set(template.fields))
            missing = sorted(set(template.fields) - set(auth.fields))
            if undeclared or missing:
                violations.append(_violation(
                    context, scope, text,
                    f"Template {auth.template_id!r} substitution mismatch - undeclared "
                    f"field(s) {undeclared}, missing field(s) {missing}. Only declared "
                    f"fields may be substituted."))
                continue
            mistyped = sorted(
                name for name, value in auth.fields.items()
                if not FIELD_TYPES[template.fields[name]](value)
            )
            if mistyped:
                violations.append(_violation(
                    context, scope, text,
                    f"Template {auth.template_id!r} substitution(s) {mistyped} do not "
                    f"satisfy their declared type. A substitution is restricted to a "
                    f"typed value so a template cannot become a channel for prose."))
                continue
            if text != template.render(auth.fields):
                violations.append(_violation(
                    context, scope, text,
                    f"Text does not match template {auth.template_id!r} rendered with "
                    f"the declared substitutions. The rendered template is the "
                    f"canonical persisted decision."))
            continue

        violations.append(_violation(
            context, scope, text,
            "Authorization presented neither a statement_id nor a template_id."))
    return tuple(violations)


#### `p2mem/wellframe.py` — auditable well-frame assembly

Assembles one well's LOCKED canonical curve arrays alongside the LOCKED MD→TVD/TVDSS mapping. It never opens a file, never recomputes minimum curvature, never chooses a depth basis, and never modifies a curve sample.

**Why this module calls the locked mapper on a subset.** The locked `map_las_md_to_tvd_tvdss` is deliberately all-or-nothing: if *any* LAS MD sample falls outside the survey's station MD coverage it raises `ExtrapolationRejectedError` for the whole array, because silently extending a trajectory past its surveyed range is exactly the failure mode Increment 3 was written to prevent. That is correct for a mapping primitive and is not weakened here. But a well frame must still be assemblable when a log runs slightly past the last survey station — the honest result there is "these $N$ samples have no defensible TVD", not "the whole well is unusable" and certainly not "hold the last TVD constant". So this module reads the trajectory's own MD coverage through the locked selector, builds an in-coverage mask, calls the **locked** mapper on the in-coverage subset (so every TVD/TVDSS number in a frame is produced by already-reviewed code), and writes the results back into full-length arrays with NaN at out-of-coverage positions.

The result: `n_extrapolated` is 0 **by construction**, and the coverage gap is reported honestly as `n_depth_unmapped`. When every sample is in coverage — the case for all four approved wells — this reduces to a single call on the whole array.

In [ ]:
%%writefile p2mem/wellframe.py
"""
p2mem.wellframe - Increment 6 well-frame assembly.

What this module does
---------------------
Takes ONE well's already-loaded, already-contract-validated LOCKED
Increment 2.1.1 `LasFileResult` and its LOCKED Increment 3.1.1
`DeviationWellResult`, and assembles a typed `WellFrame`: the canonical
curve arrays, the MD -> TVD/TVDSS mapping computed from that well's
explicitly selected survey basis, per-sample validity masks, per-curve
provenance, and QC flags.

What this module explicitly does NOT do
---------------------------------------
* It does not open, read, parse, or re-parse any file. Both inputs are
  already-validated in-memory results produced by locked modules.
* It does not recompute minimum curvature, does not choose a depth basis,
  and does not replace `petrel_source_trace`. The basis is whatever the
  locked `DeviationWellResult.depth_basis.selected_basis` already says,
  and the mapping arithmetic is performed by the locked
  `p2mem.depth_mapping.map_las_md_to_tvd_tvdss` - never re-implemented
  here.
* It does not extrapolate. See "Coverage handling" below.
* It does not modify, resample, reorder, gap-fill, smooth, or delete any
  canonical curve sample. Sample count and order are preserved exactly;
  invalidity is expressed only through masks.
* It does not read, transfer, or re-derive formation tops.

Coverage handling (why this module calls the locked mapper twice)
-----------------------------------------------------------------
The locked `map_las_md_to_tvd_tvdss` is deliberately all-or-nothing: if
ANY LAS MD sample falls outside the survey's own station MD coverage it
raises `ExtrapolationRejectedError` for the whole array, because silently
extending a trajectory beyond its surveyed range is exactly the failure
mode Increment 3 was written to prevent. That is the correct behavior for
a mapping primitive, and this module does not weaken it.

A well frame, however, must still be assemblable when a log runs a little
beyond the last survey station - the scientifically honest result there is
"these N samples have no defensible TVD", not "the whole well is
unusable" and certainly not "hold the last TVD constant". So this module:

  1. reads the selected trajectory's own MD coverage via the locked
     `select_survey_trajectory_for_mapping` (no re-derivation);
  2. builds a per-sample in-coverage mask;
  3. calls the LOCKED mapper on the in-coverage subset only - so every
     TVD/TVDSS number in a well frame is produced by the locked,
     already-reviewed interpolation, never by code in this module;
  4. writes those results back into full-length arrays at their original
     positions, leaving out-of-coverage samples as NaN with
     `depth_valid_mask == False`.

The result is that a well frame never contains an extrapolated depth:
`n_extrapolated` is 0 by construction, and `n_depth_unmapped` reports the
honest coverage gap instead. When every sample is in coverage (the case
for all four approved wells in this project's real data), step 3 is a
single call on the whole array and the result is bit-for-bit what the
locked mapper would have returned on its own.
"""

from __future__ import annotations

from pathlib import Path
from typing import Dict, Optional, Tuple

import numpy as np

from p2mem.deviation_models import DeviationWellResult
from p2mem.depth_mapping import (
    DepthMappingError,
    map_las_md_to_tvd_tvdss,
    select_survey_trajectory_for_mapping,
)
from p2mem.models import LasFileResult
from p2mem.wellframe_models import (
    DEPTH_MAP_STATUS_FULL,
    DEPTH_MAP_STATUS_NONE,
    DEPTH_MAP_STATUS_PARTIAL,
    EVIDENCE_CLASS_MEASURED,
    CurveSlot,
    WellFrame,
    WellFrameAssemblyFailure,
)

__all__ = [
    "WellFrameAssemblyError",
    "QC_FLAG_DEPTH_COVERAGE_LIMITED",
    "QC_FLAG_NO_DEPTH_COVERAGE",
    "QC_FLAG_GR_FAMILY_ABSENT",
    "QC_FLAG_SPARSE_CURVE_COVERAGE",
    "SPARSE_COVERAGE_FRACTION",
    "assemble_well_frame",
    "assemble_well_frames",
]


class WellFrameAssemblyError(RuntimeError):
    """
    Raised when a well frame cannot be assembled from otherwise-valid
    inputs (for example a LAS result carrying no canonical `MD_m` array,
    or a curve array whose length disagrees with `MD_m`). This is a
    structural precondition failure, deliberately distinct from the
    locked layers' own ingestion exceptions.
    """


QC_FLAG_DEPTH_COVERAGE_LIMITED = "DEPTH_COVERAGE_LIMITED_SOME_SAMPLES_UNMAPPED"
QC_FLAG_NO_DEPTH_COVERAGE = "NO_SURVEY_MD_COVERAGE_OVERLAP"
QC_FLAG_GR_FAMILY_ABSENT = "GR_FAMILY_CURVE_ABSENT"
QC_FLAG_SPARSE_CURVE_COVERAGE = "SPARSE_CURVE_COVERAGE"

# A curve whose valid fraction falls below this is flagged as sparse. This
# is a REPORTING threshold only - it never removes, rejects, or rescales a
# curve, and it never by itself makes a well ineligible for anything.
SPARSE_COVERAGE_FRACTION = 0.10


def _readonly(arr: np.ndarray) -> np.ndarray:
    """
    Return a read-only VIEW of `arr` (never a copy, so no per-sample data
    is duplicated in memory, and never a mutable alias, so a downstream
    consumer cannot silently corrupt a locked loader's array through a
    well frame). The locked source array itself is left untouched.
    """
    view = arr.view()
    view.setflags(write=False)
    return view


def _map_depths_without_extrapolation(
    well_key: str,
    las_md_m: np.ndarray,
    dev_result: DeviationWellResult,
) -> Tuple[np.ndarray, np.ndarray, np.ndarray, str, str, float, float, float, str]:
    """
    Map `las_md_m` onto TVD/TVDSS using the LOCKED mapper, restricted to
    samples inside the selected survey trajectory's own MD coverage.

    Returns
    -------
    (tvd_full, tvdss_full, depth_valid_mask, depth_basis_used,
     interpolation_method, datum_elevation_m, survey_md_min_m,
     survey_md_max_m, depth_map_status)

    `tvd_full` / `tvdss_full` are full-length float arrays with NaN at
    every out-of-coverage sample. No value in them is extrapolated: each
    finite entry was produced by the locked mapper from in-coverage input.
    """
    survey_md, _survey_tvd, _basis = select_survey_trajectory_for_mapping(dev_result)
    if survey_md.size < 2:
        raise DepthMappingError(
            f"{well_key}: selected survey trajectory has fewer than 2 stations; no MD->TVD "
            f"relationship can be interpolated."
        )

    survey_md_min = float(np.min(survey_md))
    survey_md_max = float(np.max(survey_md))

    md = np.asarray(las_md_m, dtype=np.float64)
    in_coverage = np.isfinite(md) & (md >= survey_md_min) & (md <= survey_md_max)

    tvd_full = np.full(md.shape, np.nan, dtype=np.float64)
    tvdss_full = np.full(md.shape, np.nan, dtype=np.float64)

    n_in = int(np.count_nonzero(in_coverage))
    if n_in == 0:
        # Nothing can be mapped. This is reported, never worked around.
        datum = float(dev_result.header.datum_elevation_m)
        return (
            tvd_full,
            tvdss_full,
            in_coverage,
            dev_result.depth_basis.selected_basis,
            "not_applicable_no_coverage_overlap",
            datum,
            survey_md_min,
            survey_md_max,
            DEPTH_MAP_STATUS_NONE,
        )

    # Delegate every actual TVD/TVDSS number to the LOCKED mapper. When all
    # samples are in coverage this is one call on the whole array and the
    # result is exactly what the locked mapper returns unaided.
    mapped = map_las_md_to_tvd_tvdss(well_key, md[in_coverage], dev_result)
    tvd_full[in_coverage] = mapped.tvd_mapped_m
    tvdss_full[in_coverage] = mapped.tvdss_mapped_m

    status = DEPTH_MAP_STATUS_FULL if n_in == md.size else DEPTH_MAP_STATUS_PARTIAL
    return (
        tvd_full,
        tvdss_full,
        in_coverage,
        mapped.depth_basis_used,
        mapped.interpolation_method,
        float(mapped.datum_elevation_m),
        survey_md_min,
        survey_md_max,
        status,
    )


def assemble_well_frame(
    well_key: str,
    las_result: LasFileResult,
    dev_result: DeviationWellResult,
    *,
    las_path: str,
    survey_path: str,
    gr_family_canonical_name: Optional[str] = None,
) -> WellFrame:
    """
    Assemble one `WellFrame` from a LOCKED `LasFileResult` and a LOCKED
    `DeviationWellResult`.

    `gr_family_canonical_name`, when given, names which of this well's
    canonical curves is its gamma-ray-family curve (e.g. `"GR_api"`,
    `"GRD_api"`, `"ECGR_api"`). It is recorded, never guessed: this
    project never adjudicates tool identity from the data, and two wells
    sharing a canonical GR name are still two different measurements.
    If the named curve is absent, a QC flag is raised and the frame's
    `gr_family_canonical_name` is left None - it is never silently
    substituted with another well's or another mnemonic's curve.

    Raises
    ------
    WellFrameAssemblyError
        If the LAS result carries no canonical `MD_m`, or any canonical
        curve's length disagrees with `MD_m` (a structural inconsistency
        that must never be papered over by truncation or padding).
    DepthMappingError
        Propagated from the locked depth layer when no well-defined
        MD->TVD relationship exists at all.
    """
    md = las_result.canonical_data.get("MD_m")
    if md is None:
        raise WellFrameAssemblyError(
            f"{well_key}: LAS result has no canonical 'MD_m' array; a well frame cannot be "
            f"assembled without the depth index."
        )
    md = np.asarray(md)
    if md.ndim != 1 or md.size == 0:
        raise WellFrameAssemblyError(f"{well_key}: canonical 'MD_m' must be a non-empty 1-D array.")

    n_samples = int(md.size)
    for canonical_name, arr in las_result.canonical_data.items():
        a = np.asarray(arr)
        if a.ndim != 1 or a.size != n_samples:
            raise WellFrameAssemblyError(
                f"{well_key}: canonical curve {canonical_name!r} has shape {a.shape} but 'MD_m' has "
                f"{n_samples} samples; a well frame never truncates, pads, or resamples to "
                f"reconcile a length disagreement."
            )

    (
        tvd_full,
        tvdss_full,
        depth_valid_mask,
        depth_basis_used,
        interpolation_method,
        datum_elevation_m,
        survey_md_min_m,
        survey_md_max_m,
        depth_map_status,
    ) = _map_depths_without_extrapolation(well_key, md, dev_result)

    # Per-curve provenance, taken from the locked loader's own resolutions
    # and stats - never re-derived here.
    stats_by_canonical = {s.canonical_name: s for s in las_result.curve_stats}
    resolution_by_canonical = {r.canonical_name: r for r in las_result.resolutions}
    las_basename = Path(las_path).name
    survey_basename = Path(survey_path).name

    curves: Dict[str, CurveSlot] = {}
    qc_flags = []
    for canonical_name, arr in las_result.canonical_data.items():
        values = np.asarray(arr)
        valid_mask = np.isfinite(values)
        valid_count = int(np.count_nonzero(valid_mask))
        st = stats_by_canonical.get(canonical_name)
        rs = resolution_by_canonical.get(canonical_name)
        curves[canonical_name] = CurveSlot(
            canonical_name=canonical_name,
            source_curve_name=(st.source_curve_name if st is not None else (rs.source_curve_name if rs is not None else "")),
            raw_mnemonic=(st.raw_mnemonic if st is not None else (rs.matched_raw_mnemonic or "")),
            raw_unit=(st.raw_unit if st is not None else (rs.matched_raw_unit or "")),
            canonical_unit=(st.canonical_unit if st is not None else ""),
            conversion_function=(st.conversion_function if st is not None else (rs.conversion_applied or "")),
            source_filename=las_basename,
            evidence_class=EVIDENCE_CLASS_MEASURED,
            values=_readonly(values),
            valid_mask=_readonly(valid_mask),
            n_samples=n_samples,
            valid_count=valid_count,
            valid_fraction=(valid_count / n_samples) if n_samples else 0.0,
        )
        if n_samples and canonical_name != "MD_m" and (valid_count / n_samples) < SPARSE_COVERAGE_FRACTION:
            qc_flags.append(f"{QC_FLAG_SPARSE_CURVE_COVERAGE}:{canonical_name}")

    resolved_gr_name: Optional[str] = None
    resolved_gr_source: Optional[str] = None
    if gr_family_canonical_name is not None:
        slot = curves.get(gr_family_canonical_name)
        if slot is None:
            qc_flags.append(f"{QC_FLAG_GR_FAMILY_ABSENT}:{gr_family_canonical_name}")
        else:
            resolved_gr_name = slot.canonical_name
            resolved_gr_source = slot.source_curve_name

    n_depth_unmapped = int(n_samples - np.count_nonzero(depth_valid_mask))
    if depth_map_status == DEPTH_MAP_STATUS_PARTIAL:
        qc_flags.append(f"{QC_FLAG_DEPTH_COVERAGE_LIMITED}:{n_depth_unmapped}")
    elif depth_map_status == DEPTH_MAP_STATUS_NONE:
        qc_flags.append(QC_FLAG_NO_DEPTH_COVERAGE)

    return WellFrame(
        well_key=well_key,
        source_las_filename=las_basename,
        source_survey_filename=survey_basename,
        n_samples=n_samples,
        MD_m=_readonly(md),
        TVD_m=_readonly(tvd_full),
        TVDSS_m=_readonly(tvdss_full),
        depth_valid_mask=_readonly(np.asarray(depth_valid_mask, dtype=bool)),
        depth_basis_used=depth_basis_used,
        interpolation_method=interpolation_method,
        datum_elevation_m=datum_elevation_m,
        depth_map_status=depth_map_status,
        survey_md_min_m=survey_md_min_m,
        survey_md_max_m=survey_md_max_m,
        las_md_min_m=float(np.min(md)),
        las_md_max_m=float(np.max(md)),
        n_depth_unmapped=n_depth_unmapped,
        n_extrapolated=0,
        curves=curves,
        gr_family_canonical_name=resolved_gr_name,
        gr_family_source_curve_name=resolved_gr_source,
        # LAS well identity is genuinely verified from file CONTENT: the
        # locked contract resolver compares `FileContract
        # .expected_well_identifier` against the file's own `WELL` header
        # line and raises a contract ERROR on disagreement, so a
        # successfully loaded `LasFileResult` has already had its declared
        # well name checked against the file itself. This is a stronger
        # evidence status than the Increment 5 formation-top files earned
        # (filename-only association), and is recorded as such rather than
        # being flattened to the weaker label for uniformity's sake.
        well_identity_evidence_status="verified_against_file_well_header",
        qc_flags=tuple(qc_flags),
    )


def assemble_well_frames(
    las_results: Dict[str, LasFileResult],
    dev_results: Dict[str, DeviationWellResult],
    *,
    las_paths: Dict[str, str],
    survey_paths: Dict[str, str],
    gr_family_by_well: Optional[Dict[str, str]] = None,
) -> Tuple[Dict[str, WellFrame], Dict[str, WellFrameAssemblyFailure]]:
    """
    Assemble well frames for a batch of wells, isolating per-well failures
    exactly as the locked batch loaders do: one well's failure never stops
    the others, and each well key ends up in exactly one of the two
    returned dicts, never both.

    A well present in `las_results` but absent from `dev_results` (or vice
    versa) is recorded as a typed failure with the correct
    `failure_origin` - it is never assembled against another well's survey
    and never given a substituted trajectory.

    Results are keyed and iterated deterministically (sorted by well key)
    so that downstream output tables are byte-reproducible.
    """
    gr_family_by_well = gr_family_by_well or {}
    frames: Dict[str, WellFrame] = {}
    failures: Dict[str, WellFrameAssemblyFailure] = {}

    for well_key in sorted(set(las_results) | set(dev_results)):
        las_path = las_paths.get(well_key, "")
        survey_path = survey_paths.get(well_key, "")
        las_result = las_results.get(well_key)
        dev_result = dev_results.get(well_key)

        if las_result is None:
            failures[well_key] = WellFrameAssemblyFailure(
                well_key=well_key,
                failure_origin="las",
                error_type="missing_las_result",
                message=f"{well_key}: no successfully loaded LAS result was supplied for this well.",
                las_path=las_path or None,
                survey_path=survey_path or None,
            )
            continue
        if dev_result is None:
            failures[well_key] = WellFrameAssemblyFailure(
                well_key=well_key,
                failure_origin="survey",
                error_type="missing_survey_result",
                message=(
                    f"{well_key}: no successfully loaded deviation-survey result was supplied for "
                    f"this well; a well frame is never assembled against another well's trajectory."
                ),
                las_path=las_path or None,
                survey_path=survey_path or None,
            )
            continue

        try:
            frames[well_key] = assemble_well_frame(
                well_key,
                las_result,
                dev_result,
                las_path=las_path,
                survey_path=survey_path,
                gr_family_canonical_name=gr_family_by_well.get(well_key),
            )
        except WellFrameAssemblyError as exc:
            failures[well_key] = WellFrameAssemblyFailure(
                well_key=well_key, failure_origin="curve_assembly", error_type="assembly_failure",
                message=str(exc), las_path=las_path or None, survey_path=survey_path or None, exception=exc,
            )
        except DepthMappingError as exc:
            failures[well_key] = WellFrameAssemblyFailure(
                well_key=well_key, failure_origin="depth_mapping", error_type="depth_mapping_failure",
                message=str(exc), las_path=las_path or None, survey_path=survey_path or None, exception=exc,
            )

    return frames, failures


#### `p2mem/petrophysics_models.py` — typed GR/QC/scenario/proxy structures

The naming here is load-bearing. `GrEndpointScenario.evidence_class` is fixed at `"assumed_configured"`; `GrProxyResult` retains `igr_unclipped_frac` beside `igr_clipped_frac`; and the proxy array is named `VSH_GR_linear_proxy_frac` so the word "proxy" travels with the quantity everywhere it is referenced.

In [ ]:
%%writefile p2mem/petrophysics_models.py
"""
p2mem.petrophysics_models - Typed data structures for the Increment 6
gamma-ray QC, endpoint-sensitivity, and screening-proxy layer.

Scientific boundary encoded in these types
------------------------------------------
Every quantity defined here is either a FACTUAL MEASUREMENT SUMMARY of a
gamma-ray-family curve, or a DIMENSIONLESS SCREENING PROXY derived from
one under explicitly assumed, configured endpoints. Nothing here is a
calibrated shale volume, and nothing here is a lithology.

The naming is deliberate and load-bearing:

* `GrEndpointScenario.gr_low_endpoint_api` / `gr_high_endpoint_api` are
  ASSUMED values produced by a configured percentile rule. The field
  `evidence_class` on every scenario is `"assumed_configured"` and there
  is no code path in this project that can set it to anything stronger.
* `GrProxyResult.igr_unclipped_frac` is retained beside
  `igr_clipped_frac` so the amount of clipping - i.e. how far the real
  data fell outside the assumed bracket - is always visible.
* The linear proxy field is named `VSH_GR_linear_proxy_frac`: the word
  "proxy" is part of the identifier itself, so the quantity cannot be
  referenced anywhere downstream without carrying its own disclaimer.
  It is never named `VSH`, `Vshale`, or `shale_volume`.

Confidence classes (`GR_PROXY_HIGH` and friends) describe DATA AND PROXY
CONFIDENCE ONLY. Every class string generated by this layer is checked
against `p2mem.wellframe_models.PROHIBITED_LITHOLOGY_TERMS`.
"""

from __future__ import annotations

from dataclasses import dataclass, field
from typing import Dict, Optional, Tuple

import numpy as np

__all__ = [
    "USE_STATUS_PROXY_ALLOWED",
    "USE_STATUS_PROXY_ALLOWED_DEPTH_TIED",
    "USE_STATUS_QC_ONLY_EXCLUDED",
    "VALID_USE_STATUSES",
    "PROXY_PERMITTED_USE_STATUSES",
    "EXCLUSION_REASON_BOREAS_ECGR",
    "GR_PROXY_HIGH",
    "GR_PROXY_INTERMEDIATE",
    "GR_PROXY_LOW",
    "GR_NOT_AVAILABLE",
    "GR_EXCLUDED_UNRESOLVED_SCALE",
    "VALID_GR_CONFIDENCE_CLASSES",
    "GrFamilyDisposition",
    "GrFamilyQcStats",
    "GrEndpointScenario",
    "GrProxyResult",
    "PetrophysicsIssue",
    "PetrophysicsEligibilityConfig",
]

# ---------------------------------------------------------------------------
# GR-family use-status vocabulary
# ---------------------------------------------------------------------------
USE_STATUS_PROXY_ALLOWED = "screening_proxy_allowed"
USE_STATUS_PROXY_ALLOWED_DEPTH_TIED = "screening_proxy_allowed_depth_tied"
USE_STATUS_QC_ONLY_EXCLUDED = "qc_only_excluded"

VALID_USE_STATUSES: Tuple[str, ...] = (
    USE_STATUS_PROXY_ALLOWED,
    USE_STATUS_PROXY_ALLOWED_DEPTH_TIED,
    USE_STATUS_QC_ONLY_EXCLUDED,
)

# The ONLY two statuses under which any GR-derived quantity may be computed.
# A well whose status is not in this tuple gets no GR index, no proxy, no
# flag, no lithology-dependent mask, and no NCT-donor status - enforced in
# `p2mem.petrophysics` and re-checked by the test suite.
PROXY_PERMITTED_USE_STATUSES: Tuple[str, ...] = (
    USE_STATUS_PROXY_ALLOWED,
    USE_STATUS_PROXY_ALLOWED_DEPTH_TIED,
)

EXCLUSION_REASON_BOREAS_ECGR = "BOREAS_ECGR_SCALE_UNRESOLVED"

# ---------------------------------------------------------------------------
# Data/proxy confidence classes - NEVER lithology classes
# ---------------------------------------------------------------------------
GR_PROXY_HIGH = "GR_PROXY_HIGH"
GR_PROXY_INTERMEDIATE = "GR_PROXY_INTERMEDIATE"
GR_PROXY_LOW = "GR_PROXY_LOW"
GR_NOT_AVAILABLE = "GR_NOT_AVAILABLE"
GR_EXCLUDED_UNRESOLVED_SCALE = "GR_EXCLUDED_UNRESOLVED_SCALE"

VALID_GR_CONFIDENCE_CLASSES: Tuple[str, ...] = (
    GR_PROXY_HIGH,
    GR_PROXY_INTERMEDIATE,
    GR_PROXY_LOW,
    GR_NOT_AVAILABLE,
    GR_EXCLUDED_UNRESOLVED_SCALE,
)


@dataclass(frozen=True)
class GrFamilyDisposition:
    """
    One well's human-authored gamma-ray-family disposition, read verbatim
    from `config/petrophysics_eligibility.yml`.

    This object is never inferred from data. `use_status` and
    `exclusion_reason` are review decisions; the code's only job is to
    honour them exactly and to refuse to compute anything a
    `qc_only_excluded` status forbids.
    """

    well_key: str
    source_las_filename: str
    gr_family_canonical_name: str
    gr_family_source_curve_name: str
    use_status: str
    exclusion_reason: Optional[str]
    evidence_class: str
    has_approved_formation_tops: bool
    notes: str = ""

    def __post_init__(self) -> None:
        if self.use_status not in VALID_USE_STATUSES:
            raise ValueError(
                f"GrFamilyDisposition {self.well_key!r}: use_status {self.use_status!r} is not one "
                f"of {VALID_USE_STATUSES}."
            )
        if self.use_status == USE_STATUS_QC_ONLY_EXCLUDED and not self.exclusion_reason:
            raise ValueError(
                f"GrFamilyDisposition {self.well_key!r}: a 'qc_only_excluded' well must carry a "
                f"machine-readable exclusion_reason; an unexplained exclusion is not auditable."
            )

    @property
    def proxy_permitted(self) -> bool:
        """True only when this well's configured status permits any
        GR-derived calculation at all."""
        return self.use_status in PROXY_PERMITTED_USE_STATUSES


@dataclass(frozen=True)
class GrFamilyQcStats:
    """
    Purely factual, non-judgemental statistics for one well's GR-family
    curve. Computed for EVERY well - including a `qc_only_excluded` one,
    because factual QC reporting is exactly what an excluded well remains
    available for.

    `n_samples_above_seabed` is Optional: it is a genuine integer only for
    a well whose seabed marker exists in the LOCKED Increment 5
    survey-corrected output. For a well with no approved formation tops it
    is None, meaning "not determinable from approved data" - never 0, which
    would falsely assert that no such samples exist.
    """

    well_key: str
    gr_family_canonical_name: str
    gr_family_source_curve_name: str
    source_las_filename: str
    unit: str
    n_samples: int
    valid_count: int
    valid_fraction: float
    min_api: Optional[float]
    max_api: Optional[float]
    median_api: Optional[float]
    p01_api: Optional[float]
    p05_api: Optional[float]
    p10_api: Optional[float]
    p25_api: Optional[float]
    p50_api: Optional[float]
    p75_api: Optional[float]
    p90_api: Optional[float]
    p95_api: Optional[float]
    p99_api: Optional[float]
    dynamic_range_p05_p95_api: Optional[float]
    n_negative_samples: int
    n_zero_samples: int
    n_samples_above_seabed: Optional[int]
    seabed_basis: str
    n_valid_blocks: int
    longest_valid_block_samples: int
    longest_missing_block_samples: int
    longest_missing_block_md_start_m: Optional[float]
    longest_missing_block_md_end_m: Optional[float]
    statistics_basis: str


@dataclass(frozen=True)
class GrEndpointScenario:
    """
    One well's MEASURED endpoint values under one configured scenario.

    `gr_low_endpoint_api` / `gr_high_endpoint_api` are the values the
    scenario's percentile rule actually produced when applied to THIS
    well's own valid, depth-mapped samples. They are recorded so that a
    reader can see the numbers the rule generated, not merely the rule.

    `evidence_class` is fixed at `"assumed_configured"`. There is no
    constructor path, config key, or code branch anywhere in this project
    that can promote an endpoint scenario to a calibrated status.
    """

    well_key: str
    scenario_name: str
    low_percentile: float
    high_percentile: float
    gr_low_endpoint_api: float
    gr_high_endpoint_api: float
    endpoint_separation_api: float
    n_samples_used_for_endpoints: int
    endpoint_sample_basis: str
    description: str
    evidence_class: str = "assumed_configured"
    calibration_status: str = "uncalibrated_assumed_no_calibration_data_exists"


@dataclass(frozen=True)
class GrProxyResult:
    """
    One well's GR-index and linear screening-proxy arrays under ONE
    endpoint scenario.

    All arrays are full-length and aligned sample-for-sample with the well
    frame's `MD_m`. A sample that is invalid (non-finite GR, or an
    unmapped depth) is NaN in every array and False in `valid_mask` - it
    is never filled, interpolated, or dropped, so array positions remain
    directly comparable with the well frame and with other scenarios.

    `igr_unclipped_frac` may legitimately fall outside [0, 1]: that is the
    honest signal that the well's real data exceeded the assumed endpoint
    bracket, and `n_clipped_low` / `n_clipped_high` quantify it.

    `VSH_GR_linear_proxy_frac` is numerically identical to
    `igr_clipped_frac` under this increment's only permitted transform
    (linear identity). It is carried as a separately named array anyway,
    because the two mean different things: one is a normalized curve
    position, the other is the quantity a later increment might be tempted
    to treat as a shale volume - and the name is the guardrail.
    """

    well_key: str
    scenario_name: str
    gr_family_canonical_name: str
    gr_low_endpoint_api: float
    gr_high_endpoint_api: float
    igr_unclipped_frac: np.ndarray
    igr_clipped_frac: np.ndarray
    VSH_GR_linear_proxy_frac: np.ndarray
    valid_mask: np.ndarray
    n_valid: int
    n_clipped_low: int
    n_clipped_high: int
    clipped_fraction: float
    proxy_median: Optional[float]
    proxy_p25: Optional[float]
    proxy_p75: Optional[float]
    transform_name: str = "linear_identity_of_clipped_igr"
    calibration_status: str = "screening_proxy_uncalibrated_not_a_shale_volume"


@dataclass(frozen=True)
class PetrophysicsIssue:
    """
    One reportable fact about the Increment 6 petrophysics layer, mirroring
    the LOCKED `IngestionIssue` shape (severity / code / message / context)
    so that every increment's issue tables stay structurally comparable.
    """

    severity: str  # "ERROR" | "WARNING" | "INFO"
    code: str
    message: str
    context: str


@dataclass(frozen=True)
class PetrophysicsEligibilityConfig:
    """
    The fully parsed, validated contents of
    `config/petrophysics_eligibility.yml`.

    Held as a typed object rather than a raw dict so that a missing or
    misspelled key fails loudly at load time - at the one place a human
    can fix it - instead of silently defaulting deep inside a calculation.
    """

    schema_version: str
    increment: int
    assurance_tier: str
    policy: Dict
    endpoint_scenarios: Tuple[Dict, ...]
    wells: Dict[str, GrFamilyDisposition]
    gr_proxy_confidence: Dict
    method_eligibility: Dict
    source_filename: str = ""
    raw_policy_notes: Tuple[str, ...] = field(default_factory=tuple)

    @property
    def scenario_names(self) -> Tuple[str, ...]:
        return tuple(str(s["scenario_name"]) for s in self.endpoint_scenarios)


#### `p2mem/petrophysics.py` — GR QC, endpoint resolution, and the screening proxy

Implements the three questions this layer is allowed to answer (factual QC statistics, measured endpoints under an assumed rule, and the dimensionless index/proxy), and refuses everything else.

**Input-validation split**, honouring the established project precedent exactly: `TypeError` for a *type-class* defect (boolean, string/bytes, complex, non-numeric dtype); `PetrophysicsInputError` for a *structural or value* defect (wrong dimensionality, length mismatch, non-finite/mis-ordered/identical endpoints, insufficient separation). An incidental `IndexError` or bare comparison `TypeError` from deep inside a calculation is never an acceptable substitute.

**Endpoint degeneracy is rejected, never nudged.** Identical endpoints would put an exact zero in the denominator; a reversed pair would invert the proxy's sense with no visible error; a separation below the configured minimum would amplify ordinary log noise to full IGR scale. All three raise.

In [ ]:
%%writefile p2mem/petrophysics.py
"""
p2mem.petrophysics - Increment 6 gamma-ray QC, endpoint sensitivity, and
dimensionless screening-proxy calculation.

Scope
-----
This module answers exactly three questions, and refuses to answer any
other:

  1. What does each well's gamma-ray-family curve factually look like?
     (`compute_gr_family_qc_stats` - descriptive statistics only, computed
     for every well including an excluded one.)

  2. Under a configured endpoint rule, what endpoint values does this
     well's own data produce? (`resolve_endpoint_scenarios` - measured
     values from an ASSUMED rule, never a calibration.)

  3. Given those endpoints, what is the dimensionless GR index and the
     linear screening proxy? (`compute_gr_proxy` - retaining clipped and
     unclipped results side by side.)

It does NOT interpret lithology, does not correct or rescale any curve,
does not compute a calibrated shale volume, and does not implement any
nonlinear Vsh transform.

The GR index
------------
    IGR_unclipped = (GR - GR_low_endpoint) / (GR_high_endpoint - GR_low_endpoint)
    IGR_clipped   = clip(IGR_unclipped, 0, 1)

Both are retained. `IGR_unclipped` outside [0, 1] is not an error - it is
the measurable statement that the well's real data ran past the assumed
endpoint bracket, and suppressing it would hide exactly the sensitivity
this increment exists to quantify.

Input validation philosophy
---------------------------
This module's public entry points accept caller-supplied in-memory arrays,
so they validate their own inputs rather than trusting them. The
established project split is honoured exactly (see the LOCKED
`p2mem.io.tops` Increment 5.1 precedent):

  * `TypeError`             - a TYPE-CLASS defect (boolean, string/bytes,
                              complex, or otherwise non-numeric dtype).
  * `PetrophysicsInputError` - a STRUCTURAL or VALUE defect (wrong
                              dimensionality, length mismatch, non-finite
                              or mis-ordered endpoints, identical
                              endpoints, insufficient separation).

An incidental `IndexError`, broadcasting error, or bare comparison
`TypeError` from deep inside a calculation is never an acceptable
substitute for one of these.
"""

from __future__ import annotations

from pathlib import Path
from typing import Dict, List, Optional, Sequence, Tuple

import numpy as np
import yaml

from p2mem.petrophysics_models import (
    EXCLUSION_REASON_BOREAS_ECGR,
    USE_STATUS_PROXY_ALLOWED,
    USE_STATUS_PROXY_ALLOWED_DEPTH_TIED,
    GR_EXCLUDED_UNRESOLVED_SCALE,
    GR_NOT_AVAILABLE,
    GR_PROXY_HIGH,
    GR_PROXY_INTERMEDIATE,
    GR_PROXY_LOW,
    PROXY_PERMITTED_USE_STATUSES,
    GrEndpointScenario,
    GrFamilyDisposition,
    GrFamilyQcStats,
    GrProxyResult,
    PetrophysicsEligibilityConfig,
    PetrophysicsIssue,
)
from p2mem.wellframe_models import WellFrame, assert_no_lithology_vocabulary

__all__ = [
    "PetrophysicsConfigError",
    "PetrophysicsInputError",
    "PetrophysicsExclusionError",
    "load_petrophysics_eligibility_config",
    "compute_gr_family_qc_stats",
    "resolve_endpoint_scenarios",
    "compute_gr_index",
    "compute_gr_proxy",
    "classify_gr_proxy_confidence",
    "find_contiguous_blocks",
]


class PetrophysicsConfigError(ValueError):
    """Raised when `config/petrophysics_eligibility.yml` is missing,
    malformed, internally inconsistent, or missing a required key. A
    configuration problem is never silently defaulted."""


class PetrophysicsInputError(ValueError):
    """Raised for a structural or value defect in caller-supplied
    in-memory data (wrong dimensionality, length mismatch, non-finite or
    mis-ordered endpoints). Deliberately distinct from `TypeError`, which
    this module reserves for type-class defects."""


class PetrophysicsExclusionError(RuntimeError):
    """Raised when a GR-derived calculation is attempted for a well whose
    configured disposition forbids it (e.g. Boreas 1 under
    `BOREAS_ECGR_SCALE_UNRESOLVED`). This is a hard scientific boundary,
    enforced as an exception rather than a silent no-op so that a caller
    cannot mistake an empty result for a computed one."""


# ---------------------------------------------------------------------------
# Input validation helpers
# ---------------------------------------------------------------------------

def _reject_ambiguous_dtype(raw: np.ndarray, context: str) -> None:
    """
    Reject boolean, string/bytes, complex, and otherwise non-numeric dtype
    input with `TypeError` before any numeric use.

    Documented local copy of the identical check already established in
    the LOCKED `p2mem.units`, `p2mem.time_depth` (Increment 4.1.1) and
    `p2mem.io.tops` (Increment 5.1) modules. Those modules remain
    unmodified; duplicating ~8 lines here is deliberately preferred over
    editing a locked module to export a private helper.
    """
    kind = raw.dtype.kind
    if kind == "b":
        raise TypeError(f"{context}: boolean input is not accepted as a numeric quantity.")
    if kind in ("U", "S"):
        raise TypeError(f"{context}: string/bytes input is not accepted as a numeric quantity.")
    if kind == "c":
        raise TypeError(f"{context}: complex input is not accepted as a numeric quantity.")
    if kind not in ("i", "u", "f"):
        raise TypeError(f"{context}: unsupported array dtype {raw.dtype!r} for a numeric quantity.")


def _validate_numeric_1d(values, context: str) -> np.ndarray:
    """
    Validate that `values` is a 1-D numeric array and return it as float64.

    Order of checks is deliberate: dtype class first (so a string array
    raises a clear `TypeError` rather than a confusing cast failure), then
    dimensionality. A 0-d, 2-D, or higher-dimensional array is rejected
    with `PetrophysicsInputError` - never silently ravelled, which would
    destroy the sample-position correspondence every mask in this
    increment depends on.
    """
    if isinstance(values, bool):
        raise TypeError(f"{context}: boolean input is not accepted as a numeric array.")
    arr = np.asarray(values)
    _reject_ambiguous_dtype(arr, context)
    if arr.ndim != 1:
        raise PetrophysicsInputError(
            f"{context}: expected a 1-D array, got shape {arr.shape} ({arr.ndim}-D). A "
            f"multidimensional array is never silently flattened - sample positions must stay "
            f"aligned with the well frame's depth arrays."
        )
    return arr.astype(np.float64, copy=True)


def _validate_endpoints(low, high, context: str, min_separation: float) -> Tuple[float, float]:
    """
    Validate an endpoint pair and return it as plain Python floats.

    Rejects (with `TypeError`) boolean, string/bytes and complex values;
    rejects (with `PetrophysicsInputError`) non-finite values, a high
    endpoint that is not strictly greater than the low endpoint - including
    the exactly-identical case, which would otherwise divide by zero - and
    a separation below the configured minimum, which would turn ordinary
    log noise into full-scale IGR swings.
    """
    for label, value in (("low", low), ("high", high)):
        if isinstance(value, bool):
            raise TypeError(f"{context}: {label} endpoint must not be boolean, got {value!r}.")
        if isinstance(value, (str, bytes)):
            raise TypeError(f"{context}: {label} endpoint must not be a string/bytes, got {value!r}.")
        if isinstance(value, complex):
            raise TypeError(f"{context}: {label} endpoint must not be complex, got {value!r}.")
        try:
            float(value)
        except (TypeError, ValueError) as exc:
            raise TypeError(f"{context}: {label} endpoint is not a real numeric scalar: {value!r}.") from exc

    lo = float(low)
    hi = float(high)
    if not np.isfinite(lo) or not np.isfinite(hi):
        raise PetrophysicsInputError(
            f"{context}: endpoints must both be finite; got low={lo!r}, high={hi!r}."
        )
    if hi == lo:
        raise PetrophysicsInputError(
            f"{context}: low and high endpoints are identical ({lo}); the GR-index denominator "
            f"would be exactly zero. An identical endpoint pair is rejected, never nudged."
        )
    if hi < lo:
        raise PetrophysicsInputError(
            f"{context}: high endpoint ({hi}) must be strictly greater than low endpoint ({lo}); "
            f"a reversed pair would invert the proxy's sense without any visible error."
        )
    if (hi - lo) < float(min_separation):
        raise PetrophysicsInputError(
            f"{context}: endpoint separation {hi - lo:.6f} is below the configured minimum "
            f"{float(min_separation):.6f}; such a narrow bracket amplifies ordinary log noise to "
            f"full IGR scale and is rejected."
        )
    return lo, hi


# ---------------------------------------------------------------------------
# Configuration
# ---------------------------------------------------------------------------

class _NoDuplicateKeySafeLoader(yaml.SafeLoader):
    """YAML loader that rejects duplicate mapping keys, mirroring the
    LOCKED `p2mem.io.deviation` precedent: a silently overwritten
    duplicate key in a human-authored config is a real, hard-to-see
    configuration defect."""


def _construct_mapping_no_duplicates(loader, node, deep: bool = False):
    mapping = {}
    for key_node, value_node in node.value:
        key = loader.construct_object(key_node, deep=deep)
        if key in mapping:
            raise PetrophysicsConfigError(
                f"Duplicate key {key!r} in petrophysics eligibility config at line "
                f"{key_node.start_mark.line + 1}; a duplicated key silently overwrites the first "
                f"value and is rejected."
            )
        mapping[key] = loader.construct_object(value_node, deep=deep)
    return mapping


_NoDuplicateKeySafeLoader.add_constructor(
    yaml.resolver.BaseResolver.DEFAULT_MAPPING_TAG, _construct_mapping_no_duplicates
)

_REQUIRED_TOP_KEYS = (
    "schema_version", "increment", "assurance_tier", "policy",
    "endpoint_scenarios", "wells", "gr_proxy_confidence", "method_eligibility",
)
_REQUIRED_WELL_KEYS = (
    "source_las_filename", "gr_family_canonical_name", "gr_family_source_curve_name",
    "use_status", "exclusion_reason", "evidence_class", "has_approved_formation_tops",
)
_REQUIRED_POLICY_KEYS = (
    "endpoint_estimation_method", "cross_well_shared_endpoints_allowed",
    "endpoint_sample_basis", "clipping_policy", "clip_lower", "clip_upper",
    "min_endpoint_separation_api", "shale_proxy_transform",
    "nonlinear_vsh_transforms_enabled", "contiguity",
    "nct_candidate_proxy_thresholds", "physical_bounds",
)


def load_petrophysics_eligibility_config(yaml_path: str) -> PetrophysicsEligibilityConfig:
    """
    Load and validate `config/petrophysics_eligibility.yml`.

    Every required key is checked here so a typo fails at load time rather
    than deep inside a calculation. Two policy invariants are enforced as
    hard errors because violating either would silently cross an Increment
    6 scientific boundary:

      * `cross_well_shared_endpoints_allowed` must be false - a shared
        cross-well endpoint pair would assert a tool equivalence no
        evidence supports;
      * `nonlinear_vsh_transforms_enabled` must be false - no nonlinear
        Vsh transform has a closed primary-source method record in this
        project.
    """
    path = Path(yaml_path)
    if not path.is_file():
        raise PetrophysicsConfigError(f"Petrophysics eligibility config not found: {yaml_path!r}.")
    with path.open("r", encoding="utf-8") as fh:
        raw = yaml.load(fh, Loader=_NoDuplicateKeySafeLoader)
    if not isinstance(raw, dict):
        raise PetrophysicsConfigError(f"{path.name}: top level must be a mapping.")

    for key in _REQUIRED_TOP_KEYS:
        if key not in raw:
            raise PetrophysicsConfigError(f"{path.name}: required top-level key {key!r} is missing.")

    policy = raw["policy"]
    if not isinstance(policy, dict):
        raise PetrophysicsConfigError(f"{path.name}: 'policy' must be a mapping.")
    for key in _REQUIRED_POLICY_KEYS:
        if key not in policy:
            raise PetrophysicsConfigError(f"{path.name}: required policy key {key!r} is missing.")

    if bool(policy["cross_well_shared_endpoints_allowed"]):
        raise PetrophysicsConfigError(
            f"{path.name}: cross_well_shared_endpoints_allowed must be false. A single universal "
            f"cross-well endpoint pair would assert an equivalence between four differently named "
            f"GR-family tools that no calibration evidence in this project supports."
        )
    if bool(policy["nonlinear_vsh_transforms_enabled"]):
        raise PetrophysicsConfigError(
            f"{path.name}: nonlinear_vsh_transforms_enabled must be false in Increment 6. A "
            f"nonlinear Vsh transform requires a retrieved, verified primary-source method record "
            f"and a closed method-register entry; none exists in this project."
        )

    scenarios = raw["endpoint_scenarios"]
    if not isinstance(scenarios, list) or len(scenarios) < 3:
        raise PetrophysicsConfigError(
            f"{path.name}: 'endpoint_scenarios' must be a list of at least three scenarios "
            f"(low/base/high endpoint sensitivity is mandatory in Increment 6)."
        )
    seen_names = set()
    for sc in scenarios:
        for key in ("scenario_name", "low_percentile", "high_percentile", "description"):
            if key not in sc:
                raise PetrophysicsConfigError(
                    f"{path.name}: endpoint scenario is missing required key {key!r}: {sc!r}."
                )
        name = str(sc["scenario_name"])
        if name in seen_names:
            raise PetrophysicsConfigError(f"{path.name}: duplicate scenario_name {name!r}.")
        seen_names.add(name)
        lo_p = float(sc["low_percentile"])
        hi_p = float(sc["high_percentile"])
        if not (0.0 <= lo_p < hi_p <= 100.0):
            raise PetrophysicsConfigError(
                f"{path.name}: scenario {name!r} requires 0 <= low_percentile < high_percentile "
                f"<= 100; got low={lo_p}, high={hi_p}."
            )

    me = raw["method_eligibility"]
    if not isinstance(me, dict):
        raise PetrophysicsConfigError(f"{path.name}: 'method_eligibility' must be a mapping.")
    for required_mask in (
        "eligible_density_for_sv", "eligible_dynamic_elastic", "eligible_sonic_nct_candidate",
    ):
        if required_mask not in me:
            raise PetrophysicsConfigError(
                f"{path.name}: required method_eligibility entry {required_mask!r} is missing. "
                f"All three Increment 6 masks must be declared and described."
            )
        if "purpose" not in me[required_mask]:
            raise PetrophysicsConfigError(
                f"{path.name}: method_eligibility entry {required_mask!r} is missing 'purpose'; "
                f"an undocumented eligibility mask is not auditable."
            )

    wells_raw = raw["wells"]
    if not isinstance(wells_raw, dict) or not wells_raw:
        raise PetrophysicsConfigError(f"{path.name}: 'wells' must be a non-empty mapping.")
    wells: Dict[str, GrFamilyDisposition] = {}
    for well_key, spec in wells_raw.items():
        if not isinstance(spec, dict):
            raise PetrophysicsConfigError(f"{path.name}: well {well_key!r} entry must be a mapping.")
        for key in _REQUIRED_WELL_KEYS:
            if key not in spec:
                raise PetrophysicsConfigError(
                    f"{path.name}: well {well_key!r} is missing required key {key!r}."
                )
        use_status = str(spec["use_status"])
        has_tops = bool(spec["has_approved_formation_tops"])
        # Increment 6.1 (Finding 2) invariant: a proxy-permitted well with NO
        # approved formation tops must carry the DEPTH-TIED status. The plain
        # `screening_proxy_allowed` status asserts that results can be tied to
        # approved stratigraphy; for a well with no tops that is false, and the
        # contradiction must fail loudly at config-load time rather than
        # surviving as a machine-readable claim that contradicts the prose.
        if use_status == USE_STATUS_PROXY_ALLOWED and not has_tops:
            raise PetrophysicsConfigError(
                f"{path.name}: well {well_key!r} declares use_status="
                f"{USE_STATUS_PROXY_ALLOWED!r} but has_approved_formation_tops=false. A "
                f"proxy-permitted well with no approved formation tops must use "
                f"{USE_STATUS_PROXY_ALLOWED_DEPTH_TIED!r}, because its results cannot be tied "
                f"to approved stratigraphy. Fix the config; this contradiction is never "
                f"silently accepted."
            )
        # The converse is also checked: claiming depth-tied status for a well
        # that DOES have approved tops understates what the data support and is
        # equally a config/reality mismatch.
        if use_status == USE_STATUS_PROXY_ALLOWED_DEPTH_TIED and has_tops:
            raise PetrophysicsConfigError(
                f"{path.name}: well {well_key!r} declares use_status="
                f"{USE_STATUS_PROXY_ALLOWED_DEPTH_TIED!r} but has_approved_formation_tops=true. "
                f"The depth-tied status is reserved for wells with NO approved formation tops; "
                f"a well that has them must use {USE_STATUS_PROXY_ALLOWED!r}."
            )

        wells[str(well_key)] = GrFamilyDisposition(
            well_key=str(well_key),
            source_las_filename=str(spec["source_las_filename"]),
            gr_family_canonical_name=str(spec["gr_family_canonical_name"]),
            gr_family_source_curve_name=str(spec["gr_family_source_curve_name"]),
            use_status=str(spec["use_status"]),
            exclusion_reason=(None if spec["exclusion_reason"] is None else str(spec["exclusion_reason"])),
            evidence_class=str(spec["evidence_class"]),
            has_approved_formation_tops=bool(spec["has_approved_formation_tops"]),
            notes=str(spec.get("notes", "") or ""),
        )

    return PetrophysicsEligibilityConfig(
        schema_version=str(raw["schema_version"]),
        increment=int(raw["increment"]),
        assurance_tier=str(raw["assurance_tier"]),
        policy=policy,
        endpoint_scenarios=tuple(scenarios),
        wells=wells,
        gr_proxy_confidence=raw["gr_proxy_confidence"],
        method_eligibility=raw["method_eligibility"],
        source_filename=path.name,
    )


# ---------------------------------------------------------------------------
# Contiguous-block detection (shared by QC stats and eligibility intervals)
# ---------------------------------------------------------------------------

def find_contiguous_blocks(
    mask: np.ndarray,
    depth_m: Optional[np.ndarray] = None,
    *,
    max_gap_samples: int = 0,
    max_gap_depth_m: float = 0.0,
) -> List[Tuple[int, int]]:
    """
    Return `[(start_index, end_index_inclusive), ...]` for runs of True in
    `mask`, optionally bridging short interruptions.

    A gap is bridged into the surrounding block ONLY when BOTH conditions
    hold:
      * its length is <= `max_gap_samples`, AND
      * (when `depth_m` is given) the physical depth span across the gap
        is <= `max_gap_depth_m`.

    Requiring both is the whole point: sample-count continuity and
    physical-depth continuity are different things, and a 2-sample gap
    that spans a 400 m depth jump is not a continuous interval. With the
    defaults (`max_gap_samples=0`) nothing is ever bridged.

    A missing interval is never silently bridged beyond these explicit,
    configured, tested tolerances, and nothing here extrapolates: bridged
    samples remain False in the caller's own mask - only the reported
    BLOCK spans them, and the caller can always recover the true valid
    count from the mask itself.
    """
    m = np.asarray(mask, dtype=bool)
    if m.ndim != 1:
        raise PetrophysicsInputError(
            f"find_contiguous_blocks: mask must be 1-D, got shape {m.shape}."
        )
    if depth_m is not None:
        d = np.asarray(depth_m, dtype=np.float64)
        if d.shape != m.shape:
            raise PetrophysicsInputError(
                f"find_contiguous_blocks: depth array shape {d.shape} does not match mask shape "
                f"{m.shape}."
            )
    else:
        d = None

    if not m.any():
        return []

    idx = np.flatnonzero(m)
    blocks: List[Tuple[int, int]] = []
    start = int(idx[0])
    prev = int(idx[0])
    for i in idx[1:]:
        i = int(i)
        gap = i - prev - 1
        bridge = False
        if 0 < gap <= int(max_gap_samples):
            if d is None:
                bridge = True
            else:
                span = abs(float(d[i]) - float(d[prev]))
                bridge = np.isfinite(span) and span <= float(max_gap_depth_m)
        if gap == 0 or bridge:
            prev = i
            continue
        blocks.append((start, prev))
        start = i
        prev = i
    blocks.append((start, prev))
    return blocks


# ---------------------------------------------------------------------------
# GR-family QC statistics
# ---------------------------------------------------------------------------

def compute_gr_family_qc_stats(
    frame: WellFrame,
    disposition: GrFamilyDisposition,
    *,
    seabed_mdrt_m: Optional[float] = None,
) -> GrFamilyQcStats:
    """
    Compute purely factual statistics for one well's GR-family curve.

    Computed for EVERY well regardless of `use_status`: factual QC
    reporting and availability disclosure are exactly what an excluded
    well remains available for, and the numbers that justify an exclusion
    must themselves be measured and published, not asserted.

    `seabed_mdrt_m`, when supplied, must come from the LOCKED Increment 5
    survey-corrected formation-top output. When it is None,
    `n_samples_above_seabed` is None and `seabed_basis` records that the
    quantity is not determinable from approved data - never 0, which would
    falsely assert that no such samples exist.
    """
    canonical = disposition.gr_family_canonical_name
    slot = frame.curve(canonical)
    if slot is None:
        return GrFamilyQcStats(
            well_key=frame.well_key,
            gr_family_canonical_name=canonical,
            gr_family_source_curve_name=disposition.gr_family_source_curve_name,
            source_las_filename=frame.source_las_filename,
            unit="API",
            n_samples=frame.n_samples,
            valid_count=0, valid_fraction=0.0,
            min_api=None, max_api=None, median_api=None,
            p01_api=None, p05_api=None, p10_api=None, p25_api=None, p50_api=None,
            p75_api=None, p90_api=None, p95_api=None, p99_api=None,
            dynamic_range_p05_p95_api=None,
            n_negative_samples=0, n_zero_samples=0,
            n_samples_above_seabed=None,
            seabed_basis="not_applicable_gr_family_curve_absent",
            n_valid_blocks=0, longest_valid_block_samples=0,
            longest_missing_block_samples=frame.n_samples,
            longest_missing_block_md_start_m=None, longest_missing_block_md_end_m=None,
            statistics_basis=(
                f"GR-family curve {canonical!r} is not present in this well's contract-resolved "
                f"curves; reported as a factual data gap, never substituted from another well."
            ),
        )

    gr = np.asarray(slot.values, dtype=np.float64)
    md = np.asarray(frame.MD_m, dtype=np.float64)
    valid = np.asarray(slot.valid_mask, dtype=bool)
    n = int(gr.size)
    vc = int(np.count_nonzero(valid))

    if vc:
        v = gr[valid]
        pcts = np.percentile(v, [1, 5, 10, 25, 50, 75, 90, 95, 99])
        p01, p05, p10, p25, p50, p75, p90, p95, p99 = (float(x) for x in pcts)
        vmin, vmax, vmed = float(np.min(v)), float(np.max(v)), float(np.median(v))
        dyn = p95 - p05
        n_neg = int(np.count_nonzero(v < 0.0))
        n_zero = int(np.count_nonzero(v == 0.0))
    else:
        p01 = p05 = p10 = p25 = p50 = p75 = p90 = p95 = p99 = None
        vmin = vmax = vmed = None
        dyn = None
        n_neg = n_zero = 0

    if seabed_mdrt_m is not None and np.isfinite(float(seabed_mdrt_m)):
        above = int(np.count_nonzero(md < float(seabed_mdrt_m)))
        basis = (
            f"Counted where canonical LAS MD_m < {float(seabed_mdrt_m):.4f} m MDRT, the seabed "
            f"marker's reconciled MDRT from the LOCKED Increment 5 survey-corrected output."
        )
    else:
        above = None
        basis = (
            "Not determinable: this well has no approved formation-top file, so no seabed marker "
            "exists in the locked Increment 5 output. Reported as None (unknown), never as 0."
        )

    valid_blocks = find_contiguous_blocks(valid)
    longest_valid = max((e - s + 1 for s, e in valid_blocks), default=0)
    missing_blocks = find_contiguous_blocks(~valid)
    if missing_blocks:
        s, e = max(missing_blocks, key=lambda b: b[1] - b[0])
        longest_missing = e - s + 1
        miss_start_md: Optional[float] = float(md[s])
        miss_end_md: Optional[float] = float(md[e])
    else:
        longest_missing = 0
        miss_start_md = miss_end_md = None

    return GrFamilyQcStats(
        well_key=frame.well_key,
        gr_family_canonical_name=canonical,
        gr_family_source_curve_name=slot.source_curve_name,
        source_las_filename=frame.source_las_filename,
        unit=slot.canonical_unit or "API",
        n_samples=n,
        valid_count=vc,
        valid_fraction=(vc / n) if n else 0.0,
        min_api=vmin, max_api=vmax, median_api=vmed,
        p01_api=p01, p05_api=p05, p10_api=p10, p25_api=p25, p50_api=p50,
        p75_api=p75, p90_api=p90, p95_api=p95, p99_api=p99,
        dynamic_range_p05_p95_api=dyn,
        n_negative_samples=n_neg, n_zero_samples=n_zero,
        n_samples_above_seabed=above, seabed_basis=basis,
        n_valid_blocks=len(valid_blocks),
        longest_valid_block_samples=int(longest_valid),
        longest_missing_block_samples=int(longest_missing),
        longest_missing_block_md_start_m=miss_start_md,
        longest_missing_block_md_end_m=miss_end_md,
        statistics_basis=(
            "Descriptive statistics over finite samples of this well's OWN GR-family curve, in its "
            "own recorded API units. No environmental correction, rescaling, normalization, or "
            "cross-well transfer of any kind has been applied. These numbers describe the curve as "
            "recorded and imply no lithology."
        ),
    )


# ---------------------------------------------------------------------------
# Endpoint scenarios
# ---------------------------------------------------------------------------

def resolve_endpoint_scenarios(
    frame: WellFrame,
    disposition: GrFamilyDisposition,
    config: PetrophysicsEligibilityConfig,
) -> Tuple[GrEndpointScenario, ...]:
    """
    Compute the MEASURED endpoint values each configured scenario produces
    for THIS well, from THIS well's own valid, depth-mapped GR samples.

    Raises
    ------
    PetrophysicsExclusionError
        If this well's configured `use_status` forbids GR-derived
        calculation. An excluded well gets no endpoints at all - not even
        "for reference" - because a published endpoint pair is exactly the
        artefact someone would later be tempted to rescale the curve with.
    PetrophysicsInputError
        If the well has no valid, depth-mapped GR samples to estimate from,
        or if a scenario's rule produces a degenerate endpoint pair.
    """
    if not disposition.proxy_permitted:
        raise PetrophysicsExclusionError(
            f"{frame.well_key}: GR-derived calculation is forbidden for this well "
            f"(use_status={disposition.use_status!r}, exclusion_reason="
            f"{disposition.exclusion_reason!r}). No endpoint, index, proxy, flag, "
            f"lithology-dependent mask, or NCT-donor status may be computed for it."
        )

    slot = frame.curve(disposition.gr_family_canonical_name)
    if slot is None:
        raise PetrophysicsInputError(
            f"{frame.well_key}: GR-family curve {disposition.gr_family_canonical_name!r} is absent; "
            f"endpoints cannot be estimated and are never borrowed from another well."
        )

    gr = np.asarray(slot.values, dtype=np.float64)
    usable = np.asarray(slot.valid_mask, dtype=bool) & np.asarray(frame.depth_valid_mask, dtype=bool)
    n_used = int(np.count_nonzero(usable))
    if n_used == 0:
        raise PetrophysicsInputError(
            f"{frame.well_key}: no finite, depth-mapped GR-family sample exists; endpoints cannot "
            f"be estimated from zero valid coverage."
        )

    v = gr[usable]
    min_sep = float(config.policy["min_endpoint_separation_api"])
    basis = str(config.policy["endpoint_sample_basis"])

    out: List[GrEndpointScenario] = []
    for sc in config.endpoint_scenarios:
        name = str(sc["scenario_name"])
        lo_p = float(sc["low_percentile"])
        hi_p = float(sc["high_percentile"])
        lo_raw = float(np.percentile(v, lo_p))
        hi_raw = float(np.percentile(v, hi_p))
        lo, hi = _validate_endpoints(
            lo_raw, hi_raw,
            f"{frame.well_key}/{name} endpoints (p{lo_p:g}/p{hi_p:g} of "
            f"{disposition.gr_family_canonical_name})",
            min_sep,
        )
        out.append(
            GrEndpointScenario(
                well_key=frame.well_key,
                scenario_name=name,
                low_percentile=lo_p,
                high_percentile=hi_p,
                gr_low_endpoint_api=lo,
                gr_high_endpoint_api=hi,
                endpoint_separation_api=hi - lo,
                n_samples_used_for_endpoints=n_used,
                endpoint_sample_basis=basis,
                description=str(sc["description"]).strip(),
            )
        )
    return tuple(out)


# ---------------------------------------------------------------------------
# GR index and screening proxy
# ---------------------------------------------------------------------------

def compute_gr_index(
    gr_values,
    gr_low_endpoint,
    gr_high_endpoint,
    *,
    context: str = "compute_gr_index",
    min_endpoint_separation_api: float = 1.0,
    valid_mask=None,
) -> Tuple[np.ndarray, np.ndarray, np.ndarray]:
    """
    Compute `(igr_unclipped, igr_clipped, valid_mask)` from a GR array and
    an explicit endpoint pair.

        IGR_unclipped = (GR - low) / (high - low)
        IGR_clipped   = clip(IGR_unclipped, 0, 1)

    Both arrays are full-length and aligned with `gr_values`. A sample
    that is NaN on input stays NaN in both outputs and False in the
    returned mask - a NaN is never coerced to 0, to an endpoint, or to a
    neighbouring value. An infinite input sample is explicitly INVALIDATED
    (masked out and set to NaN) rather than propagating a signed infinity
    through the division: an infinite gamma-ray reading is not a
    measurement, and silently normalizing it would produce a
    plus/minus-infinity IGR that clipping would then quietly turn into a
    clean-looking 0 or 1.

    `valid_mask`, when supplied, is ANDed with the finiteness mask - used
    by callers to additionally require a mapped depth.
    """
    arr = _validate_numeric_1d(gr_values, f"{context}: gr_values")
    lo, hi = _validate_endpoints(
        gr_low_endpoint, gr_high_endpoint, context, min_endpoint_separation_api
    )

    finite = np.isfinite(arr)
    if valid_mask is not None:
        extra = np.asarray(valid_mask)
        if extra.dtype != np.bool_:
            raise TypeError(
                f"{context}: valid_mask must be a boolean array, got dtype {extra.dtype!r}."
            )
        if extra.shape != arr.shape:
            raise PetrophysicsInputError(
                f"{context}: valid_mask shape {extra.shape} does not match gr_values shape "
                f"{arr.shape}; a length mismatch is never reconciled by truncation or padding."
            )
        finite = finite & extra

    igr_unclipped = np.full(arr.shape, np.nan, dtype=np.float64)
    np.divide(arr - lo, hi - lo, out=igr_unclipped, where=finite)
    igr_clipped = np.full(arr.shape, np.nan, dtype=np.float64)
    np.clip(igr_unclipped, 0.0, 1.0, out=igr_clipped, where=finite)
    # `np.clip(..., where=)` leaves untouched positions at their `out`
    # initial value (NaN), which is exactly what is wanted; re-assert it
    # explicitly so the invariant does not depend on that subtlety.
    igr_clipped[~finite] = np.nan
    igr_unclipped[~finite] = np.nan
    return igr_unclipped, igr_clipped, finite


def compute_gr_proxy(
    frame: WellFrame,
    disposition: GrFamilyDisposition,
    scenario: GrEndpointScenario,
    config: PetrophysicsEligibilityConfig,
) -> GrProxyResult:
    """
    Compute one well's GR index and linear screening proxy under ONE
    endpoint scenario.

    The proxy is the LINEAR IDENTITY of the clipped index - the only
    transform Increment 6 permits. It is named
    `VSH_GR_linear_proxy_frac` and is documented, in every field and
    every export, as an uncalibrated screening proxy that is NOT a shale
    volume and NOT a lithology.

    Raises `PetrophysicsExclusionError` for a well whose disposition
    forbids GR-derived calculation.
    """
    if not disposition.proxy_permitted:
        raise PetrophysicsExclusionError(
            f"{frame.well_key}: GR-derived screening proxy is forbidden for this well "
            f"(use_status={disposition.use_status!r}, exclusion_reason="
            f"{disposition.exclusion_reason!r})."
        )
    if scenario.well_key != frame.well_key:
        raise PetrophysicsInputError(
            f"Endpoint scenario belongs to well {scenario.well_key!r} but was supplied for well "
            f"{frame.well_key!r}; endpoints are per-well and are never transferred between wells."
        )

    slot = frame.curve(disposition.gr_family_canonical_name)
    if slot is None:
        raise PetrophysicsInputError(
            f"{frame.well_key}: GR-family curve {disposition.gr_family_canonical_name!r} is absent."
        )

    igr_unclipped, igr_clipped, valid = compute_gr_index(
        np.asarray(slot.values, dtype=np.float64),
        scenario.gr_low_endpoint_api,
        scenario.gr_high_endpoint_api,
        context=f"{frame.well_key}/{scenario.scenario_name}",
        min_endpoint_separation_api=float(config.policy["min_endpoint_separation_api"]),
        valid_mask=np.asarray(frame.depth_valid_mask, dtype=bool),
    )

    proxy = igr_clipped.copy()  # linear identity transform, by policy
    n_valid = int(np.count_nonzero(valid))
    n_low = int(np.count_nonzero(valid & (igr_unclipped < 0.0)))
    n_high = int(np.count_nonzero(valid & (igr_unclipped > 1.0)))

    if n_valid:
        pv = proxy[valid]
        med: Optional[float] = float(np.median(pv))
        q25: Optional[float] = float(np.percentile(pv, 25))
        q75: Optional[float] = float(np.percentile(pv, 75))
    else:
        med = q25 = q75 = None

    return GrProxyResult(
        well_key=frame.well_key,
        scenario_name=scenario.scenario_name,
        gr_family_canonical_name=disposition.gr_family_canonical_name,
        gr_low_endpoint_api=scenario.gr_low_endpoint_api,
        gr_high_endpoint_api=scenario.gr_high_endpoint_api,
        igr_unclipped_frac=igr_unclipped,
        igr_clipped_frac=igr_clipped,
        VSH_GR_linear_proxy_frac=proxy,
        valid_mask=valid,
        n_valid=n_valid,
        n_clipped_low=n_low,
        n_clipped_high=n_high,
        clipped_fraction=((n_low + n_high) / n_valid) if n_valid else 0.0,
        proxy_median=med, proxy_p25=q25, proxy_p75=q75,
    )


# ---------------------------------------------------------------------------
# Data/proxy confidence classification
# ---------------------------------------------------------------------------

def classify_gr_proxy_confidence(
    stats: GrFamilyQcStats,
    disposition: GrFamilyDisposition,
    proxy_results: Sequence[GrProxyResult],
    config: PetrophysicsEligibilityConfig,
) -> Tuple[str, str]:
    """
    Return `(confidence_class, rationale)` describing confidence in the
    DATA and the PROXY - never a lithology.

    The returned class is asserted against the prohibited named-lithology
    vocabulary before it is returned, so a rock name can never leave this
    function even if the config were edited to introduce one.

    Ordering of the checks matters: an explicitly excluded well is
    classified as excluded regardless of how good its coverage statistics
    look, because the exclusion is about an unresolved SCALE anomaly that
    good coverage does nothing to resolve.
    """
    if not disposition.proxy_permitted:
        cls = GR_EXCLUDED_UNRESOLVED_SCALE
        rationale = (
            f"Formally excluded from every GR-derived calculation "
            f"(exclusion_reason={disposition.exclusion_reason!r}). The curve remains available for "
            f"factual raw QC display and availability reporting only. Good coverage does not "
            f"resolve an unresolved scale/acquisition anomaly, so coverage statistics do not "
            f"override this classification."
        )
    elif stats.valid_count == 0 or stats.dynamic_range_p05_p95_api is None:
        cls = GR_NOT_AVAILABLE
        rationale = "No finite GR-family sample is available for this well; reported as a factual data gap."
    else:
        rules = config.gr_proxy_confidence["rules"]
        medians = [p.proxy_median for p in proxy_results if p.proxy_median is not None]
        spread = (max(medians) - min(medians)) if len(medians) >= 2 else 0.0
        vf = float(stats.valid_fraction)
        dr = float(stats.dynamic_range_p05_p95_api)
        hi = rules["high"]
        mid = rules["intermediate"]
        if (
            vf >= float(hi["min_valid_fraction"])
            and dr >= float(hi["min_dynamic_range_api"])
            and spread <= float(hi["max_proxy_median_spread_across_scenarios"])
        ):
            cls = GR_PROXY_HIGH
        elif (
            vf >= float(mid["min_valid_fraction"])
            and dr >= float(mid["min_dynamic_range_api"])
            and spread <= float(mid["max_proxy_median_spread_across_scenarios"])
        ):
            cls = GR_PROXY_INTERMEDIATE
        else:
            cls = GR_PROXY_LOW
        rationale = (
            f"valid_fraction={vf:.4f}, dynamic_range_p05_p95={dr:.3f} API, "
            f"proxy_median_spread_across_{len(medians)}_scenarios={spread:.4f}. Describes "
            f"confidence in the DATA and the SCREENING PROXY only; asserts no lithology and no "
            f"calibration."
        )

    assert_no_lithology_vocabulary(cls, f"{stats.well_key}: gr_proxy_confidence class")
    return cls, rationale


#### `p2mem/method_eligibility.py` — the three masks and contiguous-interval registers

Each mask answers only *"is this sample technically admissible as INPUT to a later method?"* — never whether the method is appropriate, whether its result would be defensible, whether an interval is normally compacted, or what rock it is.

Masks 1 and 2 are lithology-independent and are computed for **every** well including a GR-excluded one (excluding Boreas 1 from GR-derived work says nothing about whether its density or sonic samples are finite). Mask 3 consumes the GR proxy and is therefore **never** computed for a GR-excluded well — attempting it raises.

In [ ]:
%%writefile p2mem/method_eligibility.py
"""
p2mem.method_eligibility - Increment 6 method-eligibility masks and
contiguous-interval registers.

ELIGIBILITY IS NOT VALIDITY
---------------------------
Every mask in this module answers one narrow question:

    "Is this sample technically ADMISSIBLE as INPUT to a later method?"

It never answers any of these:

    "Is that method appropriate here?"
    "Would its result be defensible?"
    "Is this interval normally compacted?"
    "What rock is this?"

Eligibility is a NECESSARY, not a SUFFICIENT, condition. A sample can be
eligible for a density-based overburden integration and that integration
can still be indefensible - for instance because the density log starts
hundreds of metres below the seabed, so no amount of per-sample
eligibility supplies the missing shallow section. This distinction is the
reason these masks are computed in a separate increment from the methods
they gate, and it is restated in every exported table.

The three masks
---------------
1. `eligible_density_for_sv`
   Finite, physically plausible RHOB at a mapped depth. Increment 6 does
   NOT fill missing density and does NOT compute Sv.

2. `eligible_dynamic_elastic`
   Finite, positive, physically plausible VP, VS and RHOB at a mapped
   depth, with VP > VS and a Vp/Vs ratio that passes the CONFIGURED
   NON-NEGATIVE-POISSON-RATIO APPLICABILITY SCREEN - an INCLUSIVE
   `Vp/Vs >= sqrt(2)` bound, together with a configured plausibility
   maximum. That screen is a conservative PROJECT POLICY about what to
   admit to a later dynamic-elastic calculation; it is NOT a
   physical-possibility test and NOT a boundary of the mathematical
   Poisson domain. `Vp/Vs = sqrt(2)` gives a Poisson's ratio of exactly
   zero and is ACCEPTED; ratios below it are excluded by policy and are
   diagnosed BY REGIME (non-positive bulk modulus versus positive bulk
   modulus with negative Poisson's ratio), never aggregated under a
   single "non-physical" label. See `compute_dynamic_elastic_eligibility`
   for the full derivation. Increment 6 computes NO elastic property -
   not Young's modulus, not Poisson's ratio, not bulk or shear modulus.
   This is an input-admissibility mask only.

3. `eligible_sonic_nct_candidate`
   CANDIDATE DATA for a later sonic normal-compaction-trend analysis:
   an approved GR-family disposition, finite VP, a finite screening
   proxy at or above a scenario-specific threshold, and a mapped depth.
   It does NOT fit a trend, does NOT select a donor interval, does NOT
   claim normal compaction, and does NOT claim overpressure. A candidate
   mask is emphatically NOT proof that an interval is normally compacted,
   and it names no lithology.

Both mask 1 and mask 2 are lithology-independent and are therefore
computed for EVERY well, including a GR-excluded one - excluding Boreas 1
from GR-derived work says nothing about whether its density or sonic
samples are finite. Mask 3 is lithology-dependent (it consumes the GR
screening proxy) and is therefore NEVER computed for a GR-excluded well.
"""

from __future__ import annotations

from typing import Dict, List, Optional, Sequence, Tuple

import math

import numpy as np

from p2mem.petrophysics import PetrophysicsInputError, find_contiguous_blocks
from p2mem.petrophysics_models import (
    GrFamilyDisposition,
    GrProxyResult,
    PetrophysicsEligibilityConfig,
)
from p2mem.wellframe_models import WellFrame

__all__ = [
    "MASK_DENSITY_FOR_SV",
    "MASK_DYNAMIC_ELASTIC",
    "MASK_SONIC_NCT_CANDIDATE",
    "VALID_MASK_NAMES",
    "EligibilityMaskResult",
    "EligibilityInterval",
    "compute_density_eligibility",
    "compute_dynamic_elastic_eligibility",
    "compute_sonic_nct_candidate_eligibility",
    "build_eligibility_intervals",
    "CONTIGUITY_POLICY_CONFIGURED",
    "CONTIGUITY_POLICY_STRICT",
    "VALID_CONTIGUITY_POLICIES",
]

MASK_DENSITY_FOR_SV = "eligible_density_for_sv"
MASK_DYNAMIC_ELASTIC = "eligible_dynamic_elastic"
MASK_SONIC_NCT_CANDIDATE = "eligible_sonic_nct_candidate"

VALID_MASK_NAMES: Tuple[str, ...] = (
    MASK_DENSITY_FOR_SV,
    MASK_DYNAMIC_ELASTIC,
    MASK_SONIC_NCT_CANDIDATE,
)


class EligibilityMaskResult:
    """
    One well's boolean eligibility mask for one method, plus the per-
    criterion counts that explain it.

    `criteria_counts` records how many samples PASSED each individual
    criterion, so a reader can see which requirement actually limited the
    result rather than only the final total. `limiting_criterion` names
    the single most restrictive one. Neither is an interpretation - both
    are counts.
    """

    __slots__ = (
        "well_key", "mask_name", "scenario_name", "proxy_threshold", "mask",
        "n_samples", "n_eligible", "eligible_fraction", "criteria_counts",
        "diagnostic_counts", "limiting_criterion", "lithology_dependent",
        "purpose", "notes",
    )

    def __init__(
        self,
        well_key: str,
        mask_name: str,
        mask: np.ndarray,
        criteria_counts: Dict[str, int],
        *,
        diagnostic_counts: Optional[Dict[str, int]] = None,
        scenario_name: Optional[str] = None,
        proxy_threshold: Optional[float] = None,
        lithology_dependent: bool = False,
        purpose: str = "",
        notes: str = "",
    ) -> None:
        if mask_name not in VALID_MASK_NAMES:
            raise ValueError(f"Unknown mask_name {mask_name!r}; expected one of {VALID_MASK_NAMES}.")
        m = np.asarray(mask, dtype=bool)
        if m.ndim != 1:
            raise PetrophysicsInputError(f"{well_key}/{mask_name}: mask must be 1-D, got {m.shape}.")
        self.well_key = well_key
        self.mask_name = mask_name
        self.scenario_name = scenario_name
        self.proxy_threshold = proxy_threshold
        self.mask = m
        self.n_samples = int(m.size)
        self.n_eligible = int(np.count_nonzero(m))
        self.eligible_fraction = (self.n_eligible / self.n_samples) if self.n_samples else 0.0
        self.criteria_counts = dict(criteria_counts)
        # Reported alongside, but deliberately EXCLUDED from the
        # limiting-criterion comparison: these are conditional counts whose
        # magnitude is bounded by another criterion, so comparing them
        # directly against unconditional pass counts would misattribute the
        # cause of a low eligible fraction.
        self.diagnostic_counts = dict(diagnostic_counts or {})
        self.limiting_criterion = (
            min(criteria_counts, key=lambda k: criteria_counts[k]) if criteria_counts else ""
        )
        self.lithology_dependent = bool(lithology_dependent)
        self.purpose = purpose
        self.notes = notes

    def __repr__(self) -> str:  # pragma: no cover - debug convenience only
        return (
            f"EligibilityMaskResult({self.well_key!r}, {self.mask_name!r}, "
            f"scenario={self.scenario_name!r}, n_eligible={self.n_eligible}/{self.n_samples})"
        )


class EligibilityInterval:
    """
    One contiguous block of eligible samples, reported on all three depth
    references (MD, TVD, TVDSS) so a reader never has to guess which basis
    a thickness refers to.

    Increment 6.1 (Finding 4): thickness is reported in THREE explicitly
    named forms, because the single unqualified "thickness" of Increment 6
    conflated them:

      * `gross_thickness_*_m` - the block's ENDPOINT SPAN, first sample to
        last. Under the configured contiguity policy this span may CONTAIN
        explicitly bridged ineligible samples, so it is a gross figure and
        is named as one. It is never called simply "eligible thickness".

      * `net_thickness_*_m` - the sum of the depth spans of the maximal
        STRICTLY-CONTIGUOUS eligible sub-runs inside this block, i.e. the
        gross span minus the spans of the bridged gaps. This is the
        rigorous "how much eligible section is actually here" figure.

    Increment 6.1.1 (Finding 3) replaces the ambiguous interruption fields
    with three that each state exactly what they count:

      * `n_bridged_samples`   - total ineligible samples absorbed INSIDE the
                                gross block;
      * `n_bridged_gaps`      - number of DISTINCT bridged False runs;
      * `n_eligible_subruns`  - number of strict contiguous eligible sub-runs
                                inside the gross block.

    The previous `n_interruptions` was a boolean-like flag masquerading as a
    count (it was 1 for every one of the 327 bridged blocks in the real data,
    whatever the actual number of gaps), and `n_interrupted_subruns` did not
    say what it counted. Both are gone; no ambiguous alias survives in the
    active exports.

    Invariants, enforced at construction and asserted by tests:

        no gap            -> n_bridged_samples == 0
                             n_bridged_gaps    == 0
                             n_eligible_subruns == 1
        two bridged gaps  -> n_bridged_gaps    == 2
                             n_eligible_subruns == 3
        general           -> n_eligible_subruns > 0 implies
                             n_bridged_gaps == n_eligible_subruns - 1

    `contiguity_policy` records which decomposition produced this record:
    `configured_bridging` (the project's configured gap tolerances) or
    `strict_no_gap` (no bridging at all). Under `strict_no_gap`, gross and
    net are equal by construction.

    A thickness is None when either end of the relevant span has no mapped
    depth - never 0, and never silently substituted with the MD span, which
    in a deviated well is a different quantity.
    """

    __slots__ = (
        "well_key", "mask_name", "scenario_name", "proxy_threshold", "block_index",
        "start_index", "end_index", "n_samples", "n_eligible_samples",
        "md_start_m", "md_end_m", "gross_thickness_md_m", "net_thickness_md_m",
        "tvd_start_m", "tvd_end_m", "gross_thickness_tvd_m", "net_thickness_tvd_m",
        "tvdss_start_m", "tvdss_end_m", "gross_thickness_tvdss_m", "net_thickness_tvdss_m",
        "n_bridged_samples", "n_bridged_gaps", "n_eligible_subruns",
        "contiguity_policy", "meets_configured_minimums", "limiting_reason",
        "depth_basis_used",
    )

    #: The three interruption-count fields, validated together as one record.
    COUNT_FIELDS = ("n_bridged_samples", "n_bridged_gaps", "n_eligible_subruns")

    #: Field names removed in Increment 6.1.1. Named explicitly so a caller
    #: still using the ambiguous schema gets a message that says so, rather
    #: than the generic unknown-keyword message.
    REMOVED_COUNT_FIELDS = ("n_interruptions", "n_interrupted_subruns")

    def _require_count(self, name: str) -> int:
        """Return a validated non-negative integer count for `name`.

        Increment 6.1.2 enforced the relational rules but let the TYPE gate
        through in three ways an audit found: `0.0 / 0.0 / 1.0` was accepted
        although the contract requires integers, and NaN / Inf escaped as a
        bare `ValueError` / `OverflowError` from `int()` rather than as a
        typed `PetrophysicsInputError`. Increment 6.1.3 decides the type
        before any conversion is attempted, so no arithmetic on an
        unvalidated value can raise first.
        """
        value = getattr(self, name)
        where = f"{self.well_key}/{self.mask_name}"
        if value is None:
            raise PetrophysicsInputError(
                f"{where}: interval record is missing required count {name!r}. "
                f"All of {self.COUNT_FIELDS} must be supplied together."
            )
        # `bool` is a subclass of `int`; a True/False count is a category
        # error - it is the very confusion the removed `n_interruptions` flag
        # embodied - and must not be silently read as 1/0.
        if isinstance(value, bool):
            raise PetrophysicsInputError(
                f"{where}: {name}={value!r} is a boolean, not a count. "
                f"A count says HOW MANY, not whether."
            )
        if isinstance(value, (int, np.integer)):
            return self._check_non_negative(name, int(value), where)
        # Floats are rejected OUTRIGHT, including whole-valued ones. The
        # contract is an integer count; `0.0` is not `0`, and accepting it
        # would mean the type gate depends on the value. Non-finite values are
        # named specifically, because `float('nan')` reaching `int()` is what
        # produced an untyped exception before.
        if isinstance(value, (float, np.floating)):
            if not math.isfinite(float(value)):
                raise PetrophysicsInputError(
                    f"{where}: {name}={value!r} is not finite; a count must be a "
                    f"finite integer."
                )
            raise PetrophysicsInputError(
                f"{where}: {name}={value!r} is a float; an integer count is required "
                f"({float(value)!r} is not {int(value)!r}). Counts are never coerced "
                f"from another numeric type."
            )
        raise PetrophysicsInputError(
            f"{where}: {name}={value!r} has type {type(value).__name__}; an integer "
            f"count is required (strings, complex values and other types are never "
            f"coerced)."
        )

    def _check_non_negative(self, name: str, ivalue: int, where: str) -> int:
        if ivalue < 0:
            raise PetrophysicsInputError(
                f"{where}: {name}={ivalue} is negative; counts cannot be negative."
            )
        return ivalue

    def __init__(self, **kwargs) -> None:
        # Increment 6.1.3: keyword acceptance is a WHITELIST, not a denylist.
        # Increment 6.1.2 rejected only the two known legacy names, so a typo
        # such as `n_bridged_sample=99` was silently ignored and the record was
        # built with that count unset - the same category error as guarding a
        # prohibition with an allowlist. Only real slots are accepted now.
        unknown = sorted(set(kwargs) - set(self.__slots__))
        if unknown:
            legacy = [k for k in unknown if k in self.REMOVED_COUNT_FIELDS]
            if legacy:
                raise PetrophysicsInputError(
                    f"{kwargs.get('well_key')}/{kwargs.get('mask_name')}: field(s) "
                    f"{legacy} were removed in Increment 6.1.1 and have no replacement "
                    f"alias. Use {self.COUNT_FIELDS} instead."
                )
            raise PetrophysicsInputError(
                f"{kwargs.get('well_key')}/{kwargs.get('mask_name')}: unknown field(s) "
                f"{unknown}. An interval record accepts only its declared fields, so a "
                f"misspelled count cannot silently leave the real one unset."
            )
        for slot in self.__slots__:
            setattr(self, slot, kwargs.get(slot))

        # Per-field validation first, so an error names the offending field.
        n_samples = self._require_count("n_bridged_samples")
        n_gaps = self._require_count("n_bridged_gaps")
        n_runs = self._require_count("n_eligible_subruns")
        self.n_bridged_samples, self.n_bridged_gaps = n_samples, n_gaps
        self.n_eligible_subruns = n_runs
        where = f"{self.well_key}/{self.mask_name}"

        # A block exists, so it has at least one eligible sub-run.
        if n_runs < 1:
            raise PetrophysicsInputError(
                f"{where}: n_eligible_subruns={n_runs}; a constructed interval "
                f"record describes an existing block and must have at least one "
                f"strictly contiguous eligible sub-run."
            )
        # The gap/sub-run relationship is structural, not incidental: cutting a
        # block into k pieces takes exactly k-1 cuts.
        if n_gaps != n_runs - 1:
            raise PetrophysicsInputError(
                f"{where}: inconsistent interval record - "
                f"n_eligible_subruns={n_runs} requires n_bridged_gaps={n_runs - 1}, "
                f"got {n_gaps}."
            )
        # Samples and gaps must agree about whether any bridging happened at
        # all. Either both are zero or both are positive - "5 bridged samples
        # in 0 gaps" and "2 gaps holding 0 samples" are each impossible.
        if (n_samples == 0) != (n_gaps == 0):
            raise PetrophysicsInputError(
                f"{where}: n_bridged_samples={n_samples} and n_bridged_gaps="
                f"{n_gaps} disagree - bridged samples exist if and only if a "
                f"bridged gap exists."
            )
        # Every distinct gap holds at least one sample, so samples >= gaps.
        if n_gaps > 0 and n_samples < n_gaps:
            raise PetrophysicsInputError(
                f"{where}: n_bridged_samples={n_samples} is fewer than "
                f"n_bridged_gaps={n_gaps}; every distinct bridged gap contains "
                f"at least one ineligible sample."
            )

    def as_dict(self) -> Dict:
        """Plain-Python dict view, JSON/CSV-safe (no NumPy scalars)."""
        out = {}
        for slot in self.__slots__:
            v = getattr(self, slot)
            if isinstance(v, (np.floating,)):
                v = float(v)
            elif isinstance(v, (np.integer,)):
                v = int(v)
            elif isinstance(v, (np.bool_,)):
                v = bool(v)
            out[slot] = v
        return out


def _purpose(config: PetrophysicsEligibilityConfig, mask_name: str) -> str:
    """The configured human-authored purpose string for a mask.

    Returns "" when the config carries no descriptive entry. Descriptive
    metadata is validated at CONFIG LOAD time (see
    `p2mem.petrophysics.load_petrophysics_eligibility_config`), which is
    where a real misconfiguration must fail loudly; a missing description
    must never abort a numerical computation that is otherwise correct.
    """
    entry = (config.method_eligibility or {}).get(mask_name) or {}
    return str(entry.get("purpose", "")).strip()


def _finite_within(values: Optional[np.ndarray], lo: float, hi: float) -> np.ndarray:
    """True where `values` is finite AND within [lo, hi]. An absent curve
    yields an all-False mask of the caller's length - a factual data gap,
    never an implicit pass."""
    if values is None:
        return None  # caller substitutes a correctly sized all-False mask
    v = np.asarray(values, dtype=np.float64)
    return np.isfinite(v) & (v >= lo) & (v <= hi)


def compute_density_eligibility(
    frame: WellFrame, config: PetrophysicsEligibilityConfig
) -> EligibilityMaskResult:
    """
    Mask samples technically admissible as input to a LATER vertical-stress
    (Sv) integration: finite, physically plausible RHOB at a mapped depth.

    Lithology-independent, so computed for every well including a
    GR-excluded one. This function does not fill missing density and does
    not compute Sv.
    """
    n = frame.n_samples
    bounds = config.policy["physical_bounds"]
    rhob = frame.values_or_none("RHOB_kg_m3")

    if rhob is None:
        rhob_finite = np.zeros(n, dtype=bool)
        rhob_in_bounds = np.zeros(n, dtype=bool)
    else:
        r = np.asarray(rhob, dtype=np.float64)
        rhob_finite = np.isfinite(r)
        rhob_in_bounds = rhob_finite & (r >= float(bounds["rhob_min_kg_m3"])) & (
            r <= float(bounds["rhob_max_kg_m3"])
        )

    depth_ok = np.asarray(frame.depth_valid_mask, dtype=bool)
    mask = rhob_in_bounds & depth_ok

    return EligibilityMaskResult(
        frame.well_key, MASK_DENSITY_FOR_SV, mask,
        {
            "rhob_finite": int(np.count_nonzero(rhob_finite)),
            "rhob_within_physical_bounds": int(np.count_nonzero(rhob_in_bounds)),
            "depth_mapped": int(np.count_nonzero(depth_ok)),
        },
        lithology_dependent=False,
        purpose=_purpose(config, MASK_DENSITY_FOR_SV),
        notes=(
            "Eligibility is a necessary, not sufficient, condition. Increment 6 does not fill "
            "missing density and does not compute vertical stress. A log that begins well below "
            "the seabed cannot support an overburden integral from surface regardless of how many "
            "of its own samples are eligible."
        ),
    )


def compute_dynamic_elastic_eligibility(
    frame: WellFrame, config: PetrophysicsEligibilityConfig
) -> EligibilityMaskResult:
    r"""
    Mask samples technically admissible as input to a LATER dynamic-elastic
    calculation: finite, physically plausible VP, VS and RHOB at a mapped
    depth, with VP > VS and a Vp/Vs ratio that passes the CONFIGURED
    NON-NEGATIVE-POISSON-RATIO APPLICABILITY SCREEN.

    The Vp/Vs domain, stated precisely (Increment 6.1 correction)
    ------------------------------------------------------------
    For an isotropic elastic solid with r = Vp/Vs:

        nu = (r^2 - 2) / (2 * (r^2 - 1))          [dynamic Poisson's ratio]
        K  = rho * (Vp^2 - (4/3) * Vs^2)          [bulk modulus]

    from which four DISTINCT regimes follow, which Increment 6 wrongly
    collapsed into a single "non-physical" label:

      * r <= sqrt(4/3) ~= 1.154701
            K <= 0. A non-positive bulk modulus IS outside the isotropic
            elastic model - this is the only regime that genuinely
            warrants the word "non-physical" under that model.

      * sqrt(4/3) < r < sqrt(2)
            K > 0 but nu < 0. A negative Poisson's ratio is UNUSUAL and
            is outside this project's conservative applicability policy,
            but it is NOT mathematically impossible and NOT "non-physical"
            - auxetic behavior is a real, if rare, elastic response. These
            samples are excluded by POLICY, and are diagnosed separately
            so a reader can see that the exclusion is a policy choice
            rather than a physical impossibility.

      * r = sqrt(2) EXACTLY
            nu = 0 exactly. A policy requiring a NON-NEGATIVE Poisson's
            ratio must ACCEPT this value. Increment 6 used a strictly
            exclusive bound and therefore wrongly rejected nu = 0; the
            bound is INCLUSIVE here (`ratio >= sqrt(2)`).

      * r > ratio_max (configured, 4.0)
            nu ~= 0.467 at r = 4 - an entirely ordinary Poisson's ratio.
            This bound is a CONFIGURED PLAUSIBILITY LIMIT reflecting what
            this project is willing to treat as a credible logged ratio,
            NOT a boundary of the mathematical Poisson domain.

    The screen is applied on the RATIO directly, with an inclusive
    `>= sqrt(2)` comparison against the configured constant, rather than
    on a floating-point evaluation of `nu >= 0`. This matters: evaluating
    the nu expression at r = sqrt(2) in IEEE-754 returns ~2.2e-16 rather
    than exactly 0, so a naive `nu >= 0` test would be decided by rounding
    noise at precisely the boundary the policy is about.

    Excluded samples are never deleted from the well frame and never
    "corrected". Increment 6 computes NO elastic property whatsoever -
    no Young's modulus, no Poisson's ratio curve, no bulk or shear
    modulus. The nu and K relations above are stated to DEFINE the
    screen's boundaries, not to compute anything.
    """
    n = frame.n_samples
    b = config.policy["physical_bounds"]
    vp_arr = frame.values_or_none("VP_m_s")
    vs_arr = frame.values_or_none("VS_m_s")
    rhob_arr = frame.values_or_none("RHOB_kg_m3")

    vp_ok = _finite_within(vp_arr, float(b["vp_min_m_s"]), float(b["vp_max_m_s"]))
    vs_ok = _finite_within(vs_arr, float(b["vs_min_m_s"]), float(b["vs_max_m_s"]))
    rhob_ok = _finite_within(rhob_arr, float(b["rhob_min_kg_m3"]), float(b["rhob_max_kg_m3"]))
    vp_ok = np.zeros(n, dtype=bool) if vp_ok is None else vp_ok
    vs_ok = np.zeros(n, dtype=bool) if vs_ok is None else vs_ok
    rhob_ok = np.zeros(n, dtype=bool) if rhob_ok is None else rhob_ok

    r_nu_min = float(b["vp_vs_ratio_nonnegative_poisson_min_inclusive"])
    r_k_bound = float(b["vp_vs_ratio_positive_bulk_modulus_min_exclusive"])
    r_max = float(b["vp_vs_ratio_plausibility_max"])

    both = vp_ok & vs_ok
    vp_gt_vs = np.zeros(n, dtype=bool)
    screen_ok = np.zeros(n, dtype=bool)
    nonpositive_K = np.zeros(n, dtype=bool)
    posK_negnu = np.zeros(n, dtype=bool)
    above_max = np.zeros(n, dtype=bool)

    if both.any():
        vp = np.asarray(vp_arr, dtype=np.float64)
        vs = np.asarray(vs_arr, dtype=np.float64)
        vp_gt_vs[both] = vp[both] > vs[both]
        with np.errstate(divide="ignore", invalid="ignore"):
            ratio = np.full(n, np.nan, dtype=np.float64)
            np.divide(vp, vs, out=ratio, where=both & (vs != 0.0))
        finite_r = np.isfinite(ratio)
        # Mutually exclusive, exhaustive classification of the finite-ratio
        # samples. `screen_ok` uses the INCLUSIVE sqrt(2) bound so that
        # nu == 0 passes a non-negative-nu policy.
        nonpositive_K = finite_r & (ratio <= r_k_bound)
        posK_negnu = finite_r & (ratio > r_k_bound) & (ratio < r_nu_min)
        above_max = finite_r & (ratio > r_max)
        screen_ok = finite_r & (ratio >= r_nu_min) & (ratio <= r_max)

    depth_ok = np.asarray(frame.depth_valid_mask, dtype=bool)
    mask = vp_ok & vs_ok & rhob_ok & vp_gt_vs & screen_ok & depth_ok

    # The Vp/Vs conditions are only MEANINGFUL where both velocities exist,
    # so their raw pass counts are structurally bounded by VS availability.
    # Using such a bounded count to pick the "limiting criterion" would
    # misreport a DATA-COVERAGE problem (VS is the scarcest curve in these
    # wells) as a PHYSICS problem. The counts compared for the limiting
    # criterion are therefore the independently meaningful per-curve
    # availability criteria plus depth mapping; the Vp/Vs conditions are
    # reported separately, and SEPARATELY BY REGIME - never aggregated
    # under a single "non-physical" label.
    return EligibilityMaskResult(
        frame.well_key, MASK_DYNAMIC_ELASTIC, mask,
        {
            "vp_finite_positive_in_bounds": int(np.count_nonzero(vp_ok)),
            "vs_finite_positive_in_bounds": int(np.count_nonzero(vs_ok)),
            "rhob_finite_in_bounds": int(np.count_nonzero(rhob_ok)),
            "depth_mapped": int(np.count_nonzero(depth_ok)),
        },
        diagnostic_counts={
            "n_vp_and_vs_both_valid": int(np.count_nonzero(both)),
            "n_vp_not_greater_than_vs": int(np.count_nonzero(both & ~vp_gt_vs)),
            "n_ratio_nonpositive_bulk_modulus": int(np.count_nonzero(nonpositive_K)),
            "n_ratio_positive_bulk_but_negative_poisson": int(np.count_nonzero(posK_negnu)),
            "n_ratio_above_configured_plausibility_max": int(np.count_nonzero(above_max)),
            "n_passes_nonnegative_poisson_screen": int(np.count_nonzero(screen_ok)),
        },
        lithology_dependent=False,
        purpose=_purpose(config, MASK_DYNAMIC_ELASTIC),
        notes=(
            "Input-admissibility only. Increment 6 computes no Young's modulus, Poisson ratio, "
            "bulk modulus, or shear modulus. The Vp/Vs condition is a CONFIGURED NON-NEGATIVE-"
            "POISSON-RATIO APPLICABILITY SCREEN (inclusive at Vp/Vs = sqrt(2), where nu = 0 "
            "exactly), not a test of physical possibility. Excluded ratios are diagnosed by "
            "regime: non-positive bulk modulus (Vp/Vs <= sqrt(4/3), genuinely outside the "
            "isotropic elastic model); positive bulk modulus with negative Poisson ratio "
            "(sqrt(4/3) < Vp/Vs < sqrt(2), unusual and outside this project's conservative "
            "policy, but NOT non-physical); and above the configured plausibility maximum "
            "(Vp/Vs > 4, a project credibility limit, not a Poisson-domain boundary). These are "
            "never aggregated into a single 'non-physical' count. No excluded sample is deleted "
            "from the well frame or corrected."
        ),
    )


def compute_sonic_nct_candidate_eligibility(
    frame: WellFrame,
    disposition: GrFamilyDisposition,
    proxy: GrProxyResult,
    proxy_threshold: float,
    config: PetrophysicsEligibilityConfig,
) -> EligibilityMaskResult:
    """
    Mask CANDIDATE DATA for a LATER sonic normal-compaction-trend analysis.

    Requires an approved GR-family disposition, finite VP, a finite
    screening proxy at or above `proxy_threshold`, and a mapped depth.

    Raises `PetrophysicsInputError` if called for a well whose disposition
    forbids GR-derived work - such a well can never have a candidate mask,
    because the mask consumes a GR proxy that must not exist for it.

    This mask fits nothing, selects no donor interval, and claims neither
    normal compaction nor overpressure. It must never be read as proof
    that an interval is normally compacted, and it names no lithology.
    """
    if not disposition.proxy_permitted:
        raise PetrophysicsInputError(
            f"{frame.well_key}: sonic-NCT candidate eligibility is lithology-dependent and cannot "
            f"be computed for a well excluded from GR-derived work "
            f"(use_status={disposition.use_status!r}, exclusion_reason="
            f"{disposition.exclusion_reason!r})."
        )
    if proxy.well_key != frame.well_key:
        raise PetrophysicsInputError(
            f"Proxy result belongs to well {proxy.well_key!r} but was supplied for "
            f"{frame.well_key!r}; a proxy is never transferred between wells."
        )

    n = frame.n_samples
    b = config.policy["physical_bounds"]
    vp_ok = _finite_within(frame.values_or_none("VP_m_s"), float(b["vp_min_m_s"]), float(b["vp_max_m_s"]))
    vp_ok = np.zeros(n, dtype=bool) if vp_ok is None else vp_ok

    proxy_vals = np.asarray(proxy.VSH_GR_linear_proxy_frac, dtype=np.float64)
    if proxy_vals.size != n:
        raise PetrophysicsInputError(
            f"{frame.well_key}: proxy array length {proxy_vals.size} does not match well-frame "
            f"sample count {n}."
        )
    proxy_finite = np.isfinite(proxy_vals)
    thr = float(proxy_threshold)
    if not np.isfinite(thr):
        raise PetrophysicsInputError(f"{frame.well_key}: proxy_threshold must be finite, got {thr!r}.")
    proxy_at_or_above = proxy_finite & (proxy_vals >= thr)

    depth_ok = np.asarray(frame.depth_valid_mask, dtype=bool)
    mask = vp_ok & proxy_at_or_above & depth_ok

    return EligibilityMaskResult(
        frame.well_key, MASK_SONIC_NCT_CANDIDATE, mask,
        {
            "gr_disposition_approved": n,  # gate already passed above, else this raised
            "vp_finite_in_bounds": int(np.count_nonzero(vp_ok)),
            "proxy_finite": int(np.count_nonzero(proxy_finite)),
            "proxy_at_or_above_threshold": int(np.count_nonzero(proxy_at_or_above)),
            "depth_mapped": int(np.count_nonzero(depth_ok)),
        },
        scenario_name=proxy.scenario_name,
        proxy_threshold=thr,
        lithology_dependent=True,
        purpose=_purpose(config, MASK_SONIC_NCT_CANDIDATE),
        notes=(
            "CANDIDATE DATA ONLY - no NCT fitted. This mask does not fit a trend, does not select "
            "a donor interval, does not claim normal compaction, and does not claim overpressure. "
            "It is not proof that any interval is normally compacted, and it assigns no lithology."
        ),
    )


CONTIGUITY_POLICY_CONFIGURED = "configured_bridging"
CONTIGUITY_POLICY_STRICT = "strict_no_gap"
VALID_CONTIGUITY_POLICIES: Tuple[str, ...] = (
    CONTIGUITY_POLICY_CONFIGURED,
    CONTIGUITY_POLICY_STRICT,
)


def _span(depth: np.ndarray, i0: int, i1: int) -> Optional[float]:
    """Depth span between two indices, or None if either end is unmapped."""
    a, b = depth[i0], depth[i1]
    if not (np.isfinite(a) and np.isfinite(b)):
        return None
    return float(b) - float(a)


def _net_span(depth: np.ndarray, mask: np.ndarray, i0: int, i1: int) -> Optional[float]:
    """
    Sum of the depth spans of the maximal STRICTLY-CONTIGUOUS True runs
    inside `[i0, i1]` - i.e. the block's gross span minus the spans of any
    bridged gaps.

    Returns None if any contributing run has an unmapped endpoint, so a
    partially-unmapped block reports "unknown" rather than an
    understated number.
    """
    sub = mask[i0 : i1 + 1]
    if not sub.any():
        return 0.0
    total = 0.0
    for a, b in find_contiguous_blocks(sub):
        sp = _span(depth, i0 + a, i0 + b)
        if sp is None:
            return None
        total += sp
    return total


def build_eligibility_intervals(
    frame: WellFrame,
    mask_result: EligibilityMaskResult,
    config: PetrophysicsEligibilityConfig,
    *,
    contiguity_policy: str = CONTIGUITY_POLICY_CONFIGURED,
) -> List[EligibilityInterval]:
    """
    Convert one eligibility mask into contiguous interval records on MD,
    TVD and TVDSS.

    `contiguity_policy` selects the decomposition:

      * `configured_bridging` (default) - uses the project's configured
        gap tolerances. A gap is bridged into a block only when BOTH the
        sample gap AND the physical depth span are within tolerance, so a
        two-sample gap across a large depth jump correctly starts a new
        block. Bridged samples are always disclosed via
        `n_bridged_samples` / `n_bridged_gaps` / `n_eligible_subruns`,
        and the block's
        `gross_thickness_*_m` is explicitly labelled gross because it
        spans them. `net_thickness_*_m` reports the same block with those
        gaps removed.

      * `strict_no_gap` - no bridging at all. Gross and net coincide by
        construction, giving the conservative comparison figure.

    Nothing is extrapolated under either policy. A block is emitted even
    when it fails the configured minimum sample count or thickness -
    `meets_configured_minimums` is False and `limiting_reason` records
    which requirement it failed - so sub-threshold eligible data is
    disclosed rather than silently dropped.
    """
    if contiguity_policy not in VALID_CONTIGUITY_POLICIES:
        raise PetrophysicsInputError(
            f"Unknown contiguity_policy {contiguity_policy!r}; expected one of "
            f"{VALID_CONTIGUITY_POLICIES}."
        )
    cont = config.policy["contiguity"]
    if contiguity_policy == CONTIGUITY_POLICY_STRICT:
        max_gap_samples, max_gap_depth_m = 0, 0.0
    else:
        max_gap_samples = int(cont["max_gap_samples"])
        max_gap_depth_m = float(cont["max_gap_depth_m"])
    min_block_samples = int(cont["min_block_samples"])
    min_block_thickness_m = float(cont["min_block_thickness_m"])

    md = np.asarray(frame.MD_m, dtype=np.float64)
    tvd = np.asarray(frame.TVD_m, dtype=np.float64)
    tvdss = np.asarray(frame.TVDSS_m, dtype=np.float64)
    mask = mask_result.mask

    blocks = find_contiguous_blocks(
        mask, md, max_gap_samples=max_gap_samples, max_gap_depth_m=max_gap_depth_m
    )

    out: List[EligibilityInterval] = []
    for i, (s0, e0) in enumerate(blocks):
        span_n = e0 - s0 + 1
        n_elig = int(np.count_nonzero(mask[s0 : e0 + 1]))
        n_bridged = span_n - n_elig
        # Strict (unbridged) eligible sub-runs inside this gross block, and the
        # distinct bridged False runs that separate them.
        sub = mask[s0 : e0 + 1]
        n_eligible_subruns = len(find_contiguous_blocks(sub))
        n_bridged_gaps = len(find_contiguous_blocks(~sub)) if n_bridged else 0

        md_s, md_e = float(md[s0]), float(md[e0])
        tvd_s = float(tvd[s0]) if np.isfinite(tvd[s0]) else None
        tvd_e = float(tvd[e0]) if np.isfinite(tvd[e0]) else None
        ss_s = float(tvdss[s0]) if np.isfinite(tvdss[s0]) else None
        ss_e = float(tvdss[e0]) if np.isfinite(tvdss[e0]) else None

        gross_md = md_e - md_s
        gross_tvd = (tvd_e - tvd_s) if (tvd_s is not None and tvd_e is not None) else None
        gross_ss = (ss_e - ss_s) if (ss_s is not None and ss_e is not None) else None
        net_md = _net_span(md, mask, s0, e0)
        net_tvd = _net_span(tvd, mask, s0, e0)
        net_ss = _net_span(tvdss, mask, s0, e0)

        reasons = []
        if span_n < min_block_samples:
            reasons.append(f"below_min_block_samples({span_n}<{min_block_samples})")
        thickness_for_test = gross_ss if gross_ss is not None else gross_md
        if thickness_for_test is not None and thickness_for_test < min_block_thickness_m:
            reasons.append(
                f"below_min_block_thickness({thickness_for_test:.3f}m<{min_block_thickness_m}m)"
            )
        meets = not reasons
        limiting = ";".join(reasons) if reasons else "none_meets_all_configured_minimums"

        out.append(
            EligibilityInterval(
                well_key=frame.well_key,
                mask_name=mask_result.mask_name,
                scenario_name=mask_result.scenario_name,
                proxy_threshold=mask_result.proxy_threshold,
                block_index=i,
                start_index=int(s0),
                end_index=int(e0),
                n_samples=int(span_n),
                n_eligible_samples=n_elig,
                md_start_m=md_s, md_end_m=md_e,
                gross_thickness_md_m=gross_md, net_thickness_md_m=net_md,
                tvd_start_m=tvd_s, tvd_end_m=tvd_e,
                gross_thickness_tvd_m=gross_tvd, net_thickness_tvd_m=net_tvd,
                tvdss_start_m=ss_s, tvdss_end_m=ss_e,
                gross_thickness_tvdss_m=gross_ss, net_thickness_tvdss_m=net_ss,
                n_bridged_samples=int(n_bridged),
                n_bridged_gaps=int(n_bridged_gaps),
                n_eligible_subruns=int(n_eligible_subruns),
                contiguity_policy=contiguity_policy,
                meets_configured_minimums=bool(meets),
                limiting_reason=limiting,
                depth_basis_used=frame.depth_basis_used,
            )
        )
    return out


#### `p2mem/io/petrophysics_inventory.py` — deterministic, sanitized export builders

Two disciplines are enforced here and tested directly: **no per-sample real-data array is ever exported** (that would effectively reproduce the private source logs inside a deliverable), and **no absolute path is ever exported** (every path-shaped field is reduced to a basename, and every free-text message is sanitized against *both* candidate source paths).

#### `p2mem/io/output_policy.py` — schema-driven, failure-atomic output authorization

**Technical objective (Increment 6.1.7):** validate the exact schema of all eight artifacts independently of their values, authorize every declared string occurrence, serialize to an isolated staging directory, re-read and re-validate the candidate bytes, and compare every typed field and row before publication. Any rejected export must leave the official destination unchanged.


In [ ]:
%%writefile p2mem/io/output_policy.py
"""
Increment 6.1.7 - SCHEMA-DRIVEN, FAILURE-ATOMIC OUTPUT AUTHORIZATION.

Increment 6.1.5 introduced positive authorization and closed
`validate_no_prohibited_interpretation`. It then validated a scope that the
manifest builder RECONSTRUCTED from dispositions, confidences and masks - a
parallel object, not the records actually written to disk. The consequence was
demonstrable: a `GrEndpointScenario` carrying
`description="The interval is chalk."` was persisted verbatim by
`build_gr_endpoint_scenario_rows()`, while that string never entered the
143-field scope and so could not move `named_lithology_assigned`.

The unit validator was closed. The 6.1.6 export path still discovered fields
from their values: missing fields, empty strings, non-string substitutions and
unknown numeric-looking fields could escape the occurrence collector. It also
wrote directly into the official directory before its post-write check and
compared aggregate counts rather than every authorized value.

This module closes those gaps with independent, exact artifact schemas. Schema
validation checks artifact inventory, keys, ordered CSV columns, requiredness,
types and finite numeric values before content authorization. Export writes a
complete candidate set into an isolated sibling directory, re-reads it,
re-validates it, and compares a type-aware canonical representation of every
field and row before publishing. Validation or serializer failure leaves the
official destination unchanged.

CATEGORIES (closed; an unknown category, artifact, column or JSON path fails):

  structural_enum        fixed vocabulary enumerated per field
  identifier             approved well / scenario identifiers
  filename               bare basename, no separators, approved extension
  typed_label            APPROVED_LABELS[(field_kind, value)] - both required
  registered_statement   exact text of a registered statement, by id
  controlled_template    registered template rendered with typed substitutions
  structured_diagnostic  machine-generated, grammar-constrained; no free prose
  sanitized_diagnostic   dynamic operator text under an explicit safety contract

There is deliberately NO general-purpose category admitting arbitrary prose.

SCOPE OF THE GUARANTEE. `structured_diagnostic` and `sanitized_diagnostic` are
NOT covered by the controlled-interpretation guarantee, and are counted and
reported separately. `structured_diagnostic` values must match a declared
grammar of `name=integer` / `name(number<number)` / `CODE:token` items, which
admits no sentence. `sanitized_diagnostic` covers the operator-facing issue
`message` and `context` only; its contract is bounded length, a restricted
character set, no path separator, no newline and no absolute path, and it is
tested separately. It is not claimed to be interpretation-controlled.
"""

from typing import Dict, Optional, Tuple

from p2mem.wellframe_models import (
    APPROVED_LABELS,
    Authorization,
    FIELD_TYPES,
    RegisteredStatement,
    RegisteredTemplate,
    approved_label,
    find_prohibited_lithology_terms,
)

__all__ = [
    "CATEGORIES", "SCOPE_OUTPUT", "FieldPolicy", "OUTPUT_FIELD_POLICY",
    "OUTPUT_STATEMENTS", "OUTPUT_TEMPLATES", "OUTPUT_ARTIFACTS", "CSV_SCHEMAS",
    "FieldOccurrence", "CoverageReport",
    "collect_string_fields", "authorize_occurrence", "validate_artifact_schema",
    "validate_emitted_records", "canonicalize_emitted_records",
    "export_authorized_outputs", "OutputAuthorizationError",
    "SANITIZED_DIAGNOSTIC_MAX_LEN", "SANITIZED_DIAGNOSTIC_CHARSET",
]

#: Statements registered because they are EMITTED, distinct from the
#: scope-level registry in `wellframe_models`.
SCOPE_OUTPUT = "output"

CATEGORIES: Tuple[str, ...] = (
    "structural_enum", "identifier", "filename", "typed_label",
    "registered_statement", "controlled_template",
    "structured_diagnostic", "sanitized_diagnostic",
)

#: Categories covered by the controlled-interpretation guarantee.
CONTROLLED_CATEGORIES: Tuple[str, ...] = (
    "typed_label", "registered_statement", "controlled_template",
)
#: Categories that carry no interpretation and are counted separately.
STRUCTURAL_CATEGORIES: Tuple[str, ...] = (
    "structural_enum", "identifier", "filename",
)
#: Explicitly OUTSIDE the controlled-interpretation guarantee.
UNGUARANTEED_CATEGORIES: Tuple[str, ...] = (
    "structured_diagnostic", "sanitized_diagnostic",
)


class FieldPolicy:
    """The declared classification of one emitted string field."""

    __slots__ = ("artifact", "field", "category", "field_kind",
                 "allowed_values", "statement_ids", "template_id",
                 "allow_empty")

    def __init__(self, artifact, field, category, field_kind=None,
                 allowed_values=(), statement_ids=(), template_id=None,
                 allow_empty=False):
        # `allowed_values` on a controlled_template declares enumerated SENTINELS
        # (e.g. "not_applicable_..."), and `statement_ids` declares registered
        # alternatives. Together they are the field's complete closed form set.
        if category not in CATEGORIES:
            raise ValueError(f"{artifact}:{field}: unknown category {category!r}")
        self.artifact = artifact
        self.field = field
        self.category = category
        self.field_kind = field_kind
        self.allowed_values = tuple(allowed_values)
        self.statement_ids = tuple(statement_ids)
        self.template_id = template_id
        self.allow_empty = bool(allow_empty)

    def __repr__(self):  # pragma: no cover - diagnostic only
        return f"FieldPolicy({self.artifact!r}, {self.field!r}, {self.category!r})"


_OUTPUT_STATEMENT_LIST: Tuple[RegisteredStatement, ...] = (
    RegisteredStatement(
        statement_id="boreas_excluded_confidence_rationale",
        scope=SCOPE_OUTPUT,
        text="Formally excluded from every GR-derived calculation (exclusion_reason='BOREAS_ECGR_SCALE_UNRESOLVED'). The curve remains available for factual raw QC display and availability reporting only. Good coverage does not resolve an unresolved scale/acquisition anomaly, so coverage statistics do not override this classification.",
        purpose="The per-well confidence rationale emitted for the excluded well, where "
                "no measured proxy statistics exist to render the template.",
        provenance="Enumerated from the ACTUAL emitted manifest record."),
    RegisteredStatement(
        statement_id="eligibility_interval_register_limitations",
        scope=SCOPE_OUTPUT,
        text="Candidate/eligible DATA extent only. GROSS thickness is the block's endpoint span and, under the configured contiguity policy, may include explicitly bridged ineligible samples (see n_bridged_samples); NET thickness removes those gaps. Neither is an unqualified 'eligible thickness'. A sonic-NCT-candidate interval is not proof of normal compaction, is not a fitted trend, and is not a selected donor interval. Data/proxy confidence only - asserts no named lithology and no calibration.",
        purpose="Persisted as the 'limitations' field of eligibility_interval_register.csv.",
        provenance="Enumerated from the ACTUAL emitted eligibility_interval_register.csv record. Authorization is by id and exact text at the emission boundary, not from a parallel reconstructed scope."),
    RegisteredStatement(
        statement_id="gr_endpoint_scenarios_description_1",
        scope=SCOPE_OUTPUT,
        text="Conventional mid-range bracket. Chosen for comparability with common screening practice, NOT because any evidence in this project supports it over the other two.",
        purpose="Persisted as the 'description' field of gr_endpoint_scenarios.csv.",
        provenance="Enumerated from the ACTUAL emitted gr_endpoint_scenarios.csv record. Authorization is by id and exact text at the emission boundary, not from a parallel reconstructed scope."),
    RegisteredStatement(
        statement_id="gr_endpoint_scenarios_description_2",
        scope=SCOPE_OUTPUT,
        text="Narrow bracket: a high low-endpoint and a low high-endpoint. This compresses the normalization range, so more samples clip at both ends and the proxy saturates sooner. Reported to show the upper bound of apparent proxy magnitude.",
        purpose="Persisted as the 'description' field of gr_endpoint_scenarios.csv.",
        provenance="Enumerated from the ACTUAL emitted gr_endpoint_scenarios.csv record. Authorization is by id and exact text at the emission boundary, not from a parallel reconstructed scope."),
    RegisteredStatement(
        statement_id="gr_endpoint_scenarios_description_3",
        scope=SCOPE_OUTPUT,
        text="Wide bracket: a low low-endpoint and a high high-endpoint. This expands the normalization range, so fewer samples clip and the proxy is damped. Reported to show the lower bound of apparent proxy magnitude.",
        purpose="Persisted as the 'description' field of gr_endpoint_scenarios.csv.",
        provenance="Enumerated from the ACTUAL emitted gr_endpoint_scenarios.csv record. Authorization is by id and exact text at the emission boundary, not from a parallel reconstructed scope."),
    RegisteredStatement(
        statement_id="gr_endpoint_scenarios_limitations",
        scope=SCOPE_OUTPUT,
        text="Endpoints are ASSUMED, configured percentile values estimated from this well's OWN samples. They are not calibrated, are not shared across wells, and must never be presented as a validated endpoint pair. Data/proxy confidence only - asserts no named lithology and no calibration.",
        purpose="Persisted as the 'limitations' field of gr_endpoint_scenarios.csv.",
        provenance="Enumerated from the ACTUAL emitted gr_endpoint_scenarios.csv record. Authorization is by id and exact text at the emission boundary, not from a parallel reconstructed scope."),
    RegisteredStatement(
        statement_id="gr_family_qc_summary_limitations",
        scope=SCOPE_OUTPUT,
        text="Descriptive statistics of the curve AS RECORDED. No environmental correction, rescaling, or cross-well normalization applied. Data/proxy confidence only - asserts no named lithology and no calibration.",
        purpose="Persisted as the 'limitations' field of gr_family_qc_summary.csv.",
        provenance="Enumerated from the ACTUAL emitted gr_family_qc_summary.csv record. Authorization is by id and exact text at the emission boundary, not from a parallel reconstructed scope."),
    RegisteredStatement(
        statement_id="gr_family_qc_summary_statistics_basis",
        scope=SCOPE_OUTPUT,
        text="Descriptive statistics over finite samples of this well's OWN GR-family curve, in its own recorded API units. No environmental correction, rescaling, normalization, or cross-well transfer of any kind has been applied. These numbers describe the curve as recorded and imply no lithology.",
        purpose="Persisted as the 'statistics_basis' field of gr_family_qc_summary.csv.",
        provenance="Enumerated from the ACTUAL emitted gr_family_qc_summary.csv record. Authorization is by id and exact text at the emission boundary, not from a parallel reconstructed scope."),
    RegisteredStatement(
        statement_id="gr_proxy_sensitivity_summary_limitations",
        scope=SCOPE_OUTPUT,
        text="IGR and the linear screening proxy are dimensionless quantities derived under ASSUMED endpoints. The proxy is NOT a calibrated shale volume and NOT a lithology. Clipped and unclipped indices are computed and retained together in memory; the clipped counts here quantify how far the real data fell outside the assumed endpoint bracket. Data/proxy confidence only - asserts no named lithology and no calibration.",
        purpose="Persisted as the 'limitations' field of gr_proxy_sensitivity_summary.csv.",
        provenance="Enumerated from the ACTUAL emitted gr_proxy_sensitivity_summary.csv record. Authorization is by id and exact text at the emission boundary, not from a parallel reconstructed scope."),
    RegisteredStatement(
        statement_id="method_eligibility_summary_limitations_1",
        scope=SCOPE_OUTPUT,
        text="ELIGIBILITY IS NOT VALIDITY. This is a necessary, not sufficient, condition for a LATER method; the method itself is not implemented, not fitted, and not validated in Increment 6. CANDIDATE DATA ONLY - no NCT fitted. This mask does not fit a trend, does not select a donor interval, does not claim normal compaction, and does not claim overpressure. It is not proof that any interval is normally compacted, and it assigns no lithology. Data/proxy confidence only - asserts no named lithology and no calibration.",
        purpose="Persisted as the 'limitations' field of method_eligibility_summary.csv.",
        provenance="Enumerated from the ACTUAL emitted method_eligibility_summary.csv record. Authorization is by id and exact text at the emission boundary, not from a parallel reconstructed scope."),
    RegisteredStatement(
        statement_id="method_eligibility_summary_limitations_2",
        scope=SCOPE_OUTPUT,
        text="ELIGIBILITY IS NOT VALIDITY. This is a necessary, not sufficient, condition for a LATER method; the method itself is not implemented, not fitted, and not validated in Increment 6. Eligibility is a necessary, not sufficient, condition. Increment 6 does not fill missing density and does not compute vertical stress. A log that begins well below the seabed cannot support an overburden integral from surface regardless of how many of its own samples are eligible. Data/proxy confidence only - asserts no named lithology and no calibration.",
        purpose="Persisted as the 'limitations' field of method_eligibility_summary.csv.",
        provenance="Enumerated from the ACTUAL emitted method_eligibility_summary.csv record. Authorization is by id and exact text at the emission boundary, not from a parallel reconstructed scope."),
    RegisteredStatement(
        statement_id="method_eligibility_summary_limitations_3",
        scope=SCOPE_OUTPUT,
        text="ELIGIBILITY IS NOT VALIDITY. This is a necessary, not sufficient, condition for a LATER method; the method itself is not implemented, not fitted, and not validated in Increment 6. Input-admissibility only. Increment 6 computes no Young's modulus, Poisson ratio, bulk modulus, or shear modulus. The Vp/Vs condition is a CONFIGURED NON-NEGATIVE-POISSON-RATIO APPLICABILITY SCREEN (inclusive at Vp/Vs = sqrt(2), where nu = 0 exactly), not a test of physical possibility. Excluded ratios are diagnosed by regime: non-positive bulk modulus (Vp/Vs <= sqrt(4/3), genuinely outside the isotropic elastic model); positive bulk modulus with negative Poisson ratio (sqrt(4/3) < Vp/Vs < sqrt(2), unusual and outside this project's conservative policy, but NOT non-physical); and above the configured plausibility maximum (Vp/Vs > 4, a project credibility limit, not a Poisson-domain boundary). These are never aggregated into a single 'non-physical' count. No excluded sample is deleted from the well frame or corrected. Data/proxy confidence only - asserts no named lithology and no calibration.",
        purpose="Persisted as the 'limitations' field of method_eligibility_summary.csv.",
        provenance="Enumerated from the ACTUAL emitted method_eligibility_summary.csv record. Authorization is by id and exact text at the emission boundary, not from a parallel reconstructed scope."),
    RegisteredStatement(
        statement_id="method_eligibility_summary_purpose_1",
        scope=SCOPE_OUTPUT,
        text="Marks samples technically admissible as input to a LATER dynamic-elastic calculation. Increment 6 computes NO elastic property - no Young's modulus, no Poisson ratio, no bulk or shear modulus. It only records where the three required inputs coexist and pass the configured non-negative-Poisson-ratio applicability screen. Excluded Vp/Vs values are diagnosed BY REGIME (non-positive bulk modulus; positive bulk modulus with negative Poisson ratio; above the configured plausibility maximum) and are never aggregated under a single \"non-physical\" label.",
        purpose="Persisted as the 'purpose' field of method_eligibility_summary.csv.",
        provenance="Enumerated from the ACTUAL emitted method_eligibility_summary.csv record. Authorization is by id and exact text at the emission boundary, not from a parallel reconstructed scope."),
    RegisteredStatement(
        statement_id="method_eligibility_summary_purpose_2",
        scope=SCOPE_OUTPUT,
        text="Marks samples technically admissible as input to a LATER vertical-stress (Sv) integration. Increment 6 neither fills missing density nor computes Sv; a sample being eligible says nothing about whether an integration over it would be defensible, since a density log that starts at ~470 m TVDSS cannot by itself support an overburden integral from surface.",
        purpose="Persisted as the 'purpose' field of method_eligibility_summary.csv.",
        provenance="Enumerated from the ACTUAL emitted method_eligibility_summary.csv record. Authorization is by id and exact text at the emission boundary, not from a parallel reconstructed scope."),
    RegisteredStatement(
        statement_id="method_eligibility_summary_purpose_3",
        scope=SCOPE_OUTPUT,
        text="Marks samples that are CANDIDATE DATA for a LATER sonic normal-compaction -trend analysis. This is a data-admissibility mask and nothing more. It does NOT fit a trend, does NOT select a donor interval, does NOT claim normal compaction, does NOT claim overpressure, and must never be read as evidence that any interval is normally compacted or is any named lithology.",
        purpose="Persisted as the 'purpose' field of method_eligibility_summary.csv.",
        provenance="Enumerated from the ACTUAL emitted method_eligibility_summary.csv record. Authorization is by id and exact text at the emission boundary, not from a parallel reconstructed scope."),
    RegisteredStatement(
        statement_id="petrophysics_eligibility_manifest_calibration_data_available_statement",
        scope=SCOPE_OUTPUT,
        text="No RFT, MDT, DST, FIT, LOT, XLOT or DFIT data exist for this project. The supplied Vp/Vs text file is derived from the existing sonic curves and is NOT independent calibration data. Every quantity in this increment therefore remains uncalibrated.",
        purpose="Persisted as the '/calibration_data_available/statement' field of petrophysics_eligibility_manifest.json.",
        provenance="Enumerated from the ACTUAL emitted petrophysics_eligibility_manifest.json record. Authorization is by id and exact text at the emission boundary, not from a parallel reconstructed scope."),
    RegisteredStatement(
        statement_id="petrophysics_eligibility_manifest_increment_title",
        scope=SCOPE_OUTPUT,
        text="Gamma-Ray QC, Shale-Proxy Sensitivity, Well-Frame Assembly, and Method-Eligibility Framework",
        purpose="Persisted as the '/increment_title' field of petrophysics_eligibility_manifest.json.",
        provenance="Enumerated from the ACTUAL emitted petrophysics_eligibility_manifest.json record. Authorization is by id and exact text at the emission boundary, not from a parallel reconstructed scope."),
    RegisteredStatement(
        statement_id="petrophysics_eligibility_manifest_limitations_1",
        scope=SCOPE_OUTPUT,
        text="A sonic-NCT-candidate interval is candidate DATA only. It is not a fitted trend, not a selected donor interval, and not evidence of normal compaction or overpressure.",
        purpose="Persisted as the '/limitations[]' field of petrophysics_eligibility_manifest.json.",
        provenance="Enumerated from the ACTUAL emitted petrophysics_eligibility_manifest.json record. Authorization is by id and exact text at the emission boundary, not from a parallel reconstructed scope."),
    RegisteredStatement(
        statement_id="petrophysics_eligibility_manifest_limitations_2",
        scope=SCOPE_OUTPUT,
        text="Boreas 1 is formally excluded from all GR-derived work under BOREAS_ECGR_SCALE_UNRESOLVED and is retained for factual raw QC display only. It is excluded rather than corrected because no calibration evidence exists to support any correction.",
        purpose="Persisted as the '/limitations[]' field of petrophysics_eligibility_manifest.json.",
        provenance="Enumerated from the ACTUAL emitted petrophysics_eligibility_manifest.json record. Authorization is by id and exact text at the emission boundary, not from a parallel reconstructed scope."),
    RegisteredStatement(
        statement_id="petrophysics_eligibility_manifest_limitations_3",
        scope=SCOPE_OUTPUT,
        text="Eligibility is a necessary, not sufficient, condition. No method gated by these masks is implemented, fitted, or validated in this increment.",
        purpose="Persisted as the '/limitations[]' field of petrophysics_eligibility_manifest.json.",
        provenance="Enumerated from the ACTUAL emitted petrophysics_eligibility_manifest.json record. Authorization is by id and exact text at the emission boundary, not from a parallel reconstructed scope."),
    RegisteredStatement(
        statement_id="petrophysics_eligibility_manifest_limitations_4",
        scope=SCOPE_OUTPUT,
        text="GR endpoints are ASSUMED configured percentiles of each well's own samples, never calibrated and never shared across wells.",
        purpose="Persisted as the '/limitations[]' field of petrophysics_eligibility_manifest.json.",
        provenance="Enumerated from the ACTUAL emitted petrophysics_eligibility_manifest.json record. Authorization is by id and exact text at the emission boundary, not from a parallel reconstructed scope."),
    RegisteredStatement(
        statement_id="petrophysics_eligibility_manifest_limitations_5",
        scope=SCOPE_OUTPUT,
        text="No named lithology is assigned, and the available data do not support assigning one.",
        purpose="Persisted as the '/limitations[]' field of petrophysics_eligibility_manifest.json.",
        provenance="Enumerated from the ACTUAL emitted petrophysics_eligibility_manifest.json record. Authorization is by id and exact text at the emission boundary, not from a parallel reconstructed scope."),
    RegisteredStatement(
        statement_id="petrophysics_eligibility_manifest_limitations_6",
        scope=SCOPE_OUTPUT,
        text="Poseidon North 1 and Proteus 1ST2 have no approved formation tops; their results are depth-tied and stratigraphically unvalidated.",
        purpose="Persisted as the '/limitations[]' field of petrophysics_eligibility_manifest.json.",
        provenance="Enumerated from the ACTUAL emitted petrophysics_eligibility_manifest.json record. Authorization is by id and exact text at the emission boundary, not from a parallel reconstructed scope."),
    RegisteredStatement(
        statement_id="petrophysics_eligibility_manifest_limitations_7",
        scope=SCOPE_OUTPUT,
        text="Tier C - screening-level and uncalibrated. Nothing here is validated against independent measurement.",
        purpose="Persisted as the '/limitations[]' field of petrophysics_eligibility_manifest.json.",
        provenance="Enumerated from the ACTUAL emitted petrophysics_eligibility_manifest.json record. Authorization is by id and exact text at the emission boundary, not from a parallel reconstructed scope."),
    RegisteredStatement(
        statement_id="lithology_validation_derivation",
        scope=SCOPE_OUTPUT,
        text='named_lithology_assigned is DERIVED at a SCHEMA-DRIVEN EMISSION BOUNDARY. Exact artifact schemas validate inventory, keys, ordered CSV columns, requiredness, types and finite numeric values independently of content. Every declared string occurrence - including empty and numeric-looking strings - is then positively authorized as an approved (field_kind, value) label, a registered statement matched by id and exact text, a registered template rendered with typed substitutions, an enumerated structural value, or a declared machine-diagnostic grammar. Export writes only to an isolated staging directory, re-reads and re-validates the candidate bytes, and requires a type-aware canonical match of every field and row before publication; a rejected export leaves the official destination unchanged. This is NOT a claim that the software recognizes natural-language lithology; the prohibited-term list is a diagnostic linter that authorizes nothing. Machine diagnostics and operator-facing issue text are counted separately and are outside the controlled-interpretation guarantee.',
        purpose="Persisted as manifest lithology_validation.derivation. Describes the "
                "Increment 6.1.7 schema-driven, failure-atomic emission-boundary model "
                "that is actually implemented, superseding the incomplete 6.1.6 wording.",
        provenance="Registered so the assurance prose is itself authorized like any other "
                   "emitted field, and cannot drift from the implementation silently."),
    RegisteredStatement(
        statement_id="petrophysics_eligibility_manifest_named_lithology_statement",
        scope=SCOPE_OUTPUT,
        text="NO named lithology is assigned anywhere in Increment 6. Gamma-ray response is not uniquely diagnostic of rock type, and no independent lithological evidence (core, cuttings description, image log, spectral GR, or calibrated multi-mineral solution) is available in this project. Low-GR intervals occur independently in each of the three GR-eligible wells; cross-well stratigraphic persistence is NOT established, and cannot be, because two of those wells have no approved formation tops. Those intervals therefore remain UNRESOLVED in lithology. All classifications in this increment describe DATA AND PROXY CONFIDENCE ONLY.",
        purpose="Persisted as the '/named_lithology_statement' field of petrophysics_eligibility_manifest.json.",
        provenance="Enumerated from the ACTUAL emitted petrophysics_eligibility_manifest.json record. Authorization is by id and exact text at the emission boundary, not from a parallel reconstructed scope."),
    RegisteredStatement(
        statement_id="petrophysics_eligibility_manifest_nonlinear_vsh_deferral_statement",
        scope=SCOPE_OUTPUT,
        text="No nonlinear Vsh transform (Larionov, Clavier, Stieber, or any other) is implemented. Each would require a retrieved, verified primary-source method record and a closed method-register entry; none exists in this project.",
        purpose="Persisted as the '/nonlinear_vsh_deferral_statement' field of petrophysics_eligibility_manifest.json.",
        provenance="Enumerated from the ACTUAL emitted petrophysics_eligibility_manifest.json record. Authorization is by id and exact text at the emission boundary, not from a parallel reconstructed scope."),
    RegisteredStatement(
        statement_id="thickness_sensitivity_summary_limitations",
        scope=SCOPE_OUTPUT,
        text="Eligible/candidate DATA extent only. No gated method is implemented, fitted, or validated. Data/proxy confidence only - asserts no named lithology and no calibration.",
        purpose="Persisted as the 'limitations' field of thickness_sensitivity_summary.csv.",
        provenance="Enumerated from the ACTUAL emitted thickness_sensitivity_summary.csv record. Authorization is by id and exact text at the emission boundary, not from a parallel reconstructed scope."),
)

OUTPUT_STATEMENTS = {s.statement_id: s for s in _OUTPUT_STATEMENT_LIST}

_SEABED_TEMPLATE = (
    "Counted where canonical LAS MD_m < {seabed_mdrt_m} m MDRT, the seabed marker's "
    "reconciled MDRT from the LOCKED Increment 5 survey-corrected output."
)
_POPULATION_TEMPLATE = (
    "Totals cover the {n_qualifying} block(s) meeting the configured minimums out of "
    "{n_found} found under the {policy} contiguity policy. GROSS is the sum of block "
    "endpoint spans and includes {n_bridged_samples} disclosed bridged sample(s) across "
    "{n_bridged_gaps} bridged gap(s) in {n_interrupted} interrupted block(s); NET removes "
    "those gaps."
)
_RATIONALE_TEMPLATE = (
    "valid_fraction={valid_fraction}, dynamic_range_p05_p95={dynamic_range_p05_p95} API, "
    "proxy_median_spread_across_3_scenarios={proxy_median_spread}. Describes confidence in "
    "the DATA and the SCREENING PROXY only; asserts no lithology and no calibration."
)

_OUTPUT_TEMPLATE_LIST: Tuple[RegisteredTemplate, ...] = (
    RegisteredTemplate(
        template_id="seabed_basis", scope=SCOPE_OUTPUT, template=_SEABED_TEMPLATE,
        fields={"seabed_mdrt_m": "decimal"},
        purpose="Persisted as the 'seabed_basis' field of gr_family_qc_summary.csv.",
        provenance="Fixed prose with one MEASURED depth. Enumerated from the actual "
                   "emitted record; the substitution is restricted to a decimal literal."),
    RegisteredTemplate(
        template_id="population_statement", scope=SCOPE_OUTPUT,
        template=_POPULATION_TEMPLATE,
        fields={"n_qualifying": "integer", "n_found": "integer",
                "policy": "quoted_policy_name", "n_bridged_samples": "integer",
                "n_bridged_gaps": "integer", "n_interrupted": "integer"},
        purpose="Persisted as the 'population_statement' field of "
                "thickness_sensitivity_summary.csv.",
        provenance="Fixed prose with five MEASURED counts and one enumerated policy "
                   "name. Enumerated from the actual emitted record."),
    RegisteredTemplate(
        template_id="gr_proxy_confidence_rationale", scope=SCOPE_OUTPUT,
        template=_RATIONALE_TEMPLATE,
        fields={"valid_fraction": "decimal", "dynamic_range_p05_p95": "decimal",
                "proxy_median_spread": "decimal"},
        purpose="Persisted as the per-well 'gr_proxy_confidence_rationale'.",
        provenance="Fixed prose with three MEASURED numbers. Enumerated from the "
                   "actual emitted record."),
)

OUTPUT_TEMPLATES = {t.template_id: t for t in _OUTPUT_TEMPLATE_LIST}

_dupe = sorted({i for i in
                [s.statement_id for s in _OUTPUT_STATEMENT_LIST]
                + [t.template_id for t in _OUTPUT_TEMPLATE_LIST]
                if ([s.statement_id for s in _OUTPUT_STATEMENT_LIST]
                    + [t.template_id for t in _OUTPUT_TEMPLATE_LIST]).count(i) > 1})
if _dupe:  # pragma: no cover - fails at import if violated
    raise ValueError(f"duplicate output registry ids: {_dupe}")
del _dupe


def _is_integer_literal(value):
    if isinstance(value, bool) or not isinstance(value, str) or not value:
        return False
    body = value[1:] if value[0] in "+-" else value
    return bool(body) and body.isdigit()


def _is_quoted_policy_name(value):
    return isinstance(value, str) and value in ("'configured_bridging'", "'strict_no_gap'")


#: Substitution types available to OUTPUT templates. Every one is a predicate
#: over the substituted STRING - never a judgement about its meaning.
OUTPUT_FIELD_TYPES = dict(FIELD_TYPES)
OUTPUT_FIELD_TYPES["integer"] = _is_integer_literal
OUTPUT_FIELD_TYPES["quoted_policy_name"] = _is_quoted_policy_name


OUTPUT_FIELD_POLICY_LIST: Tuple[FieldPolicy, ...] = (
    # Increment 6.1.6 assurance-metadata paths. The `violations[]` records are
    # DIAGNOSTIC: they exist only when something failed, and they are explicitly
    # outside the controlled-interpretation guarantee (counted separately).
    FieldPolicy(
        artifact="petrophysics_eligibility_manifest.json",
        field="/lithology_validation/model",
        category="structural_enum",
        allowed_values=("schema_driven_transactional_authorization_at_emission_boundary",)),
    FieldPolicy(
        artifact="petrophysics_eligibility_manifest.json",
        field="/lithology_validation/violations[]/context",
        category="sanitized_diagnostic"),
    FieldPolicy(
        artifact="petrophysics_eligibility_manifest.json",
        field="/lithology_validation/violations[]/reason",
        category="sanitized_diagnostic"),
    FieldPolicy(
        artifact="petrophysics_eligibility_manifest.json",
        field="/lithology_validation/violations[]/scope",
        category="sanitized_diagnostic"),
    FieldPolicy(
        artifact="petrophysics_eligibility_manifest.json",
        field="/lithology_validation/violations[]/terms[]",
        category="sanitized_diagnostic"),
    FieldPolicy(
        artifact="petrophysics_eligibility_manifest.json",
        field="/lithology_validation/emitted_field_coverage/artifacts_inspected[]",
        category="identifier",
        allowed_values=(
            "eligibility_interval_register.csv", "gr_endpoint_scenarios.csv",
            "gr_family_qc_summary.csv", "gr_proxy_sensitivity_summary.csv",
            "method_eligibility_summary.csv", "petrophysics_eligibility_issues.csv",
            "petrophysics_eligibility_manifest.json",
            "thickness_sensitivity_summary.csv")),
    FieldPolicy(
        artifact="petrophysics_eligibility_manifest.json",
        field="/lithology_validation/emitted_field_coverage/status",
        category="structural_enum", allowed_values=("not_supplied",)),
    FieldPolicy(
        artifact="petrophysics_eligibility_manifest.json",
        field="/lithology_validation/emitted_field_coverage/note",
        category="sanitized_diagnostic"),
    FieldPolicy(
        artifact="eligibility_interval_register.csv",
        field="assurance_tier",
        category="structural_enum",
        allowed_values=(
            "Tier C - Screening-Level / Uncalibrated Educational",
        )),
    FieldPolicy(
        artifact="eligibility_interval_register.csv",
        field="contiguity_policy",
        category="structural_enum",
        allowed_values=(
            "configured_bridging",
            "strict_no_gap",
        )),
    FieldPolicy(
        artifact="eligibility_interval_register.csv",
        field="depth_basis_used",
        category="structural_enum",
        allowed_values=(
            "petrel_source_trace",
        )),
    FieldPolicy(
        artifact="eligibility_interval_register.csv",
        field="limitations",
        category="registered_statement",
        statement_ids=("eligibility_interval_register_limitations",)),
    FieldPolicy(
        artifact="eligibility_interval_register.csv",
        field="limiting_reason",
        category="structured_diagnostic",
        allowed_values=("none_meets_all_configured_minimums",)),
    FieldPolicy(
        artifact="eligibility_interval_register.csv",
        field="mask_name",
        category="typed_label",
        field_kind="mask_name"),
    FieldPolicy(
        artifact="eligibility_interval_register.csv",
        field="scenario_name",
        category="identifier",
        allowed_values=(
            "base",
            "high",
            "low",
        ),
        allow_empty=True),
    FieldPolicy(
        artifact="eligibility_interval_register.csv",
        field="unit",
        category="structural_enum",
        allowed_values=(
            "metres",
        )),
    FieldPolicy(
        artifact="eligibility_interval_register.csv",
        field="well_key",
        category="identifier",
        allowed_values=(
            "Boreas_1",
            "Poseidon_2",
            "Poseidon_North_1",
            "Proteus_1ST2",
        )),
    FieldPolicy(
        artifact="gr_endpoint_scenarios.csv",
        field="assurance_tier",
        category="structural_enum",
        allowed_values=(
            "Tier C - Screening-Level / Uncalibrated Educational",
        )),
    FieldPolicy(
        artifact="gr_endpoint_scenarios.csv",
        field="calibration_status",
        category="structural_enum",
        allowed_values=(
            "uncalibrated_assumed_no_calibration_data_exists",
        )),
    FieldPolicy(
        artifact="gr_endpoint_scenarios.csv",
        field="description",
        category="registered_statement",
        statement_ids=("gr_endpoint_scenarios_description_1", "gr_endpoint_scenarios_description_2", "gr_endpoint_scenarios_description_3",)),
    FieldPolicy(
        artifact="gr_endpoint_scenarios.csv",
        field="endpoint_sample_basis",
        category="structural_enum",
        allowed_values=(
            "finite_and_depth_mapped_samples_only",
        )),
    FieldPolicy(
        artifact="gr_endpoint_scenarios.csv",
        field="evidence_class",
        category="typed_label",
        field_kind="evidence_class"),
    FieldPolicy(
        artifact="gr_endpoint_scenarios.csv",
        field="limitations",
        category="registered_statement",
        statement_ids=("gr_endpoint_scenarios_limitations",)),
    FieldPolicy(
        artifact="gr_endpoint_scenarios.csv",
        field="scenario_name",
        category="identifier",
        allowed_values=(
            "base",
            "high",
            "low",
        )),
    FieldPolicy(
        artifact="gr_endpoint_scenarios.csv",
        field="unit",
        category="structural_enum",
        allowed_values=(
            "API",
        )),
    FieldPolicy(
        artifact="gr_endpoint_scenarios.csv",
        field="well_key",
        category="identifier",
        allowed_values=(
            "Poseidon_2",
            "Poseidon_North_1",
            "Proteus_1ST2",
        )),
    FieldPolicy(
        artifact="gr_family_qc_summary.csv",
        field="assurance_tier",
        category="structural_enum",
        allowed_values=(
            "Tier C - Screening-Level / Uncalibrated Educational",
        )),
    FieldPolicy(
        artifact="gr_family_qc_summary.csv",
        field="depth_basis",
        category="structural_enum",
        allowed_values=(
            "MDRT (measured depth below rotary table), metres",
        )),
    FieldPolicy(
        artifact="gr_family_qc_summary.csv",
        field="evidence_class",
        category="typed_label",
        field_kind="evidence_class"),
    FieldPolicy(
        artifact="gr_family_qc_summary.csv",
        field="exclusion_reason",
        category="typed_label",
        field_kind="exclusion_reason",
        allow_empty=True),
    FieldPolicy(
        artifact="gr_family_qc_summary.csv",
        field="gr_family_canonical_name",
        category="typed_label",
        field_kind="gr_family_canonical_name"),
    FieldPolicy(
        artifact="gr_family_qc_summary.csv",
        field="gr_family_source_curve_name",
        category="typed_label",
        field_kind="gr_family_source_curve_name"),
    FieldPolicy(
        artifact="gr_family_qc_summary.csv",
        field="gr_proxy_confidence_class",
        category="typed_label",
        field_kind="gr_proxy_confidence_class"),
    FieldPolicy(
        artifact="gr_family_qc_summary.csv",
        field="limitations",
        category="registered_statement",
        statement_ids=("gr_family_qc_summary_limitations",)),
    FieldPolicy(
        artifact="gr_family_qc_summary.csv",
        field="seabed_basis",
        category="controlled_template",
        template_id="seabed_basis",
        allowed_values=('Not determinable: this well has no approved formation-top file, so no seabed marker exists in the locked Increment 5 output. Reported as None (unknown), never as 0.',)),
    FieldPolicy(
        artifact="gr_family_qc_summary.csv",
        field="source_las_filename",
        category="filename"),
    FieldPolicy(
        artifact="gr_family_qc_summary.csv",
        field="statistics_basis",
        category="registered_statement",
        statement_ids=("gr_family_qc_summary_statistics_basis",)),
    FieldPolicy(
        artifact="gr_family_qc_summary.csv",
        field="unit",
        category="structural_enum",
        allowed_values=(
            "API",
        )),
    FieldPolicy(
        artifact="gr_family_qc_summary.csv",
        field="use_status",
        category="typed_label",
        field_kind="use_status"),
    FieldPolicy(
        artifact="gr_family_qc_summary.csv",
        field="well_key",
        category="identifier",
        allowed_values=(
            "Boreas_1",
            "Poseidon_2",
            "Poseidon_North_1",
            "Proteus_1ST2",
        )),
    FieldPolicy(
        artifact="gr_proxy_sensitivity_summary.csv",
        field="assurance_tier",
        category="structural_enum",
        allowed_values=(
            "Tier C - Screening-Level / Uncalibrated Educational",
        )),
    FieldPolicy(
        artifact="gr_proxy_sensitivity_summary.csv",
        field="calibration_status",
        category="structural_enum",
        allowed_values=(
            "screening_proxy_uncalibrated_not_a_shale_volume",
        )),
    FieldPolicy(
        artifact="gr_proxy_sensitivity_summary.csv",
        field="evidence_class",
        category="typed_label",
        field_kind="evidence_class"),
    FieldPolicy(
        artifact="gr_proxy_sensitivity_summary.csv",
        field="gr_family_canonical_name",
        category="typed_label",
        field_kind="gr_family_canonical_name"),
    FieldPolicy(
        artifact="gr_proxy_sensitivity_summary.csv",
        field="limitations",
        category="registered_statement",
        statement_ids=("gr_proxy_sensitivity_summary_limitations",)),
    FieldPolicy(
        artifact="gr_proxy_sensitivity_summary.csv",
        field="proxy_field_name",
        category="structural_enum",
        allowed_values=(
            "VSH_GR_linear_proxy_frac",
        )),
    FieldPolicy(
        artifact="gr_proxy_sensitivity_summary.csv",
        field="scenario_name",
        category="identifier",
        allowed_values=(
            "base",
            "high",
            "low",
        )),
    FieldPolicy(
        artifact="gr_proxy_sensitivity_summary.csv",
        field="transform_name",
        category="structural_enum",
        allowed_values=(
            "linear_identity_of_clipped_igr",
        )),
    FieldPolicy(
        artifact="gr_proxy_sensitivity_summary.csv",
        field="unit",
        category="structural_enum",
        allowed_values=(
            "dimensionless_fraction",
        )),
    FieldPolicy(
        artifact="gr_proxy_sensitivity_summary.csv",
        field="use_status",
        category="typed_label",
        field_kind="use_status"),
    FieldPolicy(
        artifact="gr_proxy_sensitivity_summary.csv",
        field="well_key",
        category="identifier",
        allowed_values=(
            "Poseidon_2",
            "Poseidon_North_1",
            "Proteus_1ST2",
        )),
    FieldPolicy(
        artifact="method_eligibility_summary.csv",
        field="assurance_tier",
        category="structural_enum",
        allowed_values=(
            "Tier C - Screening-Level / Uncalibrated Educational",
        )),
    FieldPolicy(
        artifact="method_eligibility_summary.csv",
        field="criteria_counts",
        category="structured_diagnostic"),
    FieldPolicy(
        artifact="method_eligibility_summary.csv",
        field="depth_basis_used",
        category="structural_enum",
        allowed_values=(
            "petrel_source_trace",
        )),
    FieldPolicy(
        artifact="method_eligibility_summary.csv",
        field="depth_map_status",
        category="structural_enum",
        allowed_values=(
            "fully_mapped_within_survey_coverage",
        )),
    FieldPolicy(
        artifact="method_eligibility_summary.csv",
        field="diagnostic_counts",
        category="structured_diagnostic",
        allow_empty=True),
    FieldPolicy(
        artifact="method_eligibility_summary.csv",
        field="exclusion_reason",
        category="typed_label",
        field_kind="exclusion_reason",
        allow_empty=True),
    FieldPolicy(
        artifact="method_eligibility_summary.csv",
        field="interpolation_method",
        category="structural_enum",
        allowed_values=(
            "piecewise_linear_station_interpolation",
        )),
    FieldPolicy(
        artifact="method_eligibility_summary.csv",
        field="limitations",
        category="registered_statement",
        statement_ids=("method_eligibility_summary_limitations_1", "method_eligibility_summary_limitations_2", "method_eligibility_summary_limitations_3",)),
    FieldPolicy(
        artifact="method_eligibility_summary.csv",
        field="limiting_criterion",
        category="structural_enum",
        allowed_values=(
            # Enumerated from the criterion vocabulary the mask builder can
            # emit, not merely from the values one dataset happened to hit.
            "depth_mapped",
            "gr_disposition_approved",
            "proxy_at_or_above_threshold",
            "proxy_finite",
            "rhob_finite",
            "rhob_finite_in_bounds",
            "rhob_within_physical_bounds",
            "sonic_finite",
            "vp_finite_in_bounds",
            "vp_finite_positive_in_bounds",
            "vs_finite_positive_in_bounds",
        )),
    FieldPolicy(
        artifact="method_eligibility_summary.csv",
        field="mask_name",
        category="typed_label",
        field_kind="mask_name"),
    FieldPolicy(
        artifact="method_eligibility_summary.csv",
        field="purpose",
        category="registered_statement",
        statement_ids=("method_eligibility_summary_purpose_1", "method_eligibility_summary_purpose_2", "method_eligibility_summary_purpose_3",)),
    FieldPolicy(
        artifact="method_eligibility_summary.csv",
        field="scenario_name",
        category="identifier",
        allowed_values=(
            "base",
            "high",
            "low",
        ),
        allow_empty=True),
    FieldPolicy(
        artifact="method_eligibility_summary.csv",
        field="unit",
        category="structural_enum",
        allowed_values=(
            "sample_count_and_fraction",
        )),
    FieldPolicy(
        artifact="method_eligibility_summary.csv",
        field="use_status",
        category="typed_label",
        field_kind="use_status"),
    FieldPolicy(
        artifact="method_eligibility_summary.csv",
        field="well_key",
        category="identifier",
        allowed_values=(
            "Boreas_1",
            "Poseidon_2",
            "Poseidon_North_1",
            "Proteus_1ST2",
        )),
    FieldPolicy(
        artifact="petrophysics_eligibility_issues.csv",
        field="assurance_tier",
        category="structural_enum",
        allowed_values=(
            "Tier C - Screening-Level / Uncalibrated Educational",
        )),
    FieldPolicy(
        artifact="petrophysics_eligibility_issues.csv",
        field="code",
        category="structural_enum",
        allowed_values=(
            "GR_DERIVED_CALCULATION_SKIPPED_BY_EXCLUSION",
        )),
    FieldPolicy(
        artifact="petrophysics_eligibility_issues.csv",
        field="context",
        category="sanitized_diagnostic"),
    FieldPolicy(
        artifact="petrophysics_eligibility_issues.csv",
        field="message",
        category="sanitized_diagnostic"),
    FieldPolicy(
        artifact="petrophysics_eligibility_issues.csv",
        field="severity",
        category="structural_enum",
        allowed_values=(
            "WARNING",
        )),
    FieldPolicy(
        artifact="petrophysics_eligibility_manifest.json",
        field="/assurance_tier",
        category="structural_enum",
        allowed_values=(
            "Tier C - Screening-Level / Uncalibrated Educational",
        )),
    FieldPolicy(
        artifact="petrophysics_eligibility_manifest.json",
        field="/calibration_data_available/statement",
        category="registered_statement",
        statement_ids=("petrophysics_eligibility_manifest_calibration_data_available_statement",)),
    FieldPolicy(
        artifact="petrophysics_eligibility_manifest.json",
        field="/config_filename",
        category="structural_enum",
        allowed_values=(
            "petrophysics_eligibility.yml",
        )),
    FieldPolicy(
        artifact="petrophysics_eligibility_manifest.json",
        field="/config_schema_version",
        category="structural_enum",
        allowed_values=(
            "6.0",
        )),
    FieldPolicy(
        artifact="petrophysics_eligibility_manifest.json",
        field="/depth_reference_convention",
        category="structural_enum",
        allowed_values=(
            "MD and TVD are referenced to the well datum (rotary table), increasing downward; TVDSS_m = TVD_m - DatumElevation_m, with datum elevation referenced to MSL, positive upward. Identical to the LOCKED Increment 3.1.1 convention - not re-derived here.",
        )),
    FieldPolicy(
        artifact="petrophysics_eligibility_manifest.json",
        field="/increment_title",
        category="registered_statement",
        statement_ids=("petrophysics_eligibility_manifest_increment_title",)),
    FieldPolicy(
        artifact="petrophysics_eligibility_manifest.json",
        field="/issues[]/code",
        category="structural_enum",
        allowed_values=(
            "GR_DERIVED_CALCULATION_SKIPPED_BY_EXCLUSION",
        )),
    FieldPolicy(
        artifact="petrophysics_eligibility_manifest.json",
        field="/issues[]/context",
        category="sanitized_diagnostic"),
    FieldPolicy(
        artifact="petrophysics_eligibility_manifest.json",
        field="/issues[]/message",
        category="sanitized_diagnostic"),
    FieldPolicy(
        artifact="petrophysics_eligibility_manifest.json",
        field="/issues[]/severity",
        category="structural_enum",
        allowed_values=(
            "WARNING",
        )),
    FieldPolicy(
        artifact="petrophysics_eligibility_manifest.json",
        field="/limitations[]",
        category="registered_statement",
        statement_ids=("petrophysics_eligibility_manifest_limitations_1", "petrophysics_eligibility_manifest_limitations_2", "petrophysics_eligibility_manifest_limitations_3", "petrophysics_eligibility_manifest_limitations_4", "petrophysics_eligibility_manifest_limitations_5", "petrophysics_eligibility_manifest_limitations_6", "petrophysics_eligibility_manifest_limitations_7",)),
    FieldPolicy(
        artifact="petrophysics_eligibility_manifest.json",
        field="/lithology_validation/derivation",
        category="registered_statement",
        statement_ids=("lithology_validation_derivation",)),
    FieldPolicy(
        artifact="petrophysics_eligibility_manifest.json",
        field="/methods_not_implemented[]",
        category="structural_enum",
        allowed_values=(
            "bowers_pore_pressure",
            "density_reconstruction_or_extrapolation",
            "dynamic_elastic_property_calculation",
            "eaton_resistivity_pore_pressure",
            "eaton_sonic_pore_pressure",
            "environmental_gr_correction",
            "friction_angle_modelling",
            "gr_rescaling_or_normalization_across_wells",
            "hydrostatic_pressure_modelling",
            "mud_weight_recommendation",
            "named_lithology_interpretation",
            "normal_compaction_trend_fitting",
            "rock_strength_correlation",
            "shmin_shmax_modelling",
            "static_elastic_conversion",
            "stress_polygon_construction",
            "vertical_stress_integration",
            "wellbore_stability_analysis",
        )),
    FieldPolicy(
        artifact="petrophysics_eligibility_manifest.json",
        field="/named_lithology_statement",
        category="registered_statement",
        statement_ids=("petrophysics_eligibility_manifest_named_lithology_statement",)),
    FieldPolicy(
        artifact="petrophysics_eligibility_manifest.json",
        field="/nonlinear_vsh_deferral_statement",
        category="registered_statement",
        statement_ids=("petrophysics_eligibility_manifest_nonlinear_vsh_deferral_statement",)),
    FieldPolicy(
        artifact="petrophysics_eligibility_manifest.json",
        field="/wells/*/depth_basis_used",
        category="structural_enum",
        allowed_values=(
            "petrel_source_trace",
        )),
    FieldPolicy(
        artifact="petrophysics_eligibility_manifest.json",
        field="/wells/*/depth_map_status",
        category="structural_enum",
        allowed_values=(
            "fully_mapped_within_survey_coverage",
        )),
    FieldPolicy(
        artifact="petrophysics_eligibility_manifest.json",
        field="/wells/*/endpoint_scenarios[]/calibration_status",
        category="structural_enum",
        allowed_values=(
            "uncalibrated_assumed_no_calibration_data_exists",
        )),
    FieldPolicy(
        artifact="petrophysics_eligibility_manifest.json",
        field="/wells/*/endpoint_scenarios[]/evidence_class",
        category="typed_label",
        field_kind="evidence_class"),
    FieldPolicy(
        artifact="petrophysics_eligibility_manifest.json",
        field="/wells/*/endpoint_scenarios[]/scenario_name",
        category="identifier",
        allowed_values=(
            "base",
            "high",
            "low",
        )),
    FieldPolicy(
        artifact="petrophysics_eligibility_manifest.json",
        field="/wells/*/exclusion_reason",
        category="typed_label",
        field_kind="exclusion_reason"),
    FieldPolicy(
        artifact="petrophysics_eligibility_manifest.json",
        field="/wells/*/gr_family_canonical_name",
        category="typed_label",
        field_kind="gr_family_canonical_name"),
    FieldPolicy(
        artifact="petrophysics_eligibility_manifest.json",
        field="/wells/*/gr_family_source_curve_name",
        category="typed_label",
        field_kind="gr_family_source_curve_name"),
    FieldPolicy(
        artifact="petrophysics_eligibility_manifest.json",
        field="/wells/*/gr_proxy_confidence_class",
        category="typed_label",
        field_kind="gr_proxy_confidence_class"),
    FieldPolicy(
        artifact="petrophysics_eligibility_manifest.json",
        field="/wells/*/gr_proxy_confidence_rationale",
        category="controlled_template",
        template_id="gr_proxy_confidence_rationale",
        statement_ids=("boreas_excluded_confidence_rationale",)),
    FieldPolicy(
        artifact="petrophysics_eligibility_manifest.json",
        field="/wells/*/interpolation_method",
        category="structural_enum",
        allowed_values=(
            "piecewise_linear_station_interpolation",
        )),
    FieldPolicy(
        artifact="petrophysics_eligibility_manifest.json",
        field="/wells/*/method_eligibility[]/limiting_criterion",
        category="structural_enum",
        allowed_values=(
            # Enumerated from the criterion vocabulary the mask builder can
            # emit, not merely from the values one dataset happened to hit.
            "depth_mapped",
            "gr_disposition_approved",
            "proxy_at_or_above_threshold",
            "proxy_finite",
            "rhob_finite",
            "rhob_finite_in_bounds",
            "rhob_within_physical_bounds",
            "sonic_finite",
            "vp_finite_in_bounds",
            "vp_finite_positive_in_bounds",
            "vs_finite_positive_in_bounds",
        )),
    FieldPolicy(
        artifact="petrophysics_eligibility_manifest.json",
        field="/wells/*/method_eligibility[]/mask_name",
        category="typed_label",
        field_kind="mask_name"),
    FieldPolicy(
        artifact="petrophysics_eligibility_manifest.json",
        field="/wells/*/method_eligibility[]/scenario_name",
        category="identifier",
        allowed_values=(
            "base",
            "high",
            "low",
        ),
        allow_empty=True),
    FieldPolicy(
        artifact="petrophysics_eligibility_manifest.json",
        field="/wells/*/qc_flags[]",
        category="structured_diagnostic"),
    FieldPolicy(
        artifact="petrophysics_eligibility_manifest.json",
        field="/wells/*/source_las_filename",
        category="filename"),
    FieldPolicy(
        artifact="petrophysics_eligibility_manifest.json",
        field="/wells/*/source_survey_filename",
        category="filename"),
    FieldPolicy(
        artifact="petrophysics_eligibility_manifest.json",
        field="/wells/*/use_status",
        category="typed_label",
        field_kind="use_status"),
    FieldPolicy(
        artifact="petrophysics_eligibility_manifest.json",
        field="/wells/*/well_identity_evidence_status",
        category="structural_enum",
        allowed_values=(
            "verified_against_file_well_header",
        )),
    FieldPolicy(
        artifact="thickness_sensitivity_summary.csv",
        field="assurance_tier",
        category="structural_enum",
        allowed_values=(
            "Tier C - Screening-Level / Uncalibrated Educational",
        )),
    FieldPolicy(
        artifact="thickness_sensitivity_summary.csv",
        field="contiguity_policy",
        category="structural_enum",
        allowed_values=(
            "configured_bridging",
            "strict_no_gap",
        )),
    FieldPolicy(
        artifact="thickness_sensitivity_summary.csv",
        field="limitations",
        category="registered_statement",
        statement_ids=("thickness_sensitivity_summary_limitations",)),
    FieldPolicy(
        artifact="thickness_sensitivity_summary.csv",
        field="mask_name",
        category="typed_label",
        field_kind="mask_name"),
    FieldPolicy(
        artifact="thickness_sensitivity_summary.csv",
        field="population_statement",
        category="controlled_template",
        template_id="population_statement"),
    FieldPolicy(
        artifact="thickness_sensitivity_summary.csv",
        field="scenario_name",
        category="identifier",
        allowed_values=(
            "base",
            "high",
            "low",
        ),
        allow_empty=True),
    FieldPolicy(
        artifact="thickness_sensitivity_summary.csv",
        field="unit",
        category="structural_enum",
        allowed_values=(
            "metres",
        )),
    FieldPolicy(
        artifact="thickness_sensitivity_summary.csv",
        field="well_key",
        category="identifier",
        allowed_values=(
            "Boreas_1",
            "Poseidon_2",
            "Poseidon_North_1",
            "Proteus_1ST2",
        )),
)

OUTPUT_FIELD_POLICY: Dict[Tuple[str, str], FieldPolicy] = {}
for _pol in OUTPUT_FIELD_POLICY_LIST:
    _key = (_pol.artifact, _pol.field)
    if _key in OUTPUT_FIELD_POLICY:  # pragma: no cover - fails at import
        raise ValueError(f"duplicate output field policy {_key}")
    OUTPUT_FIELD_POLICY[_key] = _pol
del _pol, _key

# ---------------------------------------------------------------------------
# Closed artifact schemas.  Unlike the 6.1.6 collector, these declarations do
# not infer a field's existence or type from its value.  Keys are validated
# first, including empty and non-string values, and content authorization is a
# separate second operation.
# ---------------------------------------------------------------------------


class CsvArtifactSchema:
    """Exact ordered columns and pre/post-serialization types for one CSV."""

    __slots__ = ("columns", "integer_fields", "number_fields", "boolean_fields",
                 "nullable_fields", "min_rows")

    def __init__(self, columns, integer_fields=(), number_fields=(),
                 boolean_fields=(), nullable_fields=(), min_rows=1):
        self.columns = tuple(columns)
        self.integer_fields = frozenset(integer_fields)
        self.number_fields = frozenset(number_fields)
        self.boolean_fields = frozenset(boolean_fields)
        self.nullable_fields = frozenset(nullable_fields)
        self.min_rows = int(min_rows)
        declared = set(self.columns)
        typed = self.integer_fields | self.number_fields | self.boolean_fields
        if len(declared) != len(self.columns):
            raise ValueError("CSV schema contains duplicate columns")
        if not typed <= declared or not self.nullable_fields <= declared:
            raise ValueError("CSV schema type/nullability references an unknown column")
        if ((self.integer_fields & self.number_fields)
                or (self.integer_fields & self.boolean_fields)
                or (self.number_fields & self.boolean_fields)):
            raise ValueError("CSV schema type partitions overlap")

    @property
    def string_fields(self):
        return frozenset(self.columns) - self.integer_fields - self.number_fields - self.boolean_fields

    def kind(self, field):
        if field in self.integer_fields:
            return "integer"
        if field in self.number_fields:
            return "number"
        if field in self.boolean_fields:
            return "boolean"
        return "string"


CSV_SCHEMAS = {
    "eligibility_interval_register.csv": CsvArtifactSchema(
        columns=(
            "well_key", "mask_name", "contiguity_policy", "scenario_name",
            "proxy_threshold", "block_index", "start_index", "end_index",
            "n_samples", "n_eligible_samples", "n_bridged_samples",
            "n_bridged_gaps", "n_eligible_subruns", "meets_configured_minimums",
            "md_start_m", "md_end_m", "gross_thickness_md_m", "net_thickness_md_m",
            "tvd_start_m", "tvd_end_m", "gross_thickness_tvd_m",
            "net_thickness_tvd_m", "tvdss_start_m", "tvdss_end_m",
            "gross_thickness_tvdss_m", "net_thickness_tvdss_m", "limiting_reason",
            "depth_basis_used", "unit", "assurance_tier", "limitations",
        ),
        integer_fields=("block_index", "start_index", "end_index", "n_samples",
                        "n_eligible_samples", "n_bridged_samples", "n_bridged_gaps",
                        "n_eligible_subruns"),
        number_fields=("proxy_threshold", "md_start_m", "md_end_m",
                       "gross_thickness_md_m", "net_thickness_md_m", "tvd_start_m",
                       "tvd_end_m", "gross_thickness_tvd_m", "net_thickness_tvd_m",
                       "tvdss_start_m", "tvdss_end_m", "gross_thickness_tvdss_m",
                       "net_thickness_tvdss_m"),
        boolean_fields=("meets_configured_minimums",),
        nullable_fields=("proxy_threshold", "md_start_m", "md_end_m",
                         "gross_thickness_md_m", "net_thickness_md_m", "tvd_start_m",
                         "tvd_end_m", "gross_thickness_tvd_m", "net_thickness_tvd_m",
                         "tvdss_start_m", "tvdss_end_m", "gross_thickness_tvdss_m",
                         "net_thickness_tvdss_m")),
    "gr_endpoint_scenarios.csv": CsvArtifactSchema(
        columns=(
            "well_key", "scenario_name", "low_percentile", "high_percentile",
            "gr_low_endpoint_api", "gr_high_endpoint_api", "endpoint_separation_api",
            "n_samples_used_for_endpoints", "endpoint_sample_basis", "unit",
            "evidence_class", "calibration_status", "assurance_tier", "description",
            "limitations",
        ),
        integer_fields=("n_samples_used_for_endpoints",),
        number_fields=("low_percentile", "high_percentile", "gr_low_endpoint_api",
                       "gr_high_endpoint_api", "endpoint_separation_api")),
    "gr_family_qc_summary.csv": CsvArtifactSchema(
        columns=(
            "well_key", "source_las_filename", "gr_family_canonical_name",
            "gr_family_source_curve_name", "unit", "use_status", "exclusion_reason",
            "evidence_class", "gr_proxy_confidence_class",
            "has_approved_formation_tops", "n_samples", "valid_count", "valid_fraction",
            "min_api", "max_api", "median_api", "p01_api", "p05_api", "p10_api",
            "p25_api", "p50_api", "p75_api", "p90_api", "p95_api", "p99_api",
            "dynamic_range_p05_p95_api", "n_negative_samples", "n_zero_samples",
            "n_samples_above_seabed", "seabed_basis", "n_valid_blocks",
            "longest_valid_block_samples", "longest_missing_block_samples",
            "longest_missing_block_md_start_m", "longest_missing_block_md_end_m",
            "depth_basis", "assurance_tier", "statistics_basis", "limitations",
        ),
        integer_fields=("n_samples", "valid_count", "n_negative_samples",
                        "n_zero_samples", "n_samples_above_seabed", "n_valid_blocks",
                        "longest_valid_block_samples", "longest_missing_block_samples"),
        number_fields=("valid_fraction", "min_api", "max_api", "median_api",
                       "p01_api", "p05_api", "p10_api", "p25_api", "p50_api",
                       "p75_api", "p90_api", "p95_api", "p99_api",
                       "dynamic_range_p05_p95_api", "longest_missing_block_md_start_m",
                       "longest_missing_block_md_end_m"),
        boolean_fields=("has_approved_formation_tops",),
        nullable_fields=("valid_fraction", "min_api", "max_api", "median_api",
                         "p01_api", "p05_api", "p10_api", "p25_api", "p50_api",
                         "p75_api", "p90_api", "p95_api", "p99_api",
                         "dynamic_range_p05_p95_api", "n_samples_above_seabed",
                         "longest_missing_block_md_start_m", "longest_missing_block_md_end_m")),
    "gr_proxy_sensitivity_summary.csv": CsvArtifactSchema(
        columns=(
            "well_key", "scenario_name", "gr_family_canonical_name", "use_status",
            "gr_low_endpoint_api", "gr_high_endpoint_api", "transform_name",
            "proxy_field_name", "unit", "n_valid", "n_clipped_low", "n_clipped_high",
            "clipped_fraction", "proxy_median", "proxy_p25", "proxy_p75",
            "calibration_status", "evidence_class", "assurance_tier", "limitations",
        ),
        integer_fields=("n_valid", "n_clipped_low", "n_clipped_high"),
        number_fields=("gr_low_endpoint_api", "gr_high_endpoint_api", "clipped_fraction",
                       "proxy_median", "proxy_p25", "proxy_p75")),
    "method_eligibility_summary.csv": CsvArtifactSchema(
        columns=(
            "well_key", "mask_name", "scenario_name", "proxy_threshold",
            "lithology_dependent", "use_status", "exclusion_reason", "n_samples",
            "n_eligible", "eligible_fraction", "limiting_criterion", "criteria_counts",
            "diagnostic_counts", "depth_basis_used", "interpolation_method",
            "depth_map_status", "n_depth_unmapped", "n_extrapolated", "unit",
            "assurance_tier", "purpose", "limitations",
        ),
        integer_fields=("n_samples", "n_eligible", "n_depth_unmapped", "n_extrapolated"),
        number_fields=("proxy_threshold", "eligible_fraction"),
        boolean_fields=("lithology_dependent",),
        nullable_fields=("proxy_threshold", "n_depth_unmapped", "n_extrapolated")),
    "petrophysics_eligibility_issues.csv": CsvArtifactSchema(
        columns=("severity", "code", "context", "message", "assurance_tier"),
        min_rows=0),
    "thickness_sensitivity_summary.csv": CsvArtifactSchema(
        columns=(
            "well_key", "mask_name", "contiguity_policy", "scenario_name",
            "proxy_threshold", "n_blocks_all", "n_blocks_qualifying",
            "n_blocks_rejected_below_minimums",
            "n_bridged_samples_in_qualifying_blocks",
            "n_bridged_gaps_in_qualifying_blocks", "n_interrupted_qualifying_blocks",
            "gross_qualifying_thickness_tvdss_m", "net_qualifying_thickness_tvdss_m",
            "gross_all_block_thickness_tvdss_m", "n_eligible_samples_qualifying",
            "unit", "population_statement", "assurance_tier", "limitations",
        ),
        integer_fields=("n_blocks_all", "n_blocks_qualifying",
                        "n_blocks_rejected_below_minimums",
                        "n_bridged_samples_in_qualifying_blocks",
                        "n_bridged_gaps_in_qualifying_blocks",
                        "n_interrupted_qualifying_blocks", "n_eligible_samples_qualifying"),
        number_fields=("proxy_threshold", "gross_qualifying_thickness_tvdss_m",
                       "net_qualifying_thickness_tvdss_m",
                       "gross_all_block_thickness_tvdss_m"),
        nullable_fields=("proxy_threshold", "gross_qualifying_thickness_tvdss_m",
                         "net_qualifying_thickness_tvdss_m",
                         "gross_all_block_thickness_tvdss_m")),
}

MANIFEST_ARTIFACT = "petrophysics_eligibility_manifest.json"
APPROVED_WELL_KEYS = frozenset((
    "Boreas_1", "Poseidon_2", "Poseidon_North_1", "Proteus_1ST2",
))

# Exact object keys.  `None` denotes the dynamic `/wells` mapping, whose keys
# must equal the supplied well-key set.  No key is discovered from a leaf value.
JSON_OBJECT_KEYS = {
    "": frozenset((
        "assembly_failures", "assurance_tier", "calibration_data_available",
        "config_filename", "config_schema_version", "depth_reference_convention",
        "increment", "increment_title", "issues", "limitations",
        "lithology_validation", "methods_not_implemented", "n_wells_frame_assembled",
        "n_wells_frame_failed", "n_wells_gr_excluded", "n_wells_gr_proxy_permitted",
        "named_lithology_assigned", "named_lithology_statement",
        "nct_candidate_proxy_thresholds", "nonlinear_vsh_deferral_statement",
        "nonlinear_vsh_transforms_implemented", "total_samples_extrapolated", "wells",
    )),
    "/calibration_data_available": frozenset((
        "pressure_rft_mdt_dst", "stress_fit_lot_xlot_dfit",
        "independent_vp_vs_calibration", "statement",
    )),
    "/lithology_validation": frozenset((
        "model", "derivation", "scope_object_fields_checked",
        "scope_object_violations", "emitted_field_coverage", "violations",
    )),
    "/lithology_validation/emitted_field_coverage": frozenset((
        "n_string_field_occurrences", "n_controlled_occurrences",
        "n_structural_occurrences", "n_unguaranteed_occurrences",
        "n_unclassified_fields", "n_unauthorized_controlled_fields",
        "n_field_kind_mismatches", "n_schema_violations", "violations",
        "artifacts_inspected", "distinct_statements_used", "distinct_templates_used",
        "distinct_labels_used",
    )),
    "/lithology_validation/violations[]": frozenset((
        "context", "scope", "terms", "reason",
    )),
    "/issues[]": frozenset(("severity", "code", "context", "message")),
    "/assembly_failures[]": frozenset((
        "well_key", "failure_origin", "error_type", "message",
    )),
    "/wells": None,
    "/wells/*": frozenset((
        "source_las_filename", "source_survey_filename", "well_frame_assembled",
        "n_samples", "depth_basis_used", "interpolation_method", "depth_map_status",
        "n_depth_unmapped", "n_extrapolated", "datum_elevation_m",
        "well_identity_evidence_status", "qc_flags", "gr_family_canonical_name",
        "gr_family_source_curve_name", "use_status", "exclusion_reason",
        "has_approved_formation_tops", "gr_proxy_confidence_class",
        "gr_proxy_confidence_rationale", "gr_valid_fraction", "gr_median_api",
        "gr_min_api", "gr_max_api", "gr_n_samples_above_seabed",
        "endpoint_scenarios", "method_eligibility",
    )),
    "/wells/*/endpoint_scenarios[]": frozenset((
        "scenario_name", "gr_low_endpoint_api", "gr_high_endpoint_api",
        "endpoint_separation_api", "evidence_class", "calibration_status",
    )),
    "/wells/*/method_eligibility[]": frozenset((
        "mask_name", "scenario_name", "proxy_threshold", "n_eligible",
        "eligible_fraction", "limiting_criterion", "lithology_dependent",
    )),
}

JSON_LIST_ITEM_KINDS = {
    "/assembly_failures": "object", "/issues": "object", "/limitations": "string",
    "/lithology_validation/emitted_field_coverage/artifacts_inspected": "string",
    "/lithology_validation/violations": "object",
    "/lithology_validation/violations[]/terms": "string",
    "/methods_not_implemented": "string", "/nct_candidate_proxy_thresholds": "number",
    "/wells/*/endpoint_scenarios": "object", "/wells/*/method_eligibility": "object",
    "/wells/*/qc_flags": "string",
}

JSON_BOOLEAN_PATHS = frozenset((
    "/calibration_data_available/independent_vp_vs_calibration",
    "/calibration_data_available/pressure_rft_mdt_dst",
    "/calibration_data_available/stress_fit_lot_xlot_dfit",
    "/named_lithology_assigned", "/nonlinear_vsh_transforms_implemented",
    "/wells/*/has_approved_formation_tops",
    "/wells/*/method_eligibility[]/lithology_dependent",
    "/wells/*/well_frame_assembled",
))
JSON_INTEGER_PATHS = frozenset((
    "/increment", "/n_wells_frame_assembled", "/n_wells_frame_failed",
    "/n_wells_gr_excluded", "/n_wells_gr_proxy_permitted",
    "/total_samples_extrapolated",
    "/lithology_validation/emitted_field_coverage/distinct_labels_used",
    "/lithology_validation/emitted_field_coverage/distinct_statements_used",
    "/lithology_validation/emitted_field_coverage/distinct_templates_used",
    "/lithology_validation/emitted_field_coverage/n_controlled_occurrences",
    "/lithology_validation/emitted_field_coverage/n_field_kind_mismatches",
    "/lithology_validation/emitted_field_coverage/n_schema_violations",
    "/lithology_validation/emitted_field_coverage/n_string_field_occurrences",
    "/lithology_validation/emitted_field_coverage/n_structural_occurrences",
    "/lithology_validation/emitted_field_coverage/n_unauthorized_controlled_fields",
    "/lithology_validation/emitted_field_coverage/n_unclassified_fields",
    "/lithology_validation/emitted_field_coverage/n_unguaranteed_occurrences",
    "/lithology_validation/emitted_field_coverage/violations",
    "/lithology_validation/scope_object_fields_checked",
    "/lithology_validation/scope_object_violations", "/wells/*/n_depth_unmapped",
    "/wells/*/n_extrapolated", "/wells/*/n_samples",
    "/wells/*/gr_n_samples_above_seabed",
    "/wells/*/method_eligibility[]/n_eligible",
))
JSON_NUMBER_PATHS = frozenset((
    "/nct_candidate_proxy_thresholds[]", "/wells/*/datum_elevation_m",
    "/wells/*/endpoint_scenarios[]/endpoint_separation_api",
    "/wells/*/endpoint_scenarios[]/gr_high_endpoint_api",
    "/wells/*/endpoint_scenarios[]/gr_low_endpoint_api", "/wells/*/gr_max_api",
    "/wells/*/gr_median_api", "/wells/*/gr_min_api", "/wells/*/gr_valid_fraction",
    "/wells/*/method_eligibility[]/eligible_fraction",
    "/wells/*/method_eligibility[]/proxy_threshold",
))
JSON_NULLABLE_PATHS = frozenset((
    "/wells/*/exclusion_reason", "/wells/*/gr_n_samples_above_seabed",
    "/wells/*/method_eligibility[]/proxy_threshold",
))

#: The eight Increment 6 deterministic artifacts this policy governs.
OUTPUT_ARTIFACTS: Tuple[str, ...] = tuple(sorted(tuple(CSV_SCHEMAS) + (MANIFEST_ARTIFACT,)))

# The string side of each schema must be classified exactly once.  This import-
# time assertion prevents a future schema/policy drift from weakening the gate.
_csv_policy_fields = {}
for (_artifact, _field), _policy in OUTPUT_FIELD_POLICY.items():
    if _artifact in CSV_SCHEMAS:
        _csv_policy_fields.setdefault(_artifact, set()).add(_field)
for _artifact, _schema in CSV_SCHEMAS.items():
    if _csv_policy_fields.get(_artifact, set()) != set(_schema.string_fields):
        raise ValueError(
            f"{_artifact}: CSV schema string fields and authorization policy differ: "
            f"schema_only={sorted(set(_schema.string_fields) - _csv_policy_fields.get(_artifact, set()))}, "
            f"policy_only={sorted(_csv_policy_fields.get(_artifact, set()) - set(_schema.string_fields))}")
del _artifact, _field, _policy, _schema, _csv_policy_fields


# ---------------------------------------------------------------------------
# Collection: the ACTUAL string fields of the ACTUAL records
# ---------------------------------------------------------------------------

#: The sanitized-diagnostic contract, stated explicitly and tested separately.
SANITIZED_DIAGNOSTIC_MAX_LEN = 1200
SANITIZED_DIAGNOSTIC_CHARSET = set(
    "abcdefghijklmnopqrstuvwxyzABCDEFGHIJKLMNOPQRSTUVWXYZ0123456789"
    " .,;:()[]{}<>=+-_/'\"%&*#@!?"
)
_FORBIDDEN_IN_DIAGNOSTIC = ("\n", "\r", "\t", "\\", "://")


def _normalize_json_path(path, well_keys):
    for w in sorted(well_keys, key=len, reverse=True):
        path = path.replace("/" + w + "/", "/*/")
        if path.endswith("/" + w):
            path = path[: -len(w)] + "*"
    out, i = [], 0
    while i < len(path):
        if path[i] == "[":
            j = path.index("]", i)
            out.append("[]")
            i = j + 1
        else:
            out.append(path[i])
            i += 1
    return "".join(out)


class FieldOccurrence:
    """One string value at one place in one emitted record."""

    __slots__ = ("artifact", "field", "value", "location")

    def __init__(self, artifact, field, value, location):
        self.artifact = artifact
        self.field = field
        self.value = value
        self.location = location

    def __repr__(self):  # pragma: no cover - diagnostic only
        return f"FieldOccurrence({self.artifact!r}, {self.field!r}, {self.location!r})"


def _schema_violation(artifact, field, location, reason):
    return {
        "artifact": artifact, "field": field, "location": location,
        "reason": "SCHEMA: " + reason, "linter_terms": [],
    }


def _csv_stage(payload, schema, requested):
    if requested in ("pre", "post"):
        return requested
    for row in payload if isinstance(payload, list) else ():
        if not isinstance(row, dict):
            continue
        for field in schema.integer_fields | schema.number_fields | schema.boolean_fields:
            if field in row and row[field] is not None and not isinstance(row[field], str):
                return "pre"
    return "post"


def _valid_decimal_string(value):
    import decimal
    if not isinstance(value, str) or not value:
        return False
    try:
        return decimal.Decimal(value).is_finite()
    except decimal.InvalidOperation:
        return False


def _validate_csv_schema(artifact, payload, serialization_stage="auto"):
    """Validate keys, order, requiredness and types without inspecting prose."""
    schema = CSV_SCHEMAS[artifact]
    out = []
    if not isinstance(payload, list):
        return [_schema_violation(
            artifact, None, None, "CSV payload must be a list of row dictionaries")]
    if len(payload) < schema.min_rows:
        out.append(_schema_violation(
            artifact, None, None,
            f"CSV requires at least {schema.min_rows} row(s), found {len(payload)}"))
    stage = _csv_stage(payload, schema, serialization_stage)
    for i, row in enumerate(payload):
        location = f"row[{i}]"
        if not isinstance(row, dict):
            out.append(_schema_violation(
                artifact, None, location, "CSV row must be a dictionary"))
            continue
        actual = tuple(row.keys())
        if actual != schema.columns:
            missing = [c for c in schema.columns if c not in row]
            unknown = [c for c in actual if c not in schema.columns]
            out.append(_schema_violation(
                artifact, None, location,
                f"ordered columns differ; missing={missing}, unknown={unknown}, "
                f"expected={list(schema.columns)}, actual={list(actual)}"))
        for field in schema.columns:
            if field not in row:
                continue
            value = row[field]
            nullable = field in schema.nullable_fields
            kind = schema.kind(field)
            if kind != "string" and (
                    (stage == "pre" and value is None)
                    or (stage == "post" and value == "")):
                if nullable:
                    continue
                out.append(_schema_violation(
                    artifact, field, f"{location}.{field}",
                    "required value is null/empty after serialization"))
                continue
            valid = False
            if kind == "string":
                valid = isinstance(value, str)
            elif kind == "integer":
                valid = ((type(value) is int) if stage == "pre"
                         else isinstance(value, str) and _is_integer_literal(value))
            elif kind == "number":
                if stage == "pre":
                    import math
                    valid = (type(value) in (int, float) and math.isfinite(value))
                else:
                    valid = _valid_decimal_string(value)
            elif kind == "boolean":
                valid = ((type(value) is bool) if stage == "pre"
                         else value in ("True", "False"))
            if not valid:
                out.append(_schema_violation(
                    artifact, field, f"{location}.{field}",
                    f"expected {kind} at {stage}-serialization stage, got "
                    f"{type(value).__name__} {value!r}"))
    return out


def _expected_json_primitive_kind(path):
    if path in JSON_BOOLEAN_PATHS:
        return "boolean"
    if path in JSON_INTEGER_PATHS:
        return "integer"
    if path in JSON_NUMBER_PATHS:
        return "number"
    if (MANIFEST_ARTIFACT, path) in OUTPUT_FIELD_POLICY:
        return "string"
    return None


def _validate_json_schema(artifact, payload, well_keys=()):
    """Validate the complete manifest tree, including empty/non-string leaves."""
    out = []
    supplied_wells = frozenset(well_keys)
    unknown_wells = supplied_wells - APPROVED_WELL_KEYS
    if unknown_wells:
        out.append(_schema_violation(
            artifact, "/wells", "/wells",
            f"well_keys contains unapproved identifier(s): {sorted(unknown_wells)}"))

    def walk(node, path):
        norm = _normalize_json_path(path, well_keys)
        if isinstance(node, dict):
            expected = JSON_OBJECT_KEYS.get(norm, "__missing__")
            if expected == "__missing__":
                out.append(_schema_violation(
                    artifact, norm, path, "object path is not declared"))
                return
            if expected is None:
                expected = supplied_wells
            actual = frozenset(node)
            if actual != expected:
                out.append(_schema_violation(
                    artifact, norm, path,
                    f"object keys differ; missing={sorted(expected - actual)}, "
                    f"unknown={sorted(actual - expected)}"))
            for key in sorted(actual & expected):
                walk(node[key], f"{path}/{key}")
            return
        if isinstance(node, list):
            expected_kind = JSON_LIST_ITEM_KINDS.get(norm)
            if expected_kind is None:
                out.append(_schema_violation(
                    artifact, norm, path, "list path is not declared"))
                return
            for i, value in enumerate(node):
                if expected_kind == "object" and not isinstance(value, dict):
                    out.append(_schema_violation(
                        artifact, norm, f"{path}[{i}]", "list item must be an object"))
                elif expected_kind == "string" and not isinstance(value, str):
                    out.append(_schema_violation(
                        artifact, norm, f"{path}[{i}]", "list item must be a string"))
                elif expected_kind == "number" and not (
                        type(value) in (int, float) and __import__("math").isfinite(value)):
                    out.append(_schema_violation(
                        artifact, norm, f"{path}[{i}]", "list item must be a finite number"))
                else:
                    walk(value, f"{path}[{i}]")
            return
        kind = _expected_json_primitive_kind(norm)
        if kind is None:
            out.append(_schema_violation(
                artifact, norm, path, "primitive path is not declared"))
            return
        if node is None:
            if norm not in JSON_NULLABLE_PATHS:
                out.append(_schema_violation(
                    artifact, norm, path, "required value is null"))
            return
        if kind == "string":
            valid = isinstance(node, str)
        elif kind == "boolean":
            valid = type(node) is bool
        elif kind == "integer":
            valid = type(node) is int
        else:
            import math
            valid = type(node) in (int, float) and math.isfinite(node)
        if not valid:
            out.append(_schema_violation(
                artifact, norm, path,
                f"expected {kind}, got {type(node).__name__} {node!r}"))

    if not isinstance(payload, dict):
        return [_schema_violation(
            artifact, None, None, "JSON manifest payload must be an object")]
    walk(payload, "")
    return out


def validate_artifact_schema(artifact, payload, well_keys=(), serialization_stage="auto"):
    if artifact in CSV_SCHEMAS:
        return _validate_csv_schema(artifact, payload, serialization_stage)
    if artifact == MANIFEST_ARTIFACT:
        return _validate_json_schema(artifact, payload, well_keys)
    return [_schema_violation(
        artifact, None, None, "artifact has no declared schema")]


def collect_string_fields(artifact, payload, well_keys=()):
    """Collect every schema-declared string occurrence, including ``""``.

    Field discovery is schema-driven. Numeric-looking text in a declared prose
    field is therefore still prose and must authorize; unknown or missing keys
    are handled independently by :func:`validate_artifact_schema`.
    """
    out = []
    if artifact in CSV_SCHEMAS and isinstance(payload, list):
        schema = CSV_SCHEMAS[artifact]
        for i, row in enumerate(payload):
            if not isinstance(row, dict):
                continue
            for column in schema.columns:
                value = row.get(column)
                if column in schema.string_fields and isinstance(value, str):
                    out.append(FieldOccurrence(
                        artifact, column, value, f"row[{i}].{column}"))
        return out
    if artifact == MANIFEST_ARTIFACT and isinstance(payload, dict):
        def walk(node, path):
            if isinstance(node, dict):
                for key, value in node.items():
                    walk(value, f"{path}/{key}")
            elif isinstance(node, list):
                for i, value in enumerate(node):
                    walk(value, f"{path}[{i}]")
            elif isinstance(node, str):
                norm = _normalize_json_path(path, well_keys)
                if _expected_json_primitive_kind(norm) == "string":
                    out.append(FieldOccurrence(artifact, norm, node, path))
        walk(payload, "")
    return out


# ---------------------------------------------------------------------------
# Authorization of one occurrence
# ---------------------------------------------------------------------------

def _is_number_with_optional_unit(token):
    """A decimal literal with an optional trailing alphabetic unit ("2.589m").
    Still a TYPE test: it admits a number and a unit, and no sentence."""
    if not isinstance(token, str) or not token:
        return False
    i = len(token)
    while i > 0 and token[i - 1].isalpha():
        i -= 1
    return OUTPUT_FIELD_TYPES["decimal"](token[:i]) and token[i:].isalpha() or (
        i == len(token) and OUTPUT_FIELD_TYPES["decimal"](token))


def _authorize_structured(value):
    """Grammar for machine-generated diagnostics. Admits `name=integer`,
    `name(number<number)` and `CODE:token` items joined by `;`. It admits no
    sentence: a space anywhere outside a bracket is a rejection."""
    for item in value.split(";"):
        item = item.strip()
        if not item:
            return False
        if "=" in item:
            name, _, num = item.partition("=")
            if not (name.replace("_", "").isalnum() and _is_integer_literal(num)):
                return False
        elif item.endswith(")") and "(" in item:
            name, _, rest = item.partition("(")
            lo, sep, hi = rest[:-1].partition("<")
            if not (name.replace("_", "").isalnum() and sep
                    and _is_number_with_optional_unit(lo)
                    and _is_number_with_optional_unit(hi)):
                return False
        elif ":" in item:
            code, _, token = item.partition(":")
            if not (code.replace("_", "").isalnum() and code.isupper()
                    and token.replace("_", "").replace(".", "").replace("-", "").isalnum()):
                return False
        else:
            return False
    return True


def _authorize_sanitized(value):
    if len(value) > SANITIZED_DIAGNOSTIC_MAX_LEN:
        return False, "exceeds the declared diagnostic length bound"
    if any(bad in value for bad in _FORBIDDEN_IN_DIAGNOSTIC):
        return False, "contains a forbidden control or path-like sequence"
    if not set(value) <= SANITIZED_DIAGNOSTIC_CHARSET:
        offending = sorted(set(value) - SANITIZED_DIAGNOSTIC_CHARSET)
        return False, f"contains characters outside the declared charset: {offending}"
    return True, None


def _parse_template(template, text):
    """Parse `text` back against `template`, returning the substitutions or None.
    The literal parts must match character for character."""
    names, literals, buf, rest = [], [], "", template
    while "{" in rest:
        head, _, rest = rest.partition("{")
        name, _, rest = rest.partition("}")
        literals.append(buf + head)
        names.append(name)
        buf = ""
    literals.append(buf + rest)
    if not text.startswith(literals[0]):
        return None
    remainder, values = text[len(literals[0]):], {}
    for name, nxt in zip(names, literals[1:]):
        if nxt:
            value, sep, remainder = remainder.partition(nxt)
            if not sep:
                return None
        else:
            value, remainder = remainder, ""
        values[name] = value
    return None if remainder else values


def authorize_occurrence(occ):
    """Authorize ONE emitted occurrence.

    Returns `(ok, authorization, reason)`. `authorization` is the credential
    that travels with the value: a statement id, a template id plus its typed
    substitutions, an `(field_kind, value)` label, or a category marker. It is
    retained by the caller until serialization so the emitted bytes can be
    re-checked against what was authorized.
    """
    policy = OUTPUT_FIELD_POLICY.get((occ.artifact, occ.field))
    if policy is None:
        return False, None, (
            f"UNCLASSIFIED output field {occ.artifact}:{occ.field!r}. Every emitted "
            f"string column and JSON path must carry a declared policy classification; "
            f"an unknown artifact, column or JSON path fails closed.")
    if occ.value == "":
        if policy.allow_empty:
            return True, ("declared_empty", policy.category), None
        return False, None, (
            f"empty string is not allowed for required field "
            f"{occ.artifact}:{occ.field!r}")
    cat = policy.category
    if cat in ("structural_enum", "identifier"):
        if occ.value not in policy.allowed_values:
            return False, None, (
                f"value is not in the enumerated {cat} vocabulary declared for "
                f"{occ.artifact}:{occ.field!r}")
        return True, (cat, occ.value), None
    if cat == "filename":
        if ("/" in occ.value or "\\" in occ.value or occ.value.startswith(".")
                or not occ.value.strip()):
            return False, None, "filename must be a bare basename with no path separator"
        return True, ("filename", occ.value), None
    if cat == "typed_label":
        lab = approved_label(policy.field_kind, occ.value)
        if lab is None:
            other = sorted(k for k in APPROVED_LABELS if occ.value in APPROVED_LABELS[k])
            extra = f" (it is approved only as {other})" if other else ""
            return False, None, (
                f"value is not approved for field_kind {policy.field_kind!r}{extra}")
        return True, ("typed_label", policy.field_kind, occ.value), None
    if cat == "registered_statement":
        for sid in policy.statement_ids:
            st = OUTPUT_STATEMENTS.get(sid)
            if st is not None and st.text == occ.value:
                return True, ("statement", sid), None
        return False, None, (
            f"text does not exactly match any statement registered for "
            f"{occ.artifact}:{occ.field!r} ({list(policy.statement_ids)}). Registered "
            f"prose is authorized by id and exact text; near-misses are rejected.")
    if cat == "controlled_template":
        if occ.value in policy.allowed_values:
            return True, ("template_sentinel", occ.value), None
        for sid in policy.statement_ids:
            st = OUTPUT_STATEMENTS.get(sid)
            if st is not None and st.text == occ.value:
                return True, ("statement", sid), None
        tpl = OUTPUT_TEMPLATES.get(policy.template_id)
        if tpl is None:
            return False, None, f"unknown template_id {policy.template_id!r}"
        values = _parse_template(tpl.template, occ.value)
        if values is None:
            return False, None, (
                f"text does not match template {tpl.template_id!r}; the fixed prose must "
                f"match character for character")
        if set(values) != set(tpl.fields):
            return False, None, f"template {tpl.template_id!r} substitution set mismatch"
        bad = sorted(n for n, v in values.items()
                     if not OUTPUT_FIELD_TYPES[tpl.fields[n]](v))
        if bad:
            return False, None, (
                f"template {tpl.template_id!r} substitution(s) {bad} do not satisfy their "
                f"declared type, so the template cannot carry prose")
        return True, ("template", tpl.template_id, values), None
    if cat == "structured_diagnostic":
        if occ.value in policy.allowed_values:
            # An enumerated sentinel token declared for this field.
            return True, ("structured_sentinel", occ.value), None
        if not _authorize_structured(occ.value):
            return False, None, (
                "value does not satisfy the declared machine-diagnostic grammar "
                "(name=integer / name(number<number) / CODE:token, joined by ';')")
        return True, ("structured_diagnostic", None), None
    if cat == "sanitized_diagnostic":
        ok, why = _authorize_sanitized(occ.value)
        return (True, ("sanitized_diagnostic", None), None) if ok else (False, None, why)
    return False, None, f"unhandled category {cat!r}"  # pragma: no cover


class CoverageReport:
    """What was actually inspected, and what it was."""

    __slots__ = ("n_string_field_occurrences", "n_controlled_occurrences",
                 "n_structural_occurrences", "n_unguaranteed_occurrences",
                 "n_unclassified_fields", "n_unauthorized_controlled_fields",
                 "n_field_kind_mismatches", "n_schema_violations",
                 "violations", "authorizations",
                 "artifacts_inspected", "distinct_statements_used",
                 "distinct_templates_used", "distinct_labels_used")

    def __init__(self, **kw):
        for slot in self.__slots__:
            setattr(self, slot, kw.get(slot))

    def as_dict(self):
        out = {s: getattr(self, s) for s in self.__slots__ if s != "authorizations"}
        # This dictionary is embedded directly in the JSON manifest before the
        # pre-serialization schema pass. Keep it JSON-native at construction;
        # do not rely on json.dumps silently converting tuples later.
        out["violations"] = [dict(v) for v in self.violations]
        out["artifacts_inspected"] = list(self.artifacts_inspected)
        return out

    @property
    def ok(self):
        return (self.n_unclassified_fields == 0
                and self.n_unauthorized_controlled_fields == 0
                and self.n_field_kind_mismatches == 0
                and self.n_schema_violations == 0
                and not self.violations)


def validate_emitted_records(payloads, well_keys=(), expected_artifacts=None,
                             serialization_stage="auto"):
    """Validate schema, then authorize every declared string occurrence.

    `payloads` maps artifact filename -> pre-serialization payload. Every
    declared artifact must be present; exact keys/order, requiredness and types
    are checked independently of values. Only after that closed schema pass are
    string values authorized. ``serialization_stage`` is ``pre``, ``post`` or
    ``auto`` (the compatibility default for callers loading packaged CSVs).
    """
    violations, authorizations = [], []
    counts = dict(controlled=0, structural=0, unguaranteed=0,
                  unclassified=0, unauthorized=0, kind_mismatch=0, schema=0)
    stmts, tpls, labels = set(), set(), set()
    # `expected_artifacts` narrows the completeness requirement. The manifest
    # builder uses it because it cannot present the manifest it is still
    # building; the export gate always requires the full declared set.
    declared = set(OUTPUT_ARTIFACTS if expected_artifacts is None else expected_artifacts)
    unknown_artifacts = sorted(set(payloads) - declared)
    missing_artifacts = sorted(declared - set(payloads))
    for name in unknown_artifacts:
        violations.append(_schema_violation(
            name, None, None, "artifact has no declared output schema"))
        counts["unclassified"] += 1
        counts["schema"] += 1
    for name in missing_artifacts:
        violations.append(_schema_violation(
            name, None, None, "declared artifact was not presented for validation"))
        counts["unclassified"] += 1
        counts["schema"] += 1
    total = 0
    for artifact in sorted(set(payloads) & declared):
        schema_bad = validate_artifact_schema(
            artifact, payloads[artifact], well_keys,
            serialization_stage=serialization_stage)
        violations.extend(schema_bad)
        counts["schema"] += len(schema_bad)
        for occ in collect_string_fields(artifact, payloads[artifact], well_keys):
            total += 1
            ok, auth, reason = authorize_occurrence(occ)
            policy = OUTPUT_FIELD_POLICY.get((artifact, occ.field))
            cat = policy.category if policy is not None else None
            if cat in CONTROLLED_CATEGORIES:
                counts["controlled"] += 1
            elif cat in STRUCTURAL_CATEGORIES:
                counts["structural"] += 1
            elif cat in UNGUARANTEED_CATEGORIES:
                counts["unguaranteed"] += 1
            if ok:
                authorizations.append((occ, auth))
                if auth[0] == "statement":
                    stmts.add(auth[1])
                elif auth[0] == "template":
                    tpls.add(auth[1])
                elif auth[0] == "typed_label":
                    labels.add((auth[1], auth[2]))
                continue
            if policy is None:
                counts["unclassified"] += 1
            elif cat == "typed_label":
                counts["kind_mismatch"] += 1
                counts["unauthorized"] += 1
            elif cat in CONTROLLED_CATEGORIES:
                counts["unauthorized"] += 1
            violations.append({
                "artifact": artifact, "field": occ.field, "location": occ.location,
                "reason": reason,
                "linter_terms": sorted(set(find_prohibited_lithology_terms(occ.value))),
            })
    return CoverageReport(
        n_string_field_occurrences=total,
        n_controlled_occurrences=counts["controlled"],
        n_structural_occurrences=counts["structural"],
        n_unguaranteed_occurrences=counts["unguaranteed"],
        n_unclassified_fields=counts["unclassified"],
        n_unauthorized_controlled_fields=counts["unauthorized"],
        n_field_kind_mismatches=counts["kind_mismatch"],
        n_schema_violations=counts["schema"],
        violations=tuple(violations), authorizations=tuple(authorizations),
        artifacts_inspected=tuple(sorted(set(payloads) & declared)),
        distinct_statements_used=len(stmts), distinct_templates_used=len(tpls),
        distinct_labels_used=len(labels),
    )


# ---------------------------------------------------------------------------
# The export gate: validate typed records, serialize to an isolated staging
# directory, compare the complete canonical round trip, then publish.
# ---------------------------------------------------------------------------


class OutputAuthorizationError(ValueError):
    """Raised when an artifact would persist an unauthorized controlled field."""


def _canonical_csv_value(value, kind, nullable, stage):
    import decimal
    if (stage == "pre" and value is None) or (stage == "post" and value == ""):
        if nullable:
            return ("null", None)
    if kind == "string":
        return ("string", value)
    if kind == "integer":
        return ("integer", int(value))
    if kind == "number":
        number = decimal.Decimal(str(value)).normalize()
        return ("number", number.as_tuple())
    if kind == "boolean":
        return ("boolean", value if stage == "pre" else value == "True")
    raise AssertionError(kind)  # pragma: no cover


def canonicalize_emitted_records(payloads, serialization_stage="auto"):
    """Return a complete, schema-aware semantic representation.

    The caller must first obtain a clean schema report. CSV values are restored
    to their declared semantic types, so harmless serialization representation
    differences do not mask or manufacture mutations. Row order and every
    declared field value remain part of the comparison.
    """
    import json as _json
    canonical = []
    for artifact in sorted(payloads):
        payload = payloads[artifact]
        if artifact in CSV_SCHEMAS:
            schema = CSV_SCHEMAS[artifact]
            stage = _csv_stage(payload, schema, serialization_stage)
            rows = tuple(
                tuple(_canonical_csv_value(
                    row[field], schema.kind(field), field in schema.nullable_fields, stage)
                    for field in schema.columns)
                for row in payload
            )
            canonical.append((artifact, schema.columns, rows))
        else:
            canonical.append((artifact, _json.dumps(
                payload, sort_keys=True, ensure_ascii=False, allow_nan=False,
                separators=(",", ":"))))
    return tuple(canonical)


def export_authorized_outputs(out_dir, payloads, well_keys=(), writer=None):
    """Schema-driven, transactional two-stage export.

    STAGE 1 validates exact artifact/field schemas, requiredness, types and
    content authorization. STAGE 2 writes only to a temporary sibling,
    validates the bytes read back, and compares every canonical record/value.
    The official destination is touched only after both stages pass.

    Returns `(report_before, report_after)`. Raises `OutputAuthorizationError`
    on any validation or serializer failure. A rejected export leaves the
    destination unchanged. Publication uses per-file atomic replacement with
    rollback; this is failure-atomic for handled process errors, not a claim of
    multi-file atomicity across power loss or operating-system failure.
    """
    import csv as _csv
    import json as _json
    import os as _os
    import pathlib as _pathlib
    import shutil as _shutil
    import tempfile as _tempfile

    out_dir = _pathlib.Path(out_dir)
    before = validate_emitted_records(
        payloads, well_keys=well_keys, serialization_stage="pre")
    if not before.ok:
        raise OutputAuthorizationError(
            f"refusing to write: {len(before.violations)} schema/authorization "
            f"output field(s); first: {before.violations[0] if before.violations else None}")
    canonical_before = canonicalize_emitted_records(payloads, serialization_stage="pre")

    allowed_destination_names = set(OUTPUT_ARTIFACTS) | {"figures"}
    if out_dir.exists():
        if not out_dir.is_dir():
            raise OutputAuthorizationError("output destination exists and is not a directory")
        stale = sorted(p.name for p in out_dir.iterdir()
                       if p.name not in allowed_destination_names)
        if stale:
            raise OutputAuthorizationError(
                f"output destination contains undeclared stale artifact(s): {stale}")

    parent = out_dir.parent
    parent.mkdir(parents=True, exist_ok=True)
    stage_dir = _pathlib.Path(_tempfile.mkdtemp(
        prefix=f".{out_dir.name}.stage-", dir=str(parent)))
    try:
        try:
            for name, payload in sorted(payloads.items()):
                target = stage_dir / name
                if writer is not None:
                    writer(target, payload)
                elif name.endswith(".json"):
                    target.write_text(_json.dumps(payload, indent=2, sort_keys=True) + "\n",
                                      encoding="utf-8")
                else:
                    schema = CSV_SCHEMAS[name]
                    with open(target, "w", encoding="utf-8", newline="") as fh:
                        w = _csv.DictWriter(fh, fieldnames=list(schema.columns))
                        w.writeheader()
                        w.writerows(payload)
        except OutputAuthorizationError:
            raise
        except Exception as exc:
            raise OutputAuthorizationError(
                f"serializer failed before publication: {exc}") from exc

        staged_names = {p.name for p in stage_dir.iterdir()}
        expected_names = set(payloads)
        if (staged_names != expected_names
                or any(p.is_symlink() or not p.is_file() for p in stage_dir.iterdir())):
            raise OutputAuthorizationError(
                f"serializer produced wrong artifact inventory: "
                f"missing={sorted(expected_names - staged_names)}, "
                f"unknown={sorted(staged_names - expected_names)}")

        reread = {}
        for name in payloads:
            target = stage_dir / name
            try:
                if name.endswith(".json"):
                    reread[name] = _json.loads(target.read_text(encoding="utf-8"))
                else:
                    with open(target, newline="", encoding="utf-8") as fh:
                        reread[name] = list(_csv.DictReader(fh))
            except Exception as exc:
                raise OutputAuthorizationError(
                    f"could not re-read staged artifact {name!r}: {exc}") from exc
        after = validate_emitted_records(
            reread, well_keys=well_keys, serialization_stage="post")
        if not after.ok:
            raise OutputAuthorizationError(
                f"post-serialization re-check failed: {len(after.violations)} field(s) "
                f"changed or became unauthorized after authorization; first: "
                f"{after.violations[0] if after.violations else None}")
        canonical_after = canonicalize_emitted_records(
            reread, serialization_stage="post")
        if canonical_after != canonical_before:
            mismatch = next((i for i, pair in enumerate(zip(
                canonical_before, canonical_after)) if pair[0] != pair[1]), None)
            raise OutputAuthorizationError(
                f"post-serialization canonical record mismatch at artifact index {mismatch}; "
                "an authorized value, field, row order or type changed")

        # Publication happens only after the complete staged set has passed.
        out_dir.mkdir(parents=True, exist_ok=True)
        backup_dir = _pathlib.Path(_tempfile.mkdtemp(
            prefix=f".{out_dir.name}.backup-", dir=str(parent)))
        replaced, created = [], []
        try:
            for name in sorted(payloads):
                target = out_dir / name
                if target.exists():
                    _shutil.copy2(target, backup_dir / name)
                    replaced.append(name)
                else:
                    created.append(name)
                _os.replace(stage_dir / name, target)
        except Exception:
            for name in created:
                target = out_dir / name
                if target.exists():
                    target.unlink()
            for name in replaced:
                backup = backup_dir / name
                if backup.exists():
                    _os.replace(backup, out_dir / name)
            raise
        finally:
            _shutil.rmtree(backup_dir, ignore_errors=True)
        return before, after
    finally:
        _shutil.rmtree(stage_dir, ignore_errors=True)


In [ ]:
%%writefile p2mem/io/petrophysics_inventory.py
"""
p2mem.io.petrophysics_inventory - Deterministic, metadata-oriented output
builders for the Increment 6 GR-QC / eligibility layer.

Mirrors the design of the LOCKED `p2mem.io.tops_inventory` (Increment 5)
and `p2mem.io.checkshot_inventory` (Increment 4): every function returns a
list of plain dicts (one per output row) ready for `csv.DictWriter`, or a
single JSON-serializable manifest dict.

Two disciplines are enforced here and tested directly:

1. NO PER-SAMPLE REAL-DATA ARRAYS ARE EVER EXPORTED. Every row is a
   summary, a scenario record, or an interval register entry. Exporting a
   full per-sample GR/VP/VS/RHOB array would effectively reproduce the
   private source logs inside a deliverable ZIP, which this project does
   not do. Interval registers carry depths and counts, never the sample
   values inside the interval.

2. NO ABSOLUTE PATH IS EVER EXPORTED. Any path-shaped field is reduced to
   its basename, and any free-text message is sanitized against every
   candidate source path (both the LAS path and the survey path, since a
   well-frame failure may originate from either - the Increment 5.1
   Finding-2 lesson applied here from the start).

Every numeric field is a plain Python float/int/None, never a NumPy
scalar, so CSV and JSON output is byte-stable across environments.
"""

from __future__ import annotations

from pathlib import Path
from typing import Dict, List, Optional, Sequence

import numpy as np

from p2mem.method_eligibility import EligibilityInterval, EligibilityMaskResult
from p2mem.petrophysics_models import (
    GrEndpointScenario,
    GrFamilyDisposition,
    GrFamilyQcStats,
    GrProxyResult,
    PetrophysicsIssue,
)
from p2mem.io.output_policy import OUTPUT_STATEMENTS
from p2mem.petrophysics import PetrophysicsInputError
from p2mem.wellframe_models import (
    SCOPE_EXPLANATORY,
    SCOPE_METHOD,
    Authorization,
    REGISTERED_STATEMENTS,
    REGISTERED_TEMPLATES,
    SCOPE_INTERPRETIVE,
    SCOPE_LABEL,
    WellFrame,
    WellFrameAssemblyFailure,
    validate_no_prohibited_interpretation,
)

__all__ = [
    "build_gr_family_qc_rows",
    "build_gr_endpoint_scenario_rows",
    "build_gr_proxy_sensitivity_rows",
    "build_method_eligibility_rows",
    "build_eligibility_interval_rows",
    "build_thickness_sensitivity_rows",
    "build_petrophysics_issue_rows",
    "build_petrophysics_manifest",
    "build_lithology_validation_scope",
]

TIER_CLASSIFICATION = "Tier C - Screening-Level / Uncalibrated Educational"

_NOT_A_LITHOLOGY = (
    "Data/proxy confidence only - asserts no named lithology and no calibration."
)


def _sanitize_message(message: str, *source_paths) -> str:
    """Replace every literal full path in `source_paths` with its
    basename. Literal substring replacement only, never a regex, so
    unrelated scientific text is never corrupted. Mirrors the LOCKED
    `p2mem.io.tops_inventory._sanitize_message`."""
    for source_path in source_paths:
        if not source_path:
            continue
        message = message.replace(str(source_path), Path(str(source_path)).name)
    return message


def _f(value) -> Optional[float]:
    """Plain Python float, or None. NaN/Inf become None so CSV/JSON never
    carries a non-finite literal that a downstream reader may parse
    inconsistently."""
    if value is None:
        return None
    v = float(value)
    return v if np.isfinite(v) else None


def _i(value) -> Optional[int]:
    return None if value is None else int(value)


def build_gr_family_qc_rows(
    stats_by_well: Dict[str, GrFamilyQcStats],
    dispositions: Dict[str, GrFamilyDisposition],
    confidence_by_well: Dict[str, str],
) -> List[Dict]:
    """
    One factual QC row per well, INCLUDING every excluded well - the
    numbers that justify an exclusion must themselves be published.
    Sorted by well key for deterministic output.
    """
    rows: List[Dict] = []
    for well_key in sorted(stats_by_well):
        s = stats_by_well[well_key]
        d = dispositions.get(well_key)
        rows.append(
            {
                "well_key": s.well_key,
                "source_las_filename": s.source_las_filename,
                "gr_family_canonical_name": s.gr_family_canonical_name,
                "gr_family_source_curve_name": s.gr_family_source_curve_name,
                "unit": s.unit,
                "use_status": (d.use_status if d else ""),
                "exclusion_reason": (d.exclusion_reason if d and d.exclusion_reason else ""),
                "evidence_class": (d.evidence_class if d else ""),
                "gr_proxy_confidence_class": confidence_by_well.get(well_key, ""),
                "has_approved_formation_tops": (bool(d.has_approved_formation_tops) if d else False),
                "n_samples": _i(s.n_samples),
                "valid_count": _i(s.valid_count),
                "valid_fraction": _f(s.valid_fraction),
                "min_api": _f(s.min_api),
                "max_api": _f(s.max_api),
                "median_api": _f(s.median_api),
                "p01_api": _f(s.p01_api),
                "p05_api": _f(s.p05_api),
                "p10_api": _f(s.p10_api),
                "p25_api": _f(s.p25_api),
                "p50_api": _f(s.p50_api),
                "p75_api": _f(s.p75_api),
                "p90_api": _f(s.p90_api),
                "p95_api": _f(s.p95_api),
                "p99_api": _f(s.p99_api),
                "dynamic_range_p05_p95_api": _f(s.dynamic_range_p05_p95_api),
                "n_negative_samples": _i(s.n_negative_samples),
                "n_zero_samples": _i(s.n_zero_samples),
                "n_samples_above_seabed": _i(s.n_samples_above_seabed),
                "seabed_basis": s.seabed_basis,
                "n_valid_blocks": _i(s.n_valid_blocks),
                "longest_valid_block_samples": _i(s.longest_valid_block_samples),
                "longest_missing_block_samples": _i(s.longest_missing_block_samples),
                "longest_missing_block_md_start_m": _f(s.longest_missing_block_md_start_m),
                "longest_missing_block_md_end_m": _f(s.longest_missing_block_md_end_m),
                "depth_basis": "MDRT (measured depth below rotary table), metres",
                "assurance_tier": TIER_CLASSIFICATION,
                "statistics_basis": s.statistics_basis,
                "limitations": (
                    "Descriptive statistics of the curve AS RECORDED. No environmental correction, "
                    "rescaling, or cross-well normalization applied. " + _NOT_A_LITHOLOGY
                ),
            }
        )
    return rows


#: The authorization-carrying derivation text, itself a registered output
#: statement so this assurance prose is authorized like any other emitted field.
_LITHOLOGY_DERIVATION = OUTPUT_STATEMENTS["lithology_validation_derivation"].text


def _assert_emitted_field_authorized(artifact, field, value, where):
    """Increment 6.1.6 (Finding 1): a row builder must not be able to emit an
    unauthorized controlled value merely because a manifest validator runs
    later. Authorization is checked HERE, on the value actually placed in the
    record, and the resolved credential is returned so it travels with the
    value until serialization."""
    from p2mem.io.output_policy import FieldOccurrence, authorize_occurrence
    ok, auth, reason = authorize_occurrence(
        FieldOccurrence(artifact, field, value, where))
    if not ok:
        raise PetrophysicsInputError(
            f"{artifact}:{field} at {where} is not authorized for emission - {reason}")
    return auth


def build_gr_endpoint_scenario_rows(
    scenarios_by_well: Dict[str, Sequence[GrEndpointScenario]],
) -> List[Dict]:
    """One row per (well, scenario), recording the MEASURED endpoint values
    each configured rule actually produced. Sorted by well then scenario
    name for deterministic output."""
    rows: List[Dict] = []
    for well_key in sorted(scenarios_by_well):
        for sc in sorted(scenarios_by_well[well_key], key=lambda s: s.scenario_name):
            rows.append(
                {
                    "well_key": sc.well_key,
                    "scenario_name": sc.scenario_name,
                    "low_percentile": _f(sc.low_percentile),
                    "high_percentile": _f(sc.high_percentile),
                    "gr_low_endpoint_api": _f(sc.gr_low_endpoint_api),
                    "gr_high_endpoint_api": _f(sc.gr_high_endpoint_api),
                    "endpoint_separation_api": _f(sc.endpoint_separation_api),
                    "n_samples_used_for_endpoints": _i(sc.n_samples_used_for_endpoints),
                    "endpoint_sample_basis": sc.endpoint_sample_basis,
                    "unit": "API",
                    "evidence_class": sc.evidence_class,
                    "calibration_status": sc.calibration_status,
                    "assurance_tier": TIER_CLASSIFICATION,
                    "description": _assert_emitted_field_authorized(
                        "gr_endpoint_scenarios.csv", "description", sc.description,
                        f"{sc.well_key}/{sc.scenario_name}") and sc.description,
                    "limitations": (
                        "Endpoints are ASSUMED, configured percentile values estimated from this "
                        "well's OWN samples. They are not calibrated, are not shared across wells, "
                        "and must never be presented as a validated endpoint pair. " + _NOT_A_LITHOLOGY
                    ),
                }
            )
    return rows


def build_gr_proxy_sensitivity_rows(
    proxies_by_well: Dict[str, Sequence[GrProxyResult]],
    dispositions: Dict[str, GrFamilyDisposition],
) -> List[Dict]:
    """One row per (well, scenario) summarizing IGR/proxy behavior -
    summary statistics and clipping counts only, never per-sample arrays."""
    rows: List[Dict] = []
    for well_key in sorted(proxies_by_well):
        d = dispositions.get(well_key)
        for p in sorted(proxies_by_well[well_key], key=lambda x: x.scenario_name):
            rows.append(
                {
                    "well_key": p.well_key,
                    "scenario_name": p.scenario_name,
                    "gr_family_canonical_name": p.gr_family_canonical_name,
                    "use_status": (d.use_status if d else ""),
                    "gr_low_endpoint_api": _f(p.gr_low_endpoint_api),
                    "gr_high_endpoint_api": _f(p.gr_high_endpoint_api),
                    "transform_name": p.transform_name,
                    "proxy_field_name": "VSH_GR_linear_proxy_frac",
                    "unit": "dimensionless_fraction",
                    "n_valid": _i(p.n_valid),
                    "n_clipped_low": _i(p.n_clipped_low),
                    "n_clipped_high": _i(p.n_clipped_high),
                    "clipped_fraction": _f(p.clipped_fraction),
                    "proxy_median": _f(p.proxy_median),
                    "proxy_p25": _f(p.proxy_p25),
                    "proxy_p75": _f(p.proxy_p75),
                    "calibration_status": p.calibration_status,
                    "evidence_class": "correlation_derived_screening_proxy_uncalibrated",
                    "assurance_tier": TIER_CLASSIFICATION,
                    "limitations": (
                        "IGR and the linear screening proxy are dimensionless quantities derived "
                        "under ASSUMED endpoints. The proxy is NOT a calibrated shale volume and "
                        "NOT a lithology. Clipped and unclipped indices are computed and retained "
                        "together in memory; the clipped counts here quantify how far the real "
                        "data fell outside the assumed endpoint bracket. " + _NOT_A_LITHOLOGY
                    ),
                }
            )
    return rows


def build_method_eligibility_rows(
    mask_results: Sequence[EligibilityMaskResult],
    frames: Dict[str, WellFrame],
    dispositions: Dict[str, GrFamilyDisposition],
) -> List[Dict]:
    """One row per (well, mask, scenario, threshold). Deterministically
    sorted by well, mask, scenario, threshold."""

    def sort_key(m: EligibilityMaskResult):
        return (
            m.well_key,
            m.mask_name,
            m.scenario_name or "",
            float("-inf") if m.proxy_threshold is None else float(m.proxy_threshold),
        )

    rows: List[Dict] = []
    for m in sorted(mask_results, key=sort_key):
        fr = frames.get(m.well_key)
        d = dispositions.get(m.well_key)
        rows.append(
            {
                "well_key": m.well_key,
                "mask_name": m.mask_name,
                "scenario_name": m.scenario_name or "",
                "proxy_threshold": _f(m.proxy_threshold),
                "lithology_dependent": bool(m.lithology_dependent),
                "use_status": (d.use_status if d else ""),
                "exclusion_reason": (d.exclusion_reason if d and d.exclusion_reason else ""),
                "n_samples": _i(m.n_samples),
                "n_eligible": _i(m.n_eligible),
                "eligible_fraction": _f(m.eligible_fraction),
                "limiting_criterion": m.limiting_criterion,
                "criteria_counts": ";".join(f"{k}={v}" for k, v in sorted(m.criteria_counts.items())),
                "diagnostic_counts": ";".join(
                    f"{k}={v}" for k, v in sorted(getattr(m, "diagnostic_counts", {}).items())
                ),
                "depth_basis_used": (fr.depth_basis_used if fr else ""),
                "interpolation_method": (fr.interpolation_method if fr else ""),
                "depth_map_status": (fr.depth_map_status if fr else ""),
                "n_depth_unmapped": _i(fr.n_depth_unmapped) if fr else None,
                "n_extrapolated": _i(fr.n_extrapolated) if fr else None,
                "unit": "sample_count_and_fraction",
                "assurance_tier": TIER_CLASSIFICATION,
                "purpose": m.purpose,
                "limitations": (
                    "ELIGIBILITY IS NOT VALIDITY. This is a necessary, not sufficient, condition "
                    "for a LATER method; the method itself is not implemented, not fitted, and not "
                    "validated in Increment 6. " + m.notes + " " + _NOT_A_LITHOLOGY
                ),
            }
        )
    return rows


def build_eligibility_interval_rows(
    intervals: Sequence[EligibilityInterval],
) -> List[Dict]:
    """
    One row per contiguous eligible block, on MD/TVD/TVDSS.

    Increment 6.1 (Finding 4): thickness is exported under explicitly
    qualified names - `gross_thickness_*_m` (endpoint span, which under the
    configured policy may contain disclosed bridged samples) and
    `net_thickness_*_m` (that span with the bridged gaps removed) - never
    an unqualified "thickness". `contiguity_policy` states which
    decomposition produced the row, and `meets_configured_minimums` states
    whether it qualifies.

    Carries depths, counts and limiting reasons only - never the sample
    values inside the interval, which would reproduce the private source
    log.
    """
    def sort_key(iv: EligibilityInterval):
        return (
            iv.well_key or "",
            iv.mask_name or "",
            iv.contiguity_policy or "",
            iv.scenario_name or "",
            float("-inf") if iv.proxy_threshold is None else float(iv.proxy_threshold),
            int(iv.start_index or 0),
        )

    rows: List[Dict] = []
    for iv in sorted(intervals, key=sort_key):
        d = iv.as_dict()
        rows.append(
            {
                "well_key": d["well_key"],
                "mask_name": d["mask_name"],
                "contiguity_policy": d["contiguity_policy"],
                "scenario_name": d["scenario_name"] or "",
                "proxy_threshold": _f(d["proxy_threshold"]),
                "block_index": _i(d["block_index"]),
                "start_index": _i(d["start_index"]),
                "end_index": _i(d["end_index"]),
                "n_samples": _i(d["n_samples"]),
                "n_eligible_samples": _i(d["n_eligible_samples"]),
                "n_bridged_samples": _i(d["n_bridged_samples"]),
                "n_bridged_gaps": _i(d["n_bridged_gaps"]),
                "n_eligible_subruns": _i(d["n_eligible_subruns"]),
                "meets_configured_minimums": bool(d["meets_configured_minimums"]),
                "md_start_m": _f(d["md_start_m"]),
                "md_end_m": _f(d["md_end_m"]),
                "gross_thickness_md_m": _f(d["gross_thickness_md_m"]),
                "net_thickness_md_m": _f(d["net_thickness_md_m"]),
                "tvd_start_m": _f(d["tvd_start_m"]),
                "tvd_end_m": _f(d["tvd_end_m"]),
                "gross_thickness_tvd_m": _f(d["gross_thickness_tvd_m"]),
                "net_thickness_tvd_m": _f(d["net_thickness_tvd_m"]),
                "tvdss_start_m": _f(d["tvdss_start_m"]),
                "tvdss_end_m": _f(d["tvdss_end_m"]),
                "gross_thickness_tvdss_m": _f(d["gross_thickness_tvdss_m"]),
                "net_thickness_tvdss_m": _f(d["net_thickness_tvdss_m"]),
                "limiting_reason": d["limiting_reason"],
                "depth_basis_used": d["depth_basis_used"],
                "unit": "metres",
                "assurance_tier": TIER_CLASSIFICATION,
                "limitations": (
                    "Candidate/eligible DATA extent only. GROSS thickness is the block's "
                    "endpoint span and, under the configured contiguity policy, may include "
                    "explicitly bridged ineligible samples (see n_bridged_samples); NET "
                    "thickness removes those gaps. Neither is an unqualified 'eligible "
                    "thickness'. A sonic-NCT-candidate interval is not proof of normal "
                    "compaction, is not a fitted trend, and is not a selected donor interval. "
                    + _NOT_A_LITHOLOGY
                ),
            }
        )
    return rows


def build_thickness_sensitivity_rows(
    intervals: Sequence[EligibilityInterval],
) -> List[Dict]:
    """
    Aggregate interval records into one row per
    (well, mask, contiguity policy, scenario, threshold), reporting the
    QUALIFYING population's gross and net thickness side by side together
    with the bridging that separates them.

    Increment 6.1 (Finding 4): every sensitivity case states its population
    explicitly - how many blocks were found, how many qualify, how many
    bridged samples the gross figure absorbed, and how many blocks were
    interrupted - so a gross span can never be read as unbroken eligible
    section.
    """
    keys = sorted({
        (iv.well_key, iv.mask_name, iv.contiguity_policy, iv.scenario_name or "",
         float("-inf") if iv.proxy_threshold is None else float(iv.proxy_threshold))
        for iv in intervals
    })
    rows: List[Dict] = []
    for wk, mask_name, policy, scenario, thr in keys:
        sel = [
            iv for iv in intervals
            if iv.well_key == wk and iv.mask_name == mask_name
            and iv.contiguity_policy == policy
            and (iv.scenario_name or "") == scenario
            and (float("-inf") if iv.proxy_threshold is None
                 else float(iv.proxy_threshold)) == thr
        ]
        qual = [iv for iv in sel if iv.meets_configured_minimums]
        def _sum(items, attr):
            vals = [getattr(iv, attr) for iv in items]
            return None if any(v is None for v in vals) else float(sum(vals))
        rows.append({
            "well_key": wk,
            "mask_name": mask_name,
            "contiguity_policy": policy,
            "scenario_name": scenario,
            "proxy_threshold": None if thr == float("-inf") else _f(thr),
            "n_blocks_all": len(sel),
            "n_blocks_qualifying": len(qual),
            "n_blocks_rejected_below_minimums": len(sel) - len(qual),
            # Increment 6.1.1 (Finding 3): three distinct quantities, each
            # named for exactly what it counts. Samples and gaps are different
            # magnitudes (one gap can absorb several samples), and the count of
            # AFFECTED BLOCKS is different again.
            "n_bridged_samples_in_qualifying_blocks": int(
                sum(iv.n_bridged_samples for iv in qual)
            ),
            "n_bridged_gaps_in_qualifying_blocks": int(
                sum(iv.n_bridged_gaps for iv in qual)
            ),
            "n_interrupted_qualifying_blocks": int(
                sum(1 for iv in qual if iv.n_bridged_gaps > 0)
            ),
            "gross_qualifying_thickness_tvdss_m": _f(_sum(qual, "gross_thickness_tvdss_m")),
            "net_qualifying_thickness_tvdss_m": _f(_sum(qual, "net_thickness_tvdss_m")),
            "gross_all_block_thickness_tvdss_m": _f(_sum(sel, "gross_thickness_tvdss_m")),
            "n_eligible_samples_qualifying": int(sum(iv.n_eligible_samples for iv in qual)),
            "unit": "metres",
            "population_statement": (
                f"Totals cover the {len(qual)} block(s) meeting the configured minimums out of "
                f"{len(sel)} found under the {policy!r} contiguity policy. GROSS is the sum of "
                f"block endpoint spans and includes "
                f"{int(sum(iv.n_bridged_samples for iv in qual))} disclosed bridged sample(s) "
                f"across {int(sum(iv.n_bridged_gaps for iv in qual))} bridged gap(s) in "
                f"{int(sum(1 for iv in qual if iv.n_bridged_gaps > 0))} interrupted block(s); "
                f"NET removes those gaps."
            ),
            "assurance_tier": TIER_CLASSIFICATION,
            "limitations": (
                "Eligible/candidate DATA extent only. No gated method is implemented, fitted, or "
                "validated. " + _NOT_A_LITHOLOGY
            ),
        })
    return rows


def build_petrophysics_issue_rows(
    issues: Sequence[PetrophysicsIssue],
    failures: Dict[str, WellFrameAssemblyFailure],
) -> List[Dict]:
    """Issue rows plus one row per well-frame assembly failure, with every
    message sanitized against BOTH candidate source paths."""
    rows: List[Dict] = []
    for iss in issues:
        rows.append(
            {
                "severity": iss.severity,
                "code": iss.code,
                "context": iss.context,
                "message": iss.message,
                "assurance_tier": TIER_CLASSIFICATION,
            }
        )
    for well_key in sorted(failures):
        f = failures[well_key]
        rows.append(
            {
                "severity": "ERROR",
                "code": f"WELLFRAME_{f.error_type.upper()}",
                "context": (
                    f"{f.well_key} (failure_origin={f.failure_origin}; "
                    f"{'+'.join(sorted({Path(p).name for p in (f.las_path, f.survey_path) if p}))})"
                ),
                "message": _sanitize_message(f.message, f.las_path, f.survey_path),
                "assurance_tier": TIER_CLASSIFICATION,
            }
        )
    return rows


_RATIONALE_TEMPLATE_ID = "gr_proxy_confidence_rationale"


def _template_binding(text):
    """Resolve `text` against the registered rationale template by PARSING the
    declared substitution slots out of it. Returns the Authorization when the
    rendered result matches exactly, else None.

    This is a lookup against a registered template, not an inference about
    meaning: the surrounding prose must match the template character for
    character, and only the declared numeric slots may differ.
    """
    tpl = REGISTERED_TEMPLATES.get(_RATIONALE_TEMPLATE_ID)
    if tpl is None:
        return None
    names, literals = [], []
    rest, buf = tpl.template, ""
    while "{" in rest:
        head, _, rest = rest.partition("{")
        name, _, rest = rest.partition("}")
        literals.append(buf + head)
        names.append(name)
        buf = ""
    literals.append(buf + rest)
    remainder, values = text, {}
    if not remainder.startswith(literals[0]):
        return None
    remainder = remainder[len(literals[0]):]
    for name, nxt in zip(names, literals[1:]):
        if nxt:
            value, sep, remainder = remainder.partition(nxt)
            if not sep:
                return None
        else:
            value, remainder = remainder, ""
        values[name] = value
    if remainder:
        return None
    auth = Authorization(template_id=_RATIONALE_TEMPLATE_ID, fields=values)
    return auth if tpl.render(values) == text else None


def _auth_for_text(text, scope):
    """Return the Authorization for an exact registered statement in `scope`,
    or a template binding, or None when nothing registered matches."""
    for statement_id, st in REGISTERED_STATEMENTS.items():
        if st.scope == scope and st.text == text:
            return Authorization(statement_id=statement_id)
    if scope == SCOPE_INTERPRETIVE:
        return _template_binding(text)
    return None


def _authorized(context, text, scope):
    """Emit one scope entry carrying whatever authorization the text resolves
    to. Unresolvable text is emitted WITHOUT authorization, so the validator
    reports it rather than this builder hiding it."""
    if not isinstance(text, str) or not text:
        return (context, text, scope)
    auth = _auth_for_text(text, scope)
    return (context, text, scope) if auth is None else (context, text, scope, auth)


def build_lithology_validation_scope(
    dispositions: Dict[str, GrFamilyDisposition],
    confidence_by_well: Dict[str, str],
    confidence_rationale_by_well: Dict[str, str],
    mask_results: Sequence[EligibilityMaskResult],
    extra_entries: Optional[Sequence] = None,
):
    """
    Assemble the ACTUAL persisted content that the named-lithology validation
    must inspect, as `(context, text, scope)` triples.

    Increment 6.1 (Finding 3): the manifest's `named_lithology_assigned` flag
    is DERIVED from running the validator over this scope. It is not a
    constant, and it cannot report "false" while a rock name sits in a
    persisted field - injecting one makes the validation, the manifest flag,
    and the completion gate all fail together.

    Everything a well is JUDGED by is scanned as INTERPRETIVE: its confidence
    class and the rationale behind it, its configured use-status, evidence
    class, exclusion reason, curve identity, the human-authored per-well
    note, and every mask name and mask note. Callers may add further entries
    (for example the manifest's own statements) via `extra_entries`.

    Increment 6.1.3 closes a real gap. Three strings that this project WRITES
    INTO PACKAGED EXPORTS had never been inside the validated scope in any
    previous increment - including a `calibration_status` label value literally
    containing `not_a_shale_volume`, persisted to every row of
    gr_proxy_sensitivity_summary.csv. Those fields are now scanned in
    SCOPE_METHOD, which admits them only by exact membership of the closed
    METHOD_STATEMENTS registry. Widening the scope raises
    `n_fields_checked`; that is the intended, disclosed consequence of
    validating content the project already ships.
    """
    entries = []
    # Every persisted field below is emitted WITH its authorization. A field
    # with no authorization is rejected by the validator, so this builder
    # cannot silently introduce an unauthorized string.
    #
    # Increment 6.1.5: authorization is by id. `_auth_for_text` resolves the
    # registered statement (or template binding) whose canonical text this
    # field carries; if no registered statement matches, it returns None and
    # the validator reports the field as unauthorized. It is a LOOKUP, never a
    # judgement about wording.
    for statement_id in sorted(REGISTERED_STATEMENTS):
        st = REGISTERED_STATEMENTS[statement_id]
        if st.scope in (SCOPE_METHOD, SCOPE_EXPLANATORY):
            entries.append((f"{st.scope}_statement.{statement_id}", st.text, st.scope,
                            Authorization(statement_id=statement_id)))
    for well_key in sorted(dispositions):
        d = dispositions[well_key]
        # LABELS - verdicts a well is stamped with. Typed, enumerated values.
        entries.extend([
            (f"{well_key}.use_status", d.use_status, SCOPE_LABEL,
             Authorization(field_kind="use_status")),
            (f"{well_key}.evidence_class", d.evidence_class, SCOPE_LABEL,
             Authorization(field_kind="evidence_class")),
            (f"{well_key}.exclusion_reason", d.exclusion_reason or "", SCOPE_LABEL,
             Authorization(field_kind="exclusion_reason")),
            (f"{well_key}.gr_family_canonical_name", d.gr_family_canonical_name, SCOPE_LABEL,
             Authorization(field_kind="gr_family_canonical_name")),
            (f"{well_key}.gr_family_source_curve_name", d.gr_family_source_curve_name,
             SCOPE_LABEL, Authorization(field_kind="gr_family_source_curve_name")),
        ])
        entries.append(_authorized(f"{well_key}.config_notes", d.notes, SCOPE_INTERPRETIVE))
    for well_key in sorted(confidence_by_well):
        entries.append(
            (f"{well_key}.gr_proxy_confidence_class", confidence_by_well[well_key],
             SCOPE_LABEL, Authorization(field_kind="gr_proxy_confidence_class"))
        )
    for well_key in sorted(confidence_rationale_by_well):
        entries.append(_authorized(
            f"{well_key}.gr_proxy_confidence_rationale",
            confidence_rationale_by_well[well_key], SCOPE_INTERPRETIVE))
    for m in mask_results:
        entries.append((f"{m.well_key}.{m.mask_name}.mask_name", m.mask_name, SCOPE_LABEL,
                        Authorization(field_kind="mask_name")))
        entries.append(_authorized(f"{m.well_key}.{m.mask_name}.notes", m.notes,
                                   SCOPE_INTERPRETIVE))
        entries.append(_authorized(f"{m.well_key}.{m.mask_name}.purpose", m.purpose,
                                   SCOPE_INTERPRETIVE))
    if extra_entries:
        entries.extend(list(extra_entries))
    return entries


def build_petrophysics_manifest(
    frames: Dict[str, WellFrame],
    dispositions: Dict[str, GrFamilyDisposition],
    stats_by_well: Dict[str, GrFamilyQcStats],
    confidence_by_well: Dict[str, str],
    confidence_rationale_by_well: Dict[str, str],
    scenarios_by_well: Dict[str, Sequence[GrEndpointScenario]],
    mask_results: Sequence[EligibilityMaskResult],
    failures: Dict[str, WellFrameAssemblyFailure],
    issues: Sequence[PetrophysicsIssue],
    *,
    config_filename: str,
    config_schema_version: str,
    nct_candidate_thresholds: Sequence[float],
    output_payloads: Optional[Dict[str, object]] = None,
) -> Dict:
    """
    Build the single JSON-serializable Increment 6 manifest.

    Contains only metadata, counts and summary statistics - never a
    per-sample array, and never an absolute path.
    """
    # The manifest's own free-text statement about lithology is itself part of
    # the validated scope - it is explanatory, so it may not attach a rock name
    # to any of this project's wells.
    # Increment 6.1.6 (Finding 5): ONE source of truth. The literal previously
    # duplicated here could diverge from the registered copy while the gate
    # stayed green, because the scope validated the registry entry rather than
    # the value actually persisted. The persisted value IS the registered text.
    _named_lithology_statement = REGISTERED_STATEMENTS["named_lithology_statement"].text
    # Increment 6.1.5: the manifest's own explanatory statement is emitted by
    # the scope builder from REGISTERED_STATEMENTS, with its authorization, so
    # it is no longer injected here as unauthorized extra text.
    _lith_scope = list(build_lithology_validation_scope(
        dispositions, confidence_by_well, confidence_rationale_by_well, mask_results,
    ))
    _lith_violations = validate_no_prohibited_interpretation(_lith_scope)

    # Increment 6.1.6 (Finding 2): authorize the ACTUAL emitted CSV records, not
    # a reconstruction of them. `output_payloads` maps artifact filename ->
    # pre-serialization rows. When it is omitted the coverage block records that
    # honestly rather than implying a check that did not run.
    _emitted_violations = []
    if output_payloads:
        from p2mem.io.output_policy import validate_emitted_records
        _rep = validate_emitted_records(dict(output_payloads), well_keys=set(frames),
                                       expected_artifacts=set(output_payloads))
        _emitted_coverage = _rep.as_dict()
        _emitted_coverage["violations"] = len(_rep.violations)
        _emitted_violations = [
            {"context": f"{v['artifact']}:{v['field']}", "scope": "emitted_output",
             "terms": v.get("linter_terms", []), "reason": v["reason"]}
            for v in _rep.violations]
    else:
        _emitted_coverage = {"status": "not_supplied",
                             "note": "No emitted CSV records were presented to this "
                                     "builder, so no emission-boundary coverage is "
                                     "claimed here. The export gate enforces it."}

    wells_block = {}
    for well_key in sorted(set(frames) | set(dispositions) | set(stats_by_well)):
        fr = frames.get(well_key)
        d = dispositions.get(well_key)
        s = stats_by_well.get(well_key)
        well_masks = [m for m in mask_results if m.well_key == well_key]
        wells_block[well_key] = {
            "source_las_filename": (fr.source_las_filename if fr else (d.source_las_filename if d else "")),
            "source_survey_filename": (fr.source_survey_filename if fr else ""),
            "well_frame_assembled": fr is not None,
            "n_samples": (_i(fr.n_samples) if fr else None),
            "depth_basis_used": (fr.depth_basis_used if fr else ""),
            "interpolation_method": (fr.interpolation_method if fr else ""),
            "depth_map_status": (fr.depth_map_status if fr else ""),
            "n_depth_unmapped": (_i(fr.n_depth_unmapped) if fr else None),
            "n_extrapolated": (_i(fr.n_extrapolated) if fr else None),
            "datum_elevation_m": (_f(fr.datum_elevation_m) if fr else None),
            "well_identity_evidence_status": (fr.well_identity_evidence_status if fr else ""),
            "qc_flags": list(fr.qc_flags) if fr else [],
            "gr_family_canonical_name": (d.gr_family_canonical_name if d else ""),
            "gr_family_source_curve_name": (d.gr_family_source_curve_name if d else ""),
            "use_status": (d.use_status if d else ""),
            "exclusion_reason": (d.exclusion_reason if d and d.exclusion_reason else None),
            "has_approved_formation_tops": (bool(d.has_approved_formation_tops) if d else None),
            "gr_proxy_confidence_class": confidence_by_well.get(well_key, ""),
            "gr_proxy_confidence_rationale": confidence_rationale_by_well.get(well_key, ""),
            "gr_valid_fraction": (_f(s.valid_fraction) if s else None),
            "gr_median_api": (_f(s.median_api) if s else None),
            "gr_min_api": (_f(s.min_api) if s else None),
            "gr_max_api": (_f(s.max_api) if s else None),
            "gr_n_samples_above_seabed": (_i(s.n_samples_above_seabed) if s else None),
            "endpoint_scenarios": [
                {
                    "scenario_name": sc.scenario_name,
                    "gr_low_endpoint_api": _f(sc.gr_low_endpoint_api),
                    "gr_high_endpoint_api": _f(sc.gr_high_endpoint_api),
                    "endpoint_separation_api": _f(sc.endpoint_separation_api),
                    "evidence_class": sc.evidence_class,
                    "calibration_status": sc.calibration_status,
                }
                for sc in sorted(scenarios_by_well.get(well_key, ()), key=lambda x: x.scenario_name)
            ],
            "method_eligibility": [
                {
                    "mask_name": m.mask_name,
                    "scenario_name": m.scenario_name or "",
                    "proxy_threshold": _f(m.proxy_threshold),
                    "n_eligible": _i(m.n_eligible),
                    "eligible_fraction": _f(m.eligible_fraction),
                    "limiting_criterion": m.limiting_criterion,
                    "lithology_dependent": bool(m.lithology_dependent),
                }
                for m in sorted(
                    well_masks,
                    key=lambda x: (
                        x.mask_name,
                        x.scenario_name or "",
                        float("-inf") if x.proxy_threshold is None else float(x.proxy_threshold),
                    ),
                )
            ],
        }

    return {
        "increment": 6,
        "increment_title": (
            "Gamma-Ray QC, Shale-Proxy Sensitivity, Well-Frame Assembly, and "
            "Method-Eligibility Framework"
        ),
        "assurance_tier": TIER_CLASSIFICATION,
        "config_filename": config_filename,
        "config_schema_version": config_schema_version,
        "depth_reference_convention": (
            "MD and TVD are referenced to the well datum (rotary table), increasing downward; "
            "TVDSS_m = TVD_m - DatumElevation_m, with datum elevation referenced to MSL, positive "
            "upward. Identical to the LOCKED Increment 3.1.1 convention - not re-derived here."
        ),
        "nct_candidate_proxy_thresholds": [float(t) for t in nct_candidate_thresholds],
        "n_wells_frame_assembled": len(frames),
        "n_wells_frame_failed": len(failures),
        "n_wells_gr_proxy_permitted": sum(1 for d in dispositions.values() if d.proxy_permitted),
        "n_wells_gr_excluded": sum(1 for d in dispositions.values() if not d.proxy_permitted),
        "total_samples_extrapolated": int(sum(fr.n_extrapolated for fr in frames.values())),
        # DERIVED, never hardcoded (Increment 6.1, Finding 3): this is the
        # result of running the validator over the scope assembled above from
        # real persisted content. If a prohibited term were injected anywhere
        # in that scope, this flag would flip to true and the completion gate
        # would fail.
        "named_lithology_assigned": bool(_lith_violations) or bool(_emitted_violations),
        "lithology_validation": {
            "model": "schema_driven_transactional_authorization_at_emission_boundary",
            "derivation": _LITHOLOGY_DERIVATION,
            # Increment 6.1.6 (§6): the counts below say WHAT they count.
            # `scope_object_*` describes the constructed validation scope - the
            # object Increment 6.1.5 called "every persisted field", which it was
            # not. `emitted_field_coverage` describes the ACTUAL CSV records this
            # run will write; this manifest is itself re-validated by the export
            # gate after serialization, which is why it cannot count itself here.
            "scope_object_fields_checked": len(_lith_scope),
            "scope_object_violations": len(_lith_violations),
            "emitted_field_coverage": _emitted_coverage,
            "violations": [dict(v) for v in _lith_violations] + list(_emitted_violations),
        },
        "named_lithology_statement": _named_lithology_statement,
        "nonlinear_vsh_transforms_implemented": False,
        "nonlinear_vsh_deferral_statement": (
            "No nonlinear Vsh transform (Larionov, Clavier, Stieber, or any other) is implemented. "
            "Each would require a retrieved, verified primary-source method record and a closed "
            "method-register entry; none exists in this project."
        ),
        "methods_not_implemented": [
            "vertical_stress_integration", "hydrostatic_pressure_modelling",
            "normal_compaction_trend_fitting", "eaton_sonic_pore_pressure",
            "eaton_resistivity_pore_pressure", "bowers_pore_pressure",
            "dynamic_elastic_property_calculation", "static_elastic_conversion",
            "rock_strength_correlation", "friction_angle_modelling",
            "shmin_shmax_modelling", "stress_polygon_construction",
            "wellbore_stability_analysis", "mud_weight_recommendation",
            "named_lithology_interpretation", "environmental_gr_correction",
            "gr_rescaling_or_normalization_across_wells", "density_reconstruction_or_extrapolation",
        ],
        "calibration_data_available": {
            "pressure_rft_mdt_dst": False,
            "stress_fit_lot_xlot_dfit": False,
            "independent_vp_vs_calibration": False,
            "statement": (
                "No RFT, MDT, DST, FIT, LOT, XLOT or DFIT data exist for this project. The "
                "supplied Vp/Vs text file is derived from the existing sonic curves and is NOT "
                "independent calibration data. Every quantity in this increment therefore remains "
                "uncalibrated."
            ),
        },
        "wells": wells_block,
        "issues": [
            {"severity": i.severity, "code": i.code, "context": i.context, "message": i.message}
            for i in issues
        ],
        "assembly_failures": [
            {
                "well_key": failures[k].well_key,
                "failure_origin": failures[k].failure_origin,
                "error_type": failures[k].error_type,
                "message": _sanitize_message(
                    failures[k].message, failures[k].las_path, failures[k].survey_path
                ),
            }
            for k in sorted(failures)
        ],
        "limitations": [
            "Tier C - screening-level and uncalibrated. Nothing here is validated against "
            "independent measurement.",
            "Eligibility is a necessary, not sufficient, condition. No method gated by these masks "
            "is implemented, fitted, or validated in this increment.",
            "A sonic-NCT-candidate interval is candidate DATA only. It is not a fitted trend, not a "
            "selected donor interval, and not evidence of normal compaction or overpressure.",
            "GR endpoints are ASSUMED configured percentiles of each well's own samples, never "
            "calibrated and never shared across wells.",
            "Boreas 1 is formally excluded from all GR-derived work under "
            "BOREAS_ECGR_SCALE_UNRESOLVED and is retained for factual raw QC display only. It is "
            "excluded rather than corrected because no calibration evidence exists to support any "
            "correction.",
            "Poseidon North 1 and Proteus 1ST2 have no approved formation tops; their results are "
            "depth-tied and stratigraphically unvalidated.",
            "No named lithology is assigned, and the available data do not support assigning one.",
        ],
    }


---

## Tests

#### Step 7 — Write the Increment 6 test suite

**Technical objective:** write the synthetic-only regression tests for this increment. Every LAS result, deviation result, curve array, disposition and config in these tests is fabricated in memory — **no real or private project file is read, referenced, or packaged as a fixture.**

The shared synthetic builders construct the LOCKED prior increments' REAL dataclasses rather than mocks, so a field renamed in a locked model breaks these builders loudly instead of letting a mock drift silently out of step with reality.

In [ ]:
%%writefile tests/synthetic_inc6.py
"""
Shared SYNTHETIC builders for the Increment 6 test modules.

Everything here is fictional and generated in memory. No real or private
project LAS, deviation, checkshot, or formation-top file is read,
referenced, copied, or packaged by this module or by any test that uses
it.

These builders construct the LOCKED prior increments' REAL dataclasses
(rather than mocks) so the Increment 6 tests stay honest about the actual
shapes those locked layers produce - a field renamed in a locked model
would break these builders loudly instead of letting a mock drift
silently out of step with reality.

`PROJECT_ROOT` is resolved from this file's own location so that every
test is independent of the process working directory.
"""

from pathlib import Path

import numpy as np

from p2mem.deviation_models import (
    DEPTH_BASIS_PETREL_SOURCE,
    DepthBasisSelection,
    DeviationHeaderInfo,
    DeviationStationData,
    DeviationWellResult,
)
from p2mem.models import CurveStats, FileContract, LasFileResult, LasHeaderInfo

__all__ = ["PROJECT_ROOT", "synthetic_survey", "synthetic_las"]

PROJECT_ROOT = Path(__file__).resolve().parent.parent


def synthetic_survey(
    well_key="Synth_1",
    md=(0.0, 100.0, 200.0, 300.0, 400.0),
    tvd=(0.0, 100.0, 199.0, 297.0, 394.0),
    datum_elevation_m=25.0,
):
    """A minimal, fictional `DeviationWellResult` on the Petrel-source basis.

    Only the fields the depth layer and the well-frame layer actually read
    are meaningfully populated; the rest carry inert placeholders.
    """
    md = np.asarray(md, dtype=float)
    tvd = np.asarray(tvd, dtype=float)
    z = np.zeros(md.size)
    raw = DeviationStationData(
        MD_source_m=md,
        X_source_m=z.copy(),
        Y_source_m=z.copy(),
        Z_source_m=datum_elevation_m - tvd,
        TVD_source_m=tvd,
        DX_source_m=z.copy(),
        DY_source_m=z.copy(),
        AZIM_TN_source_deg=z.copy(),
        INCL_source_deg=z.copy(),
        DLS_source_deg_per_30m=z.copy(),
        AZIM_GN_source_deg=z.copy(),
    )
    header = DeviationHeaderInfo(
        source_path=f"/synthetic/{well_key}_dev.txt",
        source_filename=f"{well_key}_dev.txt",
        sha256="0" * 64,
        well_name=well_key,
        survey_name="synthetic",
        wellhead_x_m=0.0,
        wellhead_y_m=0.0,
        datum_elevation_m=datum_elevation_m,
        datum_reference="synthetic RT",
        well_type="synthetic",
        coordinate_reference_system="synthetic",
        depth_reference_statement="synthetic",
        angle_unit_statement="degrees",
        dx_dy_statement="synthetic",
        z_statement="synthetic",
        column_names=("MD", "X", "Y", "Z", "TVD", "DX", "DY", "AZIM_TN", "INCL", "DLS", "AZIM_GN"),
        header_line_count=0,
        data_line_offset=0,
    )
    basis = DepthBasisSelection(
        well_key=well_key,
        selected_basis=DEPTH_BASIS_PETREL_SOURCE,
        rationale="synthetic test fixture",
    )
    return DeviationWellResult(
        header=header, contract=None, raw=raw, mc=None,
        validation=None, depth_basis=basis, issues=(), contract_status="PASSED",
    )


def synthetic_las(canonical, well_name="Synth_1", source_filename="Synth_1_logs.las"):
    """A minimal, fictional `LasFileResult` carrying only what the
    well-frame layer reads: `canonical_data`, `curve_stats`, `contract`."""
    stats = tuple(
        CurveStats(
            source_curve_name=name.split("_")[0],
            raw_mnemonic=name.split("_")[0],
            raw_description=f"synthetic {name}",
            raw_unit="API" if ("GR" in name or "ECGR" in name) else "SYN",
            canonical_name=name,
            canonical_unit="API" if ("GR" in name or "ECGR" in name) else "SYN",
            conversion_function="identity",
            n_samples=int(np.asarray(arr).size),
            valid_count=int(np.count_nonzero(np.isfinite(arr))),
            null_count=int(np.count_nonzero(~np.isfinite(arr))),
            valid_fraction=float(np.count_nonzero(np.isfinite(arr)) / np.asarray(arr).size),
            raw_min=None, raw_max=None, canonical_min=None, canonical_max=None,
            statistics_basis="synthetic",
        )
        for name, arr in canonical.items()
    )
    header = LasHeaderInfo(
        source_path=f"/synthetic/{source_filename}",
        source_filename=source_filename,
        sha256="0" * 64,
        las_version="2.0",
        wrap="NO",
        well_name=well_name,
        declared_null=-999.25,
        declared_strt=None,
        declared_stop=None,
        declared_step=None,
        version_section=(),
        well_section=(),
        parameter_section=(),
        curve_headers=(),
        data_section_line_offset=0,
    )
    contract = FileContract(
        source_filename=source_filename, expected_sha256="0" * 64,
        expected_well_identifier=well_name, expected_las_version="2.0",
        expected_wrap="NO", expected_null_value=-999.25,
        expected_curve_count=len(canonical), expected_data_layout="ascii", curves=(),
    )
    return LasFileResult(
        header=header, contract=contract, resolutions=(), depth=None,
        curve_stats=stats, raw_data=np.zeros((1, 1)),
        canonical_data=dict(canonical), issues=(), contract_status="PASSED",
    )


In [ ]:
%%writefile tests/test_wellframe.py
"""
Increment 6 - well-frame assembly tests.

SYNTHETIC ONLY. Every LAS result, deviation result, and curve array in
this file is fabricated in-memory from fictional numbers. No real or
private project file is read, referenced, or packaged.
"""

import numpy as np
import pytest

from p2mem.depth_mapping import DepthMappingError
from p2mem.wellframe import (
    QC_FLAG_DEPTH_COVERAGE_LIMITED,
    QC_FLAG_GR_FAMILY_ABSENT,
    QC_FLAG_NO_DEPTH_COVERAGE,
    WellFrameAssemblyError,
    assemble_well_frame,
    assemble_well_frames,
)
from p2mem.wellframe_models import (
    DEPTH_MAP_STATUS_FULL,
    DEPTH_MAP_STATUS_NONE,
    DEPTH_MAP_STATUS_PARTIAL,
    PROHIBITED_LITHOLOGY_TERMS,
    assert_no_lithology_vocabulary,
)


# ---------------------------------------------------------------------------
# Synthetic builders (shared, cwd-independent - see tests/synthetic_inc6.py)
# ---------------------------------------------------------------------------

from synthetic_inc6 import synthetic_las as _synthetic_las  # noqa: E402
from synthetic_inc6 import synthetic_survey as _synthetic_survey  # noqa: E402


def _basic_frame(**kw):
    md = np.array([50.0, 100.0, 150.0, 200.0, 250.0, 300.0])
    las = _synthetic_las({"MD_m": md, "GR_api": np.array([10.0, 20.0, np.nan, 40.0, 50.0, 60.0])})
    dev = _synthetic_survey()
    return assemble_well_frame(
        "Synth_1", las, dev, las_path="/synthetic/Synth_1_logs.las",
        survey_path="/synthetic/Synth_1_dev.txt", gr_family_canonical_name="GR_api", **kw
    )


# ---------------------------------------------------------------------------
# Sample/order preservation
# ---------------------------------------------------------------------------

def test_well_frame_preserves_sample_count_and_order():
    """A well frame never truncates, pads, resamples, sorts, or reorders."""
    md = np.array([50.0, 100.0, 150.0, 200.0, 250.0, 300.0])
    gr = np.array([10.0, 20.0, np.nan, 40.0, 50.0, 60.0])
    frame = _basic_frame()
    assert frame.n_samples == md.size
    np.testing.assert_array_equal(frame.MD_m, md)
    np.testing.assert_array_equal(frame.curve("GR_api").values, gr)
    assert frame.TVD_m.size == md.size
    assert frame.TVDSS_m.size == md.size


def test_well_frame_never_deletes_failed_samples():
    """An invalid sample stays in place as NaN and is expressed only in the
    mask - never dropped, filled, or interpolated."""
    frame = _basic_frame()
    slot = frame.curve("GR_api")
    assert slot.n_samples == 6
    assert slot.valid_count == 5
    assert np.isnan(slot.values[2])
    assert slot.valid_mask[2] is np.False_ or slot.valid_mask[2] == False  # noqa: E712
    assert np.count_nonzero(slot.valid_mask) == 5


def test_well_frame_arrays_are_read_only():
    """Downstream consumers cannot mutate a frame's arrays in place."""
    frame = _basic_frame()
    assert not frame.MD_m.flags.writeable
    assert not frame.TVDSS_m.flags.writeable
    assert not frame.curve("GR_api").values.flags.writeable
    with pytest.raises(ValueError):
        frame.MD_m[0] = 1.0


def test_well_frame_length_mismatch_rejected_with_typed_error():
    """A curve whose length disagrees with MD_m is a structural defect and
    is never reconciled by truncation or padding."""
    las = _synthetic_las(
        {"MD_m": np.array([50.0, 100.0, 150.0]), "GR_api": np.array([1.0, 2.0])}
    )
    with pytest.raises(WellFrameAssemblyError, match="never truncates, pads, or resamples"):
        assemble_well_frame(
            "Synth_1", las, _synthetic_survey(),
            las_path="/synthetic/a.las", survey_path="/synthetic/a_dev.txt",
        )


def test_well_frame_missing_md_rejected():
    las = _synthetic_las({"GR_api": np.array([1.0, 2.0, 3.0])})
    with pytest.raises(WellFrameAssemblyError, match="no canonical 'MD_m'"):
        assemble_well_frame(
            "Synth_1", las, _synthetic_survey(),
            las_path="/synthetic/a.las", survey_path="/synthetic/a_dev.txt",
        )


# ---------------------------------------------------------------------------
# Depth mapping: no extrapolation, ever
# ---------------------------------------------------------------------------

def test_no_extrapolation_all_samples_within_coverage():
    frame = _basic_frame()
    assert frame.depth_map_status == DEPTH_MAP_STATUS_FULL
    assert frame.n_extrapolated == 0
    assert frame.n_depth_unmapped == 0
    assert bool(np.all(frame.depth_valid_mask))


def test_samples_beyond_survey_coverage_are_unmapped_never_extrapolated():
    """A LAS sample past the last survey station gets NaN depth and a False
    mask - never a clamped or held-constant TVD."""
    md = np.array([100.0, 200.0, 300.0, 400.0, 500.0, 900.0])  # last two beyond 400 m coverage
    las = _synthetic_las({"MD_m": md, "GR_api": np.full(6, 30.0)})
    frame = assemble_well_frame(
        "Synth_1", las, _synthetic_survey(),
        las_path="/s/a.las", survey_path="/s/a_dev.txt", gr_family_canonical_name="GR_api",
    )
    assert frame.depth_map_status == DEPTH_MAP_STATUS_PARTIAL
    assert frame.n_extrapolated == 0
    assert frame.n_depth_unmapped == 2
    assert np.isnan(frame.TVD_m[-1]) and np.isnan(frame.TVDSS_m[-1])
    assert not frame.depth_valid_mask[-1]
    # The in-coverage samples still carry genuine, locked-mapper values.
    assert np.isfinite(frame.TVD_m[0]) and np.isfinite(frame.TVDSS_m[3])
    assert any(f.startswith(QC_FLAG_DEPTH_COVERAGE_LIMITED) for f in frame.qc_flags)


def test_sample_below_survey_start_is_unmapped():
    md = np.array([-50.0, 100.0, 200.0])
    las = _synthetic_las({"MD_m": md, "GR_api": np.full(3, 30.0)})
    frame = assemble_well_frame(
        "Synth_1", las, _synthetic_survey(md=(0.0, 100.0, 200.0), tvd=(0.0, 100.0, 199.0)),
        las_path="/s/a.las", survey_path="/s/a_dev.txt",
    )
    assert frame.n_depth_unmapped == 1
    assert np.isnan(frame.TVD_m[0])
    assert frame.n_extrapolated == 0


def test_zero_coverage_overlap_reports_none_status():
    md = np.array([9000.0, 9100.0, 9200.0])
    las = _synthetic_las({"MD_m": md, "GR_api": np.full(3, 30.0)})
    frame = assemble_well_frame(
        "Synth_1", las, _synthetic_survey(),
        las_path="/s/a.las", survey_path="/s/a_dev.txt",
    )
    assert frame.depth_map_status == DEPTH_MAP_STATUS_NONE
    assert frame.n_depth_unmapped == 3
    assert frame.n_extrapolated == 0
    assert QC_FLAG_NO_DEPTH_COVERAGE in frame.qc_flags
    assert bool(np.all(np.isnan(frame.TVDSS_m)))


def test_tvdss_equals_tvd_minus_datum_elevation():
    """The locked depth-reference convention is reproduced exactly, not
    re-derived with a different sign."""
    frame = _basic_frame()
    finite = np.isfinite(frame.TVD_m)
    np.testing.assert_allclose(
        frame.TVDSS_m[finite], frame.TVD_m[finite] - frame.datum_elevation_m
    )


def test_single_station_survey_rejected():
    las = _synthetic_las({"MD_m": np.array([10.0, 20.0]), "GR_api": np.array([1.0, 2.0])})
    dev = _synthetic_survey(md=(0.0,), tvd=(0.0,))
    with pytest.raises(DepthMappingError):
        assemble_well_frame(
            "Synth_1", las, dev, las_path="/s/a.las", survey_path="/s/a_dev.txt"
        )


# ---------------------------------------------------------------------------
# Per-file GR identity preservation
# ---------------------------------------------------------------------------

def test_gr_family_identity_is_recorded_never_guessed():
    """The GR-family curve is the one the caller names, and its per-file
    source identity is preserved."""
    md = np.arange(5, dtype=float) * 50.0 + 50.0
    las = _synthetic_las({"MD_m": md, "ECGR_api": np.full(5, 8.0)}, source_filename="B_logs.las")
    frame = assemble_well_frame(
        "B_1", las, _synthetic_survey(), las_path="/s/B_logs.las",
        survey_path="/s/B_dev.txt", gr_family_canonical_name="ECGR_api",
    )
    assert frame.gr_family_canonical_name == "ECGR_api"
    assert frame.gr_family_source_curve_name == "ECGR"
    assert frame.curve("ECGR_api").source_filename == "B_logs.las"


def test_absent_gr_family_curve_is_flagged_never_substituted():
    """A well missing its declared GR-family curve is flagged as a data
    gap - another curve is never silently used instead."""
    md = np.arange(5, dtype=float) * 50.0 + 50.0
    las = _synthetic_las({"MD_m": md, "RHOB_kg_m3": np.full(5, 2400.0)})
    frame = assemble_well_frame(
        "Synth_1", las, _synthetic_survey(), las_path="/s/a.las",
        survey_path="/s/a_dev.txt", gr_family_canonical_name="GRD_api",
    )
    assert frame.gr_family_canonical_name is None
    assert any(f.startswith(QC_FLAG_GR_FAMILY_ABSENT) for f in frame.qc_flags)
    assert frame.curve("GRD_api") is None


def test_two_wells_sharing_canonical_gr_name_stay_distinct():
    """Two wells whose contracts both canonicalize to GR_api remain
    separate frames with separate provenance - never merged."""
    md = np.arange(5, dtype=float) * 50.0 + 50.0
    a = _synthetic_las({"MD_m": md, "GR_api": np.full(5, 30.0)},
                       well_name="A_1", source_filename="A_1_logs.las")
    b = _synthetic_las({"MD_m": md, "GR_api": np.full(5, 90.0)},
                       well_name="B_1", source_filename="B_1_logs.las")
    frames, failures = assemble_well_frames(
        {"A_1": a, "B_1": b},
        {"A_1": _synthetic_survey("A_1"), "B_1": _synthetic_survey("B_1")},
        las_paths={"A_1": "/s/A_1_logs.las", "B_1": "/s/B_1_logs.las"},
        survey_paths={"A_1": "/s/A_1_dev.txt", "B_1": "/s/B_1_dev.txt"},
        gr_family_by_well={"A_1": "GR_api", "B_1": "GR_api"},
    )
    assert not failures
    assert frames["A_1"].curve("GR_api").source_filename == "A_1_logs.las"
    assert frames["B_1"].curve("GR_api").source_filename == "B_1_logs.las"
    assert frames["A_1"].curve("GR_api").values[0] != frames["B_1"].curve("GR_api").values[0]


# ---------------------------------------------------------------------------
# Batch failure isolation
# ---------------------------------------------------------------------------

def test_batch_failure_isolation_missing_survey():
    md = np.arange(5, dtype=float) * 50.0 + 50.0
    good = _synthetic_las({"MD_m": md, "GR_api": np.full(5, 30.0)}, well_name="G_1")
    bad = _synthetic_las({"MD_m": md, "GR_api": np.full(5, 30.0)}, well_name="B_1")
    frames, failures = assemble_well_frames(
        {"G_1": good, "B_1": bad},
        {"G_1": _synthetic_survey("G_1")},
        las_paths={"G_1": "/s/G.las", "B_1": "/s/B.las"},
        survey_paths={"G_1": "/s/G_dev.txt", "B_1": "/s/B_dev.txt"},
    )
    assert set(frames) == {"G_1"}
    assert set(failures) == {"B_1"}
    assert failures["B_1"].failure_origin == "survey"
    assert "never assembled against another well's trajectory" in failures["B_1"].message


def test_batch_failure_isolation_missing_las():
    frames, failures = assemble_well_frames(
        {}, {"X_1": _synthetic_survey("X_1")},
        las_paths={}, survey_paths={"X_1": "/s/X_dev.txt"},
    )
    assert frames == {}
    assert failures["X_1"].failure_origin == "las"


def test_batch_failure_isolation_bad_curve_length():
    md = np.arange(5, dtype=float) * 50.0 + 50.0
    good = _synthetic_las({"MD_m": md, "GR_api": np.full(5, 30.0)}, well_name="G_1")
    bad = _synthetic_las({"MD_m": md, "GR_api": np.full(3, 30.0)}, well_name="B_1")
    frames, failures = assemble_well_frames(
        {"G_1": good, "B_1": bad},
        {"G_1": _synthetic_survey("G_1"), "B_1": _synthetic_survey("B_1")},
        las_paths={"G_1": "/s/G.las", "B_1": "/s/B.las"},
        survey_paths={"G_1": "/s/G_dev.txt", "B_1": "/s/B_dev.txt"},
    )
    assert set(frames) == {"G_1"}
    assert failures["B_1"].failure_origin == "curve_assembly"
    assert "B_1" not in frames


def test_batch_results_are_deterministically_ordered():
    md = np.arange(5, dtype=float) * 50.0 + 50.0
    las = {k: _synthetic_las({"MD_m": md, "GR_api": np.full(5, 30.0)}, well_name=k)
           for k in ("Z_1", "A_1", "M_1")}
    dev = {k: _synthetic_survey(k) for k in ("Z_1", "A_1", "M_1")}
    frames, _ = assemble_well_frames(
        las, dev,
        las_paths={k: f"/s/{k}.las" for k in las},
        survey_paths={k: f"/s/{k}_dev.txt" for k in las},
    )
    assert list(frames) == ["A_1", "M_1", "Z_1"]


# ---------------------------------------------------------------------------
# Prohibited named-lithology vocabulary guard
# ---------------------------------------------------------------------------

@pytest.mark.parametrize(
    "bad", ["shale", "SAND", "probable-sandstone", "carbonate_zone", "net pay",
            "limestone", "MARL", "reservoir rock"],
)
def test_lithology_vocabulary_guard_rejects_rock_names(bad):
    with pytest.raises(ValueError, match="prohibited named-lithology term"):
        assert_no_lithology_vocabulary(bad, "test")


@pytest.mark.parametrize(
    "ok", ["GR_PROXY_HIGH", "GR_PROXY_INTERMEDIATE", "GR_PROXY_LOW",
           "GR_NOT_AVAILABLE", "GR_EXCLUDED_UNRESOLVED_SCALE",
           "VSH_GR_linear_proxy_frac", "eligible_sonic_nct_candidate",
           "a thousand samples"],
)
def test_lithology_vocabulary_guard_accepts_confidence_and_proxy_names(ok):
    assert_no_lithology_vocabulary(ok, "test")


def test_lithology_guard_rejects_non_string():
    with pytest.raises(TypeError):
        assert_no_lithology_vocabulary(123, "test")


def test_prohibited_vocabulary_covers_the_specified_rock_names():
    """Every rock name the Increment 6 specification names as prohibited is
    actually in the enforced vocabulary."""
    for required in ("shale", "sand", "sandstone", "carbonate", "limestone", "marl"):
        assert required in PROHIBITED_LITHOLOGY_TERMS


In [ ]:
%%writefile tests/test_petrophysics.py
"""
Increment 6 - GR QC, endpoint-sensitivity, and screening-proxy tests.

SYNTHETIC ONLY. Every array, disposition, and config in this file is
fabricated in-memory. No real or private project file is read or packaged.
"""

import json

import numpy as np
import pytest

from p2mem.petrophysics import (
    PetrophysicsConfigError,
    PetrophysicsExclusionError,
    PetrophysicsInputError,
    classify_gr_proxy_confidence,
    compute_gr_family_qc_stats,
    compute_gr_index,
    compute_gr_proxy,
    find_contiguous_blocks,
    load_petrophysics_eligibility_config,
)
from p2mem.petrophysics_models import (
    EXCLUSION_REASON_BOREAS_ECGR,
    GR_EXCLUDED_UNRESOLVED_SCALE,
    GR_NOT_AVAILABLE,
    GR_PROXY_HIGH,
    GR_PROXY_INTERMEDIATE,
    GR_PROXY_LOW,
    USE_STATUS_PROXY_ALLOWED,
    USE_STATUS_QC_ONLY_EXCLUDED,
    GrEndpointScenario,
    GrFamilyDisposition,
    PetrophysicsEligibilityConfig,
)
from p2mem.wellframe_models import assert_no_lithology_vocabulary

from synthetic_inc6 import PROJECT_ROOT  # noqa: E402
from synthetic_inc6 import synthetic_las as _synthetic_las  # noqa: E402
from synthetic_inc6 import synthetic_survey as _synthetic_survey  # noqa: E402
from p2mem.wellframe import assemble_well_frame  # noqa: E402


# ---------------------------------------------------------------------------
# Synthetic helpers
# ---------------------------------------------------------------------------

def _frame_with_gr(gr_values, canonical="GR_api", well_key="Synth_1", md=None):
    gr = np.asarray(gr_values, dtype=float)
    if md is None:
        md = np.linspace(50.0, 350.0, gr.size)
    las = _synthetic_las({"MD_m": np.asarray(md, dtype=float), canonical: gr}, well_name=well_key)
    dev = _synthetic_survey(well_key, md=(0.0, 200.0, 400.0), tvd=(0.0, 199.0, 396.0))
    return assemble_well_frame(
        well_key, las, dev, las_path=f"/s/{well_key}.las", survey_path=f"/s/{well_key}_dev.txt",
        gr_family_canonical_name=canonical,
    )


def _disposition(well_key="Synth_1", canonical="GR_api", status=USE_STATUS_PROXY_ALLOWED,
                 reason=None, tops=True):
    return GrFamilyDisposition(
        well_key=well_key, source_las_filename=f"{well_key}_logs.las",
        gr_family_canonical_name=canonical, gr_family_source_curve_name=canonical.split("_")[0],
        use_status=status, exclusion_reason=reason, evidence_class="measured",
        has_approved_formation_tops=tops, notes="synthetic",
    )


def _config(**policy_overrides) -> PetrophysicsEligibilityConfig:
    policy = {
        "endpoint_estimation_method": "per_well_percentile_of_own_valid_samples",
        "cross_well_shared_endpoints_allowed": False,
        "endpoint_sample_basis": "finite_and_depth_mapped_samples_only",
        "clipping_policy": "retain_unclipped_and_clipped_side_by_side",
        "clip_lower": 0.0, "clip_upper": 1.0,
        "min_endpoint_separation_api": 1.0,
        "shale_proxy_transform": "linear_identity_of_clipped_igr",
        "nonlinear_vsh_transforms_enabled": False,
        "contiguity": {"max_gap_samples": 2, "max_gap_depth_m": 1.0,
                       "min_block_samples": 2, "min_block_thickness_m": 1.0},
        "nct_candidate_proxy_thresholds": [0.5, 0.6, 0.7],
        "physical_bounds": {
            "rhob_min_kg_m3": 1000.0, "rhob_max_kg_m3": 3500.0,
            "vp_min_m_s": 1000.0, "vp_max_m_s": 8000.0,
            "vs_min_m_s": 300.0, "vs_max_m_s": 5000.0,
            "vp_vs_ratio_nonnegative_poisson_min_inclusive": np.sqrt(2.0),
            "vp_vs_ratio_positive_bulk_modulus_min_exclusive": np.sqrt(4.0 / 3.0),
            "vp_vs_ratio_plausibility_max": 4.0,
        },
    }
    policy.update(policy_overrides)
    return PetrophysicsEligibilityConfig(
        schema_version="6.0", increment=6, assurance_tier="Tier C", policy=policy,
        endpoint_scenarios=(
            {"scenario_name": "low", "low_percentile": 10.0, "high_percentile": 85.0,
             "description": "narrow"},
            {"scenario_name": "base", "low_percentile": 5.0, "high_percentile": 95.0,
             "description": "mid"},
            {"scenario_name": "high", "low_percentile": 1.0, "high_percentile": 99.0,
             "description": "wide"},
        ),
        wells={},
        method_eligibility={
            "eligible_density_for_sv": {"purpose": "synthetic purpose"},
            "eligible_dynamic_elastic": {"purpose": "synthetic purpose"},
            "eligible_sonic_nct_candidate": {"purpose": "synthetic purpose"},
        },
        gr_proxy_confidence={"classes": [], "rules": {
            "high": {"min_valid_fraction": 0.90, "min_dynamic_range_api": 60.0,
                     "max_proxy_median_spread_across_scenarios": 0.10},
            "intermediate": {"min_valid_fraction": 0.70, "min_dynamic_range_api": 40.0,
                             "max_proxy_median_spread_across_scenarios": 0.20},
        }},
        source_filename="synthetic.yml",
    )


def _registered_endpoint_description():
    """Increment 6.1.6: the endpoint description is a CONTROLLED emitted field,
    so a synthetic one is correctly refused by the builder."""
    from p2mem.io.output_policy import OUTPUT_STATEMENTS
    return sorted(st.text for st in OUTPUT_STATEMENTS.values()
                  if st.statement_id.startswith("gr_endpoint_scenarios_description"))[0]


def _scenario(low=10.0, high=110.0, well_key="Synth_1", name="base"):
    return GrEndpointScenario(
        well_key=well_key, scenario_name=name, low_percentile=5.0, high_percentile=95.0,
        gr_low_endpoint_api=low, gr_high_endpoint_api=high,
        endpoint_separation_api=high - low, n_samples_used_for_endpoints=100,
        endpoint_sample_basis="finite_and_depth_mapped_samples_only",
        description=_registered_endpoint_description(),
    )


# ---------------------------------------------------------------------------
# GR index arithmetic
# ---------------------------------------------------------------------------

def test_gr_index_basic_arithmetic():
    gr = np.array([0.0, 50.0, 100.0])
    unc, clip, valid = compute_gr_index(gr, 0.0, 100.0)
    np.testing.assert_allclose(unc, [0.0, 0.5, 1.0])
    np.testing.assert_allclose(clip, [0.0, 0.5, 1.0])
    assert bool(np.all(valid))


def test_clipped_and_unclipped_differ_outside_endpoints():
    """Both results are retained; the unclipped one preserves the honest
    signal that data ran past the assumed bracket."""
    gr = np.array([-20.0, 50.0, 200.0])
    unc, clip, _ = compute_gr_index(gr, 0.0, 100.0)
    np.testing.assert_allclose(unc, [-0.2, 0.5, 2.0])
    np.testing.assert_allclose(clip, [0.0, 0.5, 1.0])
    assert unc[0] < 0.0 and unc[2] > 1.0
    assert clip.min() >= 0.0 and clip.max() <= 1.0


def test_nan_input_stays_masked_and_nan_in_both_outputs():
    """A NaN is never coerced to 0, to an endpoint, or to a neighbour."""
    gr = np.array([10.0, np.nan, 90.0])
    unc, clip, valid = compute_gr_index(gr, 0.0, 100.0)
    assert np.isnan(unc[1]) and np.isnan(clip[1])
    assert not valid[1]
    assert valid[0] and valid[2]


@pytest.mark.parametrize("bad", [np.inf, -np.inf])
def test_infinite_input_is_explicitly_invalidated(bad):
    """An infinite GR sample is not a measurement: it is masked out and set
    to NaN, never normalized into a plus/minus-infinity IGR that clipping
    would quietly turn into a clean-looking 0 or 1."""
    gr = np.array([10.0, bad, 90.0])
    unc, clip, valid = compute_gr_index(gr, 0.0, 100.0)
    assert not valid[1]
    assert np.isnan(unc[1]) and np.isnan(clip[1])
    assert np.count_nonzero(valid) == 2


def test_valid_mask_argument_is_anded_with_finiteness():
    gr = np.array([10.0, 50.0, 90.0])
    extra = np.array([True, False, True])
    _, _, valid = compute_gr_index(gr, 0.0, 100.0, valid_mask=extra)
    np.testing.assert_array_equal(valid, [True, False, True])


# ---------------------------------------------------------------------------
# Endpoint validation
# ---------------------------------------------------------------------------

def test_identical_endpoints_rejected():
    with pytest.raises(PetrophysicsInputError, match="identical"):
        compute_gr_index(np.array([1.0, 2.0]), 50.0, 50.0)


def test_reversed_endpoints_rejected():
    with pytest.raises(PetrophysicsInputError, match="strictly greater"):
        compute_gr_index(np.array([1.0, 2.0]), 100.0, 10.0)


def test_endpoint_separation_below_minimum_rejected():
    with pytest.raises(PetrophysicsInputError, match="below the configured minimum"):
        compute_gr_index(np.array([1.0, 2.0]), 10.0, 10.5, min_endpoint_separation_api=1.0)


@pytest.mark.parametrize("bad", [np.nan, np.inf, -np.inf])
def test_non_finite_endpoints_rejected(bad):
    with pytest.raises(PetrophysicsInputError, match="finite"):
        compute_gr_index(np.array([1.0, 2.0]), 0.0, bad)


@pytest.mark.parametrize("bad", [True, False, "50", b"50", complex(1, 2)])
def test_endpoint_type_class_defects_raise_typeerror(bad):
    with pytest.raises(TypeError):
        compute_gr_index(np.array([1.0, 2.0]), bad, 100.0)


# ---------------------------------------------------------------------------
# Input array validation
# ---------------------------------------------------------------------------

def test_boolean_array_input_rejected_with_typeerror():
    with pytest.raises(TypeError, match="boolean"):
        compute_gr_index(np.array([True, False]), 0.0, 100.0)


def test_string_array_input_rejected_with_typeerror():
    with pytest.raises(TypeError, match="string/bytes"):
        compute_gr_index(np.array(["10", "20"]), 0.0, 100.0)


def test_bytes_array_input_rejected_with_typeerror():
    with pytest.raises(TypeError, match="string/bytes"):
        compute_gr_index(np.array([b"10", b"20"]), 0.0, 100.0)


def test_complex_array_input_rejected_with_typeerror():
    with pytest.raises(TypeError, match="complex"):
        compute_gr_index(np.array([1 + 2j, 3 + 4j]), 0.0, 100.0)


def test_object_array_input_rejected_with_typeerror():
    with pytest.raises(TypeError, match="unsupported array dtype"):
        compute_gr_index(np.array([{"a": 1}, {"b": 2}], dtype=object), 0.0, 100.0)


@pytest.mark.parametrize("shape", [(2, 3), (2, 2, 2)])
def test_multidimensional_array_rejected(shape):
    """A multidimensional array is never silently ravelled - that would
    destroy sample-position alignment with the well frame."""
    with pytest.raises(PetrophysicsInputError, match="never silently flattened"):
        compute_gr_index(np.zeros(shape), 0.0, 100.0)


def test_length_mismatch_between_values_and_mask_rejected():
    with pytest.raises(PetrophysicsInputError, match="never reconciled by truncation or padding"):
        compute_gr_index(np.array([1.0, 2.0, 3.0]), 0.0, 100.0,
                         valid_mask=np.array([True, False]))


def test_non_boolean_mask_rejected():
    with pytest.raises(TypeError, match="boolean array"):
        compute_gr_index(np.array([1.0, 2.0]), 0.0, 100.0, valid_mask=np.array([1, 0]))


# ---------------------------------------------------------------------------
# Boreas-style exclusion
# ---------------------------------------------------------------------------

def test_excluded_well_gets_no_endpoint_scenarios():
    from p2mem.petrophysics import resolve_endpoint_scenarios
    frame = _frame_with_gr(np.linspace(1.0, 500.0, 20), canonical="ECGR_api", well_key="Boreas_1")
    d = _disposition("Boreas_1", "ECGR_api", USE_STATUS_QC_ONLY_EXCLUDED,
                     EXCLUSION_REASON_BOREAS_ECGR)
    with pytest.raises(PetrophysicsExclusionError, match=EXCLUSION_REASON_BOREAS_ECGR):
        resolve_endpoint_scenarios(frame, d, _config())


def test_excluded_well_gets_no_proxy():
    frame = _frame_with_gr(np.linspace(1.0, 500.0, 20), canonical="ECGR_api", well_key="Boreas_1")
    d = _disposition("Boreas_1", "ECGR_api", USE_STATUS_QC_ONLY_EXCLUDED,
                     EXCLUSION_REASON_BOREAS_ECGR)
    with pytest.raises(PetrophysicsExclusionError):
        compute_gr_proxy(frame, d, _scenario(well_key="Boreas_1"), _config())


def test_excluded_well_still_gets_factual_qc_statistics():
    """Exclusion removes GR-DERIVED work, not factual QC reporting - the
    numbers justifying the exclusion must themselves be published."""
    gr = np.array([0.0, -0.001, 8.4, 519.2, 3.3, np.nan])
    frame = _frame_with_gr(gr, canonical="ECGR_api", well_key="Boreas_1")
    d = _disposition("Boreas_1", "ECGR_api", USE_STATUS_QC_ONLY_EXCLUDED,
                     EXCLUSION_REASON_BOREAS_ECGR)
    st = compute_gr_family_qc_stats(frame, d)
    assert st.valid_count == 5
    assert st.n_negative_samples == 1
    assert st.n_zero_samples == 1
    assert st.max_api == pytest.approx(519.2)


def test_excluded_well_classified_as_excluded_not_by_coverage():
    """Good coverage never overrides an unresolved-scale exclusion."""
    frame = _frame_with_gr(np.linspace(1.0, 200.0, 50), canonical="ECGR_api", well_key="Boreas_1")
    d = _disposition("Boreas_1", "ECGR_api", USE_STATUS_QC_ONLY_EXCLUDED,
                     EXCLUSION_REASON_BOREAS_ECGR)
    st = compute_gr_family_qc_stats(frame, d)
    assert st.valid_fraction == 1.0  # perfect coverage
    cls, why = classify_gr_proxy_confidence(st, d, [], _config())
    assert cls == GR_EXCLUDED_UNRESOLVED_SCALE
    assert "does not resolve an unresolved scale" in why


def test_qc_only_disposition_requires_machine_readable_reason():
    with pytest.raises(ValueError, match="machine-readable exclusion_reason"):
        GrFamilyDisposition(
            well_key="X", source_las_filename="x.las", gr_family_canonical_name="ECGR_api",
            gr_family_source_curve_name="ECGR", use_status=USE_STATUS_QC_ONLY_EXCLUDED,
            exclusion_reason=None, evidence_class="measured", has_approved_formation_tops=False,
        )


def test_unknown_use_status_rejected():
    with pytest.raises(ValueError, match="use_status"):
        GrFamilyDisposition(
            well_key="X", source_las_filename="x.las", gr_family_canonical_name="GR_api",
            gr_family_source_curve_name="GR", use_status="do_whatever_you_like",
            exclusion_reason=None, evidence_class="measured", has_approved_formation_tops=False,
        )


# ---------------------------------------------------------------------------
# Scenario resolution
# ---------------------------------------------------------------------------

def test_low_base_high_scenarios_all_computed_with_measured_endpoints():
    from p2mem.petrophysics import resolve_endpoint_scenarios
    frame = _frame_with_gr(np.linspace(5.0, 205.0, 201))
    scenarios = resolve_endpoint_scenarios(frame, _disposition(), _config())
    assert [s.scenario_name for s in scenarios] == ["low", "base", "high"]
    for s in scenarios:
        assert np.isfinite(s.gr_low_endpoint_api) and np.isfinite(s.gr_high_endpoint_api)
        assert s.gr_high_endpoint_api > s.gr_low_endpoint_api
        assert s.evidence_class == "assumed_configured"
        assert "uncalibrated" in s.calibration_status
    # The wide "high" scenario must bracket the narrow "low" one.
    lo, base, hi = scenarios
    assert hi.gr_low_endpoint_api < lo.gr_low_endpoint_api
    assert hi.gr_high_endpoint_api > lo.gr_high_endpoint_api
    assert hi.endpoint_separation_api > base.endpoint_separation_api > lo.endpoint_separation_api


def test_endpoint_scenario_is_never_calibrated():
    s = _scenario()
    assert s.evidence_class == "assumed_configured"
    assert "uncalibrated" in s.calibration_status


def test_zero_valid_coverage_rejected_for_endpoints():
    from p2mem.petrophysics import resolve_endpoint_scenarios
    frame = _frame_with_gr(np.full(10, np.nan))
    with pytest.raises(PetrophysicsInputError, match="zero valid coverage"):
        resolve_endpoint_scenarios(frame, _disposition(), _config())


def test_absent_gr_curve_rejected_for_endpoints_never_borrowed():
    from p2mem.petrophysics import resolve_endpoint_scenarios
    frame = _frame_with_gr(np.linspace(5.0, 105.0, 20), canonical="GR_api")
    d = _disposition(canonical="GRD_api")  # names a curve this well does not have
    with pytest.raises(PetrophysicsInputError, match="never borrowed from another well"):
        resolve_endpoint_scenarios(frame, d, _config())


def test_proxy_from_another_wells_scenario_rejected():
    frame = _frame_with_gr(np.linspace(5.0, 105.0, 20), well_key="A_1")
    other = _scenario(well_key="B_1")
    with pytest.raises(PetrophysicsInputError, match="never transferred between wells"):
        compute_gr_proxy(frame, _disposition("A_1"), other, _config())


# ---------------------------------------------------------------------------
# Proxy behavior and threshold sensitivity
# ---------------------------------------------------------------------------

def test_proxy_equals_clipped_igr_under_linear_identity():
    frame = _frame_with_gr(np.linspace(0.0, 200.0, 41))
    p = compute_gr_proxy(frame, _disposition(), _scenario(10.0, 110.0), _config())
    np.testing.assert_array_equal(
        p.VSH_GR_linear_proxy_frac[p.valid_mask], p.igr_clipped_frac[p.valid_mask]
    )
    assert p.transform_name == "linear_identity_of_clipped_igr"
    assert "not_a_shale_volume" in p.calibration_status


def test_proxy_clipping_counts_are_reported():
    frame = _frame_with_gr(np.array([0.0, 50.0, 60.0, 200.0]))
    p = compute_gr_proxy(frame, _disposition(), _scenario(10.0, 110.0), _config())
    assert p.n_clipped_low == 1   # 0 API is below the low endpoint
    assert p.n_clipped_high == 1  # 200 API is above the high endpoint
    assert p.clipped_fraction == pytest.approx(0.5)


def test_wider_endpoints_damp_the_proxy():
    """Endpoint sensitivity is real and measurable, not cosmetic."""
    gr = np.linspace(5.0, 205.0, 201)
    frame = _frame_with_gr(gr)
    narrow = compute_gr_proxy(frame, _disposition(), _scenario(50.0, 100.0, name="narrow"), _config())
    wide = compute_gr_proxy(frame, _disposition(), _scenario(5.0, 205.0, name="wide"), _config())
    assert narrow.clipped_fraction > wide.clipped_fraction
    assert narrow.proxy_median != wide.proxy_median


def test_threshold_sensitivity_is_monotonic():
    """Raising a proxy threshold can only reduce (never increase) the count
    of samples at or above it."""
    frame = _frame_with_gr(np.linspace(0.0, 200.0, 201))
    p = compute_gr_proxy(frame, _disposition(), _scenario(10.0, 110.0), _config())
    counts = [
        int(np.count_nonzero(np.isfinite(p.VSH_GR_linear_proxy_frac)
                             & (p.VSH_GR_linear_proxy_frac >= t)))
        for t in (0.5, 0.6, 0.7)
    ]
    assert counts[0] >= counts[1] >= counts[2]


def test_unmapped_depth_samples_are_excluded_from_the_proxy():
    """A sample with no defensible depth cannot contribute to a
    depth-resolved proxy."""
    gr = np.full(6, 50.0)
    md = np.array([50.0, 100.0, 200.0, 300.0, 380.0, 900.0])  # last beyond coverage
    frame = _frame_with_gr(gr, md=md)
    p = compute_gr_proxy(frame, _disposition(), _scenario(10.0, 110.0), _config())
    assert not p.valid_mask[-1]
    assert np.isnan(p.VSH_GR_linear_proxy_frac[-1])
    assert p.n_valid == 5


# ---------------------------------------------------------------------------
# QC statistics
# ---------------------------------------------------------------------------

def test_qc_stats_percentiles_and_dynamic_range():
    frame = _frame_with_gr(np.linspace(0.0, 100.0, 101))
    st = compute_gr_family_qc_stats(frame, _disposition())
    assert st.min_api == pytest.approx(0.0)
    assert st.max_api == pytest.approx(100.0)
    assert st.median_api == pytest.approx(50.0)
    assert st.dynamic_range_p05_p95_api == pytest.approx(90.0)


def test_qc_stats_samples_above_seabed_is_none_when_not_determinable():
    """A well with no approved tops reports None (unknown), never 0, which
    would falsely assert that no such samples exist."""
    frame = _frame_with_gr(np.linspace(10.0, 100.0, 10))
    st = compute_gr_family_qc_stats(frame, _disposition(tops=False), seabed_mdrt_m=None)
    assert st.n_samples_above_seabed is None
    assert "never as 0" in st.seabed_basis


def test_qc_stats_counts_samples_above_seabed_when_determinable():
    md = np.array([100.0, 200.0, 300.0, 380.0])
    frame = _frame_with_gr(np.full(4, 20.0), md=md)
    st = compute_gr_family_qc_stats(frame, _disposition(), seabed_mdrt_m=250.0)
    assert st.n_samples_above_seabed == 2


def test_qc_stats_zero_valid_coverage():
    frame = _frame_with_gr(np.full(8, np.nan))
    st = compute_gr_family_qc_stats(frame, _disposition())
    assert st.valid_count == 0
    assert st.valid_fraction == 0.0
    assert st.median_api is None
    assert st.dynamic_range_p05_p95_api is None
    assert st.longest_missing_block_samples == 8


def test_qc_stats_absent_curve_is_a_factual_gap():
    frame = _frame_with_gr(np.linspace(1.0, 10.0, 5), canonical="GR_api")
    st = compute_gr_family_qc_stats(frame, _disposition(canonical="GRD_api"))
    assert st.valid_count == 0
    assert "never substituted from another well" in st.statistics_basis


def test_qc_stats_reports_valid_and_missing_blocks():
    gr = np.array([1.0, 2.0, np.nan, np.nan, np.nan, 6.0, 7.0])
    frame = _frame_with_gr(gr)
    st = compute_gr_family_qc_stats(frame, _disposition())
    assert st.n_valid_blocks == 2
    assert st.longest_valid_block_samples == 2
    assert st.longest_missing_block_samples == 3


# ---------------------------------------------------------------------------
# Confidence classification
# ---------------------------------------------------------------------------

def test_confidence_class_never_contains_lithology_vocabulary():
    for cls in (GR_PROXY_HIGH, GR_PROXY_INTERMEDIATE, GR_PROXY_LOW,
                GR_NOT_AVAILABLE, GR_EXCLUDED_UNRESOLVED_SCALE):
        assert_no_lithology_vocabulary(cls, "test")


def test_confidence_not_available_when_no_valid_samples():
    frame = _frame_with_gr(np.full(8, np.nan))
    st = compute_gr_family_qc_stats(frame, _disposition())
    cls, _ = classify_gr_proxy_confidence(st, _disposition(), [], _config())
    assert cls == GR_NOT_AVAILABLE


def test_confidence_low_when_dynamic_range_is_poor():
    frame = _frame_with_gr(np.linspace(40.0, 45.0, 50))  # 5 API of range
    st = compute_gr_family_qc_stats(frame, _disposition())
    cls, why = classify_gr_proxy_confidence(st, _disposition(), [], _config())
    assert cls == GR_PROXY_LOW
    assert "dynamic_range" in why


def test_confidence_high_for_well_covered_wide_range_stable_proxy():
    from p2mem.petrophysics import resolve_endpoint_scenarios
    frame = _frame_with_gr(np.linspace(5.0, 205.0, 401))
    d = _disposition()
    cfg = _config()
    st = compute_gr_family_qc_stats(frame, d)
    proxies = [compute_gr_proxy(frame, d, s, cfg) for s in resolve_endpoint_scenarios(frame, d, cfg)]
    cls, _ = classify_gr_proxy_confidence(st, d, proxies, cfg)
    assert cls in (GR_PROXY_HIGH, GR_PROXY_INTERMEDIATE)


# ---------------------------------------------------------------------------
# Contiguous blocks and gap tolerance
# ---------------------------------------------------------------------------

def test_contiguous_blocks_no_bridging_by_default():
    mask = np.array([True, True, False, True, True])
    assert find_contiguous_blocks(mask) == [(0, 1), (3, 4)]


def test_contiguous_blocks_bridges_small_gap_when_both_rules_permit():
    mask = np.array([True, True, False, True, True])
    depth = np.array([0.0, 0.5, 1.0, 1.5, 2.0])
    assert find_contiguous_blocks(mask, depth, max_gap_samples=2, max_gap_depth_m=2.0) == [(0, 4)]


def test_contiguous_blocks_does_not_bridge_large_physical_gap():
    """Sample-count continuity is not physical-depth continuity: a 1-sample
    gap spanning 400 m must start a new block."""
    mask = np.array([True, True, False, True, True])
    depth = np.array([0.0, 0.5, 200.0, 400.0, 400.5])
    assert find_contiguous_blocks(mask, depth, max_gap_samples=2, max_gap_depth_m=1.0) == [(0, 1), (3, 4)]


def test_contiguous_blocks_gap_exactly_at_sample_tolerance_is_bridged():
    mask = np.array([True, False, False, True])
    depth = np.array([0.0, 0.2, 0.4, 0.6])
    assert find_contiguous_blocks(mask, depth, max_gap_samples=2, max_gap_depth_m=1.0) == [(0, 3)]


def test_contiguous_blocks_gap_one_beyond_sample_tolerance_is_not_bridged():
    mask = np.array([True, False, False, False, True])
    depth = np.linspace(0.0, 0.8, 5)
    assert find_contiguous_blocks(mask, depth, max_gap_samples=2, max_gap_depth_m=1.0) == [(0, 0), (4, 4)]


def test_contiguous_blocks_depth_exactly_at_tolerance_is_bridged():
    mask = np.array([True, False, True])
    depth = np.array([0.0, 0.5, 1.0])
    assert find_contiguous_blocks(mask, depth, max_gap_samples=1, max_gap_depth_m=1.0) == [(0, 2)]


def test_contiguous_blocks_empty_mask():
    assert find_contiguous_blocks(np.zeros(5, dtype=bool)) == []


def test_contiguous_blocks_all_true():
    assert find_contiguous_blocks(np.ones(5, dtype=bool)) == [(0, 4)]


def test_contiguous_blocks_rejects_2d_mask():
    with pytest.raises(PetrophysicsInputError, match="1-D"):
        find_contiguous_blocks(np.ones((2, 2), dtype=bool))


def test_contiguous_blocks_rejects_depth_shape_mismatch():
    with pytest.raises(PetrophysicsInputError, match="does not match mask shape"):
        find_contiguous_blocks(np.ones(3, dtype=bool), np.zeros(2))


# ---------------------------------------------------------------------------
# Configuration loading
# ---------------------------------------------------------------------------

def _write_cfg(tmp_path, mutate=None):
    import yaml
    base = {
        "schema_version": "6.0", "increment": 6, "assurance_tier": "Tier C",
        "policy": {
            "endpoint_estimation_method": "per_well_percentile_of_own_valid_samples",
            "cross_well_shared_endpoints_allowed": False,
            "endpoint_sample_basis": "finite_and_depth_mapped_samples_only",
            "clipping_policy": "retain", "clip_lower": 0.0, "clip_upper": 1.0,
            "min_endpoint_separation_api": 1.0,
            "shale_proxy_transform": "linear_identity_of_clipped_igr",
            "nonlinear_vsh_transforms_enabled": False,
            "contiguity": {"max_gap_samples": 2, "max_gap_depth_m": 1.0,
                           "min_block_samples": 20, "min_block_thickness_m": 5.0},
            "nct_candidate_proxy_thresholds": [0.5, 0.6, 0.7],
            "physical_bounds": {"rhob_min_kg_m3": 1000.0, "rhob_max_kg_m3": 3500.0,
                                "vp_min_m_s": 1000.0, "vp_max_m_s": 8000.0,
                                "vs_min_m_s": 300.0, "vs_max_m_s": 5000.0,
                                "vp_vs_ratio_nonnegative_poisson_min_inclusive": 1.4142135623730951,
                                "vp_vs_ratio_positive_bulk_modulus_min_exclusive": 1.1547005383792515,
                                "vp_vs_ratio_plausibility_max": 4.0},
        },
        "endpoint_scenarios": [
            {"scenario_name": "low", "low_percentile": 10.0, "high_percentile": 85.0, "description": "n"},
            {"scenario_name": "base", "low_percentile": 5.0, "high_percentile": 95.0, "description": "m"},
            {"scenario_name": "high", "low_percentile": 1.0, "high_percentile": 99.0, "description": "w"},
        ],
        "wells": {
            "W_1": {"source_las_filename": "W_1.las", "gr_family_canonical_name": "GR_api",
                    "gr_family_source_curve_name": "GR", "use_status": "screening_proxy_allowed",
                    "exclusion_reason": None, "evidence_class": "measured",
                    "has_approved_formation_tops": True, "notes": "n"},
        },
        "gr_proxy_confidence": {"classes": [], "rules": {
            "high": {"min_valid_fraction": 0.9, "min_dynamic_range_api": 60.0,
                     "max_proxy_median_spread_across_scenarios": 0.1},
            "intermediate": {"min_valid_fraction": 0.7, "min_dynamic_range_api": 40.0,
                             "max_proxy_median_spread_across_scenarios": 0.2}}},
        "method_eligibility": {
            "eligible_density_for_sv": {"purpose": "p"},
            "eligible_dynamic_elastic": {"purpose": "p"},
            "eligible_sonic_nct_candidate": {"purpose": "p"},
        },
    }
    if mutate:
        mutate(base)
    p = tmp_path / "cfg.yml"
    p.write_text(yaml.safe_dump(base), encoding="utf-8")
    return str(p)


def test_config_loads_and_validates(tmp_path):
    cfg = load_petrophysics_eligibility_config(_write_cfg(tmp_path))
    assert cfg.schema_version == "6.0"
    assert cfg.scenario_names == ("low", "base", "high")
    assert cfg.wells["W_1"].proxy_permitted


def test_config_missing_file_rejected():
    with pytest.raises(PetrophysicsConfigError, match="not found"):
        load_petrophysics_eligibility_config("/nonexistent/nope.yml")


def test_config_rejects_shared_cross_well_endpoints(tmp_path):
    def m(b):
        b["policy"]["cross_well_shared_endpoints_allowed"] = True
    with pytest.raises(PetrophysicsConfigError, match="cross_well_shared_endpoints_allowed"):
        load_petrophysics_eligibility_config(_write_cfg(tmp_path, m))


def test_config_rejects_enabled_nonlinear_vsh(tmp_path):
    def m(b):
        b["policy"]["nonlinear_vsh_transforms_enabled"] = True
    with pytest.raises(PetrophysicsConfigError, match="nonlinear Vsh transform"):
        load_petrophysics_eligibility_config(_write_cfg(tmp_path, m))


def test_config_rejects_fewer_than_three_scenarios(tmp_path):
    def m(b):
        b["endpoint_scenarios"] = b["endpoint_scenarios"][:2]
    with pytest.raises(PetrophysicsConfigError, match="at least three scenarios"):
        load_petrophysics_eligibility_config(_write_cfg(tmp_path, m))


def test_config_rejects_reversed_scenario_percentiles(tmp_path):
    def m(b):
        b["endpoint_scenarios"][0]["low_percentile"] = 99.0
    with pytest.raises(PetrophysicsConfigError, match="low_percentile < high_percentile"):
        load_petrophysics_eligibility_config(_write_cfg(tmp_path, m))


def test_config_rejects_missing_required_well_key(tmp_path):
    def m(b):
        del b["wells"]["W_1"]["use_status"]
    with pytest.raises(PetrophysicsConfigError, match="missing required key 'use_status'"):
        load_petrophysics_eligibility_config(_write_cfg(tmp_path, m))


def test_config_rejects_duplicate_keys(tmp_path):
    p = tmp_path / "dup.yml"
    p.write_text("schema_version: '6.0'\nschema_version: '6.1'\n", encoding="utf-8")
    with pytest.raises(PetrophysicsConfigError, match="Duplicate key"):
        load_petrophysics_eligibility_config(str(p))


def test_real_project_config_is_valid_and_excludes_boreas():
    """The packaged, human-authored config must itself parse and must carry
    the required dispositions."""
    cfg = load_petrophysics_eligibility_config(str(PROJECT_ROOT / "config" / "petrophysics_eligibility.yml"))
    assert cfg.wells["Boreas_1"].use_status == USE_STATUS_QC_ONLY_EXCLUDED
    assert cfg.wells["Boreas_1"].exclusion_reason == EXCLUSION_REASON_BOREAS_ECGR
    assert not cfg.wells["Boreas_1"].proxy_permitted
    assert cfg.wells["Poseidon_2"].use_status == "screening_proxy_allowed"
    # Increment 6.1 (Finding 2): no approved tops -> must be the depth-tied status.
    assert cfg.wells["Poseidon_North_1"].use_status == "screening_proxy_allowed_depth_tied"
    assert cfg.wells["Poseidon_North_1"].has_approved_formation_tops is False
    assert cfg.wells["Proteus_1ST2"].use_status == "screening_proxy_allowed_depth_tied"
    # Four distinct GR-family curve names, never merged.
    assert cfg.wells["Poseidon_2"].gr_family_canonical_name == "GR_api"
    assert cfg.wells["Boreas_1"].gr_family_canonical_name == "ECGR_api"
    assert cfg.wells["Poseidon_North_1"].gr_family_canonical_name == "GRD_api"
    assert cfg.wells["Proteus_1ST2"].gr_family_canonical_name == "GR_api"


def test_real_project_config_declares_no_named_lithology():
    """No key or value anywhere in the packaged config asserts a rock
    type."""
    import yaml
    with open(PROJECT_ROOT / "config" / "petrophysics_eligibility.yml", encoding="utf-8") as fh:
        raw = yaml.safe_load(fh)
    for well_key, spec in raw["wells"].items():
        for field in ("use_status", "evidence_class", "gr_family_canonical_name"):
            assert_no_lithology_vocabulary(str(spec[field]), f"{well_key}.{field}")
    for cls in raw["gr_proxy_confidence"]["classes"]:
        assert_no_lithology_vocabulary(str(cls), "confidence class")


def test_config_rejects_missing_required_eligibility_mask(tmp_path):
    """All three Increment 6 masks must be declared in the config; an
    undeclared mask is a configuration defect, not a silent default."""
    def m(b):
        del b["method_eligibility"]["eligible_sonic_nct_candidate"]
    with pytest.raises(PetrophysicsConfigError, match="eligible_sonic_nct_candidate"):
        load_petrophysics_eligibility_config(_write_cfg(tmp_path, m))


def test_config_rejects_undocumented_eligibility_mask(tmp_path):
    """An eligibility mask with no stated purpose is not auditable."""
    def m(b):
        b["method_eligibility"]["eligible_density_for_sv"] = {}
    with pytest.raises(PetrophysicsConfigError, match="not auditable"):
        load_petrophysics_eligibility_config(_write_cfg(tmp_path, m))


def test_real_project_config_documents_all_three_masks():
    cfg = load_petrophysics_eligibility_config(
        str(PROJECT_ROOT / "config" / "petrophysics_eligibility.yml")
    )
    for mask in ("eligible_density_for_sv", "eligible_dynamic_elastic",
                 "eligible_sonic_nct_candidate"):
        assert cfg.method_eligibility[mask]["purpose"].strip()
    # The lithology-dependent mask must be the ONLY one flagged as such, and
    # must not apply to a GR-excluded well.
    assert cfg.method_eligibility["eligible_sonic_nct_candidate"]["lithology_dependent"] is True
    assert cfg.method_eligibility["eligible_sonic_nct_candidate"][
        "applies_to_excluded_gr_wells"] is False
    assert cfg.method_eligibility["eligible_density_for_sv"]["lithology_dependent"] is False
    assert cfg.method_eligibility["eligible_dynamic_elastic"]["lithology_dependent"] is False


# ---------------------------------------------------------------------------
# Increment 6.1 (Finding 2): no-tops / depth-tied config invariant
# ---------------------------------------------------------------------------

def test_config_rejects_proxy_allowed_without_approved_tops(tmp_path):
    """A proxy-permitted well with NO approved formation tops must use the
    depth-tied status. The plain status asserts a stratigraphic tie the well
    does not have, and the contradiction must fail loudly."""
    def m(b):
        b["wells"]["W_1"]["has_approved_formation_tops"] = False
        b["wells"]["W_1"]["use_status"] = "screening_proxy_allowed"
    with pytest.raises(PetrophysicsConfigError, match="screening_proxy_allowed_depth_tied"):
        load_petrophysics_eligibility_config(_write_cfg(tmp_path, m))


def test_config_rejects_depth_tied_when_tops_actually_exist(tmp_path):
    """The converse contradiction is equally a config/reality mismatch."""
    def m(b):
        b["wells"]["W_1"]["has_approved_formation_tops"] = True
        b["wells"]["W_1"]["use_status"] = "screening_proxy_allowed_depth_tied"
    with pytest.raises(PetrophysicsConfigError, match="reserved for wells with NO approved"):
        load_petrophysics_eligibility_config(_write_cfg(tmp_path, m))


def test_config_accepts_the_two_consistent_combinations(tmp_path):
    """Both self-consistent pairings load cleanly."""
    def has_tops(b):
        b["wells"]["W_1"]["has_approved_formation_tops"] = True
        b["wells"]["W_1"]["use_status"] = "screening_proxy_allowed"
    def no_tops(b):
        b["wells"]["W_1"]["has_approved_formation_tops"] = False
        b["wells"]["W_1"]["use_status"] = "screening_proxy_allowed_depth_tied"
    for mut, expected in ((has_tops, "screening_proxy_allowed"),
                          (no_tops, "screening_proxy_allowed_depth_tied")):
        cfg = load_petrophysics_eligibility_config(_write_cfg(tmp_path, mut))
        assert cfg.wells["W_1"].use_status == expected


def test_excluded_well_without_tops_is_not_forced_to_depth_tied(tmp_path):
    """The invariant applies only to PROXY-PERMITTED wells: an excluded well
    keeps its qc_only status regardless of whether it has tops."""
    def m(b):
        b["wells"]["W_1"]["has_approved_formation_tops"] = False
        b["wells"]["W_1"]["use_status"] = "qc_only_excluded"
        b["wells"]["W_1"]["exclusion_reason"] = "SOME_REASON"
    cfg = load_petrophysics_eligibility_config(_write_cfg(tmp_path, m))
    assert cfg.wells["W_1"].use_status == "qc_only_excluded"


def test_real_project_config_poseidon_north_1_is_depth_tied():
    """The real config must satisfy the invariant it now enforces."""
    cfg = load_petrophysics_eligibility_config(
        str(PROJECT_ROOT / "config" / "petrophysics_eligibility.yml")
    )
    for wk, d in cfg.wells.items():
        if d.proxy_permitted:
            assert d.has_approved_formation_tops == (
                d.use_status == "screening_proxy_allowed"
            ), f"{wk}: use_status and has_approved_formation_tops disagree"
    assert cfg.wells["Poseidon_North_1"].use_status == "screening_proxy_allowed_depth_tied"
    assert cfg.wells["Proteus_1ST2"].use_status == "screening_proxy_allowed_depth_tied"
    assert cfg.wells["Poseidon_2"].use_status == "screening_proxy_allowed"


# ---------------------------------------------------------------------------
# Increment 6.1 (Finding 1): the corrected Vp/Vs bound names in the real config
# ---------------------------------------------------------------------------

def test_real_config_declares_the_three_separated_vp_vs_bounds():
    cfg = load_petrophysics_eligibility_config(
        str(PROJECT_ROOT / "config" / "petrophysics_eligibility.yml")
    )
    b = cfg.policy["physical_bounds"]
    assert b["vp_vs_ratio_nonnegative_poisson_min_inclusive"] == pytest.approx(np.sqrt(2.0))
    assert b["vp_vs_ratio_positive_bulk_modulus_min_exclusive"] == pytest.approx(np.sqrt(4.0 / 3.0))
    assert b["vp_vs_ratio_plausibility_max"] == pytest.approx(4.0)
    # The old, conflating key must be gone.
    assert "vp_vs_ratio_min_exclusive" not in b


# ---------------------------------------------------------------------------
# Increment 6.1 (Finding 3): the prohibited vocabulary and scope model
# ---------------------------------------------------------------------------

def test_clastic_and_rock_class_terms_are_prohibited():
    from p2mem.wellframe_models import PROHIBITED_LITHOLOGY_TERMS
    for term in ("clastic", "siliciclastic", "volcanic", "basement"):
        assert term in PROHIBITED_LITHOLOGY_TERMS


def test_unknown_validation_scope_rejected():
    from p2mem.wellframe_models import validate_no_prohibited_interpretation
    with pytest.raises(ValueError, match="unknown validation scope"):
        validate_no_prohibited_interpretation([("c", "text", "whatever")])

# ---------------------------------------------------------------------------
# Increment 6.1.5: POSITIVE AUTHORIZATION.
#
# Increments 6.1 through 6.1.4 asked "does this text contain a geological
# assertion?" and were defeated four times, finally by `chalk` - a word no
# recognizer had been given. These tests assert the inverted contract: nothing
# is acceptable by default, and rejection never depends on recognizing a rock.
# ---------------------------------------------------------------------------

import p2mem.wellframe_models as _wm


def _V(text, scope, auth=None, context="audit.field"):
    entry = (context, text, scope, auth) if auth is not None else (context, text, scope)
    return _wm.validate_no_prohibited_interpretation([entry])


def _scope(name):
    return {"label": _wm.SCOPE_LABEL, "interpretive": _wm.SCOPE_INTERPRETIVE,
            "method": _wm.SCOPE_METHOD, "explanatory": _wm.SCOPE_EXPLANATORY}[name]


CONTROLLED = ["label", "interpretive", "explanatory", "method"]

# The lithologies the Increment 6.1.5 audit named, plus a fabricated word.
# NONE of these is in PROHIBITED_LITHOLOGY_TERMS - that is the point.
UNKNOWN_LITHOLOGY_ASSERTIONS = [
    "The interval is chalk.", "The interval is halite.", "The interval is gypsum.",
    "The interval is conglomerate.", "The interval is chert.", "The interval is tuff.",
    "The interval is basalt.", "The interval is dolostone.", "The interval is lignite.",
    "The interval is calcareous.", "The interval is argillaceous.",
    "The interval is arenaceous.",
]
FABRICATED_ASSERTIONS = ["The interval is qxzite.", "The interval is zzqqworp."]
INNOCENT_UNREGISTERED = [
    "Everything looks fine.", "The operator was competent.",
    "Data coverage is good in the deep section.",
]


@pytest.mark.parametrize("scope_name", CONTROLLED)
@pytest.mark.parametrize("text", UNKNOWN_LITHOLOGY_ASSERTIONS)
def test_unknown_lithology_assertions_are_rejected_in_every_controlled_scope(text, scope_name):
    assert _V(text, _scope(scope_name)), f"{text!r} in {scope_name}"


@pytest.mark.parametrize("scope_name", CONTROLLED)
@pytest.mark.parametrize("text", FABRICATED_ASSERTIONS)
def test_fabricated_words_are_rejected_because_nothing_authorized_them(text, scope_name):
    assert _V(text, _scope(scope_name)), f"{text!r} in {scope_name}"


@pytest.mark.parametrize("scope_name", CONTROLLED)
@pytest.mark.parametrize("text", INNOCENT_UNREGISTERED)
def test_innocent_unregistered_text_is_rejected_too(text, scope_name):
    """THE decisive test. These contain no rock word at all, so a detector has
    nothing to detect. They are rejected because the rule is positive
    authorization, not hidden semantic detection."""
    assert _V(text, _scope(scope_name)), f"{text!r} in {scope_name}"


@pytest.mark.parametrize(
    "text", UNKNOWN_LITHOLOGY_ASSERTIONS + FABRICATED_ASSERTIONS + INNOCENT_UNREGISTERED)
def test_the_linter_recognizes_nothing_in_any_of_these(text):
    """Proves the rejections above owe nothing to the blacklist: it is empty
    for every one of them."""
    assert _wm.find_prohibited_lithology_terms(text) == (), text


def test_the_blacklist_is_documented_as_a_linter_and_authorizes_nothing():
    doc = " ".join((_wm.find_prohibited_lithology_terms.__doc__ or "").split())
    assert "AUTHORIZES NOTHING" in doc
    assert "not exhaustive" in doc.lower()


# --- LABEL scope: typed, enumerated values only ----------------------------

def test_every_approved_label_passes_under_its_own_field_kind():
    for field_kind, bucket in _wm.APPROVED_LABELS.items():
        for value, lab in bucket.items():
            auth = _wm.Authorization(field_kind=field_kind)
            assert _V(value, _scope("label"), auth) == (), (field_kind, value)
            assert lab.field_kind == field_kind and lab.purpose
            assert len(lab.provenance) > 40


def test_every_label_fails_under_every_other_field_kind():
    """Requirement 3: a complete cross-product across all registered labels and
    field kinds. A value approved under one kind authorizes nothing elsewhere."""
    kinds = _wm.APPROVED_LABEL_FIELD_KINDS
    checked = 0
    for own_kind, bucket in _wm.APPROVED_LABELS.items():
        for value in bucket:
            for other in kinds:
                if other == own_kind or value in _wm.APPROVED_LABELS[other]:
                    continue
                checked += 1
                assert _V(value, _scope("label"), _wm.Authorization(field_kind=other)), (
                    f"{value!r} wrongly accepted as {other!r}")
    assert checked >= 100, f"cross-product must be substantive, checked {checked}"


def test_unknown_field_kind_and_missing_field_kind_fail():
    assert _V("measured", _scope("label"), _wm.Authorization(field_kind="no_such_kind"))
    assert _V("measured", _scope("label"))


def test_the_audited_field_kind_mismatches_fail():
    assert _V("GR", _scope("label"), _wm.Authorization(field_kind="use_status"))
    assert _V("measured", _scope("label"), _wm.Authorization(field_kind="mask_name"))
    assert _V("screening_proxy_allowed", _scope("label"),
              _wm.Authorization(field_kind="use_status")) == ()
    assert _V("GR", _scope("label"),
              _wm.Authorization(field_kind="gr_family_source_curve_name")) == ()


@pytest.mark.parametrize("value", [
    "screening_proxy_allowed_v2", "SCREENING_PROXY_ALLOWED", "measured ",
    "GR_PROXY_MEDIUM", "arbitrary_label", "chalk_proxy_high",
])
def test_unapproved_label_values_are_rejected(value):
    assert _V(value, _scope("label"), _wm.Authorization(field_kind="use_status")), value


# --- Registered statements: id and text validated together -----------------

def test_every_registered_statement_passes_under_its_own_id():
    for sid, st in _wm.REGISTERED_STATEMENTS.items():
        auth = _wm.Authorization(statement_id=sid)
        assert _V(st.text, st.scope, auth) == (), sid
        assert st.purpose and len(st.provenance) > 40


def _a_method_statement():
    """The longest registered method statement, so every mutation below is a
    genuine near-miss rather than a no-op."""
    sid = max((k for k, v in _wm.REGISTERED_STATEMENTS.items()
               if v.scope == _wm.SCOPE_METHOD),
              key=lambda k: len(_wm.REGISTERED_STATEMENTS[k].text))
    return sid, _wm.REGISTERED_STATEMENTS[sid]


@pytest.mark.parametrize("mutate,label", [
    (lambda t: t.upper(), "case change"),
    (lambda t: t + " ", "trailing space"),
    (lambda t: t + " Additional clause.", "added clause"),
    (lambda t: t[:-1], "truncated"),
    (lambda t: t.replace(".", ""), "punctuation removed"),
])
def test_near_miss_text_fails_against_its_registered_id(mutate, label):
    sid, st = _a_method_statement()
    mutated = mutate(st.text)
    assert mutated != st.text, f"{label} must actually mutate the statement"
    assert _V(mutated, st.scope, _wm.Authorization(statement_id=sid)), label


@pytest.mark.parametrize("sid", sorted(
    k for k, v in _wm.REGISTERED_STATEMENTS.items() if v.scope == _wm.SCOPE_METHOD))
def test_every_method_statement_rejects_a_case_change(sid):
    """Applied to EVERY method statement, so no single statement carries the
    exact-equality guarantee alone."""
    st = _wm.REGISTERED_STATEMENTS[sid]
    if st.text.upper() == st.text:
        pytest.skip("statement has no case to change")
    assert _V(st.text.upper(), st.scope, _wm.Authorization(statement_id=sid))


def test_unknown_statement_id_fails():
    sid, st = _a_method_statement()
    assert _V(st.text, st.scope, _wm.Authorization(statement_id="no_such_statement"))


def test_id_text_mismatch_fails():
    sid, st = _a_method_statement()
    other = [v for k, v in _wm.REGISTERED_STATEMENTS.items()
             if v.scope == st.scope and k != sid][0]
    assert _V(other.text, st.scope, _wm.Authorization(statement_id=sid))


def test_statement_cannot_authorize_a_different_scope():
    sid, st = _a_method_statement()
    other = _wm.SCOPE_INTERPRETIVE if st.scope != _wm.SCOPE_INTERPRETIVE else _wm.SCOPE_METHOD
    assert _V(st.text, other, _wm.Authorization(statement_id=sid))


def test_presenting_both_a_statement_id_and_a_template_id_fails():
    sid, st = _a_method_statement()
    auth = _wm.Authorization(statement_id=sid, template_id="gr_proxy_confidence_rationale")
    assert _V(st.text, st.scope, auth)


def test_registry_ids_are_unique():
    assert _wm._duplicate_registry_ids() == []


# --- Templates: controlled prose, strictly typed substitutions -------------

def _tpl():
    return _wm.REGISTERED_TEMPLATES["gr_proxy_confidence_rationale"]


GOOD_FIELDS = {"valid_fraction": "0.8131", "dynamic_range_p05_p95": "135.761",
               "proxy_median_spread": "0.1184"}


def test_template_renders_and_passes_with_typed_substitutions():
    t = _tpl()
    auth = _wm.Authorization(template_id=t.template_id, fields=GOOD_FIELDS)
    assert _V(t.render(GOOD_FIELDS), t.scope, auth) == ()


@pytest.mark.parametrize("value", ["high", "very high", "0.1.2", "", "1e5", "NaN", "abc"])
def test_template_substitution_must_satisfy_its_declared_type(value):
    """A substitution slot is restricted to a decimal literal, so a template
    can never become a channel for prose."""
    t = _tpl()
    fields = dict(GOOD_FIELDS, proxy_median_spread=value)
    text = t.template.replace("{proxy_median_spread}", value).format(
        **{k: v for k, v in fields.items() if k != "proxy_median_spread"})
    assert _V(text, t.scope, _wm.Authorization(template_id=t.template_id, fields=fields))


def test_undeclared_or_missing_template_fields_fail():
    t = _tpl()
    extra = dict(GOOD_FIELDS, sneaky="1.0")
    assert _V(t.render(GOOD_FIELDS), t.scope,
              _wm.Authorization(template_id=t.template_id, fields=extra))
    missing = {k: v for k, v in GOOD_FIELDS.items() if k != "valid_fraction"}
    assert _V(t.render(GOOD_FIELDS), t.scope,
              _wm.Authorization(template_id=t.template_id, fields=missing))


def test_unknown_template_id_fails():
    t = _tpl()
    assert _V(t.render(GOOD_FIELDS), t.scope,
              _wm.Authorization(template_id="no_such_template", fields=GOOD_FIELDS))


def test_text_not_matching_the_rendered_template_fails():
    t = _tpl()
    auth = _wm.Authorization(template_id=t.template_id, fields=GOOD_FIELDS)
    assert _V(t.render(GOOD_FIELDS) + " Extra.", t.scope, auth)


# --- The legitimate limitation statements, routed properly -----------------

@pytest.mark.parametrize("text", [
    "The well contains no shale volume estimate.",
    "This method for the interval contains a shale proxy calculation.",
])
def test_legitimate_sounding_limitations_still_need_a_registered_id(text):
    """These read as reasonable method/limitation prose. They are rejected in
    every controlled scope because no registered statement authorizes them -
    NOT by a grammar-based interpretive exemption, which no longer exists."""
    for scope_name in CONTROLLED:
        assert _V(text, _scope(scope_name)), f"{text!r} in {scope_name}"


def test_the_projects_real_limitation_statement_passes_through_its_id():
    """The need those sentences express is real, and is met by the registered
    limitation statement the project actually persists."""
    st = _wm.REGISTERED_STATEMENTS["proxy_limitations"]
    assert _V(st.text, st.scope, _wm.Authorization(statement_id=st.statement_id)) == ()
    assert "not a calibrated shale volume" in st.text.lower()


# --- No fallback path ------------------------------------------------------

def test_no_scope_accepts_text_merely_because_no_rock_term_was_found():
    """Sweep every controlled scope with text the linter cannot fault. If any
    scope accepted it, an unknown lithology would have a way in."""
    accepted = []
    for scope_name in CONTROLLED:
        for text in INNOCENT_UNREGISTERED + FABRICATED_ASSERTIONS:
            assert _wm.find_prohibited_lithology_terms(text) == ()
            if not _V(text, _scope(scope_name)):
                accepted.append((scope_name, text))
    assert not accepted, f"fallback acceptance path found: {accepted}"


def test_an_entry_without_authorization_is_rejected_in_prose_scopes():
    for scope_name in ("interpretive", "method", "explanatory"):
        v = _V("Any text at all.", _scope(scope_name))
        assert v and "authorization" in v[0]["reason"].lower()


def test_malformed_scope_entry_is_rejected_loudly():
    with pytest.raises(ValueError, match="context, text, scope"):
        _wm.validate_no_prohibited_interpretation([("a", "b")])
    with pytest.raises(ValueError, match="unknown validation scope"):
        _wm.validate_no_prohibited_interpretation([("a", "b", "nonsense")])


# --- Real packaged content is fully accounted for --------------------------

def test_every_real_persisted_field_is_positively_authorized():
    """Requirement 7/12: every real per-well note resolves to an authorization
    and every real label is approved UNDER ITS OWN FIELD KIND."""
    from p2mem.io.petrophysics_inventory import _authorized
    from p2mem.petrophysics import load_petrophysics_eligibility_config
    cfg = load_petrophysics_eligibility_config(
        str(PROJECT_ROOT / "config" / "petrophysics_eligibility.yml"))
    entries = [(f"{wk}.config_notes", d.notes, _wm.SCOPE_INTERPRETIVE)
               for wk, d in cfg.wells.items()]
    authorized = [_authorized(c, t, s) for c, t, s in entries]
    assert all(len(e) == 4 for e in authorized)
    assert _wm.validate_no_prohibited_interpretation(authorized) == ()
    for wk, d in cfg.wells.items():
        for kind, value in (("use_status", d.use_status),
                            ("evidence_class", d.evidence_class),
                            ("gr_family_canonical_name", d.gr_family_canonical_name),
                            ("gr_family_source_curve_name", d.gr_family_source_curve_name)):
            assert _wm.approved_label(kind, value) is not None, f"{wk}: {kind}={value!r}"


In [ ]:
%%writefile tests/test_method_eligibility.py
"""
Increment 6 - method-eligibility mask, interval-register, and export tests.

SYNTHETIC ONLY. No real or private project file is read or packaged.
"""

import json

import numpy as np
from p2mem.wellframe_models import Authorization
import pytest

from p2mem.io.petrophysics_inventory import (
    build_eligibility_interval_rows,
    build_gr_endpoint_scenario_rows,
    build_gr_family_qc_rows,
    build_gr_proxy_sensitivity_rows,
    build_method_eligibility_rows,
    build_petrophysics_issue_rows,
    build_petrophysics_manifest,
    build_thickness_sensitivity_rows,
)
from p2mem.method_eligibility import (
    MASK_DENSITY_FOR_SV,
    MASK_DYNAMIC_ELASTIC,
    MASK_SONIC_NCT_CANDIDATE,
    build_eligibility_intervals,
    compute_density_eligibility,
    compute_dynamic_elastic_eligibility,
    compute_sonic_nct_candidate_eligibility,
)
from p2mem.petrophysics import (
    PetrophysicsInputError,
    compute_gr_family_qc_stats,
    compute_gr_proxy,
)
from p2mem.petrophysics_models import (
    EXCLUSION_REASON_BOREAS_ECGR,
    USE_STATUS_QC_ONLY_EXCLUDED,
    GrFamilyDisposition,
    PetrophysicsIssue,
)
from p2mem.wellframe import assemble_well_frame
from p2mem.wellframe_models import (
    WellFrameAssemblyFailure,
    assert_no_lithology_vocabulary,
)

from synthetic_inc6 import synthetic_las as _synthetic_las  # noqa: E402
from synthetic_inc6 import synthetic_survey as _synthetic_survey  # noqa: E402
from test_petrophysics import _config, _disposition, _scenario  # noqa: E402


# ---------------------------------------------------------------------------
# Synthetic frame builder with the full curve set
# ---------------------------------------------------------------------------

def _full_frame(
    n=10, rhob=2400.0, vp=4000.0, vs=2200.0, gr=50.0, well_key="Synth_1", md=None,
):
    """A frame carrying RHOB/VP/VS/GR. Scalars broadcast to length n;
    arrays are used verbatim."""
    def arr(x):
        return np.asarray(x, dtype=float) if np.ndim(x) else np.full(n, float(x))

    md = np.linspace(50.0, 350.0, n) if md is None else np.asarray(md, dtype=float)
    las = _synthetic_las(
        {"MD_m": md, "RHOB_kg_m3": arr(rhob), "VP_m_s": arr(vp),
         "VS_m_s": arr(vs), "GR_api": arr(gr)},
        well_name=well_key,
    )
    dev = _synthetic_survey(well_key, md=(0.0, 200.0, 400.0), tvd=(0.0, 199.0, 396.0))
    return assemble_well_frame(
        well_key, las, dev, las_path=f"/private/build/{well_key}.las",
        survey_path=f"/private/build/{well_key}_dev.txt", gr_family_canonical_name="GR_api",
    )


# ---------------------------------------------------------------------------
# Density eligibility
# ---------------------------------------------------------------------------

def test_density_mask_all_eligible_when_valid():
    m = compute_density_eligibility(_full_frame(), _config())
    assert m.mask_name == MASK_DENSITY_FOR_SV
    assert m.n_eligible == 10
    assert m.eligible_fraction == pytest.approx(1.0)
    assert not m.lithology_dependent


def test_density_mask_rejects_nan_density():
    rhob = np.full(10, 2400.0)
    rhob[3] = np.nan
    m = compute_density_eligibility(_full_frame(rhob=rhob), _config())
    assert m.n_eligible == 9
    assert not m.mask[3]


@pytest.mark.parametrize("bad", [-2400.0, 0.0, 100.0, 9000.0, np.inf, -np.inf])
def test_density_mask_rejects_non_physical_density(bad):
    """Non-physical density is excluded from the mask - never repaired."""
    rhob = np.full(10, 2400.0)
    rhob[5] = bad
    m = compute_density_eligibility(_full_frame(rhob=rhob), _config())
    assert not m.mask[5]
    assert m.n_eligible == 9


def test_density_mask_absent_curve_yields_all_false_not_implicit_pass():
    md = np.linspace(50.0, 350.0, 5)
    las = _synthetic_las({"MD_m": md, "GR_api": np.full(5, 40.0)})
    frame = assemble_well_frame(
        "Synth_1", las, _synthetic_survey(md=(0.0, 200.0, 400.0), tvd=(0.0, 199.0, 396.0)),
        las_path="/s/a.las", survey_path="/s/a_dev.txt",
    )
    m = compute_density_eligibility(frame, _config())
    assert m.n_eligible == 0
    assert not m.mask.any()


def test_density_mask_requires_mapped_depth():
    md = np.array([50.0, 100.0, 200.0, 300.0, 900.0])  # last beyond coverage
    m = compute_density_eligibility(_full_frame(n=5, md=md), _config())
    assert not m.mask[-1]
    assert m.n_eligible == 4


def test_density_mask_computed_for_gr_excluded_well():
    """GR exclusion is about the GR curve, not about density: a
    lithology-independent mask is still computed for an excluded well."""
    m = compute_density_eligibility(_full_frame(well_key="Boreas_1"), _config())
    assert m.n_eligible == 10


# ---------------------------------------------------------------------------
# Dynamic-elastic eligibility
# ---------------------------------------------------------------------------

def test_dynamic_elastic_mask_all_eligible_when_physical():
    m = compute_dynamic_elastic_eligibility(_full_frame(vp=4000.0, vs=2200.0), _config())
    assert m.mask_name == MASK_DYNAMIC_ELASTIC
    assert m.n_eligible == 10


def test_dynamic_elastic_rejects_vp_not_greater_than_vs():
    vp = np.full(10, 4000.0)
    vp[2] = 2000.0  # below VS
    m = compute_dynamic_elastic_eligibility(_full_frame(vp=vp, vs=2200.0), _config())
    assert not m.mask[2]


# ---------------------------------------------------------------------------
# Increment 6.1 (Finding 1): Vp/Vs regimes, stated correctly
#
#     nu = (r^2 - 2) / (2 * (r^2 - 1)),  r = Vp/Vs
#     K  = rho * (Vp^2 - (4/3) * Vs^2)
#
# r = sqrt(2)          -> nu = 0 exactly            -> MUST PASS a
#                                                     non-negative-nu screen
# sqrt(4/3) < r < sqrt(2) -> K > 0, nu < 0          -> excluded by POLICY,
#                                                     NOT "non-physical"
# r <= sqrt(4/3)       -> K <= 0                    -> genuinely outside the
#                                                     isotropic elastic model
# r > 4                -> ordinary nu (~0.467)      -> configured PLAUSIBILITY
#                                                     limit only
# ---------------------------------------------------------------------------

def _nu(r):
    """Dynamic Poisson's ratio for an isotropic elastic solid."""
    return (r ** 2 - 2.0) / (2.0 * (r ** 2 - 1.0))


def _bulk_modulus(rho, vp, vs):
    """K = rho * (Vp^2 - 4/3 Vs^2)."""
    return rho * (vp ** 2 - (4.0 / 3.0) * vs ** 2)


def test_poisson_ratio_is_exactly_zero_at_sqrt2():
    """The analytic anchor for the inclusive bound."""
    assert _nu(np.sqrt(2.0)) == pytest.approx(0.0, abs=1e-12)
    assert _nu(4.0) == pytest.approx(7.0 / 15.0)  # ~0.4667: an ordinary ratio


def test_vp_vs_exactly_sqrt2_passes_nonnegative_poisson_screen():
    """r = sqrt(2) gives nu = 0, which a NON-NEGATIVE-nu policy must ACCEPT.
    Increment 6 used an exclusive bound and wrongly rejected it."""
    vs = np.full(10, 2200.0)
    vp = np.full(10, 4000.0)
    vp[4] = 2200.0 * np.sqrt(2.0)  # r == sqrt(2) exactly
    frame = _full_frame(vp=vp, vs=vs)
    m = compute_dynamic_elastic_eligibility(frame, _config())
    assert _nu(vp[4] / vs[4]) == pytest.approx(0.0, abs=1e-12)
    assert m.mask[4], "nu = 0 must pass a non-negative-Poisson-ratio screen"
    assert m.n_eligible == 10
    assert m.diagnostic_counts["n_ratio_positive_bulk_but_negative_poisson"] == 0
    assert m.diagnostic_counts["n_ratio_nonpositive_bulk_modulus"] == 0


def test_positive_bulk_modulus_with_negative_poisson_is_diagnosed_separately():
    """sqrt(4/3) < r < sqrt(2): K > 0 and nu < 0. Excluded by POLICY, and
    NEVER counted as non-positive-bulk-modulus or called non-physical."""
    vs = np.full(10, 2200.0)
    vp = np.full(10, 4000.0)
    vp[3] = 2200.0 * 1.30  # sqrt(4/3)=1.1547 < 1.30 < sqrt(2)=1.41421
    frame = _full_frame(vp=vp, vs=vs)
    r = vp[3] / vs[3]
    assert np.sqrt(4.0 / 3.0) < r < np.sqrt(2.0)
    assert _bulk_modulus(2400.0, vp[3], vs[3]) > 0.0, "K must be positive here"
    assert _nu(r) < 0.0, "nu must be negative here"
    m = compute_dynamic_elastic_eligibility(frame, _config())
    assert not m.mask[3]
    assert m.diagnostic_counts["n_ratio_positive_bulk_but_negative_poisson"] == 1
    assert m.diagnostic_counts["n_ratio_nonpositive_bulk_modulus"] == 0
    assert m.n_eligible == 9


def test_nonpositive_bulk_modulus_is_diagnosed_separately():
    """r <= sqrt(4/3) implies K <= 0 - genuinely outside the isotropic
    elastic model, and the ONLY regime that warrants that description."""
    vs = np.full(10, 2200.0)
    vp = np.full(10, 4000.0)
    vp[2] = 2200.0 * np.sqrt(4.0 / 3.0)  # r == sqrt(4/3) exactly -> K == 0
    vp[6] = 2200.0 * 1.10                # r < sqrt(4/3)         -> K < 0
    frame = _full_frame(vp=vp, vs=vs)
    # K here is O(1e10), so compare relative to the rho*Vp^2 scale rather than
    # against a meaningless absolute tolerance.
    _scale = 2400.0 * vp[2] ** 2
    assert abs(_bulk_modulus(2400.0, vp[2], vs[2])) < 1e-12 * _scale
    assert _bulk_modulus(2400.0, vp[6], vs[6]) < 0.0
    m = compute_dynamic_elastic_eligibility(frame, _config())
    assert not m.mask[2] and not m.mask[6]
    assert m.diagnostic_counts["n_ratio_nonpositive_bulk_modulus"] == 2
    assert m.diagnostic_counts["n_ratio_positive_bulk_but_negative_poisson"] == 0
    assert m.n_eligible == 8


def test_ratio_above_configured_plausibility_max_is_diagnosed_separately():
    """r > 4 has an ordinary Poisson's ratio; it is outside a CONFIGURED
    plausibility limit, not outside the mathematical Poisson domain."""
    vs = np.full(10, 700.0)
    vp = np.full(10, 2800.0)   # r = 4.0 exactly -> inside the inclusive max
    vp[8] = 700.0 * 5.0        # r = 5.0 -> above the configured max
    frame = _full_frame(vp=vp, vs=vs)
    assert 0.0 < _nu(5.0) < 0.5, "r = 5 still gives an ordinary Poisson ratio"
    m = compute_dynamic_elastic_eligibility(frame, _config())
    assert m.mask[0], "r = 4.0 is at the inclusive configured maximum"
    assert not m.mask[8]
    assert m.diagnostic_counts["n_ratio_above_configured_plausibility_max"] == 1
    assert m.diagnostic_counts["n_ratio_nonpositive_bulk_modulus"] == 0
    assert m.diagnostic_counts["n_ratio_positive_bulk_but_negative_poisson"] == 0


def test_vp_vs_regime_diagnostics_are_mutually_exclusive_and_exhaustive():
    """Every both-velocities-valid sample lands in exactly one regime."""
    # VS = 1000 m/s keeps every VP below the 8000 m/s plausibility bound, so
    # each sample genuinely reaches the both-velocities-valid subset.
    vs = np.full(12, 1000.0)
    vp = np.full(12, 3000.0)
    vp[0] = 1000.0 * 1.10               # K <= 0
    vp[1] = 1000.0 * np.sqrt(4.0 / 3.0) # K == 0
    vp[2] = 1000.0 * 1.30               # K > 0, nu < 0
    vp[3] = 1000.0 * np.sqrt(2.0)       # nu == 0 -> passes
    vp[4] = 1000.0 * 5.0                # above configured max
    m = compute_dynamic_elastic_eligibility(_full_frame(n=12, vp=vp, vs=vs), _config())
    d = m.diagnostic_counts
    total = (d["n_ratio_nonpositive_bulk_modulus"]
             + d["n_ratio_positive_bulk_but_negative_poisson"]
             + d["n_ratio_above_configured_plausibility_max"]
             + d["n_passes_nonnegative_poisson_screen"])
    assert total == d["n_vp_and_vs_both_valid"] == 12
    assert d["n_ratio_nonpositive_bulk_modulus"] == 2
    assert d["n_ratio_positive_bulk_but_negative_poisson"] == 1
    assert d["n_ratio_above_configured_plausibility_max"] == 1
    assert d["n_passes_nonnegative_poisson_screen"] == 8


def test_no_diagnostic_or_note_aggregates_regimes_as_non_physical():
    """The phrase must not reappear as a blanket label over all excluded
    ratios - that conflation is exactly what Increment 6.1 corrects."""
    m = compute_dynamic_elastic_eligibility(_full_frame(), _config())
    for key in m.diagnostic_counts:
        assert "non_physical" not in key.lower()
        assert "nonphysical" not in key.lower()
    low = m.notes.lower()
    assert "never aggregated" in low
    assert "not non-physical" in low or "not a test of physical possibility" in low


def test_dynamic_elastic_rejects_nan_in_any_of_the_three_inputs():
    for curve in ("vp", "vs", "rhob"):
        kwargs = {"vp": 4000.0, "vs": 2200.0, "rhob": 2400.0}
        arr = np.full(10, kwargs[curve])
        arr[7] = np.nan
        kwargs[curve] = arr
        m = compute_dynamic_elastic_eligibility(_full_frame(**kwargs), _config())
        assert not m.mask[7], f"{curve} NaN should disqualify the sample"
        assert m.n_eligible == 9


def test_dynamic_elastic_reports_limiting_criterion():
    vs = np.full(10, 2200.0)
    vs[:6] = np.nan  # VS is by far the most limiting input
    m = compute_dynamic_elastic_eligibility(_full_frame(vs=vs), _config())
    assert m.limiting_criterion == "vs_finite_positive_in_bounds"
    assert m.n_eligible == 4


def test_dynamic_elastic_does_not_compute_any_elastic_property():
    """The result exposes only a mask and counts - no modulus, no Poisson
    ratio, no elastic curve of any kind."""
    m = compute_dynamic_elastic_eligibility(_full_frame(), _config())
    exposed = set(m.__slots__)
    for forbidden in ("youngs_modulus", "poisson_ratio", "bulk_modulus", "shear_modulus",
                      "vp_vs_ratio", "elastic"):
        assert not any(forbidden in name for name in exposed)


# ---------------------------------------------------------------------------
# Sonic-NCT candidate eligibility
# ---------------------------------------------------------------------------

def _proxy_for(frame, cfg=None, low=10.0, high=110.0):
    cfg = cfg or _config()
    return compute_gr_proxy(
        frame, _disposition(frame.well_key), _scenario(low, high, well_key=frame.well_key), cfg
    )


def test_nct_candidate_mask_basic():
    frame = _full_frame(gr=np.linspace(10.0, 110.0, 10))
    p = _proxy_for(frame)
    m = compute_sonic_nct_candidate_eligibility(
        frame, _disposition(), p, 0.5, _config()
    )
    assert m.mask_name == MASK_SONIC_NCT_CANDIDATE
    assert m.lithology_dependent
    assert m.proxy_threshold == 0.5
    assert m.scenario_name == p.scenario_name
    assert 0 < m.n_eligible < 10


def test_nct_candidate_threshold_sensitivity_is_monotonic():
    frame = _full_frame(n=101, gr=np.linspace(0.0, 200.0, 101))
    p = _proxy_for(frame)
    counts = [
        compute_sonic_nct_candidate_eligibility(frame, _disposition(), p, t, _config()).n_eligible
        for t in (0.5, 0.6, 0.7)
    ]
    assert counts[0] >= counts[1] >= counts[2]
    assert counts[0] > counts[2]  # the threshold genuinely bites


def test_nct_candidate_forbidden_for_gr_excluded_well():
    """A lithology-dependent mask can never exist for a GR-excluded well."""
    frame = _full_frame(well_key="Boreas_1")
    d = _disposition("Boreas_1", "ECGR_api", USE_STATUS_QC_ONLY_EXCLUDED,
                     EXCLUSION_REASON_BOREAS_ECGR)
    p = _proxy_for(_full_frame(well_key="Boreas_1"))
    with pytest.raises(PetrophysicsInputError, match="lithology-dependent"):
        compute_sonic_nct_candidate_eligibility(frame, d, p, 0.5, _config())


def test_nct_candidate_rejects_proxy_from_another_well():
    frame = _full_frame(well_key="A_1")
    p = _proxy_for(_full_frame(well_key="B_1"))
    with pytest.raises(PetrophysicsInputError, match="never transferred between wells"):
        compute_sonic_nct_candidate_eligibility(frame, _disposition("A_1"), p, 0.5, _config())


def test_nct_candidate_rejects_non_finite_threshold():
    frame = _full_frame()
    p = _proxy_for(frame)
    with pytest.raises(PetrophysicsInputError, match="must be finite"):
        compute_sonic_nct_candidate_eligibility(frame, _disposition(), p, np.nan, _config())


def test_nct_candidate_notes_disclaim_normal_compaction():
    frame = _full_frame(gr=np.linspace(10.0, 110.0, 10))
    m = compute_sonic_nct_candidate_eligibility(
        frame, _disposition(), _proxy_for(frame), 0.5, _config()
    )
    low = m.notes.lower()
    assert "candidate data only" in low
    assert "no nct fitted" in low
    assert "not proof" in low


def test_nct_candidate_requires_finite_vp():
    vp = np.full(10, 4000.0)
    vp[1] = np.nan
    frame = _full_frame(vp=vp, gr=np.full(10, 100.0))
    m = compute_sonic_nct_candidate_eligibility(
        frame, _disposition(), _proxy_for(frame), 0.5, _config()
    )
    assert not m.mask[1]


# ---------------------------------------------------------------------------
# Contiguous interval registers
# ---------------------------------------------------------------------------

def test_intervals_report_md_tvd_and_tvdss():
    frame = _full_frame(n=30)
    m = compute_density_eligibility(frame, _config())
    ivs = build_eligibility_intervals(frame, m, _config())
    assert len(ivs) == 1
    iv = ivs[0]
    assert iv.md_start_m is not None and iv.md_end_m is not None
    assert iv.tvd_start_m is not None and iv.tvdss_start_m is not None
    assert iv.gross_thickness_md_m > 0 and iv.gross_thickness_tvdss_m > 0
    # With no bridging, gross and net coincide.
    assert iv.net_thickness_tvdss_m == pytest.approx(iv.gross_thickness_tvdss_m)
    assert iv.contiguity_policy == "configured_bridging"
    assert iv.meets_configured_minimums is True
    assert iv.n_samples == 30
    assert iv.n_eligible_samples == 30
    assert iv.n_bridged_samples == 0
    assert iv.depth_basis_used == frame.depth_basis_used


def test_intervals_split_on_large_gap():
    rhob = np.full(30, 2400.0)
    rhob[10:20] = np.nan
    frame = _full_frame(n=30, rhob=rhob)
    m = compute_density_eligibility(frame, _config())
    ivs = build_eligibility_intervals(frame, m, _config())
    assert len(ivs) == 2
    assert ivs[0].end_index == 9
    assert ivs[1].start_index == 20


def test_intervals_bridge_small_gap_and_disclose_it():
    """A bridged gap is always disclosed via n_bridged_samples - eligible
    sample count and block span are reported separately."""
    rhob = np.full(30, 2400.0)
    rhob[10] = np.nan  # single-sample gap, small physical span
    frame = _full_frame(n=30, rhob=rhob)
    cfg = _config(contiguity={"max_gap_samples": 2, "max_gap_depth_m": 100.0,
                              "min_block_samples": 2, "min_block_thickness_m": 1.0})
    ivs = build_eligibility_intervals(frame, compute_density_eligibility(frame, cfg), cfg)
    assert len(ivs) == 1
    assert ivs[0].n_samples == 30
    assert ivs[0].n_eligible_samples == 29
    assert ivs[0].n_bridged_samples == 1
    assert ivs[0].n_bridged_gaps == 1
    assert ivs[0].n_eligible_subruns == 2


def test_intervals_record_limiting_reason_for_short_blocks():
    rhob = np.full(30, np.nan)
    rhob[0:3] = 2400.0  # a 3-sample block, under the 20-sample minimum
    frame = _full_frame(n=30, rhob=rhob)
    cfg = _config(contiguity={"max_gap_samples": 0, "max_gap_depth_m": 0.0,
                              "min_block_samples": 20, "min_block_thickness_m": 5.0})
    ivs = build_eligibility_intervals(frame, compute_density_eligibility(frame, cfg), cfg)
    assert len(ivs) == 1  # disclosed, not silently dropped
    assert "below_min_block_samples" in ivs[0].limiting_reason


def test_intervals_empty_when_nothing_eligible():
    frame = _full_frame(n=10, rhob=np.full(10, np.nan))
    m = compute_density_eligibility(frame, _config())
    assert build_eligibility_intervals(frame, m, _config()) == []


def test_intervals_thickness_none_when_depth_unmapped_at_edge():
    """Physical thickness is None (unknown) when an endpoint has no mapped
    depth - never 0, and never silently replaced by the MD span."""
    md = np.array([50.0, 100.0, 200.0, 300.0, 900.0])
    frame = _full_frame(n=5, md=md)
    m = compute_density_eligibility(frame, _config())
    ivs = build_eligibility_intervals(frame, m, _config())
    assert all(iv.end_index < 4 for iv in ivs)  # unmapped sample never inside a block


# ---------------------------------------------------------------------------
# Export layer: determinism, sanitization, JSON-serializability
# ---------------------------------------------------------------------------

def _export_bundle():
    frames = {}
    stats = {}
    conf = {}
    rationale = {}
    scen = {}
    proxies = {}
    masks = []
    for wk in ("Z_1", "A_1"):
        fr = _full_frame(n=20, gr=np.linspace(10.0, 110.0, 20), well_key=wk)
        frames[wk] = fr
        d = _disposition(wk)
        stats[wk] = compute_gr_family_qc_stats(fr, d)
        conf[wk] = "GR_PROXY_INTERMEDIATE"
        rationale[wk] = "synthetic"
        s = _scenario(well_key=wk)
        scen[wk] = [s]
        p = compute_gr_proxy(fr, d, s, _config())
        proxies[wk] = [p]
        masks.append(compute_density_eligibility(fr, _config()))
        masks.append(compute_dynamic_elastic_eligibility(fr, _config()))
        masks.append(compute_sonic_nct_candidate_eligibility(fr, d, p, 0.5, _config()))
    disp = {wk: _disposition(wk) for wk in frames}
    return frames, stats, conf, rationale, scen, proxies, masks, disp


def test_export_rows_are_deterministically_ordered():
    frames, stats, conf, rationale, scen, proxies, masks, disp = _export_bundle()
    qc = build_gr_family_qc_rows(stats, disp, conf)
    assert [r["well_key"] for r in qc] == ["A_1", "Z_1"]
    el = build_method_eligibility_rows(masks, frames, disp)
    keys = [(r["well_key"], r["mask_name"], r["scenario_name"], r["proxy_threshold"]) for r in el]
    assert keys == sorted(keys, key=lambda k: (k[0], k[1], k[2], -1e18 if k[3] is None else k[3]))


def test_export_rows_contain_no_absolute_paths():
    """Every path-shaped field is reduced to a basename, so a private build
    directory can never leak into a deliverable."""
    frames, stats, conf, rationale, scen, proxies, masks, disp = _export_bundle()
    blob = json.dumps(
        build_gr_family_qc_rows(stats, disp, conf)
        + build_gr_endpoint_scenario_rows(scen)
        + build_gr_proxy_sensitivity_rows(proxies, disp)
        + build_method_eligibility_rows(masks, frames, disp)
    )
    assert "/private/build" not in blob
    assert "/home/" not in blob and "/root/" not in blob and "/content/" not in blob


def test_failure_message_is_sanitized_against_both_candidate_paths():
    """A well-frame failure may originate from either file, so BOTH paths
    are sanitized - never only one."""
    f = WellFrameAssemblyFailure(
        well_key="X_1", failure_origin="depth_mapping", error_type="depth_mapping_failure",
        message=("X_1: failed using /home/user/secret/X_1_logs.las against "
                 "/home/user/secret/X_1_dev.txt"),
        las_path="/home/user/secret/X_1_logs.las",
        survey_path="/home/user/secret/X_1_dev.txt",
    )
    rows = build_petrophysics_issue_rows([], {"X_1": f})
    assert len(rows) == 1
    assert "/home/user/secret" not in rows[0]["message"]
    assert "X_1_logs.las" in rows[0]["message"]
    assert "X_1_dev.txt" in rows[0]["message"]
    assert "/home/user/secret" not in rows[0]["context"]


def test_issue_rows_carry_severity_and_code():
    rows = build_petrophysics_issue_rows(
        [PetrophysicsIssue("WARNING", "SOME_CODE", "a message", "ctx")], {}
    )
    assert rows[0]["severity"] == "WARNING"
    assert rows[0]["code"] == "SOME_CODE"


def test_manifest_is_json_serializable_and_has_no_numpy_scalars():
    frames, stats, conf, rationale, scen, proxies, masks, disp = _export_bundle()
    man = build_petrophysics_manifest(
        frames, disp, stats, conf, rationale, scen, masks, {}, [],
        config_filename="petrophysics_eligibility.yml", config_schema_version="6.0",
        nct_candidate_thresholds=[0.5, 0.6, 0.7],
    )
    text = json.dumps(man)  # raises TypeError on any NumPy scalar
    assert json.loads(text)["increment"] == 6
    assert "/private/build" not in text


def test_manifest_declares_no_named_lithology():
    frames, stats, conf, rationale, scen, proxies, masks, disp = _export_bundle()
    man = build_petrophysics_manifest(
        frames, disp, stats, conf, rationale, scen, masks, {}, [],
        config_filename="c.yml", config_schema_version="6.0",
        nct_candidate_thresholds=[0.5],
    )
    # Increment 6.1.5: this bundle's prose is SYNTHETIC and therefore
    # unauthorized, so the derived flag is True here. That is the contract
    # working - the assertions below are about the manifest's other declared
    # facts, which are unaffected. The clean/authorized path is covered by
    # test_unchanged_real_packaged_content_keeps_the_gate_passing.
    assert man["named_lithology_assigned"] is True
    assert all(v["terms"] == [] for v in man["lithology_validation"]["violations"]), (
        "synthetic prose contains no rock word; it fails for lack of authorization"
    )
    assert man["nonlinear_vsh_transforms_implemented"] is False
    assert man["total_samples_extrapolated"] == 0
    assert man["calibration_data_available"]["pressure_rft_mdt_dst"] is False
    assert man["calibration_data_available"]["stress_fit_lot_xlot_dfit"] is False
    for method in ("normal_compaction_trend_fitting", "eaton_sonic_pore_pressure",
                   "dynamic_elastic_property_calculation", "wellbore_stability_analysis",
                   "named_lithology_interpretation"):
        assert method in man["methods_not_implemented"]


def test_no_export_row_contains_named_lithology_vocabulary():
    """Every generated classification string in every export is checked
    against the prohibited rock-name vocabulary."""
    frames, stats, conf, rationale, scen, proxies, masks, disp = _export_bundle()
    rows = (
        build_gr_family_qc_rows(stats, disp, conf)
        + build_gr_endpoint_scenario_rows(scen)
        + build_gr_proxy_sensitivity_rows(proxies, disp)
        + build_method_eligibility_rows(masks, frames, disp)
    )
    for r in rows:
        for field in ("gr_proxy_confidence_class", "use_status", "evidence_class",
                      "mask_name", "scenario_name", "gr_family_canonical_name"):
            if r.get(field):
                assert_no_lithology_vocabulary(str(r[field]), f"{field}")


def test_export_never_contains_per_sample_arrays():
    """No exported row or manifest value is an array-like of sample values -
    packaging one would effectively reproduce the private source log."""
    frames, stats, conf, rationale, scen, proxies, masks, disp = _export_bundle()
    rows = (
        build_gr_family_qc_rows(stats, disp, conf)
        + build_gr_proxy_sensitivity_rows(proxies, disp)
        + build_method_eligibility_rows(masks, frames, disp)
        + build_eligibility_interval_rows(
            build_eligibility_intervals(frames["A_1"], masks[0], _config())
        )
    )
    for r in rows:
        for k, v in r.items():
            assert not isinstance(v, (list, tuple, np.ndarray)), f"{k} exports an array"


def test_interval_rows_are_json_safe_and_sorted():
    frame = _full_frame(n=30)
    ivs = build_eligibility_intervals(frame, compute_density_eligibility(frame, _config()), _config())
    rows = build_eligibility_interval_rows(ivs)
    json.dumps(rows)
    assert rows[0]["unit"] == "metres"
    assert "not proof of normal compaction" in rows[0]["limitations"].lower() or \
           "not evidence" in rows[0]["limitations"].lower()


def test_dynamic_elastic_limiting_criterion_is_not_confounded_by_vs_sparsity():
    """The Vp/Vs criteria are only meaningful where both velocities exist, so
    their raw pass counts are structurally bounded by VS availability. They
    must NOT be allowed to win the limiting-criterion comparison and report a
    DATA-COVERAGE problem as a PHYSICS problem."""
    vs = np.full(20, 2200.0)
    vs[:15] = np.nan          # VS is by far the scarcest curve
    vp = np.full(20, 4000.0)  # every surviving ratio is comfortably physical
    m = compute_dynamic_elastic_eligibility(_full_frame(n=20, vp=vp, vs=vs), _config())
    assert m.limiting_criterion == "vs_finite_positive_in_bounds"
    assert "vp_vs_ratio_in_poisson_domain" not in m.criteria_counts
    assert m.n_eligible == 5


def test_dynamic_elastic_reports_each_excluded_regime_as_its_own_count():
    """Each exclusion regime is reported separately, under a name that says
    what it actually is."""
    vs = np.full(10, 2200.0)
    vp = np.full(10, 4000.0)
    vp[3] = 2200.0 * 1.20  # sqrt(4/3) < r < sqrt(2): K > 0, nu < 0
    vp[7] = 2000.0         # VP below VS entirely
    m = compute_dynamic_elastic_eligibility(_full_frame(vp=vp, vs=vs), _config())
    d = m.diagnostic_counts
    assert d["n_vp_and_vs_both_valid"] == 10
    assert d["n_vp_not_greater_than_vs"] == 1
    assert d["n_ratio_positive_bulk_but_negative_poisson"] == 1
    # The VP<VS sample has r < 1, so it is ALSO a non-positive-K ratio; the
    # regime counts describe the RATIO, the vp_gt_vs count describes the
    # ordering. Both are reported; neither is merged into the other.
    assert d["n_ratio_nonpositive_bulk_modulus"] == 1
    assert m.n_eligible == 8


def test_diagnostic_counts_are_exported_and_json_safe():
    frames, stats, conf, rationale, scen, proxies, masks, disp = _export_bundle()
    rows = build_method_eligibility_rows(masks, frames, disp)
    elastic = [r for r in rows if r["mask_name"] == MASK_DYNAMIC_ELASTIC]
    assert elastic and all("diagnostic_counts" in r for r in elastic)
    assert all("n_vp_and_vs_both_valid" in r["diagnostic_counts"] for r in elastic)
    json.dumps(rows)


# ---------------------------------------------------------------------------
# Increment 6.1 (Finding 4): gross vs net vs strict-no-gap thickness
# ---------------------------------------------------------------------------

def _bridging_frame():
    """30 samples with a single-sample ineligible gap at index 10, small
    enough in both sample count and depth span to be bridged."""
    rhob = np.full(30, 2400.0)
    rhob[10] = np.nan
    return _full_frame(n=30, rhob=rhob)


def _bridging_config():
    return _config(contiguity={"max_gap_samples": 2, "max_gap_depth_m": 100.0,
                               "min_block_samples": 2, "min_block_thickness_m": 1.0})


def test_gross_exceeds_net_by_exactly_the_bridged_gap_span():
    """The gross endpoint span minus the net sum of strictly-contiguous
    sub-run spans must equal the bridged gap's own depth span."""
    frame, cfg = _bridging_frame(), _bridging_config()
    ivs = build_eligibility_intervals(frame, compute_density_eligibility(frame, cfg), cfg)
    assert len(ivs) == 1
    iv = ivs[0]
    assert iv.n_bridged_samples == 1
    assert iv.n_bridged_gaps == 1
    assert iv.n_eligible_subruns == 2
    y = np.asarray(frame.TVDSS_m, dtype=float)
    # Identity: with sub-runs [s, p-1] and [p+g, e] around a gap of length g
    # starting at p, gross - net = y[p+g] - y[p-1] - the span from the last
    # eligible sample before the gap to the first eligible sample after it.
    gap_span = abs(float(y[11]) - float(y[9]))
    assert iv.gross_thickness_tvdss_m > iv.net_thickness_tvdss_m
    assert iv.gross_thickness_tvdss_m - iv.net_thickness_tvdss_m == pytest.approx(
        gap_span, abs=1e-9
    )


def test_strict_policy_produces_no_bridged_samples_and_gross_equals_net():
    frame, cfg = _bridging_frame(), _bridging_config()
    m = compute_density_eligibility(frame, cfg)
    strict = build_eligibility_intervals(frame, m, cfg, contiguity_policy="strict_no_gap")
    assert len(strict) == 2, "the gap must split the block under a strict policy"
    for iv in strict:
        assert iv.contiguity_policy == "strict_no_gap"
        assert iv.n_bridged_samples == 0
        assert iv.n_bridged_gaps == 0
        assert iv.n_eligible_subruns == 1
        assert iv.net_thickness_tvdss_m == pytest.approx(iv.gross_thickness_tvdss_m)


def test_strict_total_never_exceeds_configured_gross_total():
    """A strict decomposition can only be shorter than, or equal to, the
    bridged one - it removes span, never adds it."""
    frame, cfg = _bridging_frame(), _bridging_config()
    m = compute_density_eligibility(frame, cfg)
    conf = build_eligibility_intervals(frame, m, cfg)
    strict = build_eligibility_intervals(frame, m, cfg, contiguity_policy="strict_no_gap")
    g = sum(iv.gross_thickness_tvdss_m for iv in conf)
    s = sum(iv.gross_thickness_tvdss_m for iv in strict)
    assert s <= g


def test_unknown_contiguity_policy_rejected():
    frame, cfg = _bridging_frame(), _bridging_config()
    with pytest.raises(PetrophysicsInputError, match="Unknown contiguity_policy"):
        build_eligibility_intervals(frame, compute_density_eligibility(frame, cfg), cfg,
                                    contiguity_policy="whatever")


def test_sensitivity_rows_state_their_population_and_bridging():
    """Every sensitivity case must report how many blocks it covers, how
    many qualify, and how much bridging the gross figure absorbed."""
    from p2mem.io.petrophysics_inventory import build_thickness_sensitivity_rows
    frame, cfg = _bridging_frame(), _bridging_config()
    ivs = build_eligibility_intervals(frame, compute_density_eligibility(frame, cfg), cfg)
    rows = build_thickness_sensitivity_rows(ivs)
    assert len(rows) == 1
    r = rows[0]
    assert r["n_blocks_all"] == 1
    assert r["n_blocks_qualifying"] == 1
    assert r["n_bridged_samples_in_qualifying_blocks"] == 1
    assert r["n_interrupted_qualifying_blocks"] == 1
    assert r["gross_qualifying_thickness_tvdss_m"] > r["net_qualifying_thickness_tvdss_m"]
    assert "GROSS" in r["population_statement"] and "NET" in r["population_statement"]
    assert r["contiguity_policy"] == "configured_bridging"


def test_sensitivity_rows_separate_qualifying_from_rejected_blocks():
    """The qualifying total must exclude sub-threshold blocks, and the
    rejected count must be reported rather than hidden."""
    from p2mem.io.petrophysics_inventory import build_thickness_sensitivity_rows
    rhob = np.full(40, np.nan)
    rhob[0:25] = 2400.0   # a qualifying block
    rhob[35:38] = 2400.0  # a 3-sample block, below the 20-sample minimum
    frame = _full_frame(n=40, rhob=rhob)
    cfg = _config(contiguity={"max_gap_samples": 0, "max_gap_depth_m": 0.0,
                              "min_block_samples": 20, "min_block_thickness_m": 5.0})
    ivs = build_eligibility_intervals(frame, compute_density_eligibility(frame, cfg), cfg)
    rows = build_thickness_sensitivity_rows(ivs)
    r = rows[0]
    assert r["n_blocks_all"] == 2
    assert r["n_blocks_qualifying"] == 1
    assert r["n_blocks_rejected_below_minimums"] == 1
    assert r["gross_qualifying_thickness_tvdss_m"] < r["gross_all_block_thickness_tvdss_m"]


def test_interval_rows_never_export_an_unqualified_thickness_field():
    """No exported column may be called simply 'thickness_*' - the whole
    point of Finding 4 is that the qualifier is mandatory."""
    frame, cfg = _bridging_frame(), _bridging_config()
    ivs = build_eligibility_intervals(frame, compute_density_eligibility(frame, cfg), cfg)
    rows = build_eligibility_interval_rows(ivs)
    for r in rows:
        for key in r:
            if "thickness" in key:
                assert key.startswith("gross_") or key.startswith("net_"), key


# ---------------------------------------------------------------------------
# Increment 6.1 (Finding 3): the completion gate is DERIVED, not hardcoded
# ---------------------------------------------------------------------------

def _registered_note():
    """A REGISTERED interpretive statement. Increment 6.1.5: synthetic prose is
    no longer authorized, so a helper that means to build a CLEAN manifest must
    use content the project has actually registered."""
    from p2mem.wellframe_models import REGISTERED_STATEMENTS, SCOPE_INTERPRETIVE
    return sorted(s.text for s in REGISTERED_STATEMENTS.values()
                  if s.scope == SCOPE_INTERPRETIVE)[0]


def _registered_rationale():
    """The registered rationale TEMPLATE rendered with typed decimal values."""
    from p2mem.wellframe_models import REGISTERED_TEMPLATES
    tpl = REGISTERED_TEMPLATES["gr_proxy_confidence_rationale"]
    return tpl.render({"valid_fraction": "0.9000",
                       "dynamic_range_p05_p95": "100.000",
                       "proxy_median_spread": "0.1000"})


def _output_registered_mask(mask):
    """Set the mask's persisted prose to the statements the OUTPUT policy
    declares for method_eligibility_summary.csv, so a builder-produced payload
    carries the same controlled values the real artifact does."""
    from p2mem.io.output_policy import OUTPUT_STATEMENTS
    def _pick(prefix):
        return sorted(st.text for st in OUTPUT_STATEMENTS.values()
                      if st.statement_id.startswith(prefix))
    purposes = _pick("method_eligibility_summary_purpose")
    idx = {"eligible_density_for_sv": 0, "eligible_dynamic_elastic": 1,
           "eligible_sonic_nct_candidate": 2}.get(mask.mask_name, 0)
    object.__setattr__(mask, "purpose", purposes[min(idx, len(purposes) - 1)])
    object.__setattr__(mask, "notes", _registered_note())
    return mask


def _registered_mask(mask):
    """Increment 6.1.5: a mask's notes and purpose are persisted interpretive
    fields, so the synthetic config's "synthetic purpose" is - correctly -
    unauthorized. A helper that means to build a CLEAN manifest substitutes the
    registered mask prose the real project actually persists."""
    from p2mem.wellframe_models import REGISTERED_STATEMENTS, SCOPE_INTERPRETIVE
    registered = sorted(s.text for s in REGISTERED_STATEMENTS.values()
                        if s.scope == SCOPE_INTERPRETIVE)
    object.__setattr__(mask, "notes", registered[0])
    object.__setattr__(mask, "purpose", registered[1])
    return mask


def _clean_disposition():
    return GrFamilyDisposition(
        well_key="W_1", source_las_filename="W_1.las", gr_family_canonical_name="GR_api",
        gr_family_source_curve_name="GR", use_status="screening_proxy_allowed",
        exclusion_reason=None, evidence_class="measured", has_approved_formation_tops=True,
        notes=_registered_note(),
    )


def _manifest_with(disposition=None, confidence="GR_PROXY_INTERMEDIATE"):
    """Build a real manifest from real builders, so the gate under test is the
    one the deliverable actually uses."""
    d = disposition or _clean_disposition()
    frame = _full_frame(n=20, gr=np.linspace(10.0, 110.0, 20), well_key="W_1")
    stats = compute_gr_family_qc_stats(frame, d)
    mask = _registered_mask(compute_density_eligibility(frame, _config()))
    return build_petrophysics_manifest(
        {"W_1": frame}, {"W_1": d}, {"W_1": stats}, {"W_1": confidence},
        {"W_1": _registered_rationale()}, {"W_1": []}, [mask], {}, [],
        config_filename="c.yml", config_schema_version="6.0",
        nct_candidate_thresholds=[0.5],
    )


def _registered_endpoint_description():
    """Increment 6.1.6: an endpoint description is a CONTROLLED emitted field.
    A synthetic one is correctly refused by the builder, so a test that means to
    exercise the happy path uses a registered description."""
    from p2mem.io.output_policy import OUTPUT_STATEMENTS
    return sorted(st.text for st in OUTPUT_STATEMENTS.values()
                  if st.statement_id.startswith("gr_endpoint_scenarios_description"))[0]


def _n_viol(manifest):
    """Increment 6.1.6: violations are the union of the scope-object pass and
    the emission-boundary pass, so tests count the list itself."""
    return len(manifest["lithology_validation"]["violations"])


def _gate_passes(manifest):
    """The completion-gate condition as the notebook evaluates it: DERIVED
    from the validation result, never from a constant."""
    lv = manifest["lithology_validation"]
    return (manifest["named_lithology_assigned"] is False) and (len(lv["violations"]) == 0)


def test_manifest_lithology_flag_is_derived_from_a_real_validation_pass():
    man = _manifest_with()
    lv = man["lithology_validation"]
    assert lv["scope_object_fields_checked"] > 0, "the validation must actually inspect content"
    assert _n_viol(man) == 0
    assert man["named_lithology_assigned"] is False
    assert _gate_passes(man)
    assert "DERIVED" in lv["derivation"] or "derived" in lv["derivation"]


def test_injected_prohibited_term_in_a_per_well_note_fails_gate():
    """Injecting a prohibited interpretation must make the validation, the
    manifest flag, AND the completion gate all fail together."""
    dirty = GrFamilyDisposition(
        well_key="W_1", source_las_filename="W_1.las", gr_family_canonical_name="GR_api",
        gr_family_source_curve_name="GR", use_status="screening_proxy_allowed",
        exclusion_reason=None, evidence_class="measured", has_approved_formation_tops=True,
        notes="Distribution is consistent with a clastic-dominated section.",
    )
    man = _manifest_with(disposition=dirty)
    assert man["named_lithology_assigned"] is True
    assert _n_viol(man) == 1
    assert man["lithology_validation"]["violations"][0]["terms"] == ["clastic"]
    assert not _gate_passes(man)


def test_injected_prohibited_term_in_a_persisted_classification_fails_gate():
    man = _manifest_with(confidence="SHALE_PROXY_HIGH")
    assert man["named_lithology_assigned"] is True
    assert not _gate_passes(man)
    ctxs = [v["context"] for v in man["lithology_validation"]["violations"]]
    assert any("gr_proxy_confidence_class" in c for c in ctxs)


def test_gate_cannot_be_satisfied_by_a_constant():
    """A hardcoded `False` would keep the gate passing under injection; the
    derived flag must move with the evidence."""
    clean = _manifest_with()
    dirty = _manifest_with(confidence="LIMESTONE_PROXY_HIGH")
    assert clean["named_lithology_assigned"] != dirty["named_lithology_assigned"]
    assert _gate_passes(clean) and not _gate_passes(dirty)


def test_validation_scope_covers_labels_notes_and_manifest_statement():
    man = _manifest_with()
    # The scope is rebuilt here from the same builder the manifest uses.
    from p2mem.io.petrophysics_inventory import build_lithology_validation_scope
    scope = build_lithology_validation_scope(
        {"W_1": _clean_disposition()}, {"W_1": "GR_PROXY_HIGH"},
        {"W_1": _registered_rationale()},
        [compute_density_eligibility(_full_frame(), _config())],
    )
    contexts = {e[0] for e in scope}
    assert any("use_status" in c for c in contexts)
    assert any("config_notes" in c for c in contexts)
    assert any("gr_proxy_confidence_class" in c for c in contexts)
    assert any("mask_name" in c for c in contexts)
    # The manifest additionally folds in its own explanatory statement.
    assert man["lithology_validation"]["scope_object_fields_checked"] > len(scope) - 1


# ---------------------------------------------------------------------------
# Increment 6.1 (Finding 4): Figure 4 totals must match their stated population
# ---------------------------------------------------------------------------

def test_figure4_totals_match_the_qualifying_population_they_state():
    """The number a Figure-4 annotation prints must equal the sum over the
    blocks it says it covers - the qualifying ones - and must exclude the
    rejected sub-threshold blocks drawn separately."""
    from p2mem.io.petrophysics_inventory import build_thickness_sensitivity_rows
    rhob = np.full(40, np.nan)
    rhob[0:25] = 2400.0   # qualifying
    rhob[35:38] = 2400.0  # sub-threshold
    frame = _full_frame(n=40, rhob=rhob)
    cfg = _config(contiguity={"max_gap_samples": 0, "max_gap_depth_m": 0.0,
                              "min_block_samples": 20, "min_block_thickness_m": 5.0})
    ivs = build_eligibility_intervals(frame, compute_density_eligibility(frame, cfg), cfg)
    qual = [iv for iv in ivs if iv.meets_configured_minimums]
    rej = [iv for iv in ivs if not iv.meets_configured_minimums]
    assert len(qual) == 1 and len(rej) == 1

    # What the figure annotates:
    fig_gross = sum(float(iv.gross_thickness_tvdss_m or 0.0) for iv in qual)
    fig_net = sum(float(iv.net_thickness_tvdss_m or 0.0) for iv in qual)
    # What the exported sensitivity table reports for the same population:
    row = build_thickness_sensitivity_rows(ivs)[0]
    assert row["gross_qualifying_thickness_tvdss_m"] == pytest.approx(fig_gross)
    assert row["net_qualifying_thickness_tvdss_m"] == pytest.approx(fig_net)
    assert row["n_blocks_qualifying"] == len(qual)
    assert row["n_blocks_rejected_below_minimums"] == len(rej)
    # And the all-block figure is strictly larger, so the two populations can
    # never be silently interchanged.
    assert row["gross_all_block_thickness_tvdss_m"] > row["gross_qualifying_thickness_tvdss_m"]


# ---------------------------------------------------------------------------
# Increment 6.1.1 (Finding 2): the ACTIVE module documentation must not
# reintroduce the exclusive-boundary wording
# ---------------------------------------------------------------------------

def test_active_module_docstring_uses_corrected_vp_vs_language():
    """The Increment 6.1 implementation fix left the module's own top-level
    docstring saying "inside the Poisson domain (Vp/Vs > sqrt(2), i.e. Poisson
    ratio > 0)". Documentation that contradicts the code it documents is a real
    defect: a reader trusts the docstring. This test pins the corrected
    wording so the exclusive boundary cannot return."""
    import p2mem.method_eligibility as me
    raw = me.__doc__ or ""
    assert raw.strip(), "the module must retain a top-level docstring"
    # Compare on whitespace-normalized text: these assertions are about
    # CONTENT, and a docstring's line wrapping must not decide the outcome.
    doc = " ".join(raw.split())

    forbidden = [
        "inside the Poisson domain",
        "in the Poisson domain",
        "Vp/Vs > sqrt(2)",
        "ratio > sqrt(2)",
        "Poisson ratio > 0",
        "Poisson's ratio is non-negative only when",
    ]
    for phrase in forbidden:
        assert phrase not in doc, (
            f"active module docstring reintroduces stale exclusive-boundary wording: {phrase!r}"
        )

    required = [
        "NON-NEGATIVE-POISSON-RATIO APPLICABILITY SCREEN",
        "Vp/Vs >= sqrt(2)",
    ]
    for phrase in required:
        assert phrase in doc, f"active module docstring is missing {phrase!r}"
    low = doc.lower()
    assert "inclusive" in low
    assert "not a physical-possibility test" in low
    assert "not a boundary of the mathematical poisson domain" in low


def test_active_function_docstring_matches_the_implementation():
    """The mask function's own docstring must agree with the inclusive bound
    it actually implements."""
    from p2mem.method_eligibility import compute_dynamic_elastic_eligibility as f
    doc = " ".join((f.__doc__ or "").split())
    assert "sqrt(2)" in doc and "INCLUSIVE" in doc.upper()
    assert "inside the Poisson domain" not in doc
    assert "nu = 0" in doc or "nu = 0 EXACTLY" in doc


def test_no_active_source_file_reintroduces_the_exclusive_bound():
    """Sweep the packaged, ACTIVE p2mem sources. Historical/superseded records
    and negative assertions are out of scope by construction (this scans
    modules only, not manifests or tests)."""
    import pathlib
    import p2mem
    root = pathlib.Path(p2mem.__file__).resolve().parent
    forbidden = ["inside the Poisson domain", "Vp/Vs > sqrt(2)", "Poisson ratio > 0"]
    offenders = []
    for path in sorted(root.rglob("*.py")):
        text = path.read_text(encoding="utf-8")
        for phrase in forbidden:
            if phrase in text:
                offenders.append(f"{path.name}: {phrase!r}")
    assert not offenders, f"stale exclusive-boundary wording in active source: {offenders}"


# ---------------------------------------------------------------------------
# Increment 6.1.1 (Finding 1): the allowlist cannot hide a geological assertion
# from the DERIVED completion gate
# ---------------------------------------------------------------------------

def test_injected_shale_gas_note_fails_the_derived_gate():
    """Audited bypass case 1, injected into real manifest-building content:
    ("Poseidon_2.config_notes", "This interval contains shale gas.",
    SCOPE_INTERPRETIVE). Before Increment 6.1.1 this produced zero violations
    and the gate passed."""
    dirty = GrFamilyDisposition(
        well_key="W_1", source_las_filename="W_1.las", gr_family_canonical_name="GR_api",
        gr_family_source_curve_name="GR", use_status="screening_proxy_allowed",
        exclusion_reason=None, evidence_class="measured", has_approved_formation_tops=True,
        notes="This interval contains shale gas.",
    )
    man = _manifest_with(disposition=dirty)
    assert man["named_lithology_assigned"] is True
    assert _n_viol(man) > 0
    assert "shale" in man["lithology_validation"]["violations"][0]["terms"]
    assert not _gate_passes(man)


def test_injected_shale_gas_manifest_statement_fails_the_derived_gate():
    """Audited bypass case 2, injected into the manifest's own explanatory
    scope: ("manifest.statement", "Poseidon 2 contains shale gas.",
    SCOPE_EXPLANATORY)."""
    from p2mem.io.petrophysics_inventory import build_lithology_validation_scope
    from p2mem.wellframe_models import (
        SCOPE_EXPLANATORY, validate_no_prohibited_interpretation)
    scope = build_lithology_validation_scope(
        {"W_1": _disposition("W_1")}, {"W_1": "GR_PROXY_INTERMEDIATE"}, {"W_1": "r"},
        [compute_density_eligibility(_full_frame(), _config())],
        extra_entries=[
            ("manifest.statement", "Poseidon 2 contains shale gas.", SCOPE_EXPLANATORY),
        ],
    )
    violations = validate_no_prohibited_interpretation(scope)
    # The manifest derives its flag and its gate from exactly this call.
    named_lithology_assigned = bool(violations)
    assert named_lithology_assigned is True
    assert len(violations) > 0
    assert any(v["context"] == "manifest.statement" for v in violations)
    assert not _gate_passes({
        "named_lithology_assigned": named_lithology_assigned,
        "lithology_validation": {"violations": list(violations)},
    })


def test_clean_manifest_still_passes_after_the_allowlist_correction():
    """The correction must tighten the validator without breaking the real,
    legitimate content the manifest already carries."""
    man = _manifest_with()
    assert _n_viol(man) == 0
    assert man["named_lithology_assigned"] is False
    assert _gate_passes(man)


# ---------------------------------------------------------------------------
# Increment 6.1.1 (Finding 3): explicit, unambiguous interruption counts
# ---------------------------------------------------------------------------

def test_no_gap_block_reports_zero_gaps_and_one_subrun():
    """Identity: no gap -> 0 bridged samples, 0 bridged gaps, 1 eligible
    sub-run."""
    frame, cfg = _full_frame(n=30), _bridging_config()
    ivs = build_eligibility_intervals(frame, compute_density_eligibility(frame, cfg), cfg)
    assert len(ivs) == 1
    iv = ivs[0]
    assert iv.n_bridged_samples == 0
    assert iv.n_bridged_gaps == 0
    assert iv.n_eligible_subruns == 1


def test_two_separate_bridged_gaps_in_one_configured_block():
    """Identity: two distinct bridged gaps -> n_bridged_gaps == 2 and
    n_eligible_subruns == 3, inside a SINGLE configured block."""
    rhob = np.full(30, 2400.0)
    rhob[10] = np.nan          # gap 1: one sample
    rhob[20:22] = np.nan       # gap 2: two samples
    frame = _full_frame(n=30, rhob=rhob)
    cfg = _bridging_config()
    ivs = build_eligibility_intervals(frame, compute_density_eligibility(frame, cfg), cfg)
    assert len(ivs) == 1, "both gaps are within tolerance, so one gross block"
    iv = ivs[0]
    assert iv.n_bridged_samples == 3      # 1 + 2 samples absorbed
    assert iv.n_bridged_gaps == 2         # two DISTINCT False runs
    assert iv.n_eligible_subruns == 3     # 0..9, 11..19, 22..29
    assert iv.n_eligible_samples == 27
    assert iv.n_samples == 30


def test_bridged_gaps_and_subruns_identity_holds_generally():
    """Whenever n_eligible_subruns > 0, n_bridged_gaps == n_eligible_subruns - 1."""
    for gap_positions in ([10], [10, 20], [5, 12, 19, 26]):
        rhob = np.full(32, 2400.0)
        for g in gap_positions:
            rhob[g] = np.nan
        frame = _full_frame(n=32, rhob=rhob)
        cfg = _bridging_config()
        ivs = build_eligibility_intervals(frame, compute_density_eligibility(frame, cfg), cfg)
        assert len(ivs) == 1
        iv = ivs[0]
        assert iv.n_eligible_subruns > 0
        assert iv.n_bridged_gaps == iv.n_eligible_subruns - 1
        assert iv.n_bridged_gaps == len(gap_positions)
        assert iv.n_bridged_samples == len(gap_positions)


def test_one_gap_may_absorb_several_samples_so_gaps_and_samples_differ():
    """Samples and gaps are different magnitudes: a single gap can absorb
    several samples. Reporting only one of them was the ambiguity."""
    rhob = np.full(30, 2400.0)
    rhob[10:12] = np.nan  # ONE gap, TWO samples
    frame = _full_frame(n=30, rhob=rhob)
    cfg = _bridging_config()
    iv = build_eligibility_intervals(frame, compute_density_eligibility(frame, cfg), cfg)[0]
    assert iv.n_bridged_samples == 2
    assert iv.n_bridged_gaps == 1
    assert iv.n_eligible_subruns == 2


def test_inconsistent_interval_record_is_rejected_at_construction():
    """The gap/sub-run relationship is structural; an inconsistent record must
    not be constructible.

    Increment 6.1.2 (Finding 2): all three counts are now supplied, because a
    record that omits one is rejected earlier, by the missing-field rule. The
    mismatch under test here is therefore the ONLY defect in this record."""
    from p2mem.method_eligibility import EligibilityInterval
    with pytest.raises(PetrophysicsInputError, match="inconsistent interval record"):
        EligibilityInterval(well_key="W", mask_name=MASK_DENSITY_FOR_SV,
                            n_bridged_samples=5, n_bridged_gaps=5,
                            n_eligible_subruns=3)


def test_ambiguous_interruption_aliases_are_gone_from_active_exports():
    """No ambiguous alias may survive in an active CSV export."""
    frame, cfg = _bridging_frame(), _bridging_config()
    ivs = build_eligibility_intervals(frame, compute_density_eligibility(frame, cfg), cfg)
    rows = build_eligibility_interval_rows(ivs)
    for r in rows:
        assert "n_interruptions" not in r
        assert "n_interrupted_subruns" not in r
        assert "n_bridged_samples" in r
        assert "n_bridged_gaps" in r
        assert "n_eligible_subruns" in r
    from p2mem.method_eligibility import EligibilityInterval
    assert "n_interruptions" not in EligibilityInterval.__slots__
    assert "n_interrupted_subruns" not in EligibilityInterval.__slots__


def test_sensitivity_row_separates_samples_gaps_and_affected_blocks():
    """thickness_sensitivity_summary must distinguish all three quantities."""
    from p2mem.io.petrophysics_inventory import build_thickness_sensitivity_rows
    rhob = np.full(30, 2400.0)
    rhob[10] = np.nan
    rhob[20:22] = np.nan
    frame = _full_frame(n=30, rhob=rhob)
    cfg = _bridging_config()
    ivs = build_eligibility_intervals(frame, compute_density_eligibility(frame, cfg), cfg)
    r = build_thickness_sensitivity_rows(ivs)[0]
    assert r["n_bridged_samples_in_qualifying_blocks"] == 3
    assert r["n_bridged_gaps_in_qualifying_blocks"] == 2
    assert r["n_interrupted_qualifying_blocks"] == 1   # one block, containing both gaps
    assert "bridged gap(s)" in r["population_statement"]


# ---------------------------------------------------------------------------
# Increment 6.1.2 (Finding 2): the three interruption counts are validated as
# ONE coherent record at construction. Table-driven, both directions.
# ---------------------------------------------------------------------------

def _interval(**counts):
    from p2mem.method_eligibility import EligibilityInterval
    return EligibilityInterval(well_key="W", mask_name=MASK_DENSITY_FOR_SV, **counts)


# (label, kwargs) - every one of these MUST construct.
INTERVAL_VALID_MATRIX = [
    ("no gap: 0 samples / 0 gaps / 1 sub-run",
     dict(n_bridged_samples=0, n_bridged_gaps=0, n_eligible_subruns=1)),
    ("one gap holding two samples: 2 / 1 / 2",
     dict(n_bridged_samples=2, n_bridged_gaps=1, n_eligible_subruns=2)),
    ("two gaps holding three samples: 3 / 2 / 3",
     dict(n_bridged_samples=3, n_bridged_gaps=2, n_eligible_subruns=3)),
    ("one sample per gap is the minimum: 2 / 2 / 3",
     dict(n_bridged_samples=2, n_bridged_gaps=2, n_eligible_subruns=3)),
    ("whole-valued numpy integers are accepted",
     dict(n_bridged_samples=np.int64(2), n_bridged_gaps=np.int64(1),
          n_eligible_subruns=np.int64(2))),
]

# (label, kwargs, expected_message_fragment) - every one MUST raise.
INTERVAL_INVALID_MATRIX = [
    ("negative samples",
     dict(n_bridged_samples=-1, n_bridged_gaps=-1, n_eligible_subruns=0), "negative"),
    ("negative gaps only",
     dict(n_bridged_samples=0, n_bridged_gaps=-1, n_eligible_subruns=1), "negative"),
    ("negative sub-runs",
     dict(n_bridged_samples=0, n_bridged_gaps=0, n_eligible_subruns=-1), "negative"),
    ("zero sub-runs with gaps",
     dict(n_bridged_samples=0, n_bridged_gaps=2, n_eligible_subruns=0),
     "at least one"),
    ("zero sub-runs without gaps",
     dict(n_bridged_samples=0, n_bridged_gaps=0, n_eligible_subruns=0),
     "at least one"),
    ("samples without gaps",
     dict(n_bridged_samples=5, n_bridged_gaps=0, n_eligible_subruns=1), "disagree"),
    ("gaps without samples",
     dict(n_bridged_samples=0, n_bridged_gaps=1, n_eligible_subruns=2), "disagree"),
    ("more gaps than bridged samples",
     dict(n_bridged_samples=1, n_bridged_gaps=2, n_eligible_subruns=3), "fewer than"),
    ("gap/sub-run mismatch",
     dict(n_bridged_samples=2, n_bridged_gaps=1, n_eligible_subruns=3),
     "inconsistent interval record"),
    ("all three missing", dict(), "missing required count"),
    ("samples missing",
     dict(n_bridged_gaps=0, n_eligible_subruns=1), "missing required count"),
    ("gaps missing",
     dict(n_bridged_samples=0, n_eligible_subruns=1), "missing required count"),
    ("sub-runs missing",
     dict(n_bridged_samples=0, n_bridged_gaps=0), "missing required count"),
    ("explicit None",
     dict(n_bridged_samples=None, n_bridged_gaps=0, n_eligible_subruns=1),
     "missing required count"),
    ("boolean counts",
     dict(n_bridged_samples=True, n_bridged_gaps=False, n_eligible_subruns=True),
     "boolean"),
    ("boolean sub-runs only",
     dict(n_bridged_samples=0, n_bridged_gaps=0, n_eligible_subruns=True), "boolean"),
    ("numeric strings",
     dict(n_bridged_samples="0", n_bridged_gaps="0", n_eligible_subruns="1"),
     "integer count is required"),
    ("non-integral float",
     dict(n_bridged_samples=0.0, n_bridged_gaps=0.5, n_eligible_subruns=1.0),
     "is a float"),
    ("fractional sample count",
     dict(n_bridged_samples=2.5, n_bridged_gaps=1, n_eligible_subruns=2),
     "is a float"),
    # Increment 6.1.3: whole-valued floats are rejected too. The contract is an
    # integer count; 0.0 is not 0, and accepting it would make the type gate
    # depend on the value.
    ("whole-valued floats",
     dict(n_bridged_samples=0.0, n_bridged_gaps=0.0, n_eligible_subruns=1.0),
     "is a float"),
    ("NaN",
     dict(n_bridged_samples=float("nan"), n_bridged_gaps=0, n_eligible_subruns=1),
     "not finite"),
    ("positive infinity",
     dict(n_bridged_samples=float("inf"), n_bridged_gaps=1, n_eligible_subruns=2),
     "not finite"),
    ("negative infinity",
     dict(n_bridged_samples=float("-inf"), n_bridged_gaps=0, n_eligible_subruns=1),
     "not finite"),
    ("numpy NaN",
     dict(n_bridged_samples=np.nan, n_bridged_gaps=0, n_eligible_subruns=1),
     "not finite"),
    ("misspelled count keyword",
     dict(n_bridged_samples=0, n_bridged_gaps=0, n_eligible_subruns=1,
          n_bridged_sample=99), "unknown field"),
    ("entirely unknown keyword",
     dict(n_bridged_samples=0, n_bridged_gaps=0, n_eligible_subruns=1, zzz=1),
     "unknown field"),
    ("complex value",
     dict(n_bridged_samples=complex(0, 0), n_bridged_gaps=0, n_eligible_subruns=1),
     "integer count is required"),
    ("legacy alias n_interruptions",
     dict(n_bridged_samples=0, n_bridged_gaps=0, n_eligible_subruns=1,
          n_interruptions=1), "removed in Increment 6.1.1"),
    ("legacy alias n_interrupted_subruns",
     dict(n_bridged_samples=0, n_bridged_gaps=0, n_eligible_subruns=1,
          n_interrupted_subruns=2), "removed in Increment 6.1.1"),
]


@pytest.mark.parametrize(
    "label,counts", INTERVAL_VALID_MATRIX, ids=[r[0] for r in INTERVAL_VALID_MATRIX])
def test_interval_valid_count_records_construct(label, counts):
    iv = _interval(**counts)
    assert iv.n_bridged_gaps == iv.n_eligible_subruns - 1
    assert (iv.n_bridged_samples == 0) == (iv.n_bridged_gaps == 0)
    assert iv.n_bridged_gaps == 0 or iv.n_bridged_samples >= iv.n_bridged_gaps


@pytest.mark.parametrize(
    "label,counts,fragment", INTERVAL_INVALID_MATRIX,
    ids=[r[0] for r in INTERVAL_INVALID_MATRIX])
def test_interval_invalid_count_records_are_rejected(label, counts, fragment):
    with pytest.raises(PetrophysicsInputError, match=fragment):
        _interval(**counts)


def test_the_three_audited_invalid_records_are_now_rejected():
    """The exact records the Increment 6.1.2 audit showed were constructible
    under Increment 6.1.1."""
    audited = [
        dict(n_bridged_samples=5, n_bridged_gaps=0, n_eligible_subruns=1),
        dict(n_bridged_samples=0, n_bridged_gaps=2, n_eligible_subruns=0),
        dict(n_bridged_samples=-1, n_bridged_gaps=-1, n_eligible_subruns=0),
    ]
    for counts in audited:
        with pytest.raises(PetrophysicsInputError):
            _interval(**counts)


def test_every_error_message_names_the_field_or_the_relationship():
    """A validation error that does not say WHAT is wrong is not enforcement."""
    for label, counts, fragment in INTERVAL_INVALID_MATRIX:
        try:
            _interval(**counts)
        except PetrophysicsInputError as exc:
            msg = str(exc)
            assert "W/" in msg and MASK_DENSITY_FOR_SV in msg, label
            assert fragment in msg, f"{label}: {msg}"
        else:  # pragma: no cover - the parametrized test above would fail first
            raise AssertionError(f"{label} was accepted")


def test_real_block_builder_emits_only_valid_records():
    """The invariants must describe what the builder actually produces, not an
    aspiration the builder violates."""
    rhob = np.full(60, 2400.0)
    rhob[10:12] = np.nan
    rhob[30:31] = np.nan
    frame = _full_frame(n=60, rhob=rhob)
    cfg = _bridging_config()
    ivs = build_eligibility_intervals(frame, compute_density_eligibility(frame, cfg), cfg)
    assert ivs
    for iv in ivs:
        assert iv.n_eligible_subruns >= 1
        assert iv.n_bridged_gaps == iv.n_eligible_subruns - 1
        assert (iv.n_bridged_samples == 0) == (iv.n_bridged_gaps == 0)
        if iv.n_bridged_gaps:
            assert iv.n_bridged_samples >= iv.n_bridged_gaps


# ---------------------------------------------------------------------------
# Increment 6.1.2 (Finding 1): every audited case, driven through the REAL
# manifest-building path and the DERIVED completion gate - not a helper.
# ---------------------------------------------------------------------------

def _disposition_with_note(note):
    return GrFamilyDisposition(
        well_key="W_1", source_las_filename="W_1.las", gr_family_canonical_name="GR_api",
        gr_family_source_curve_name="GR", use_status="screening_proxy_allowed",
        exclusion_reason=None, evidence_class="measured", has_approved_formation_tops=True,
        notes=note,
    )


GATE_INJECTIONS_THAT_MUST_FAIL = [
    "Poseidon 2's shale volume is high.",
    "Poseidon 2's shale volume is 70 percent.",
    "The interval's shale volume is high.",
    "The interval's shale volume exceeds 60 percent.",
    "This interval contains shale gas.",
    "This interval has a high shale volume.",
]

# Increment 6.1.3 CONTRACT CHANGE: method wording in a per-well note is no
# longer permitted either. A per-well note is project-specific prose and has
# zero allowance; the project states its boundaries through registered METHOD
# statements instead. Asserted here so the change can never happen silently.
GATE_NOTES_THAT_MUST_ALSO_FAIL = [
    "This method contains a shale proxy calculation.",
    "The analysis shows no shale volume was computed.",
    "Excluded from every shale-proxy calculation; this is not a calibrated shale volume.",
]

# Increment 6.1.5: a note passes ONLY when it is a registered interpretive
# statement. "Sensible-looking" prose is no longer a category - the previous
# entries here were plausible sentences nobody had registered.
def _all_registered_interpretive_notes():
    from p2mem.wellframe_models import REGISTERED_STATEMENTS, SCOPE_INTERPRETIVE
    return sorted(s.text for s in REGISTERED_STATEMENTS.values()
                  if s.scope == SCOPE_INTERPRETIVE)


GATE_NOTES_THAT_MUST_PASS = _all_registered_interpretive_notes()

# Plausible, innocent, unregistered prose. Each must FAIL, which is what
# distinguishes positive authorization from semantic detection.
GATE_NOTES_UNREGISTERED_BUT_INNOCENT = [
    "The screening proxy is dimensionless and uncalibrated.",
    "This well has no approved formation tops, so results are depth-tied.",
    "Everything looks fine.",
]


@pytest.mark.parametrize("note", GATE_NOTES_UNREGISTERED_BUT_INNOCENT)
def test_innocent_unregistered_note_still_fails_the_derived_gate(note):
    """No prohibited term appears in any of these. They fail because nothing
    authorized them."""
    from p2mem.wellframe_models import find_prohibited_lithology_terms
    assert find_prohibited_lithology_terms(note) == (), note
    man = _manifest_with(disposition=_disposition_with_note(note))
    assert man["named_lithology_assigned"] is True, note
    assert _n_viol(man) > 0, note
    assert not _gate_passes(man), note


@pytest.mark.parametrize("note", GATE_INJECTIONS_THAT_MUST_FAIL
                         + GATE_NOTES_THAT_MUST_ALSO_FAIL)
def test_injected_assertion_fails_the_derived_gate(note):
    """The full derived path: real manifest builder -> real validation ->
    `named_lithology_assigned` -> completion gate."""
    man = _manifest_with(disposition=_disposition_with_note(note))
    assert man["named_lithology_assigned"] is True, note
    assert _n_viol(man) > 0, note
    assert not _gate_passes(man), note


@pytest.mark.parametrize("note", GATE_NOTES_THAT_MUST_PASS)
def test_legitimate_note_keeps_the_derived_gate_passing(note):
    """Notes that carry no rock name at all - which is what a per-well note
    must now look like - keep the gate passing."""
    man = _manifest_with(disposition=_disposition_with_note(note))
    assert man["named_lithology_assigned"] is False, note
    assert _n_viol(man) == 0, note
    assert _gate_passes(man), note


def test_registered_method_and_explanatory_statements_are_in_scope_and_pass():
    """Every registered method and explanatory statement is emitted by the real
    scope builder WITH its authorization, and all of them pass."""
    from p2mem.io.petrophysics_inventory import build_lithology_validation_scope
    from p2mem.wellframe_models import (
        REGISTERED_STATEMENTS, SCOPE_METHOD, SCOPE_EXPLANATORY,
        validate_no_prohibited_interpretation)
    scope = build_lithology_validation_scope(
        {"W_1": _clean_disposition()}, {"W_1": "GR_PROXY_INTERMEDIATE"},
        {"W_1": _registered_rationale()},
        [compute_density_eligibility(_full_frame(), _config())],
    )
    controlled = [e for e in scope if e[2] in (SCOPE_METHOD, SCOPE_EXPLANATORY)]
    expected = [s for s in REGISTERED_STATEMENTS.values()
                if s.scope in (SCOPE_METHOD, SCOPE_EXPLANATORY)]
    assert len(controlled) == len(expected)
    assert all(len(e) == 4 for e in controlled), "each must carry an Authorization"
    assert validate_no_prohibited_interpretation(controlled) == ()


def test_an_unregistered_method_statement_fails_the_derived_gate():
    """If a future edit changes a persisted method string without registering
    it, the gate must fail rather than silently ship the new wording."""
    from p2mem.io.petrophysics_inventory import build_lithology_validation_scope
    from p2mem.wellframe_models import (
        SCOPE_METHOD, validate_no_prohibited_interpretation)
    scope = build_lithology_validation_scope(
        {"W_1": _clean_disposition()}, {"W_1": "GR_PROXY_INTERMEDIATE"},
        {"W_1": _registered_rationale()},
        [compute_density_eligibility(_full_frame(), _config())],
        extra_entries=[
            ("proxy.limitations", "The proxy is not a calibrated shale volume.",
             SCOPE_METHOD),
        ],
    )
    violations = validate_no_prohibited_interpretation(scope)
    assert violations
    assert not _gate_passes({
        "named_lithology_assigned": bool(violations),
        "lithology_validation": {"violations": list(violations)},
    })


def test_explanatory_assertion_in_the_manifest_scope_fails_the_gate():
    """The manifest's own explanatory statement is inside the validated scope;
    a project-well assertion there must fail the derived gate."""
    from p2mem.io.petrophysics_inventory import build_lithology_validation_scope
    from p2mem.wellframe_models import (
        SCOPE_EXPLANATORY, validate_no_prohibited_interpretation)
    scope = build_lithology_validation_scope(
        {"W_1": _disposition("W_1")}, {"W_1": "GR_PROXY_INTERMEDIATE"}, {"W_1": "r"},
        [compute_density_eligibility(_full_frame(), _config())],
        extra_entries=[
            ("manifest.statement", "Poseidon 2's shale volume is high.", SCOPE_EXPLANATORY),
        ],
    )
    violations = validate_no_prohibited_interpretation(scope)
    assert violations and any(v["context"] == "manifest.statement" for v in violations)
    assert not _gate_passes({
        "named_lithology_assigned": bool(violations),
        "lithology_validation": {"violations": list(violations)},
    })


def test_unchanged_real_packaged_content_keeps_the_gate_passing():
    """The clean baseline must be unaffected by both corrections."""
    man = _manifest_with()
    assert man["lithology_validation"]["scope_object_fields_checked"] > 0
    assert _n_viol(man) == 0
    assert man["named_lithology_assigned"] is False
    assert _gate_passes(man)


# ---------------------------------------------------------------------------
# Increment 6.1.6: EMISSION-BOUNDARY authorization.
#
# Increment 6.1.5 closed the unit validator and left the export path open: a
# GrEndpointScenario carrying description="The interval is chalk." was persisted
# verbatim while never entering the 143-field scope. These tests run against the
# OUTPUT BUILDERS and the COMPLETE EXPORT, not against
# validate_no_prohibited_interpretation().
# ---------------------------------------------------------------------------

import copy as _copy
import json as _json
import pathlib as _pathlib

from p2mem.io import output_policy as _op
from p2mem.io.output_policy import (
    CSV_SCHEMAS, OUTPUT_FIELD_POLICY, OUTPUT_STATEMENTS, OUTPUT_TEMPLATES,
    OUTPUT_ARTIFACTS, OutputAuthorizationError, canonicalize_emitted_records,
    export_authorized_outputs, validate_emitted_records,
)

_PROJECT_ROOT = _pathlib.Path(__file__).resolve().parents[1]
_OUT = _PROJECT_ROOT / "outputs" / "06_petrophysics_eligibility"


def _real_payloads():
    """The ACTUAL packaged records when they are present, otherwise the SAME
    records produced by the SAME builders from the synthetic bundle.

    The notebook runs pytest before it writes any output, so the packaged
    artifacts do not exist at that point. Falling back to builder-produced
    records keeps every emission-boundary test genuinely executing there rather
    than silently skipping - the objects under test are real builder output
    either way.
    """
    import csv as _csv
    manifest_path = _OUT / "petrophysics_eligibility_manifest.json"
    if manifest_path.exists():
        payloads = {}
        for p in sorted(_OUT.glob("*.csv")):
            with open(p, newline="", encoding="utf-8") as fh:
                payloads[p.name] = list(_csv.DictReader(fh))
        payloads["petrophysics_eligibility_manifest.json"] = _json.loads(
            manifest_path.read_text(encoding="utf-8"))
        return payloads
    return _builder_payloads()


def _builder_payloads():
    """Every artifact, built by the REAL builders using APPROVED identifiers.

    The identifiers must be approved ones because a well key is itself a
    policy-classified emitted field - which is the point of the policy.
    """
    from p2mem.petrophysics_models import GrFamilyDisposition
    frames, stats, disp, conf, rationale, scen, proxies, masks = {}, {}, {}, {}, {}, {}, {}, []
    for wk in ("Poseidon_2", "Proteus_1ST2"):
        fr = _full_frame(n=20, gr=np.linspace(10.0, 110.0, 20), well_key=wk)
        frames[wk] = fr
        d = GrFamilyDisposition(
            well_key=wk, source_las_filename=f"{wk}_logs.las",
            gr_family_canonical_name="GR_api", gr_family_source_curve_name="GR",
            use_status="screening_proxy_allowed", exclusion_reason=None,
            evidence_class="measured", has_approved_formation_tops=True,
            notes=_registered_note())
        disp[wk] = d
        stats[wk] = compute_gr_family_qc_stats(fr, d)
        conf[wk] = "GR_PROXY_INTERMEDIATE"
        rationale[wk] = _registered_rationale()
        sc = _scenario(well_key=wk, name="base")
        scen[wk] = [sc]
        proxies[wk] = [compute_gr_proxy(fr, d, sc, _config())]
        for m in (compute_density_eligibility(fr, _config()),
                  compute_dynamic_elastic_eligibility(fr, _config())):
            masks.append(_output_registered_mask(m))
    intervals = []
    for m in masks:
        intervals.extend(build_eligibility_intervals(frames[m.well_key], m, _config()))
    payloads = {
        "gr_family_qc_summary.csv": build_gr_family_qc_rows(stats, disp, conf),
        "gr_endpoint_scenarios.csv": build_gr_endpoint_scenario_rows(scen),
        "gr_proxy_sensitivity_summary.csv": build_gr_proxy_sensitivity_rows(proxies, disp),
        "method_eligibility_summary.csv": build_method_eligibility_rows(masks, frames, disp),
        "eligibility_interval_register.csv": build_eligibility_interval_rows(intervals),
        "thickness_sensitivity_summary.csv": build_thickness_sensitivity_rows(intervals),
        "petrophysics_eligibility_issues.csv": build_petrophysics_issue_rows([], {}),
    }
    payloads["petrophysics_eligibility_manifest.json"] = build_petrophysics_manifest(
        frames, disp, stats, conf, rationale, scen, masks, {}, [],
        config_filename="petrophysics_eligibility.yml", config_schema_version="6.0",
        nct_candidate_thresholds=[0.5], output_payloads=payloads)
    return payloads


def _well_keys(payloads):
    return set(payloads["petrophysics_eligibility_manifest.json"]["wells"])


NOVEL_TOKENS = ["qxzite", "zzqqworp", "vundrelic", "morbaquin", "tesqualor"]


# --- 1-3. the builder must refuse an unauthorized description ---------------

@pytest.mark.parametrize("description", [
    "The interval is chalk.",                     # unknown lithology (test 1)
    "Everything looks fine.",                      # innocent prose      (test 2)
    "The interval is qxzite.",                     # generated novel     (test 3)
    "zzqqworp", "vundrelic sands of the upper member",
])
def test_endpoint_row_builder_rejects_unauthorized_description(description):
    sc = _scenario_for_export(description)
    with pytest.raises(PetrophysicsInputError, match="not authorized for emission"):
        build_gr_endpoint_scenario_rows({"Poseidon_2": [sc]})


def test_rejection_does_not_depend_on_lithology_recognition():
    """The decisive framing: the linter sees nothing in any of these."""
    from p2mem.wellframe_models import find_prohibited_lithology_terms
    for d in ["Everything looks fine.", "The interval is qxzite.", "zzqqworp"]:
        assert find_prohibited_lithology_terms(d) == (), d
        with pytest.raises(PetrophysicsInputError):
            build_gr_endpoint_scenario_rows({"Poseidon_2": [_scenario_for_export(d)]})


def _scenario_for_export(description):
    from p2mem.petrophysics_models import GrEndpointScenario
    return GrEndpointScenario(
        well_key="Poseidon_2", scenario_name="base", low_percentile=5.0,
        high_percentile=95.0, gr_low_endpoint_api=10.0, gr_high_endpoint_api=110.0,
        endpoint_separation_api=100.0, n_samples_used_for_endpoints=100,
        endpoint_sample_basis="finite_and_depth_mapped_samples_only",
        description=description, evidence_class="assumed_configured",
        calibration_status="uncalibrated_assumed_no_calibration_data_exists")


def test_a_registered_endpoint_description_is_accepted():
    rows = build_gr_endpoint_scenario_rows(
        {"Poseidon_2": [_scenario_for_export(_registered_endpoint_description())]})
    assert rows and rows[0]["description"] == _registered_endpoint_description()


# --- 4. mutate every controlled prose field of every builder ---------------

def _controlled_fields():
    return sorted(
        (k for k, p in OUTPUT_FIELD_POLICY.items()
         if p.category in ("registered_statement", "controlled_template")),
        key=lambda k: (k[0], k[1]))


@pytest.mark.parametrize("artifact,field", _controlled_fields())
def test_mutating_any_controlled_prose_field_is_rejected(artifact, field):
    """Requirement 4: mutate every controlled prose field produced by every
    output builder and prove the emitted artifact is rejected."""
    payloads = _real_payloads()
    wk = _well_keys(payloads)
    mutated, done = _copy.deepcopy(payloads), False
    if artifact.endswith(".json"):
        done = _mutate_json_path(mutated[artifact], field, wk)
    else:
        for row in mutated[artifact]:
            if isinstance(row.get(field), str) and row[field]:
                row[field] = row[field] + " Additional clause."
                done = True
                break
    if not done:
        pytest.skip(f"{artifact}:{field} has no occurrence in the packaged records")
    rep = validate_emitted_records(mutated, well_keys=wk)
    assert not rep.ok and rep.n_unauthorized_controlled_fields >= 1, (artifact, field)


def _mutate_json_path(node, target, well_keys, prefix="", value=None):
    """Replace the first occurrence of `target` with `value`, or - when no
    value is given - with the original text plus an extra clause."""
    if isinstance(node, dict):
        for k, v in node.items():
            if isinstance(v, str) and v and _op._normalize_json_path(
                    f"{prefix}/{k}", well_keys) == target:
                node[k] = v + " Additional clause." if value is None else value
                return True
            if _mutate_json_path(v, target, well_keys, f"{prefix}/{k}", value):
                return True
    elif isinstance(node, list):
        for i, v in enumerate(node):
            if isinstance(v, str) and v and _op._normalize_json_path(
                    f"{prefix}[{i}]", well_keys) == target:
                node[i] = v + " Additional clause." if value is None else value
                return True
            if _mutate_json_path(v, target, well_keys, f"{prefix}[{i}]", value):
                return True
    return False


# --- 5. cross-field label substitution -------------------------------------

def test_every_cross_field_label_substitution_is_rejected():
    """Requirement 5: replace each typed label with a valid label from a
    DIFFERENT field_kind; every substitution must fail."""
    from p2mem.wellframe_models import APPROVED_LABELS
    payloads = _real_payloads()
    wk = _well_keys(payloads)
    tried = 0
    for (artifact, field), pol in sorted(OUTPUT_FIELD_POLICY.items()):
        if pol.category != "typed_label" or artifact.endswith(".json"):
            continue
        others = [v for k, b in APPROVED_LABELS.items() if k != pol.field_kind
                  for v in b if v not in APPROVED_LABELS[pol.field_kind]]
        if not others:
            continue
        mutated = _copy.deepcopy(payloads)
        hit = False
        for row in mutated[artifact]:
            if isinstance(row.get(field), str) and row[field]:
                row[field] = others[0]
                hit = True
                break
        if not hit:
            continue
        tried += 1
        rep = validate_emitted_records(mutated, well_keys=wk)
        assert rep.n_field_kind_mismatches >= 1, (artifact, field, others[0])
    assert tried >= 5, f"cross-field substitution must be substantive, tried {tried}"


# --- 6-8. coverage must fail closed ----------------------------------------

def test_an_unknown_output_column_fails_coverage():
    payloads = _real_payloads()
    payloads["gr_family_qc_summary.csv"][0]["editorial_note"] = "Everything looks fine."
    rep = validate_emitted_records(payloads, well_keys=_well_keys(payloads))
    assert not rep.ok and rep.n_schema_violations >= 1


def test_an_unknown_json_path_fails_coverage():
    payloads = _real_payloads()
    payloads["petrophysics_eligibility_manifest.json"]["editorial_note"] = "zzqqworp"
    rep = validate_emitted_records(payloads, well_keys=_well_keys(payloads))
    assert not rep.ok and rep.n_schema_violations >= 1


def test_an_unknown_artifact_fails_coverage():
    payloads = _real_payloads()
    payloads["surprise_extra_output.csv"] = [{"note": "Everything looks fine."}]
    rep = validate_emitted_records(payloads, well_keys=_well_keys(payloads))
    assert not rep.ok


def test_removing_a_required_policy_entry_fails_the_gate(monkeypatch):
    """Requirement 8: remove a required field-policy entry; the gate must fail."""
    payloads = _real_payloads()
    reduced = dict(OUTPUT_FIELD_POLICY)
    victim = ("gr_family_qc_summary.csv", "limitations")
    assert victim in reduced
    del reduced[victim]
    monkeypatch.setattr(_op, "OUTPUT_FIELD_POLICY", reduced)
    rep = validate_emitted_records(payloads, well_keys=_well_keys(payloads))
    assert not rep.ok and rep.n_unclassified_fields >= 1


def test_a_missing_declared_artifact_fails_the_gate():
    payloads = _real_payloads()
    payloads.pop("thickness_sensitivity_summary.csv")
    rep = validate_emitted_records(payloads, well_keys=_well_keys(payloads))
    assert not rep.ok


# --- 9-10. the manifest statement is canonical, and immutable after auth ----

def test_manifest_uses_the_canonical_registered_statement(monkeypatch):
    """Requirement 9: change the registered text and prove the manifest follows
    it rather than a duplicated literal."""
    import p2mem.wellframe_models as wm
    original = wm.REGISTERED_STATEMENTS["named_lithology_statement"]
    altered = wm.RegisteredStatement(
        statement_id=original.statement_id, scope=original.scope,
        text="CANONICAL SENTINEL VALUE FOR THIS TEST.",
        purpose=original.purpose, provenance=original.provenance)
    monkeypatch.setitem(wm.REGISTERED_STATEMENTS, "named_lithology_statement", altered)
    man = _manifest_with()
    assert man["named_lithology_statement"] == "CANONICAL SENTINEL VALUE FOR THIS TEST."


def test_post_authorization_mutation_of_the_manifest_statement_is_caught(tmp_path):
    """Requirement 10: change the final emitted statement AFTER authorization and
    prove validation fails."""
    payloads = _builder_payloads()
    wk = _well_keys(payloads)
    assert validate_emitted_records(payloads, well_keys=wk).ok

    def _tampering_writer(target, payload):
        if str(target).endswith(".json"):
            tampered = _copy.deepcopy(payload)
            tampered["named_lithology_statement"] = "The interval is chalk."
            target.write_text(_json.dumps(tampered, indent=2, sort_keys=True) + "\n",
                              encoding="utf-8")
        else:
            import csv as _csv
            with open(target, "w", encoding="utf-8", newline="") as fh:
                if payload:
                    w = _csv.DictWriter(fh, fieldnames=list(payload[0].keys()))
                    w.writeheader()
                    w.writerows(payload)

    with pytest.raises(OutputAuthorizationError, match="post-serialization"):
        export_authorized_outputs(tmp_path, payloads, well_keys=wk,
                                  writer=_tampering_writer)


def test_export_refuses_to_write_anything_when_a_field_is_unauthorized(tmp_path):
    payloads = _builder_payloads()
    payloads["gr_endpoint_scenarios.csv"][0]["description"] = "The interval is chalk."
    with pytest.raises(OutputAuthorizationError, match="refusing to write"):
        export_authorized_outputs(tmp_path, payloads, well_keys=_well_keys(payloads))
    assert not list(tmp_path.glob("*")), "nothing may be written when authorization fails"


# --- 11-12. coverage and occurrence counting -------------------------------

def test_every_emitted_string_field_has_exactly_one_policy_classification():
    """Requirement 11."""
    payloads = _real_payloads()
    wk = _well_keys(payloads)
    seen = set()
    for artifact, payload in payloads.items():
        for occ in _op.collect_string_fields(artifact, payload, wk):
            key = (occ.artifact, occ.field)
            assert key in OUTPUT_FIELD_POLICY, f"unclassified {key}"
            seen.add(key)
    assert len(seen) >= 40
    # exactly one: the mapping is a dict, and duplicates raise at import
    assert len(OUTPUT_FIELD_POLICY) == len(set(OUTPUT_FIELD_POLICY))


def test_every_occurrence_is_validated_not_merely_each_registry_entry():
    """Requirement 12: occurrences, not distinct values."""
    payloads = _real_payloads()
    rep = validate_emitted_records(payloads, well_keys=_well_keys(payloads))
    assert rep.ok
    assert rep.n_string_field_occurrences > 200, rep.n_string_field_occurrences
    assert rep.n_controlled_occurrences > 50
    # far more occurrences than distinct registry entries - proving occurrences
    assert rep.n_controlled_occurrences > 3 * rep.distinct_statements_used


def test_coverage_counters_are_internally_consistent():
    payloads = _real_payloads()
    rep = validate_emitted_records(payloads, well_keys=_well_keys(payloads))
    assert (rep.n_controlled_occurrences + rep.n_structural_occurrences
            + rep.n_unguaranteed_occurrences) == rep.n_string_field_occurrences
    assert rep.n_unclassified_fields == 0
    assert rep.n_unauthorized_controlled_fields == 0
    assert rep.n_field_kind_mismatches == 0


# --- 13. the derived gate moves with the emitted records -------------------

def test_gate_fails_when_a_final_emitted_controlled_field_is_unauthorized():
    """Requirement 13."""
    payloads = {k: v for k, v in _real_payloads().items() if k.endswith(".csv")}
    payloads["gr_endpoint_scenarios.csv"][0]["description"] = "The interval is chalk."
    man = _manifest_with_output_payloads(payloads)
    assert man["named_lithology_assigned"] is True
    assert len(man["lithology_validation"]["violations"]) >= 1
    assert not _gate_passes(man)


def _manifest_with_output_payloads(payloads):
    d = _clean_disposition()
    frame = _full_frame(n=20, gr=np.linspace(10.0, 110.0, 20), well_key="W_1")
    stats = compute_gr_family_qc_stats(frame, d)
    mask = _registered_mask(compute_density_eligibility(frame, _config()))
    return build_petrophysics_manifest(
        {"W_1": frame}, {"W_1": d}, {"W_1": stats}, {"W_1": "GR_PROXY_INTERMEDIATE"},
        {"W_1": _registered_rationale()}, {"W_1": []}, [mask], {}, [],
        config_filename="c.yml", config_schema_version="6.0",
        nct_candidate_thresholds=[0.5], output_payloads=payloads)


def test_the_real_packaged_records_keep_the_gate_passing():
    payloads = {k: v for k, v in _real_payloads().items() if k.endswith(".csv")}
    man = _manifest_with_output_payloads(payloads)
    cov = man["lithology_validation"]["emitted_field_coverage"]
    assert cov["n_unclassified_fields"] == 0
    assert cov["n_unauthorized_controlled_fields"] == 0
    assert cov["n_field_kind_mismatches"] == 0
    assert man["named_lithology_assigned"] is False and _gate_passes(man)


def test_the_manifest_no_longer_reports_a_count_that_overstates_what_it_checked():
    """Requirement 6: `n_fields_checked=143` is gone; the counts say what they
    count, and the emitted coverage is reported separately."""
    payloads = {k: v for k, v in _real_payloads().items() if k.endswith(".csv")}
    lv = _manifest_with_output_payloads(payloads)["lithology_validation"]
    assert "n_fields_checked" not in lv
    assert lv["scope_object_fields_checked"] > 0
    assert lv["emitted_field_coverage"]["n_string_field_occurrences"] > 100
    assert lv["model"] == (
        "schema_driven_transactional_authorization_at_emission_boundary")


def test_the_derivation_describes_the_emission_boundary_model():
    """Finding 4: the stale 6.1.5 wording must be gone from the live builder."""
    lv = _manifest_with()["lithology_validation"]
    assert "Interpretive fields admit no prohibited term at all" not in lv["derivation"]
    assert "EMISSION BOUNDARY" in lv["derivation"]
    assert lv["derivation"] == OUTPUT_STATEMENTS["lithology_validation_derivation"].text


def test_no_duplicated_manifest_statement_literal_remains():
    """Finding 5: one source of truth."""
    import inspect
    import p2mem.io.petrophysics_inventory as inv
    src = inspect.getsource(inv.build_petrophysics_manifest)
    assert "NO named lithology is assigned anywhere in Increment 6." not in src
    assert 'REGISTERED_STATEMENTS["named_lithology_statement"].text' in src


# ---------------------------------------------------------------------------
# The diagnostic contract, tested SEPARATELY from the controlled guarantee
# ---------------------------------------------------------------------------
# `sanitized_diagnostic` is the one category whose content originates outside
# this package (caught exception text, file-level QC messages). It therefore
# cannot be enumerated in advance and is deliberately NOT covered by the
# controlled-interpretation guarantee. What it does carry is an explicit
# safety/provenance contract, and that contract is what these tests exercise.


def _sanitized_field():
    """A real (artifact, field) pair whose declared category is sanitized."""
    for key, pol in sorted(OUTPUT_FIELD_POLICY.items()):
        if pol.category == "sanitized_diagnostic":
            return key
    raise AssertionError("no sanitized_diagnostic field is declared")


def test_the_sanitized_diagnostic_contract_bounds_length():
    """Contract clause 1: bounded length."""
    from p2mem.io.output_policy import (
        FieldOccurrence, authorize_occurrence, SANITIZED_DIAGNOSTIC_MAX_LEN)
    artifact, field = _sanitized_field()
    at_limit = "a" * SANITIZED_DIAGNOSTIC_MAX_LEN
    over = "a" * (SANITIZED_DIAGNOSTIC_MAX_LEN + 1)
    ok_at, _, _ = authorize_occurrence(
        FieldOccurrence(artifact, field, at_limit, "probe"))
    ok_over, _, why = authorize_occurrence(
        FieldOccurrence(artifact, field, over, "probe"))
    assert ok_at is True
    assert ok_over is False and "length" in why


def test_the_sanitized_diagnostic_contract_rejects_control_and_path_sequences():
    """Contract clause 2: no newline, carriage return, tab, backslash or
    URL-ish scheme separator may survive into a written diagnostic field."""
    from p2mem.io.output_policy import FieldOccurrence, authorize_occurrence
    artifact, field = _sanitized_field()
    for bad in ("line one\nline two", "carriage\rreturn", "tab\there",
                "C:\\Users\\someone\\secret.las", "see https://example.invalid/x",
                "file:///home/someone/private"):
        ok, _, why = authorize_occurrence(
            FieldOccurrence(artifact, field, bad, "probe"))
        assert ok is False, f"{bad!r} must be refused by the diagnostic contract"
        assert why


def test_the_sanitized_diagnostic_contract_restricts_the_charset():
    """Contract clause 3: printable-ASCII subset only."""
    from p2mem.io.output_policy import FieldOccurrence, authorize_occurrence
    artifact, field = _sanitized_field()
    for bad in ("curve \u2013 missing", "temperature 25\u00b0C", "\u0000null"):
        ok, _, why = authorize_occurrence(
            FieldOccurrence(artifact, field, bad, "probe"))
        assert ok is False and "charset" in why


def test_a_sanitized_diagnostic_is_not_counted_as_a_controlled_field():
    """The claim boundary itself: a diagnostic field that PASSES its contract
    must still be counted OUTSIDE the controlled-interpretation guarantee.

    This is the test that would fail if the sanitized category were ever
    quietly folded into the controlled bucket to make a coverage number
    look better."""
    from p2mem.io.output_policy import (
        CONTROLLED_CATEGORIES, UNGUARANTEED_CATEGORIES,
        FieldOccurrence, authorize_occurrence)
    artifact, field = _sanitized_field()
    assert "sanitized_diagnostic" in UNGUARANTEED_CATEGORIES
    assert "sanitized_diagnostic" not in CONTROLLED_CATEGORIES
    assert "structured_diagnostic" in UNGUARANTEED_CATEGORIES
    assert "structured_diagnostic" not in CONTROLLED_CATEGORIES

    # An arbitrary sentence PASSES the sanitized contract - by design.
    prose = "The interval is chalk and the operator was competent."
    ok, _auth, _why = authorize_occurrence(
        FieldOccurrence(artifact, field, prose, "probe"))
    assert ok is True, ("the diagnostic contract is a safety contract, not an "
                        "interpretation guarantee - this must pass")

    # ... and passing it must move the counter that is EXCLUDED from the
    # guarantee, never the controlled counter.
    payloads = _real_payloads()
    wk = _well_keys(payloads)
    base = validate_emitted_records(payloads, well_keys=wk)
    mutated = _copy.deepcopy(payloads)
    done = False
    if artifact.endswith(".json"):
        done = _mutate_json_path(mutated[artifact], field, wk, value=prose)
    else:
        for row in mutated[artifact]:
            if isinstance(row.get(field), str) and row[field]:
                row[field] = prose
                done = True
                break
    if done:
        after = validate_emitted_records(mutated, well_keys=wk)
        assert after.ok is True, (
            "replacing a diagnostic value with prose must NOT fail the gate - "
            "the diagnostic contract makes no interpretation claim")
        assert after.n_controlled_occurrences == base.n_controlled_occurrences
        assert after.n_unguaranteed_occurrences == base.n_unguaranteed_occurrences


def test_the_structured_diagnostic_grammar_admits_no_sentence():
    """The other unguaranteed category is far tighter: it is a machine grammar
    and cannot express prose at all, which is why it needs no length bound."""
    from p2mem.io.output_policy import FieldOccurrence, authorize_occurrence
    artifact, field = next(
        k for k, p in sorted(OUTPUT_FIELD_POLICY.items())
        if p.category == "structured_diagnostic")
    for prose in ("The interval is chalk.",
                  "Everything looks fine.",
                  "gaps=3; the operator was competent",
                  "The interval is qxzite."):
        ok, _, why = authorize_occurrence(
            FieldOccurrence(artifact, field, prose, "probe"))
        assert ok is False, f"{prose!r} must not parse as a structured diagnostic"
        assert why


# ---------------------------------------------------------------------------
# Increment 6.1.7: closed schemas and transactional round-trip integrity.
# ---------------------------------------------------------------------------

def _write_payload_for_gate(target, payload, mutate=None):
    """Test serializer with the production schema's exact column order."""
    import csv as _csv
    value = _copy.deepcopy(payload)
    if mutate is not None:
        value = mutate(target.name, value)
    if target.suffix == ".json":
        target.write_text(_json.dumps(value, indent=2, sort_keys=True) + "\n",
                          encoding="utf-8")
    else:
        with open(target, "w", encoding="utf-8", newline="") as fh:
            writer = _csv.DictWriter(fh, fieldnames=list(CSV_SCHEMAS[target.name].columns))
            writer.writeheader()
            writer.writerows(value)


def test_removing_a_required_column_from_every_row_fails_schema():
    payloads = _real_payloads()
    for row in payloads["gr_endpoint_scenarios.csv"]:
        row.pop("description")
    rep = validate_emitted_records(payloads, well_keys=_well_keys(payloads))
    assert not rep.ok and rep.n_schema_violations >= 1


@pytest.mark.parametrize("bad", ["", None, 123, "123"])
def test_controlled_csv_field_cannot_disappear_by_value_or_type(bad):
    payloads = _real_payloads()
    payloads["gr_endpoint_scenarios.csv"][0]["description"] = bad
    rep = validate_emitted_records(payloads, well_keys=_well_keys(payloads))
    assert not rep.ok
    assert rep.n_schema_violations + rep.n_unauthorized_controlled_fields >= 1


@pytest.mark.parametrize("bad", ["", "123", None])
def test_unknown_csv_column_fails_independently_of_its_value(bad):
    payloads = _real_payloads()
    for row in payloads["gr_endpoint_scenarios.csv"]:
        row["rogue_column"] = bad
    rep = validate_emitted_records(payloads, well_keys=_well_keys(payloads))
    assert not rep.ok and rep.n_schema_violations >= 1


@pytest.mark.parametrize("bad", ["", 123, None])
def test_unknown_json_path_fails_independently_of_its_value(bad):
    payloads = _real_payloads()
    payloads["petrophysics_eligibility_manifest.json"]["rogue_path"] = bad
    rep = validate_emitted_records(payloads, well_keys=_well_keys(payloads))
    assert not rep.ok and rep.n_schema_violations >= 1


@pytest.mark.parametrize("bad", ["", None, 123, "123"])
def test_controlled_json_statement_requires_nonempty_registered_text(bad):
    payloads = _real_payloads()
    payloads["petrophysics_eligibility_manifest.json"][
        "named_lithology_statement"] = bad
    rep = validate_emitted_records(payloads, well_keys=_well_keys(payloads))
    assert not rep.ok
    assert rep.n_schema_violations + rep.n_unauthorized_controlled_fields >= 1


def test_authorized_to_authorized_writer_mutation_is_detected_transactionally(tmp_path):
    payloads = _builder_payloads()
    wk = _well_keys(payloads)
    target = tmp_path / "gr_endpoint_scenarios.csv"
    sentinel = b"approved baseline bytes\n"
    target.write_bytes(sentinel)
    replacement = OUTPUT_STATEMENTS["gr_endpoint_scenarios_description_2"].text

    def mutate(name, value):
        if name == "gr_endpoint_scenarios.csv":
            value[0]["description"] = replacement
        return value

    def writer(path, value):
        _write_payload_for_gate(path, value, mutate=mutate)

    with pytest.raises(OutputAuthorizationError, match="canonical record mismatch"):
        export_authorized_outputs(tmp_path, payloads, well_keys=wk, writer=writer)
    assert target.read_bytes() == sentinel
    assert {p.name for p in tmp_path.iterdir()} == {target.name}


def test_unauthorized_post_serialization_mutation_leaves_destination_unchanged(tmp_path):
    payloads = _builder_payloads()
    wk = _well_keys(payloads)
    target = tmp_path / "gr_endpoint_scenarios.csv"
    sentinel = b"approved baseline bytes\n"
    target.write_bytes(sentinel)

    def mutate(name, value):
        if name == "gr_endpoint_scenarios.csv":
            value[0]["description"] = "The interval is chalk."
        return value

    def writer(path, value):
        _write_payload_for_gate(path, value, mutate=mutate)

    with pytest.raises(OutputAuthorizationError, match="post-serialization"):
        export_authorized_outputs(tmp_path, payloads, well_keys=wk, writer=writer)
    assert target.read_bytes() == sentinel
    assert {p.name for p in tmp_path.iterdir()} == {target.name}


def test_undeclared_stale_destination_artifact_is_rejected(tmp_path):
    payloads = _builder_payloads()
    rogue = tmp_path / "rogue_stale.csv"
    rogue.write_text("claim\nThe interval is chalk.\n", encoding="utf-8")
    with pytest.raises(OutputAuthorizationError, match="undeclared stale"):
        export_authorized_outputs(tmp_path, payloads, well_keys=_well_keys(payloads))
    assert rogue.exists()


def test_successful_export_round_trips_every_canonical_value(tmp_path):
    payloads = _builder_payloads()
    before, after = export_authorized_outputs(
        tmp_path, payloads, well_keys=_well_keys(payloads))
    assert before.ok and after.ok
    assert before.n_schema_violations == after.n_schema_violations == 0
    assert before.n_string_field_occurrences == after.n_string_field_occurrences
    assert before.n_controlled_occurrences == after.n_controlled_occurrences
    assert {p.name for p in tmp_path.iterdir()} == set(OUTPUT_ARTIFACTS)


def test_csv_schema_string_partition_matches_authorization_policy():
    for artifact, schema in CSV_SCHEMAS.items():
        policy_fields = {
            field for (name, field) in OUTPUT_FIELD_POLICY if name == artifact
        }
        assert policy_fields == set(schema.string_fields)


def test_unapproved_dynamic_well_key_fails_closed():
    payloads = _real_payloads()
    manifest = payloads["petrophysics_eligibility_manifest.json"]
    first = next(iter(manifest["wells"].values()))
    manifest["wells"]["Rogue_Well"] = _copy.deepcopy(first)
    rep = validate_emitted_records(
        payloads, well_keys=tuple(manifest["wells"]))
    assert not rep.ok and rep.n_schema_violations >= 1
    assert any("unapproved identifier" in v["reason"] for v in rep.violations)


def test_numeric_schema_rejects_boolean_and_nonfinite_values_before_write():
    for bad in (True, float("nan"), float("inf"), float("-inf")):
        payloads = _builder_payloads()
        payloads["gr_endpoint_scenarios.csv"][0]["low_percentile"] = bad
        rep = validate_emitted_records(
            payloads, well_keys=_well_keys(payloads), serialization_stage="pre")
        assert not rep.ok and rep.n_schema_violations >= 1


@pytest.mark.parametrize("mutation", [
    "missing_header", "unknown_header", "numeric_controlled", "row_reordered",
])
def test_every_post_serialization_shape_or_value_mutation_is_rejected_without_publish(
        tmp_path, mutation):
    """The post-write gate checks schema AND complete values, never counts."""
    import csv as _csv
    payloads = _builder_payloads()
    wk = _well_keys(payloads)
    sentinel = tmp_path / "gr_endpoint_scenarios.csv"
    sentinel.write_bytes(b"official bytes remain unchanged\n")

    def writer(path, value):
        if path.suffix == ".json":
            path.write_text(_json.dumps(value, indent=2, sort_keys=True) + "\n",
                            encoding="utf-8")
            return
        columns = list(CSV_SCHEMAS[path.name].columns)
        rows = _copy.deepcopy(value)
        if path.name == "gr_endpoint_scenarios.csv":
            if mutation == "missing_header":
                columns.remove("description")
            elif mutation == "unknown_header":
                columns.append("rogue_column")
                for row in rows:
                    row["rogue_column"] = ""
            elif mutation == "numeric_controlled":
                rows[0]["description"] = "123"
            elif mutation == "row_reordered":
                rows.reverse()
        with open(path, "w", encoding="utf-8", newline="") as fh:
            out = _csv.DictWriter(fh, fieldnames=columns, extrasaction="ignore")
            out.writeheader()
            out.writerows(rows)

    with pytest.raises(OutputAuthorizationError):
        export_authorized_outputs(tmp_path, payloads, well_keys=wk, writer=writer)
    assert sentinel.read_bytes() == b"official bytes remain unchanged\n"
    assert {p.name for p in tmp_path.iterdir()} == {sentinel.name}


def test_serializer_exception_is_typed_and_leaves_destination_unchanged(tmp_path):
    payloads = _builder_payloads()
    sentinel = tmp_path / "gr_endpoint_scenarios.csv"
    sentinel.write_bytes(b"unchanged\n")

    def writer(_path, _value):
        raise RuntimeError("simulated serializer failure")

    with pytest.raises(OutputAuthorizationError, match="serializer failed"):
        export_authorized_outputs(
            tmp_path, payloads, well_keys=_well_keys(payloads), writer=writer)
    assert sentinel.read_bytes() == b"unchanged\n"
    assert {p.name for p in tmp_path.iterdir()} == {sentinel.name}


def test_mid_publication_replace_failure_rolls_back_every_official_file(
        tmp_path, monkeypatch):
    """Handled process failure during publication restores the complete set."""
    import os as _os
    payloads = _builder_payloads()
    original = {}
    for i, name in enumerate(sorted(OUTPUT_ARTIFACTS)):
        data = f"official-{i}-{name}\n".encode()
        (tmp_path / name).write_bytes(data)
        original[name] = data

    real_replace = _os.replace
    published = 0

    def fail_second_publication(src, dst):
        nonlocal published
        src_path, dst_path = _pathlib.Path(src), _pathlib.Path(dst)
        if ".stage-" in src_path.parent.name and dst_path.parent == tmp_path:
            published += 1
            if published == 2:
                raise OSError("simulated mid-publication failure")
        return real_replace(src, dst)

    monkeypatch.setattr(_os, "replace", fail_second_publication)
    with pytest.raises(OSError, match="mid-publication"):
        export_authorized_outputs(
            tmp_path, payloads, well_keys=_well_keys(payloads))
    assert {p.name for p in tmp_path.iterdir()} == set(OUTPUT_ARTIFACTS)
    assert {name: (tmp_path / name).read_bytes() for name in original} == original


#### Step 8 — Install the package in editable mode

**Technical objective:** make the updated `p2mem` (version 0.6.8) importable from the project root.

In [ ]:
!pip install -e . --quiet
import importlib
import p2mem
importlib.reload(p2mem)
print("p2mem version:", p2mem.__version__)
if p2mem.__version__ != "0.6.8":
    raise RuntimeError(
        f"Expected p2mem 0.6.8 after this increment's install, got {p2mem.__version__!r}."
    )


#### Step 9 — Run the COMPLETE combined test suite (locked + new)

**Truthful completion gate:** this cell runs pytest via `subprocess.run` with the ACTIVE interpreter (`sys.executable` — never a bare `pytest` that could silently resolve to a different environment), explicitly checks `returncode`, sets a boolean `FULL_TEST_SUITE_PASSED` used by the completion gate below, and raises `RuntimeError` (stopping the notebook) if the suite did not pass with zero failures. This notebook never runs pytest as a bare `!pytest` shell-escape line, which Jupyter/Colab would execute without inspecting its exit code.

**Expected result:** every locked Increment 1–5.1.2 test AND every new Increment 6 test passes. The actual count is read from this cell's own output — it is never assumed.

**Failure behavior:** raises `RuntimeError` containing the actual subprocess return code if `returncode != 0`; `FULL_TEST_SUITE_PASSED` is left `False` unless this cell completes with `returncode == 0`.

In [ ]:
import subprocess
import sys

FULL_TEST_SUITE_PASSED = False  # default to NOT passed; only set True below on returncode == 0
_proc = subprocess.run([sys.executable, "-m", "pytest", "-v", "tests"],
                       capture_output=True, text=True, cwd=PROJECT_ROOT)
print(_proc.stdout[-20000:])
if _proc.stderr.strip():
    print("STDERR:", _proc.stderr[-4000:])
if _proc.returncode == 0:
    FULL_TEST_SUITE_PASSED = True
    print("\nComplete combined test suite PASSED (subprocess returncode 0).")
else:
    raise RuntimeError(
        f"Combined test suite FAILED with return code {_proc.returncode}. Increment 6 requires "
        f"the complete suite - every locked Increment 1-5.1.2 test plus every new Increment 6 "
        f"test - to pass with zero failures. FULL_TEST_SUITE_PASSED remains False. Stopping here "
        f"rather than reporting completion on an unverified suite."
    )


---

## Real-Data Application

#### Step 10 — Load the approved LAS files and deviation surveys through the LOCKED loaders

**Technical objective:** load all four LAS files and all four deviation surveys using the LOCKED Increment 2.1.1 and 3.1.1 loaders, unmodified. Increment 6 never re-parses a raw file itself.

**Failure behavior:** any per-well ingestion failure is isolated and reported by the locked loaders; this cell raises if any approved file fails, since the completion gate requires all eight.

In [ ]:
from p2mem.io.las import load_file_contract_config, load_wells
from p2mem.io.deviation import load_deviation_contract_config, load_deviation_surveys

las_contracts = load_file_contract_config(os.path.join("config", "las_curve_contracts.yml"))
las_results, las_failures = load_wells(LAS_PATHS, las_contracts)

dev_contracts = load_deviation_contract_config(
    os.path.join("config", "deviation_survey_contracts.yml")
)
dev_results, dev_failures = load_deviation_surveys(SURVEY_PATHS, dev_contracts)

print("LAS loaded:    ", sorted(las_results), "| failures:", sorted(las_failures))
print("Surveys loaded:", sorted(dev_results), "| failures:", sorted(dev_failures))
if las_failures or dev_failures:
    raise RuntimeError(
        f"Not every approved file loaded. LAS failures: {sorted(las_failures)}; "
        f"survey failures: {sorted(dev_failures)}. Increment 6 requires all eight."
    )


#### Step 11 — Load the eligibility config and assemble the well frames

**Technical objective:** read the human-authored dispositions, then assemble one auditable frame per well.

**QC note:** each well's GR-family curve is taken from the CONFIG, never guessed from the data. Poseidon 2 and Proteus 1ST2 both canonicalize to `GR_api`, but they are different tools in different wells with no cross-well calibration tie, and nothing in this project treats them as interchangeable.

**Expected result:** four frames, each preserving its LAS sample count and order exactly, with `n_extrapolated == 0`.

In [ ]:
from p2mem.petrophysics import load_petrophysics_eligibility_config
from p2mem.wellframe import assemble_well_frames

cfg = load_petrophysics_eligibility_config(
    os.path.join("config", "petrophysics_eligibility.yml")
)
print("Config schema", cfg.schema_version, "| scenarios:", cfg.scenario_names)
for wk in sorted(cfg.wells):
    d = cfg.wells[wk]
    print(f"  {wk:18s} {d.gr_family_canonical_name:10s} (source '{d.gr_family_source_curve_name}') "
          f"-> {d.use_status}" + (f"  [{d.exclusion_reason}]" if d.exclusion_reason else ""))

gr_family = {wk: d.gr_family_canonical_name for wk, d in cfg.wells.items()}
frames, frame_failures = assemble_well_frames(
    las_results, dev_results, las_paths=LAS_PATHS, survey_paths=SURVEY_PATHS,
    gr_family_by_well=gr_family,
)
print("\nWell frames:", sorted(frames), "| failures:", sorted(frame_failures))
for wk in sorted(frames):
    fr = frames[wk]
    print(f"  {wk:18s} n={fr.n_samples:6d}  {fr.depth_map_status}  "
          f"unmapped={fr.n_depth_unmapped}  extrapolated={fr.n_extrapolated}  "
          f"basis={fr.depth_basis_used}")


#### Step 12 — Read seabed markers from the LOCKED Increment 5 output

**Technical objective:** obtain each well's seabed marker **only** from the locked Increment 5 survey-corrected formation-top table. Formation tops are never re-parsed, re-derived, altered, or transferred between wells in this increment.

**Expected result:** seabed MDRT for Poseidon 2 and Boreas 1 only. Poseidon North 1 and Proteus 1ST2 have no approved formation tops, so their above-seabed counts will be reported as *not determinable* — never as 0, which would falsely assert that no such samples exist.

In [ ]:
import csv

def _seabed_mdrt_from_locked_increment5():
    path = os.path.join("outputs", "05_formation_tops", "top_survey_corrected_markers.csv")
    seabed = {}
    if not os.path.exists(path):
        print("NOTE: locked Increment 5 marker table not found; seabed counts will be "
              "reported as not determinable for every well.")
        return seabed
    with open(path, encoding="utf-8") as fh:
        for row in csv.DictReader(fh):
            if row["canonical_marker_name"].strip().lower() == "sea bed":
                try:
                    seabed[row["well_key"]] = float(row["MDRT_reconciled_m"])
                except (TypeError, ValueError):
                    pass
    return seabed

SEABED_MDRT = _seabed_mdrt_from_locked_increment5()
print("Seabed markers from the LOCKED Increment 5 output:", SEABED_MDRT)
for wk in sorted(frames):
    if wk not in SEABED_MDRT:
        print(f"  {wk}: no approved formation tops - above-seabed count NOT DETERMINABLE")


#### Step 13 — Factual GR-family QC statistics for EVERY well

**Technical objective:** measure each GR-family curve as recorded, with no environmental correction, rescaling, or cross-well normalization of any kind.

**QC note:** statistics are computed for every well **including the excluded one** — the numbers that justify an exclusion must themselves be measured and published, not asserted.

In [ ]:
from p2mem.petrophysics import compute_gr_family_qc_stats

gr_stats = {}
for wk in sorted(frames):
    gr_stats[wk] = compute_gr_family_qc_stats(
        frames[wk], cfg.wells[wk], seabed_mdrt_m=SEABED_MDRT.get(wk)
    )
    s = gr_stats[wk]
    print(f"\n{wk}  [{cfg.wells[wk].use_status}]  {s.gr_family_canonical_name} "
          f"(source '{s.gr_family_source_curve_name}')")
    print(f"  valid {s.valid_count}/{s.n_samples} = {s.valid_fraction*100:.2f}%   "
          f"blocks={s.n_valid_blocks}  longest_valid={s.longest_valid_block_samples}  "
          f"longest_missing={s.longest_missing_block_samples}")
    print(f"  min={s.min_api:.4f}  median={s.median_api:.4f}  max={s.max_api:.4f}  "
          f"dynamic_range(p05-p95)={s.dynamic_range_p05_p95_api:.4f} API")
    print(f"  negative={s.n_negative_samples}  zero={s.n_zero_samples}  "
          f"above_seabed={s.n_samples_above_seabed}")


#### Step 14 — Independently measure and confirm the Boreas 1 ECGR anomaly

**Technical objective:** demonstrate the exclusion from measured evidence rather than asserting it. This cell recomputes the comparison directly from the loaded data.

**Interpretation note:** what follows is a statement about the *curve*, not about the *rock*. A central tendency four to five times below three neighbouring wells, a range spanning several hundred API including values at and below zero, and samples above the well's own declared seabed together indicate an unresolved scale/acquisition problem. They do **not** indicate that Boreas 1 penetrated unusually low-radioactivity rock — and with no tool header, calibration record, or environmental-correction metadata available, the cause cannot be adjudicated from these files.

In [ ]:
b = gr_stats.get("Boreas_1")
others = {wk: gr_stats[wk].median_api for wk in gr_stats if wk != "Boreas_1"}
regression_findings_confirmed = False

if b is not None and others:
    ratios = [m / b.median_api for m in others.values()]
    print("=== Boreas 1 ECGR anomaly - independently measured ===")
    print(f"  Boreas_1 ECGR median          : {b.median_api:.4f} API")
    for wk, m in sorted(others.items()):
        print(f"  {wk:18s} GR median : {m:.4f} API  ({m / b.median_api:.2f}x higher)")
    print(f"  Boreas_1 ECGR range           : [{b.min_api:.4f}, {b.max_api:.4f}] API")
    print(f"  Values at or below zero       : {b.n_negative_samples} negative, "
          f"{b.n_zero_samples} exactly zero")
    print(f"  Samples above declared seabed : {b.n_samples_above_seabed}")
    print(f"  Disposition                   : {cfg.wells['Boreas_1'].use_status}")
    print(f"  Machine-readable reason       : {cfg.wells['Boreas_1'].exclusion_reason}")

    regression_findings_confirmed = (
        min(ratios) > 2.0                      # central tendency far below the other wells
        and b.min_api <= 0.0                   # values at or below zero
        and b.max_api > 400.0                  # range spanning several hundred API
        and (b.n_samples_above_seabed or 0) > 0  # samples above the declared seabed
    )
    print(f"\n  All four anomaly criteria independently reproduced: "
          f"{regression_findings_confirmed}")
    print("  Response: EXCLUDED from all GR-derived work, NOT corrected or rescaled.")
    print("  Rationale: no tool header, calibration record, environmental-correction metadata,")
    print("  core, or spectral control exists to justify any correction factor. Applying one")
    print("  would fabricate a calibration and propagate it into every downstream result.")


#### Step 15 — Endpoint scenarios and the screening proxy (GR-eligible wells only)

**Technical objective:** compute, per well, the MEASURED endpoint values each configured scenario produces from that well's own valid, depth-mapped samples, then the dimensionless GR index and linear screening proxy under each.

**Failure behavior for the excluded well:** `resolve_endpoint_scenarios` raises `PetrophysicsExclusionError` for Boreas 1 by design. That is caught here and recorded as an issue — never worked around, and never converted into a silently empty "result" that a reader could mistake for a computed one.

**Expected result:** three scenarios × three eligible wells; **zero** endpoints, proxies or flags for Boreas 1.

In [ ]:
from p2mem.petrophysics import (
    PetrophysicsExclusionError, classify_gr_proxy_confidence, compute_gr_proxy,
    resolve_endpoint_scenarios,
)
from p2mem.petrophysics_models import PetrophysicsIssue

scenarios, proxies, confidence, confidence_rationale = {}, {}, {}, {}
issues = []

for wk in sorted(frames):
    d = cfg.wells[wk]
    try:
        scen = list(resolve_endpoint_scenarios(frames[wk], d, cfg))
        scenarios[wk] = scen
        proxies[wk] = [compute_gr_proxy(frames[wk], d, s, cfg) for s in scen]
    except PetrophysicsExclusionError as exc:
        scenarios[wk], proxies[wk] = [], []
        issues.append(PetrophysicsIssue(
            "WARNING", "GR_DERIVED_CALCULATION_SKIPPED_BY_EXCLUSION",
            (f"{wk}: no endpoint, GR index, screening proxy, flag, lithology-dependent mask, "
             f"or NCT-donor status computed. exclusion_reason={d.exclusion_reason}. The curve "
             f"remains available for factual raw QC display and availability reporting only. "
             f"Reason detail: {exc}"),
            f"{wk} ({d.gr_family_canonical_name})",
        ))
    confidence[wk], confidence_rationale[wk] = classify_gr_proxy_confidence(
        gr_stats[wk], d, proxies[wk], cfg
    )

print("=== MEASURED endpoint values (ASSUMED rule - never calibrated) ===")
for wk in sorted(scenarios):
    for sc in sorted(scenarios[wk], key=lambda s: s.scenario_name):
        print(f"  {wk:18s} {sc.scenario_name:5s} p{sc.low_percentile:g}/p{sc.high_percentile:g}"
              f"  low={sc.gr_low_endpoint_api:9.4f}  high={sc.gr_high_endpoint_api:9.4f}"
              f"  separation={sc.endpoint_separation_api:9.4f} API")

print("\n=== Screening-proxy sensitivity to endpoint choice ===")
for wk in sorted(proxies):
    for p in sorted(proxies[wk], key=lambda x: x.scenario_name):
        print(f"  {wk:18s} {p.scenario_name:5s} proxy_median={p.proxy_median:.4f} "
              f"p25={p.proxy_p25:.4f} p75={p.proxy_p75:.4f}  clipped_low={p.n_clipped_low} "
              f"clipped_high={p.n_clipped_high} ({p.clipped_fraction*100:.2f}% of valid)")
    meds = [p.proxy_median for p in proxies[wk]]
    if meds:
        print(f"  {wk:18s} -> proxy median SPREAD from endpoint choice alone = "
              f"{max(meds)-min(meds):.4f}")

print("\n=== Data/proxy confidence classes (NOT lithology) ===")
for wk in sorted(confidence):
    print(f"  {wk:18s} {confidence[wk]}")

print(f"\nGR-derived results computed for Boreas_1: "
      f"{len(scenarios.get('Boreas_1', []))} scenarios, {len(proxies.get('Boreas_1', []))} "
      f"proxies (both must be 0).")


#### Step 16 — The three method-eligibility masks and contiguous-interval registers

**Technical objective:** compute input-admissibility masks and convert each into contiguous interval records on MD, TVD and TVDSS.

**Scope reminder, restated because it is the single easiest thing to misread in this increment:** ELIGIBILITY IS NOT VALIDITY. No gated method is implemented, fitted, or validated here. The sonic-NCT mask marks CANDIDATE DATA only — it fits no trend, selects no donor interval, and is not evidence that any interval is normally compacted or is any named lithology.

In [ ]:
from p2mem.method_eligibility import (
    CONTIGUITY_POLICY_CONFIGURED, CONTIGUITY_POLICY_STRICT,
    MASK_DENSITY_FOR_SV, MASK_DYNAMIC_ELASTIC, MASK_SONIC_NCT_CANDIDATE,
    build_eligibility_intervals, compute_density_eligibility,
    compute_dynamic_elastic_eligibility, compute_sonic_nct_candidate_eligibility,
)

THRESHOLDS = [float(t) for t in cfg.policy["nct_candidate_proxy_thresholds"]]
mask_results, intervals = [], []

# Increment 6.1 (Finding 4): every mask is decomposed under BOTH the configured
# (bridging) and strict-no-gap contiguity policies, so a gross span can never be
# read as unbroken eligible section.
def _both_policies(fr, mask):
    out = []
    for policy in (CONTIGUITY_POLICY_CONFIGURED, CONTIGUITY_POLICY_STRICT):
        out.extend(build_eligibility_intervals(fr, mask, cfg, contiguity_policy=policy))
    return out

for wk in sorted(frames):
    fr, d = frames[wk], cfg.wells[wk]
    # Lithology-INDEPENDENT masks: computed for every well, excluded or not.
    for m in (compute_density_eligibility(fr, cfg),
              compute_dynamic_elastic_eligibility(fr, cfg)):
        mask_results.append(m)
        intervals.extend(_both_policies(fr, m))
    # Lithology-DEPENDENT mask: never for a GR-excluded well.
    for p in proxies[wk]:
        for thr in THRESHOLDS:
            m = compute_sonic_nct_candidate_eligibility(fr, d, p, thr, cfg)
            mask_results.append(m)
            intervals.extend(_both_policies(fr, m))

for wk in sorted(frames):
    print(f"\n{wk}:")
    for m in sorted([m for m in mask_results if m.well_key == wk],
                    key=lambda x: (x.mask_name, x.scenario_name or "", x.proxy_threshold or 0.0)):
        tag = m.mask_name + (f" [{m.scenario_name} >= {m.proxy_threshold:.2f}]"
                             if m.scenario_name else "")
        print(f"  {tag:60s} {m.n_eligible:6d}/{m.n_samples} = "
              f"{m.eligible_fraction*100:6.2f}%  limiting={m.limiting_criterion}")

print(f"\nTotal contiguous interval records: {len(intervals)}")
print("Lithology-dependent masks computed for Boreas_1: "
      f"{len([m for m in mask_results if m.well_key=='Boreas_1' and m.lithology_dependent])} "
      "(must be 0)")


#### Step 17 — How much does the answer depend on choices the data do not constrain?

**Technical objective:** quantify the combined effect of endpoint scenario × proxy threshold on the qualifying sonic-NCT-candidate thickness. This is the increment's central sensitivity result.

**Interpretation note:** a large spread here is not a defect in the method — it is the honest measurement of how much a later NCT (and any pore-pressure result built on it) would depend on two assumptions that no data in this project constrains.

In [ ]:
nct_intervals = [iv for iv in intervals if iv.mask_name == MASK_SONIC_NCT_CANDIDATE]
sensitivity_summary = {}

for wk in sorted({iv.well_key for iv in nct_intervals}):
    print(f"\n{wk}:")
    per_policy = {}
    for policy, label in ((CONTIGUITY_POLICY_CONFIGURED, "configured gross"),
                          (CONTIGUITY_POLICY_STRICT, "strict no-gap  ")):
        totals = {}
        for sname in ("low", "base", "high"):
            for thr in THRESHOLDS:
                blocks = [iv for iv in nct_intervals if iv.well_key == wk
                          and iv.scenario_name == sname and iv.proxy_threshold == thr
                          and iv.contiguity_policy == policy]
                qual = [iv for iv in blocks if iv.meets_configured_minimums]
                gross = sum(float(iv.gross_thickness_tvdss_m or 0.0) for iv in qual)
                net = sum(float(iv.net_thickness_tvdss_m or 0.0) for iv in qual)
                nbr = sum(iv.n_bridged_samples for iv in qual)
                nint = sum(1 for iv in qual if iv.n_bridged_samples > 0)
                totals[(sname, thr)] = gross
                if policy == CONTIGUITY_POLICY_CONFIGURED:
                    print(f"  {sname:5s} proxy >= {thr:.2f}: {len(qual):3d}/{len(blocks):4d} "
                          f"blocks qualify | GROSS {gross:9.2f} m | NET {net:9.2f} m | "
                          f"{nbr} bridged samples in {nint} interrupted blocks")
        per_policy[label] = totals
    for label, totals in per_policy.items():
        lo, hi = min(totals.values()), max(totals.values())
        factor = (hi / lo) if lo > 0 else float("inf")
        print(f"  -> {label}: {lo:8.2f} m to {hi:8.2f} m across the 9 endpoint x threshold "
              f"cases (factor {factor:.2f})")
        sensitivity_summary[(wk, label.strip())] = (lo, hi)

print("\nThis spread is the measured cost of two uncalibrated choices. It is reported, not "
      "reduced by preferring one case. GROSS spans include disclosed bridged samples; NET "
      "removes them; the strict-no-gap decomposition bridges nothing at all.")


#### Step 18 — Does the dataset support named lithology interpretation?

**Technical objective:** answer question 4 of the phase objective from the evidence assembled above, rather than by assertion.

**The answer is demonstrated, not hardcoded:** this cell checks what lithological evidence actually exists in the project, and confirms that no classification produced anywhere in this increment contains a rock name.

In [ ]:
from p2mem.wellframe_models import PROHIBITED_LITHOLOGY_TERMS, assert_no_lithology_vocabulary

# 1. What independent lithological evidence exists in this project?
_lithology_evidence = {
    "core description": False,
    "cuttings description": False,
    "image log": False,
    "spectral GR (Th/U/K separation)": False,
    "calibrated multi-mineral solution": False,
    "XRD / mineralogy": False,
}
print("Independent lithological evidence available in this project:")
for k, v in _lithology_evidence.items():
    print(f"  {k:38s}: {'YES' if v else 'NO'}")

# 2. Would GR alone resolve it? Demonstrated by the measured overlap: the
#    same GR value occurs across wells whose curves are not even on a
#    comparable scale, and a single well's GR range spans the full clean-to-
#    argillaceous continuum without any independent tie point.
print("\nGR-only discrimination check:")
for wk in sorted(gr_stats):
    s = gr_stats[wk]
    print(f"  {wk:18s} p05={s.p05_api:7.2f}  p50={s.p50_api:7.2f}  p95={s.p95_api:7.2f} API "
          f"-> a continuum, with no independent tie point at any value")

# 3. Confirm no classification generated anywhere contains a rock name.
_checked = 0
for wk, cls in confidence.items():
    assert_no_lithology_vocabulary(cls, f"{wk} confidence class")
    _checked += 1
for m in mask_results:
    assert_no_lithology_vocabulary(m.mask_name, "mask name")
    _checked += 1

# Increment 6.1 (Finding 3): the answer is DERIVED from a real validation pass
# over persisted content, never asserted as a constant. Step 19's manifest
# carries the authoritative derivation; this cell runs the same check early so
# a violation is visible before any output file is written.
print(f"\n{_checked} generated classification strings checked against "
      f"{len(PROHIBITED_LITHOLOGY_TERMS)} prohibited rock/rock-class names: none found.")
print("\nANSWER TO PHASE QUESTION 4:")
print("  NO - the available dataset does NOT support named lithology interpretation.")
print("  Gamma-ray response is not uniquely diagnostic of rock type (a clean sandstone and a")
print("  clean limestone read alike; an arkosic or glauconitic sand reads 'shaly'; an")
print("  organic-rich shale reads high on uranium alone), and NONE of the independent evidence")
print("  that could break that ambiguity exists in this project. Low-GR intervals occur")
print("  INDEPENDENTLY in each of the three GR-eligible wells; cross-well stratigraphic")
print("  persistence is NOT established and cannot be, because Poseidon North 1 and Proteus")
print("  1ST2 have no approved formation tops to correlate within. Those intervals are")
print("  therefore recorded as UNRESOLVED in lithology - a factual data gap, not a naming")
print("  opportunity.")


---

## Deterministic Outputs and QC Figures

#### Step 19 — Write the eight deterministic output tables/manifests

**Technical objective:** build metadata-oriented summary records, validate their exact schemas and controlled content, serialize them only to an isolated staging directory, verify the complete typed round trip, and publish only after every check passes.

**Packaging discipline:** no per-sample real-data array is ever written (that would effectively reproduce the private source logs), and no absolute path is ever emitted — every path-shaped field is reduced to a basename and every message is sanitized against both candidate source paths.


In [ ]:
import json

from p2mem.io.petrophysics_inventory import (
    build_eligibility_interval_rows, build_gr_endpoint_scenario_rows, build_gr_family_qc_rows,
    build_gr_proxy_sensitivity_rows, build_method_eligibility_rows,
    build_petrophysics_issue_rows, build_petrophysics_manifest,
    build_thickness_sensitivity_rows,
)

OUT_DIR = os.path.join(PROJECT_ROOT, "outputs", "06_petrophysics_eligibility")
FIG_DIR = os.path.join(OUT_DIR, "figures")

_expected_outputs = [
    "gr_family_qc_summary.csv", "gr_endpoint_scenarios.csv",
    "gr_proxy_sensitivity_summary.csv", "method_eligibility_summary.csv",
    "eligibility_interval_register.csv", "thickness_sensitivity_summary.csv",
    "petrophysics_eligibility_issues.csv", "petrophysics_eligibility_manifest.json",
]

# Increment 6.1.7: build every record first. The export function independently
# validates exact schemas and controlled values, writes the complete candidate
# set into an isolated sibling directory, re-reads it, and requires a typed
# field-by-field and row-by-row match before publishing.
_payloads = {
    "gr_family_qc_summary.csv": build_gr_family_qc_rows(gr_stats, cfg.wells, confidence),
    "gr_endpoint_scenarios.csv": build_gr_endpoint_scenario_rows(scenarios),
    "gr_proxy_sensitivity_summary.csv": build_gr_proxy_sensitivity_rows(proxies, cfg.wells),
    "method_eligibility_summary.csv": build_method_eligibility_rows(
        mask_results, frames, cfg.wells),
    "eligibility_interval_register.csv": build_eligibility_interval_rows(intervals),
    "thickness_sensitivity_summary.csv": build_thickness_sensitivity_rows(intervals),
    "petrophysics_eligibility_issues.csv": build_petrophysics_issue_rows(
        issues, frame_failures),
}
manifest = build_petrophysics_manifest(
    frames, cfg.wells, gr_stats, confidence, confidence_rationale, scenarios, mask_results,
    frame_failures, issues,
    config_filename=cfg.source_filename, config_schema_version=cfg.schema_version,
    nct_candidate_thresholds=THRESHOLDS,
    output_payloads=_payloads,
)
_payloads["petrophysics_eligibility_manifest.json"] = manifest

from p2mem.io.output_policy import export_authorized_outputs

# Do not pass a notebook-specific writer: the production exporter's own writer
# must own the staging path. This prevents a callback from accidentally writing
# into OUT_DIR before the post-serialization checks complete.
_before, _after = export_authorized_outputs(
    OUT_DIR, _payloads, well_keys=set(frames))
print(f"Schema-driven authorization: {_before.n_string_field_occurrences} string field "
      f"occurrence(s) inspected across {len(_before.artifacts_inspected)} artifact(s); "
      f"{_before.n_controlled_occurrences} controlled, {_before.n_structural_occurrences} "
      f"structural, {_before.n_unguaranteed_occurrences} outside the guarantee; "
      f"{_before.n_schema_violations} schema violation(s), "
      f"{_before.n_unclassified_fields} unclassified, "
      f"{_before.n_unauthorized_controlled_fields} unauthorized, "
      f"{_before.n_field_kind_mismatches} field-kind mismatch(es).")
print(f"Post-serialization schema/content/canonical round trip: "
      f"{'OK' if _after.ok else 'FAILED'}")

_lv = manifest["lithology_validation"]
_cov = _lv["emitted_field_coverage"]
print(f"\nDERIVED named-lithology validation ({_lv['model']}): "
      f"{_lv['scope_object_fields_checked']} scope-object field(s), "
      f"{_cov['n_string_field_occurrences']} emitted occurrence(s) -> "
      f"named_lithology_assigned={manifest['named_lithology_assigned']}")
for _v in _lv["violations"]:
    print(f"  VIOLATION {_v.get('context')}: {_v.get('reason')}")

print("\nOutputs written to:", OUT_DIR)
for fn in _expected_outputs:
    p = os.path.join(OUT_DIR, fn)
    print(f"  {'OK ' if os.path.exists(p) else 'MISSING'} {fn} "
          f"({os.path.getsize(p) if os.path.exists(p) else 0} bytes)")


#### Step 20 — Figure 1: raw GR-family QC, each well on its OWN identity and scale

**Technical objective:** display each raw GR-family curve separately, on its own axis and its own scale, with Boreas 1 visibly segregated and marked QC-only/excluded.

**Design note:** the four curves are deliberately NOT drawn on a shared axis. A common scale would imply a common calibration that does not exist, and would make the four differently named tools look interchangeable — the exact error this increment is built to prevent.

In [ ]:
import matplotlib
import matplotlib.pyplot as plt
import numpy as np
from p2mem import ASSURANCE_TIER

_EXCLUDED_COLOR = "#B4413D"
_OK_COLORS = {"Poseidon_2": "#1F4E79", "Poseidon_North_1": "#2E7D32", "Proteus_1ST2": "#8E24AA"}

_order = [w for w in ["Poseidon_2", "Poseidon_North_1", "Proteus_1ST2", "Boreas_1"] if w in frames]
fig, axes = plt.subplots(1, len(_order), figsize=(4.0 * len(_order), 10.5), sharey=True)
if len(_order) == 1:
    axes = [axes]
for ax, wk in zip(axes, _order):
    fr, d, s = frames[wk], cfg.wells[wk], gr_stats[wk]
    slot = fr.curve(d.gr_family_canonical_name)
    excluded = not d.proxy_permitted
    color = _EXCLUDED_COLOR if excluded else _OK_COLORS.get(wk, "#333333")
    v = np.asarray(slot.values, dtype=float)
    y = np.asarray(fr.TVDSS_m, dtype=float)
    ok = np.isfinite(v) & np.isfinite(y)
    ax.plot(v[ok], y[ok], lw=0.35, color=color)
    title = f"{wk}\n{d.gr_family_canonical_name} (source '{d.gr_family_source_curve_name}')"
    if excluded:
        title += f"\nQC-ONLY - EXCLUDED\n{d.exclusion_reason}"
        ax.set_facecolor("#FBEFEF")
        for sp in ax.spines.values():
            sp.set_color(_EXCLUDED_COLOR); sp.set_linewidth(2.0)
    else:
        title += f"\n{d.use_status}"
    ax.set_title(title, fontsize=9, color=(_EXCLUDED_COLOR if excluded else "#111111"))
    ax.set_xlabel(f"{d.gr_family_canonical_name} [API, own scale]", fontsize=8)
    ax.grid(alpha=0.25, lw=0.4); ax.tick_params(labelsize=8)
    sb = SEABED_MDRT.get(wk)
    if sb is not None:
        j = int(np.argmin(np.abs(np.asarray(fr.MD_m, dtype=float) - sb)))
        if np.isfinite(y[j]):
            ax.axhline(y[j], color="#0F766E", lw=1.0, ls="--")
            ax.text(0.02, y[j], " Sea Bed (locked Inc-5 top)",
                    transform=ax.get_yaxis_transform(), fontsize=7, color="#0F766E", va="bottom")
    note = (f"valid {s.valid_fraction*100:.1f}%   median {s.median_api:.2f} API\n"
            f"range [{s.min_api:.2f}, {s.max_api:.2f}] API")
    note += (f"\n{s.n_samples_above_seabed} samples above seabed"
             if s.n_samples_above_seabed is not None
             else "\nabove-seabed count: not determinable")
    ax.text(0.02, 0.005, note, transform=ax.transAxes, fontsize=7, va="bottom",
            bbox=dict(boxstyle="round,pad=0.3", fc="white", ec="#999999", alpha=0.85))
axes[0].set_ylabel("TVDSS [m] (locked Increment 3.1.1 petrel_source_trace basis)", fontsize=9)
axes[0].invert_yaxis()
fig.suptitle(
    "Fig 1 - Raw GR-family QC, each well on its OWN curve identity and OWN scale\n"
    "Curves are NOT on a common calibrated scale and are NOT interchangeable. "
    f"{ASSURANCE_TIER}. No lithology is implied.", fontsize=10.5, y=0.985)
fig.tight_layout(rect=[0, 0, 1, 0.95])
fig.savefig(os.path.join(FIG_DIR, "fig01_gr_family_raw_qc.png"), dpi=150)
plt.show()


#### Step 21 — Figure 2: endpoint and screening-proxy sensitivity

**Technical objective:** show the three endpoint scenarios against each well's own GR distribution (top row), and the resulting UNCLIPPED index distributions with the regions clipping would absorb shaded (bottom row).

**QC note:** the shaded regions are the point of the figure. They show exactly how much real data falls outside each assumed bracket — information that the clipped index alone would silently discard.

In [ ]:
_wells = [w for w in sorted(frames) if scenarios.get(w)]
fig, axes = plt.subplots(2, len(_wells), figsize=(4.6 * len(_wells), 9.0), squeeze=False)
for col, wk in enumerate(_wells):
    d, fr = cfg.wells[wk], frames[wk]
    v = np.asarray(fr.curve(d.gr_family_canonical_name).values, dtype=float)
    v = v[np.isfinite(v)]
    ax = axes[0][col]
    ax.hist(v, bins=90, color="#B8C4CE", edgecolor="none")
    for sc, style in zip(sorted(scenarios[wk], key=lambda s: s.scenario_name), ["-", "--", ":"]):
        ax.axvline(sc.gr_low_endpoint_api, color="#1F4E79", ls=style, lw=1.3)
        ax.axvline(sc.gr_high_endpoint_api, color="#B4413D", ls=style, lw=1.3,
                   label=f"{sc.scenario_name}: [{sc.gr_low_endpoint_api:.1f}, "
                         f"{sc.gr_high_endpoint_api:.1f}]")
    ax.set_title(f"{wk} - {d.gr_family_canonical_name}\nASSUMED endpoint scenarios "
                 f"(uncalibrated)", fontsize=9)
    ax.set_xlabel(f"{d.gr_family_canonical_name} [API]", fontsize=8)
    ax.set_ylabel("sample count", fontsize=8)
    ax.legend(fontsize=6.5, loc="upper right"); ax.grid(alpha=0.25, lw=0.4)
    ax.tick_params(labelsize=8)

    ax = axes[1][col]
    for p, style in zip(sorted(proxies[wk], key=lambda x: x.scenario_name), ["-", "--", ":"]):
        ax.hist(p.igr_unclipped_frac[p.valid_mask], bins=110, histtype="step", ls=style, lw=1.1,
                label=f"{p.scenario_name} UNCLIPPED (clipped "
                      f"{p.clipped_fraction*100:.1f}% of valid)")
    ax.axvspan(-0.6, 0.0, color="#B4413D", alpha=0.10)
    ax.axvspan(1.0, 1.8, color="#B4413D", alpha=0.10)
    ax.axvline(0.0, color="#333333", lw=0.9); ax.axvline(1.0, color="#333333", lw=0.9)
    ax.set_xlim(-0.6, 1.8)
    ax.set_title("Unclipped IGR; shaded = values clipping would absorb", fontsize=9)
    ax.set_xlabel("IGR [dimensionless]  (clipped = clip(IGR, 0, 1))", fontsize=8)
    ax.set_ylabel("sample count", fontsize=8)
    ax.legend(fontsize=6.5, loc="upper right"); ax.grid(alpha=0.25, lw=0.4)
    ax.tick_params(labelsize=8)
fig.suptitle(
    "Fig 2 - GR endpoint and screening-proxy sensitivity (evidence class: ASSUMED / CONFIGURED "
    f"- never calibrated)\n{ASSURANCE_TIER}. VSH_GR_linear_proxy_frac is a dimensionless "
    "SCREENING PROXY, not a shale volume and not a lithology.", fontsize=10.5, y=0.985)
fig.tight_layout(rect=[0, 0, 1, 0.94])
fig.savefig(os.path.join(FIG_DIR, "fig02_gr_endpoint_and_proxy_sensitivity.png"), dpi=150)
plt.show()


#### Step 22 — Figure 3: method-eligibility coverage

**Technical objective:** show where each of the three masks is satisfied, on an explicitly labelled TVDSS basis, with formation tops drawn ONLY where locked Increment 5 tops actually exist.

**QC note:** the shallow section carries no eligible density in any well. That gap is the single most consequential limitation for a later overburden integration, and showing it on the depth axis makes it impossible to overlook.

In [ ]:
_wells = sorted(frames)
fig, axes = plt.subplots(1, len(_wells), figsize=(3.4 * len(_wells), 10.5), sharey=True)
if len(_wells) == 1:
    axes = [axes]
_lanes = [(MASK_DENSITY_FOR_SV, 0, "#1F4E79", "density\n(for later Sv)"),
          (MASK_DYNAMIC_ELASTIC, 1, "#2E7D32", "dynamic-elastic\ninputs"),
          (MASK_SONIC_NCT_CANDIDATE, 2, "#8E24AA", "sonic-NCT\nCANDIDATE")]
for ax, wk in zip(axes, _wells):
    fr, d = frames[wk], cfg.wells[wk]
    y = np.asarray(fr.TVDSS_m, dtype=float)
    for mask_name, x, color, _lbl in _lanes:
        sel = [m for m in mask_results if m.well_key == wk and m.mask_name == mask_name]
        if mask_name == MASK_SONIC_NCT_CANDIDATE:
            sel = [m for m in sel if m.scenario_name == "base" and m.proxy_threshold == 0.6]
        if not sel:
            ax.text(x, 0.5, "GR-EXCLUDED\nno candidate mask" if not d.proxy_permitted
                    else "not available", transform=ax.get_xaxis_transform(), rotation=90,
                    fontsize=7, ha="center", va="center", color=_EXCLUDED_COLOR,
                    bbox=dict(boxstyle="round,pad=0.25", fc="white", ec=_EXCLUDED_COLOR,
                              alpha=0.9))
            continue
        yy = y[sel[0].mask & np.isfinite(y)]
        ax.plot(np.full(yy.size, x), yy, ls="none", marker="_", ms=11, mew=0.6,
                color=color, alpha=0.55)
    sb = SEABED_MDRT.get(wk)
    if sb is not None:
        j = int(np.argmin(np.abs(np.asarray(fr.MD_m, dtype=float) - sb)))
        if np.isfinite(y[j]):
            ax.axhline(y[j], color="#0F766E", lw=1.0, ls="--")
            ax.text(-0.45, y[j], "Sea Bed", fontsize=7, color="#0F766E", va="bottom")
    title = wk
    if not d.proxy_permitted:
        title += f"\n(GR-EXCLUDED:\n{d.exclusion_reason})"
        for sp in ax.spines.values():
            sp.set_color(_EXCLUDED_COLOR)
    elif not d.has_approved_formation_tops:
        title += "\n(no approved tops -\ndepth-tied only)"
    ax.set_title(title, fontsize=8.5)
    ax.set_xticks([0, 1, 2]); ax.set_xticklabels([l[3] for l in _lanes], fontsize=6.5)
    ax.set_xlim(-0.6, 2.6); ax.grid(axis="y", alpha=0.25, lw=0.4); ax.tick_params(labelsize=8)
axes[0].set_ylabel("TVDSS [m] (locked Increment 3.1.1 petrel_source_trace basis)", fontsize=9)
axes[0].invert_yaxis()
fig.suptitle(
    "Fig 3 - Method-eligibility coverage (sonic-NCT lane: 'base' endpoints, proxy >= 0.60)\n"
    "ELIGIBILITY IS NOT VALIDITY - no gated method is implemented, fitted, or validated. "
    f"{ASSURANCE_TIER}.\nFormation tops shown only where LOCKED Increment 5 tops exist.",
    fontsize=10.5, y=0.985)
fig.tight_layout(rect=[0, 0, 1, 0.93])
fig.savefig(os.path.join(FIG_DIR, "fig03_method_eligibility_coverage_panel.png"), dpi=150)
plt.show()


#### Step 23 — Figure 4: sonic-NCT candidate interval sensitivity

**Technical objective:** show the candidate blocks under every endpoint scenario x proxy threshold combination, explicitly labelled candidate data only.

**Increment 6.1 (Finding 4) correction:** Increment 6 drew *all* candidate blocks while annotating *qualifying-only* totals, so the picture and the number described different populations. Qualifying blocks (meeting the configured minimum sample count and thickness) are now drawn thick and coloured; rejected sub-threshold blocks are drawn thin, grey, and offset. Every annotation states GROSS and NET separately, together with how many blocks qualify, how many were rejected, and how many bridged samples the gross figure absorbed.

**Interpretation note:** the visible thinning of the bars from left to right within each panel *is* the result. It shows how much of the "candidate" section is an artefact of two uncalibrated choices rather than a property of the rock.

In [ ]:
from matplotlib.lines import Line2D

_nct_cfg = [iv for iv in nct_intervals
            if iv.contiguity_policy == CONTIGUITY_POLICY_CONFIGURED]
_wells = sorted({iv.well_key for iv in _nct_cfg}) or \
         [w for w in sorted(frames) if cfg.wells[w].proxy_permitted]
fig, axes = plt.subplots(1, len(_wells), figsize=(4.4 * len(_wells), 10.4), sharey=True)
if len(_wells) == 1:
    axes = [axes]
_colors = {"low": "#1F4E79", "base": "#2E7D32", "high": "#8E24AA"}
for ax, wk in zip(axes, _wells):
    combos = [(s, t) for s in ("low", "base", "high") for t in THRESHOLDS]
    for x, (sname, thr) in enumerate(combos):
        blocks = [iv for iv in _nct_cfg if iv.well_key == wk
                  and iv.scenario_name == sname and iv.proxy_threshold == thr
                  and iv.tvdss_start_m is not None and iv.tvdss_end_m is not None]
        qual = [iv for iv in blocks if iv.meets_configured_minimums]
        rej = [iv for iv in blocks if not iv.meets_configured_minimums]
        for iv in rej:
            ax.plot([x + 0.22, x + 0.22], [iv.tvdss_start_m, iv.tvdss_end_m],
                    lw=1.4, solid_capstyle="butt", color="#9E9E9E", alpha=0.55)
        for iv in qual:
            ax.plot([x - 0.10, x - 0.10], [iv.tvdss_start_m, iv.tvdss_end_m],
                    lw=5.5, solid_capstyle="butt", color=_colors[sname], alpha=0.70)
        gross = sum(float(iv.gross_thickness_tvdss_m or 0.0) for iv in qual)
        net = sum(float(iv.net_thickness_tvdss_m or 0.0) for iv in qual)
        nbr = sum(iv.n_bridged_samples for iv in qual)
        ax.text(x, ax.get_ylim()[0],
                f"gross {gross:,.0f} m\nnet {net:,.0f} m\n{len(qual)} qual / {len(rej)} rej"
                + (f"\n{nbr} bridged" if nbr else ""),
                fontsize=5.4, ha="center", va="top", rotation=90)
    ax.set_xticks(range(len(combos)))
    ax.set_xticklabels([f"{s}\n>={t:.2f}" for s, t in combos], fontsize=6)
    ax.set_title(f"{wk}\n{cfg.wells[wk].use_status}", fontsize=8.5)
    ax.grid(axis="y", alpha=0.25, lw=0.4)
    ax.tick_params(labelsize=8)
axes[0].set_ylabel("TVDSS [m] (locked Increment 3.1.1 basis)", fontsize=9)
axes[0].invert_yaxis()
axes[-1].legend(handles=[
    Line2D([0], [0], color="#2E7D32", lw=5.5,
           label="QUALIFYING block (meets configured minimums) - counted in totals"),
    Line2D([0], [0], color="#9E9E9E", lw=1.4,
           label="rejected sub-threshold block - shown, NOT counted in totals"),
], fontsize=6, loc="lower right", framealpha=0.92)
fig.suptitle(
    "Fig 4 - Sonic-NCT CANDIDATE interval sensitivity across endpoint scenario x proxy "
    "threshold\nTotals are GROSS QUALIFYING block span (configured contiguity policy, "
    "includes disclosed bridged samples) with NET alongside;\nrejected sub-threshold blocks "
    "are drawn separately and excluded from every total. "
    f"CANDIDATE DATA ONLY - NO NCT FITTED.\n{ASSURANCE_TIER}.",
    fontsize=9.5, y=0.985)
fig.tight_layout(rect=[0, 0, 1, 0.92])
fig.savefig(os.path.join(FIG_DIR, "fig04_sonic_nct_candidate_interval_sensitivity.png"), dpi=150)
plt.show()


#### Step 24 — Display the summary tables

In [ ]:
import pandas as pd

_qc = pd.read_csv(os.path.join(OUT_DIR, "gr_family_qc_summary.csv"))
display(_qc[["well_key", "gr_family_canonical_name", "use_status", "exclusion_reason",
             "gr_proxy_confidence_class", "valid_fraction", "median_api", "min_api", "max_api",
             "n_samples_above_seabed"]])

_ep = pd.read_csv(os.path.join(OUT_DIR, "gr_endpoint_scenarios.csv"))
display(_ep[["well_key", "scenario_name", "gr_low_endpoint_api", "gr_high_endpoint_api",
             "endpoint_separation_api", "evidence_class"]])

_el = pd.read_csv(os.path.join(OUT_DIR, "method_eligibility_summary.csv"))
display(_el[["well_key", "mask_name", "scenario_name", "proxy_threshold", "n_eligible",
             "eligible_fraction", "limiting_criterion"]].head(40))


---

## Limitations

- **Tier C — screening-level and uncalibrated.** No RFT, MDT, DST, FIT, LOT, XLOT or DFIT data exist for this project. The supplied Vp/Vs text file is derived from the existing sonic curves and is NOT independent calibration data. Nothing in this increment is validated against an independent measurement.
- **No named lithology is assigned, and the data do not support assigning one.** Low-GR intervals occur independently in each of the three GR-eligible wells; cross-well stratigraphic persistence is NOT established, and cannot be, because two of those wells have no approved formation tops. Their lithology is UNRESOLVED, and `named_lithology_assigned` is DERIVED from an actual validation pass rather than hardcoded.
- **The screening proxy is not a shale volume.** `VSH_GR_linear_proxy_frac` is the linear identity of a clipped index computed under ASSUMED endpoints. Its median moves by up to ≈0.12 across the three endpoint scenarios, and the qualifying sonic-NCT-candidate thickness moves by up to a factor of ≈8 across the endpoint × threshold grid.
- **Eligibility is not validity.** No gated method is implemented, fitted, or validated. Density and dynamic-elastic eligibility begin only around 3,900–4,800 m TVDSS in all four wells, so these logs cannot support an overburden integration from surface regardless of per-sample eligibility.
- **Boreas 1 is excluded, not corrected** (`BOREAS_ECGR_SCALE_UNRESOLVED`). It remains available for factual raw-GR QC display and availability reporting only.
- **Poseidon North 1 and Proteus 1ST2 have no approved formation tops**, so their above-seabed sample counts are *not determinable* (never reported as 0) and all their results remain depth-tied and stratigraphically unvalidated.
- **No nonlinear Vsh transform is implemented.** Larionov, Clavier, Stieber and every other nonlinear transform are DEFERRED pending a retrieved, verified primary-source method record and a closed method-register entry.
- **Increment 7 has not been started**, and no pore-pressure, elastic-property, rock-strength, stress, or wellbore-stability work exists anywhere in this increment.

### Completion Gate

> **PHASE COMPLETION GATE:** Increment 6 / 6.1 / ... / 6.1.7 (Gamma-Ray QC, Shale-Proxy Sensitivity, Well-Frame Assembly, and Method-Eligibility Framework) is complete when: (0) the complete combined test suite (Step 9) passed with ZERO failures, checked by actual subprocess return code, not assumed; (1) all four approved LAS files and all four approved deviation surveys loaded successfully; (2) all four well frames assembled, preserving row counts and original order; (3) ZERO samples were extrapolated; (4) Boreas 1 is QC-only and received zero endpoints, zero proxies, and zero lithology-dependent masks; (5) ZERO named lithology is assigned anywhere; (6) low/base/high endpoint scenarios completed for all three GR-eligible wells; (7) all three eligibility masks were generated; (8) contiguous-interval registers were generated; (9) all 8 deterministic output files are present (asserted explicitly, count and existence); (10) all 4 required figures are present, giving 12 outputs/figures in total; (11) every controlled scope rejects text that nothing authorized, probed with GENERATED novel words verified invisible to the rock blacklist, so the check proves the authorization model rather than a finite sentence list; (12) every registered statement and template authorizes only itself, exactly - unknown ids, case changes and prose substitutions all fail - and an arbitrary label value is refused; (13) every persisted controlled field in THIS run is positively authorized; (14) every interval record produced by this run satisfies the full interruption-count invariant set, and the live constructor refuses the record shapes that were constructible before Increment 6.1.2. (15) both pre- and post-serialization records satisfy exact closed schemas; (16) the complete typed canonical records, including row order, survive serialization unchanged; (17) live adversarial probes prove missing/unknown/type-mutated fields fail regardless of value; and (18) live writer-mutation and serializer-failure probes leave the official destination unchanged. All conditions are verified programmatically below, not asserted.

In [ ]:
print("=" * 78)
from p2mem.wellframe_models import (
    SCOPE_INTERPRETIVE, SCOPE_LABEL, SCOPE_METHOD, SCOPE_EXPLANATORY,
    CONTROLLED_SCOPES, APPROVED_LABELS, REGISTERED_STATEMENTS, REGISTERED_TEMPLATES,
    Authorization, find_prohibited_lithology_terms,
    validate_no_prohibited_interpretation)
from p2mem.method_eligibility import EligibilityInterval
from p2mem.petrophysics import PetrophysicsInputError
import random as _random


def _validate_one(text, scope=SCOPE_INTERPRETIVE, auth=None):
    # Run the LIVE validator on one string, with or without authorization.
    entry = (("gate.probe", text, scope, auth) if auth is not None
             else ("gate.probe", text, scope))
    return validate_no_prohibited_interpretation([entry])


def _novel_tokens(n=12):
    # GENERATED nonsense words: never seen by any recognizer, absent from every
    # blacklist, and not drawn from the regression-test sentences.
    rng = _random.Random(20260903)
    cons, vow = "bcdfghjklmnpqrstvwxz", "aeiou"
    return ["".join(rng.choice(cons) + rng.choice(vow) for _ in range(3))
            + rng.choice(cons) for _ in range(n)]


def _positive_authorization_holds():
    for word in _novel_tokens():
        text = "The interval is " + word + "."
        if find_prohibited_lithology_terms(text):
            return False          # the probe must be invisible to the linter
        for scope in CONTROLLED_SCOPES:
            if not _validate_one(text, scope=scope):
                return False      # something accepted unauthorized text
    for text in ("Everything looks fine.", "Coverage is good in the deep section."):
        if find_prohibited_lithology_terms(text):
            return False
        for scope in CONTROLLED_SCOPES:
            if not _validate_one(text, scope=scope):
                return False
    return True


def _emission_gate_refuses_unauthorized():
    # Live negative probe with a GENERATED novel token: the export gate must
    # refuse to write, and the rock blacklist must see nothing in it.
    import copy as _copy
    from p2mem.io.output_policy import (OutputAuthorizationError,
                                        export_authorized_outputs)
    probe = "The interval is " + _novel_tokens(1)[0] + "."
    if find_prohibited_lithology_terms(probe):
        return False
    tampered = {k: _copy.deepcopy(v) for k, v in _payloads.items()}
    tampered["gr_endpoint_scenarios.csv"][0]["description"] = probe
    import tempfile
    with tempfile.TemporaryDirectory() as _td:
        try:
            export_authorized_outputs(_td, tampered, well_keys=set(frames))
        except OutputAuthorizationError:
            return len(os.listdir(_td)) == 0
    return False


def _every_registered_entry_authorizes_itself():
    for sid, st in REGISTERED_STATEMENTS.items():
        if _validate_one(st.text, scope=st.scope, auth=Authorization(statement_id=sid)):
            return False
        if st.text.upper() != st.text and not _validate_one(
                st.text.upper(), scope=st.scope, auth=Authorization(statement_id=sid)):
            return False          # a case change must NOT pass
        if not _validate_one(st.text, scope=st.scope,
                             auth=Authorization(statement_id="no_such_id")):
            return False          # an unknown id must NOT pass
    tpl = REGISTERED_TEMPLATES["gr_proxy_confidence_rationale"]
    good = {"valid_fraction": "0.9000", "dynamic_range_p05_p95": "100.000",
            "proxy_median_spread": "0.1000"}
    if _validate_one(tpl.render(good), scope=tpl.scope,
                     auth=Authorization(template_id=tpl.template_id, fields=good)):
        return False
    bad = dict(good, proxy_median_spread="high")
    text = tpl.template.replace("{proxy_median_spread}", "high").format(
        **{k: v for k, v in good.items() if k != "proxy_median_spread"})
    if not _validate_one(text, scope=tpl.scope,
                         auth=Authorization(template_id=tpl.template_id, fields=bad)):
        return False              # a prose substitution must NOT pass
    return True


def _interval_record_rejected(**counts):
    # True when the LIVE constructor refuses an invalid count record.
    try:
        EligibilityInterval(well_key="GATE", mask_name="GATE", **counts)
    except PetrophysicsInputError:
        return True
    return False



def _read_written_payloads():
    import csv as _csv
    out = {}
    for name in _expected_outputs:
        path = os.path.join(OUT_DIR, name)
        if name.endswith(".json"):
            with open(path, encoding="utf-8") as fh:
                out[name] = json.load(fh)
        else:
            with open(path, newline="", encoding="utf-8") as fh:
                out[name] = list(_csv.DictReader(fh))
    return out


def _canonical_round_trip_holds():
    from p2mem.io.output_policy import canonicalize_emitted_records
    written = _read_written_payloads()
    return (canonicalize_emitted_records(_payloads, serialization_stage="pre")
            == canonicalize_emitted_records(written, serialization_stage="post"))


def _schema_adversarial_probes_hold():
    import copy as _copy
    import tempfile
    from p2mem.io.output_policy import OutputAuthorizationError, export_authorized_outputs
    mutations = []
    p = _copy.deepcopy(_payloads)
    for row in p["gr_endpoint_scenarios.csv"]:
        row.pop("description")
    mutations.append(p)
    for bad in ("", None, 123, "123"):
        p = _copy.deepcopy(_payloads)
        p["gr_endpoint_scenarios.csv"][0]["description"] = bad
        mutations.append(p)
    for bad in ("", None, 123):
        p = _copy.deepcopy(_payloads)
        for row in p["gr_endpoint_scenarios.csv"]:
            row["rogue_column"] = bad
        mutations.append(p)
    for payload in mutations:
        with tempfile.TemporaryDirectory() as td:
            try:
                export_authorized_outputs(td, payload, well_keys=set(frames))
            except OutputAuthorizationError:
                if os.listdir(td):
                    return False
            else:
                return False
    return True


def _transaction_adversarial_probes_hold():
    import copy as _copy
    import csv as _csv
    import tempfile
    from pathlib import Path as _Path
    from p2mem.io.output_policy import (
        CSV_SCHEMAS, OUTPUT_STATEMENTS, OutputAuthorizationError,
        export_authorized_outputs)

    replacement = OUTPUT_STATEMENTS["gr_endpoint_scenarios_description_2"].text
    def mutating_writer(target, payload):
        value = _copy.deepcopy(payload)
        if target.name == "gr_endpoint_scenarios.csv":
            value[0]["description"] = replacement
        if target.suffix == ".json":
            target.write_text(json.dumps(value, indent=2, sort_keys=True) + "\n",
                              encoding="utf-8")
        else:
            with open(target, "w", newline="", encoding="utf-8") as fh:
                w = _csv.DictWriter(fh, fieldnames=list(CSV_SCHEMAS[target.name].columns))
                w.writeheader()
                w.writerows(value)

    def failing_writer(_target, _payload):
        raise RuntimeError("simulated serializer failure")

    for writer in (mutating_writer, failing_writer):
        with tempfile.TemporaryDirectory() as td:
            sentinel = _Path(td) / "gr_endpoint_scenarios.csv"
            sentinel.write_bytes(b"approved baseline bytes\n")
            try:
                export_authorized_outputs(
                    td, _payloads, well_keys=set(frames), writer=writer)
            except OutputAuthorizationError:
                if (sentinel.read_bytes() != b"approved baseline bytes\n"
                        or {p.name for p in _Path(td).iterdir()} != {sentinel.name}):
                    return False
            else:
                return False
    return True


print("INCREMENT 6 / 6.1 / ... / 6.1.7 COMPLETION GATE")
print("=" * 78)

_expected_figures = [
    "fig01_gr_family_raw_qc.png",
    "fig02_gr_endpoint_and_proxy_sensitivity.png",
    "fig03_method_eligibility_coverage_panel.png",
    "fig04_sonic_nct_candidate_interval_sensitivity.png",
]

_gr_eligible = [w for w in sorted(frames) if cfg.wells[w].proxy_permitted]

gate_checks = {
    # Read via globals().get(..., False), NOT a bare FULL_TEST_SUITE_PASSED
    # name reference, so that if Step 9 were ever skipped entirely this check
    # fails CLEANLY as False rather than crashing the gate with a NameError.
    "Complete combined test suite passed with zero failures": (
        globals().get("FULL_TEST_SUITE_PASSED", False) is True
    ),
    "All 4 approved LAS files loaded (0 failures)": (
        len(las_results) == 4 and len(las_failures) == 0
    ),
    "All 4 approved deviation surveys loaded (0 failures)": (
        len(dev_results) == 4 and len(dev_failures) == 0
    ),
    "All 4 well frames assembled (0 failures)": (
        len(frames) == 4 and len(frame_failures) == 0
    ),
    "Well frames preserve LAS row counts and order": all(
        frames[wk].n_samples == las_results[wk].canonical_data["MD_m"].size
        and bool(np.array_equal(frames[wk].MD_m, las_results[wk].canonical_data["MD_m"]))
        for wk in frames
    ),
    "ZERO samples extrapolated": all(fr.n_extrapolated == 0 for fr in frames.values()),
    "Boreas 1 QC-only and excluded from all GR-derived interpretation": (
        cfg.wells["Boreas_1"].use_status == "qc_only_excluded"
        and cfg.wells["Boreas_1"].exclusion_reason == "BOREAS_ECGR_SCALE_UNRESOLVED"
        and len(scenarios.get("Boreas_1", [])) == 0
        and len(proxies.get("Boreas_1", [])) == 0
        and len([m for m in mask_results
                 if m.well_key == "Boreas_1" and m.lithology_dependent]) == 0
        and confidence.get("Boreas_1") == "GR_EXCLUDED_UNRESOLVED_SCALE"
    ),
    "Boreas 1 anomaly independently reproduced (not hardcoded)": (
        globals().get("regression_findings_confirmed", False) is True
    ),
    # DERIVED (Increment 6.1, Finding 3): read from the actual validation pass
    # over persisted content, never from a hardcoded constant. Injecting a
    # prohibited term anywhere in that scope fails this check.
    "ZERO named lithology assigned (DERIVED from validation, not hardcoded)": (
        manifest["named_lithology_assigned"] is False
        and len(manifest["lithology_validation"]["violations"]) == 0
        and manifest["lithology_validation"]["scope_object_fields_checked"] > 0
        and manifest["lithology_validation"]["emitted_field_coverage"][
            "n_string_field_occurrences"] > 0
    ),
    # Increment 6.1.5: POSITIVE AUTHORIZATION. The probes below are GENERATED
    # novel tokens - never seen by any recognizer, never in the regression
    # tests - and each is verified invisible to the rock blacklist before use,
    # so a pass proves the ARCHITECTURE rather than a finite sentence list.
    "Every controlled scope rejects unauthorized text (generated novel probes)": (
        _positive_authorization_holds()
    ),
    "Registered statements/templates authorize only themselves, exactly": (
        _every_registered_entry_authorizes_itself()
        and len(APPROVED_LABELS) > 0
        and bool(_validate_one("arbitrary_label_value", scope=SCOPE_LABEL,
                               auth=Authorization(field_kind="use_status")))
        # Increment 6.1.6: a value approved under one field kind authorizes
        # nothing under another.
        and bool(_validate_one("GR", scope=SCOPE_LABEL,
                               auth=Authorization(field_kind="use_status")))
        and not _validate_one("GR", scope=SCOPE_LABEL,
                              auth=Authorization(field_kind="gr_family_source_curve_name"))
    ),
    # Increment 6.1.6: the emission boundary. These assert the records this run
    # ACTUALLY WROTE, not a reconstruction of them.
    "Every emitted output string field is classified (0 unclassified)": (
        _before.n_unclassified_fields == 0 and _after.n_unclassified_fields == 0
        and _before.n_string_field_occurrences > 0
    ),
    "Every emitted controlled field is authorized (0 unauthorized, 0 kind mismatch)": (
        _before.n_unauthorized_controlled_fields == 0
        and _before.n_field_kind_mismatches == 0
        and _after.n_unauthorized_controlled_fields == 0
        and _after.n_field_kind_mismatches == 0
    ),
    "Exact artifact schemas pass before and after serialization": (
        _before.ok and _after.ok
        and _before.n_schema_violations == 0
        and _after.n_schema_violations == 0
        and len(_before.artifacts_inspected) == len(_expected_outputs) == 8
    ),
    "Every typed field and row survives serialization unchanged": (
        _canonical_round_trip_holds()
    ),
    "Missing, unknown, empty and type-mutated fields fail closed": (
        _schema_adversarial_probes_hold()
    ),
    "Authorized writer mutation and serializer failure are failure-atomic": (
        _transaction_adversarial_probes_hold()
    ),
    "An unauthorized emitted field would be refused (live negative probe)": (
        _emission_gate_refuses_unauthorized()
    ),
    "Every persisted controlled field in THIS run is positively authorized": (
        len(manifest["lithology_validation"]["violations"]) == 0
        and manifest["lithology_validation"]["scope_object_fields_checked"] > 0
        and manifest["lithology_validation"]["emitted_field_coverage"][
            "n_string_field_occurrences"] > 0
    ),

    # Increment 6.1.2 (Finding 2): every interval record this run produced
    # satisfies the full count invariant set, and an invalid record cannot be
    # constructed.
    "Interval count invariants hold for every record and reject invalid ones": (
        all(
            iv.n_eligible_subruns >= 1
            and iv.n_bridged_gaps == iv.n_eligible_subruns - 1
            and (iv.n_bridged_samples == 0) == (iv.n_bridged_gaps == 0)
            and (iv.n_bridged_gaps == 0 or iv.n_bridged_samples >= iv.n_bridged_gaps)
            for iv in intervals
        )
        and _interval_record_rejected(n_bridged_samples=5, n_bridged_gaps=0,
                                      n_eligible_subruns=1)
        and _interval_record_rejected(n_bridged_samples=0, n_bridged_gaps=2,
                                      n_eligible_subruns=0)
        and _interval_record_rejected(n_bridged_samples=-1, n_bridged_gaps=-1,
                                      n_eligible_subruns=0)
        # Increment 6.1.3 residual type-gate findings.
        and _interval_record_rejected(n_bridged_samples=0.0, n_bridged_gaps=0.0,
                                      n_eligible_subruns=1.0)
        and _interval_record_rejected(n_bridged_samples=float("nan"),
                                      n_bridged_gaps=0, n_eligible_subruns=1)
        and _interval_record_rejected(n_bridged_samples=float("inf"),
                                      n_bridged_gaps=1, n_eligible_subruns=2)
        and _interval_record_rejected(n_bridged_samples=0, n_bridged_gaps=0,
                                      n_eligible_subruns=1, n_bridged_sample=99)
    ),
    "Poseidon North 1 carries the depth-tied status (no approved tops)": (
        cfg.wells["Poseidon_North_1"].use_status == "screening_proxy_allowed_depth_tied"
        and cfg.wells["Poseidon_North_1"].has_approved_formation_tops is False
    ),
    "Vp/Vs exclusions reported BY REGIME, never aggregated": all(
        {"n_ratio_nonpositive_bulk_modulus",
         "n_ratio_positive_bulk_but_negative_poisson",
         "n_ratio_above_configured_plausibility_max",
         "n_vp_not_greater_than_vs"}.issubset(set(m.diagnostic_counts))
        for m in mask_results if m.mask_name == MASK_DYNAMIC_ELASTIC
    ),
    "Interval thickness reported as GROSS and NET under both contiguity policies": (
        {iv.contiguity_policy for iv in intervals}
        == {CONTIGUITY_POLICY_CONFIGURED, CONTIGUITY_POLICY_STRICT}
        and all(iv.gross_thickness_tvdss_m is not None for iv in intervals
                if iv.tvdss_start_m is not None)
    ),
    "low/base/high endpoint scenarios completed for every GR-eligible well": (
        len(_gr_eligible) == 3
        and all(sorted(s.scenario_name for s in scenarios[w]) == ["base", "high", "low"]
                for w in _gr_eligible)
    ),
    "All 3 eligibility masks generated": (
        {m.mask_name for m in mask_results} == {
            MASK_DENSITY_FOR_SV, MASK_DYNAMIC_ELASTIC, MASK_SONIC_NCT_CANDIDATE}
    ),
    "Contiguous-interval registers generated": len(intervals) > 0,
    # Increment 6.1.1 (Finding 4): the label said 7 while the notebook created
    # and checked 8. Both the count and the existence of all eight are now
    # asserted explicitly, so the label can never drift from the list again.
    "8 deterministic output files declared and present": (
        len(_expected_outputs) == 8
        and all(os.path.exists(os.path.join(OUT_DIR, fn)) for fn in _expected_outputs)
    ),
    "4 QC figures present": all(
        os.path.exists(os.path.join(FIG_DIR, fn)) for fn in _expected_figures
    ),
}
for label, passed in gate_checks.items():
    print(f"  [{'PASS' if passed else 'FAIL'}] {label}")

if all(gate_checks.values()):
    print("\nIncrement 6.1.7 (corrective patch to Increment 6.1.6) is complete. Stopping here per")
    print("the approved scope.")
    print("Not implemented (explicitly out of scope): named lithology interpretation,")
    print("environmental GR correction, Boreas ECGR rescaling, neutron-density crossplot")
    print("interpretation, nonlinear Vsh transforms, shallow-density reconstruction, density")
    print("extrapolation, vertical-stress integration, hydrostatic-pressure modelling,")
    print("velocity-effective-stress crossplots, NCT fitting, Eaton sonic, Eaton resistivity,")
    print("Bowers, pore-pressure prediction, dynamic elastic-property calculation, static")
    print("elastic conversion, rock-strength correlations, friction-angle modelling,")
    print("Shmin/SHmax modelling, stress-polygon construction, wellbore-stability analysis,")
    print("mud-weight recommendations. Increment 7 has NOT been started.")
else:
    raise RuntimeError("Increment 6.1.7 completion gate FAILED - see failed check(s) above.")
